In [1]:
# ============================================================
# Setup 0A - Mount Drive + Define B2 Experiment Paths
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

B1_RESULTS_ROOT = (
    RESULTS_ROOT
    / "B1_experiments"
)


# ------------------------------------------------------------
# Frozen dev100 subset
# ------------------------------------------------------------

DEV100_FILE = (
    B1_RESULTS_ROOT
    / "subsets"
    / "FSC147_val_dev_100.csv"
)


# ------------------------------------------------------------
# Frozen B0 results
# ------------------------------------------------------------

B0_ROOT = (
    RESULTS_ROOT
    / "B0_dev100"
)

B0_PER_IMAGE_FILE = (
    B0_ROOT
    / "B0_dev100_per_image.csv"
)


# ------------------------------------------------------------
# Frozen semantic evidence
# ------------------------------------------------------------

C_PER_IMAGE_FILE = (
    B1_RESULTS_ROOT
    / "Group_C"
    / "B1_Group_C_E_candidates_per_image.csv"
)

D_PER_IMAGE_FILE = (
    B1_RESULTS_ROOT
    / "Group_D"
    / "B1_Group_D_E_candidates_per_image.csv"
)


# ------------------------------------------------------------
# Previous diagnostic synthesis
# ------------------------------------------------------------

DIAG_ROOT = (
    B1_RESULTS_ROOT
    / "Sparse_Scene_Failure_Diagnostic"
)

Q5_DECISION_FILE = (
    DIAG_ROOT
    / "tables"
    / "Q5_architecture_decision_summary.csv"
)

Q5_MANIFEST_FILE = (
    DIAG_ROOT
    / "metadata"
    / "Q5_diagnostic_synthesis.json"
)


# ------------------------------------------------------------
# NEW B2 experiment output structure
# ------------------------------------------------------------

B2_ROOT = (
    B1_RESULTS_ROOT
    / "B2_B0_Anchored_Semantic_Validation"
)

B2_CACHE_DIR = (
    B2_ROOT
    / "cache"
)

B2_TABLES_DIR = (
    B2_ROOT
    / "tables"
)

B2_METADATA_DIR = (
    B2_ROOT
    / "metadata"
)

B2_VIS_DIR = (
    B2_ROOT
    / "visualisations"
)


for path in [
    B2_ROOT,
    B2_CACHE_DIR,
    B2_TABLES_DIR,
    B2_METADATA_DIR,
    B2_VIS_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Frozen candidate semantic pool
#
# These are carried forward as candidates, NOT yet selected
# for the B0-validation role.
# ------------------------------------------------------------

C_CANDIDATES = [
    "C1",
    "C2b",
    "C2c",
]

D_CANDIDATES = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]


# ------------------------------------------------------------
# Experiment scope
# ------------------------------------------------------------

EXPERIMENT_SCOPE = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "dataset":
        "FSC-147",

    "subset":
        "frozen val dev100",

    "num_images":
        100,

    "baseline":
        "frozen B0 unguided SAM2",

    "semantic_role":
        "B0 mask validation only",

    "semantic_recovery":
        False,

    "new_C_D_generation":
        False,

    "new_C_D_fusion":
        False,

    "test_set_accessed":
        False,
}


# ------------------------------------------------------------
# Input existence checks
# ------------------------------------------------------------

INPUT_FILES = {
    "dev100":
        DEV100_FILE,

    "B0 per-image":
        B0_PER_IMAGE_FILE,

    "C evidence":
        C_PER_IMAGE_FILE,

    "D evidence":
        D_PER_IMAGE_FILE,

    "Q5 decision":
        Q5_DECISION_FILE,

    "Q5 manifest":
        Q5_MANIFEST_FILE,
}


missing = [
    name
    for name, path in INPUT_FILES.items()
    if not path.exists()
]


assert (
    len(
        missing
    )
    == 0
), (
    "Missing required frozen inputs: "
    f"{missing}"
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("B2 SETUP 0A - B0-ANCHORED SEMANTIC VALIDATION")
print("=" * 110)

print("Project root        :", PROJECT_ROOT)
print("GPU phase           : YES - T4 appropriate for SAM2 rerun/cache")
print("Frozen dev100       :", DEV100_FILE)
print("Frozen B0 results   :", B0_PER_IMAGE_FILE)

print("\nSemantic candidate pool:")
print("C:", C_CANDIDATES)
print("D:", D_CANDIDATES)

print("\nExperiment constraints:")
print("B0 generation changed?       NO")
print("New C/D evidence generation? NO")
print("New C/D fusion?              NO")
print("Semantic recovery?           NO")
print("Current semantic role        B0 MASK VALIDATION ONLY")
print("Test set accessed?           NO")

print("\nB2 output directory:")
print(B2_ROOT)

print("\nAll required frozen inputs   : PASS")
print("=" * 110)

Mounted at /content/drive
B2 SETUP 0A - B0-ANCHORED SEMANTIC VALIDATION
Project root        : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
GPU phase           : YES - T4 appropriate for SAM2 rerun/cache
Frozen dev100       : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/subsets/FSC147_val_dev_100.csv
Frozen B0 results   : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_per_image.csv

Semantic candidate pool:
C: ['C1', 'C2b', 'C2c']
D: ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']

Experiment constraints:
B0 generation changed?       NO
New C/D evidence generation? NO
New C/D fusion?              NO
Semantic recovery?           NO
Current semantic role        B0 MASK VALIDATION ONLY
Test set accessed?           NO

B2 output directory:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation

All required frozen inputs   : PASS


In [2]:
# ============================================================
# Setup 0B - Load Frozen Tables + Exact Alignment Checks
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0B.1 Load frozen tables
# ------------------------------------------------------------

dev100_df = pd.read_csv(
    DEV100_FILE
)

b0_df = pd.read_csv(
    B0_PER_IMAGE_FILE
)

c_df = pd.read_csv(
    C_PER_IMAGE_FILE
)

d_df = pd.read_csv(
    D_PER_IMAGE_FILE
)


# ------------------------------------------------------------
# 0B.2 Basic shape checks
# ------------------------------------------------------------

assert len(
    dev100_df
) == 100, (
    f"dev100 expected 100 rows, found {len(dev100_df)}"
)

assert len(
    b0_df
) == 100, (
    f"B0 expected 100 rows, found {len(b0_df)}"
)

assert len(
    c_df
) == 300, (
    f"C expected 300 rows, found {len(c_df)}"
)

assert len(
    d_df
) == 500, (
    f"D expected 500 rows, found {len(d_df)}"
)


# ------------------------------------------------------------
# 0B.3 Required columns
# ------------------------------------------------------------

B0_REQUIRED = {
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "pred_count",
}

C_REQUIRED = {
    "candidate_id",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "pred_count",
}

D_REQUIRED = {
    "candidate_id",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "pred_count",
}


missing_b0 = (
    B0_REQUIRED
    - set(
        b0_df.columns
    )
)

missing_c = (
    C_REQUIRED
    - set(
        c_df.columns
    )
)

missing_d = (
    D_REQUIRED
    - set(
        d_df.columns
    )
)


assert (
    len(
        missing_b0
    )
    == 0
), (
    f"B0 missing columns: {sorted(missing_b0)}"
)


assert (
    len(
        missing_c
    )
    == 0
), (
    f"C missing columns: {sorted(missing_c)}"
)


assert (
    len(
        missing_d
    )
    == 0
), (
    f"D missing columns: {sorted(missing_d)}"
)


# ------------------------------------------------------------
# 0B.4 Candidate identity checks
# ------------------------------------------------------------

observed_c_candidates = sorted(
    c_df[
        "candidate_id"
    ]
    .dropna()
    .unique()
    .tolist()
)

observed_d_candidates = sorted(
    d_df[
        "candidate_id"
    ]
    .dropna()
    .unique()
    .tolist()
)


assert (
    observed_c_candidates
    == sorted(
        C_CANDIDATES
    )
), (
    "Unexpected C candidate set:\n"
    f"Observed: {observed_c_candidates}\n"
    f"Expected: {sorted(C_CANDIDATES)}"
)


assert (
    observed_d_candidates
    == sorted(
        D_CANDIDATES
    )
), (
    "Unexpected D candidate set:\n"
    f"Observed: {observed_d_candidates}\n"
    f"Expected: {sorted(D_CANDIDATES)}"
)


# ------------------------------------------------------------
# 0B.5 Build authoritative image mapping from frozen B0
# ------------------------------------------------------------

IMAGE_KEY_COLS = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
]


b0_image_map = (
    b0_df[
        IMAGE_KEY_COLS
    ]
    .copy()
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert (
    b0_image_map[
        "dataset_index"
    ]
    .is_unique
)

assert (
    b0_image_map[
        "image_id"
    ]
    .is_unique
)


# ------------------------------------------------------------
# 0B.6 Exact B0 <-> C mapping
# ------------------------------------------------------------

c_image_map = (
    c_df[
        IMAGE_KEY_COLS
    ]
    .drop_duplicates()
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        c_image_map
    )
    == 100
)


c_alignment = (
    b0_image_map
    .merge(
        c_image_map,
        on=IMAGE_KEY_COLS,
        how="outer",
        indicator=True,
    )
)


assert (
    c_alignment[
        "_merge"
    ]
    .eq(
        "both"
    )
    .all()
), (
    "B0 and C image mappings are not identical."
)


# ------------------------------------------------------------
# 0B.7 Exact B0 <-> D mapping
# ------------------------------------------------------------

d_image_map = (
    d_df[
        IMAGE_KEY_COLS
    ]
    .drop_duplicates()
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        d_image_map
    )
    == 100
)


d_alignment = (
    b0_image_map
    .merge(
        d_image_map,
        on=IMAGE_KEY_COLS,
        how="outer",
        indicator=True,
    )
)


assert (
    d_alignment[
        "_merge"
    ]
    .eq(
        "both"
    )
    .all()
), (
    "B0 and D image mappings are not identical."
)


# ------------------------------------------------------------
# 0B.8 dev100 alignment
#
# dev100 is the authoritative frozen subset file, but its exact
# stored metadata columns may be slightly different from B0.
# We therefore verify all shared identity columns.
# ------------------------------------------------------------

DEV100_IDENTITY_CANDIDATES = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
]


dev100_identity_cols = [
    col
    for col in DEV100_IDENTITY_CANDIDATES
    if (
        col in dev100_df.columns
        and col in b0_image_map.columns
    )
]


assert (
    "dataset_index"
    in dev100_identity_cols
), (
    "dev100 must contain dataset_index for exact alignment."
)


assert (
    "image_id"
    in dev100_identity_cols
), (
    "dev100 must contain image_id for exact alignment."
)


dev100_image_map = (
    dev100_df[
        dev100_identity_cols
    ]
    .drop_duplicates()
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


b0_for_dev100_check = (
    b0_image_map[
        dev100_identity_cols
    ]
    .copy()
)


dev100_alignment = (
    b0_for_dev100_check
    .merge(
        dev100_image_map,
        on=dev100_identity_cols,
        how="outer",
        indicator=True,
    )
)


assert (
    len(
        dev100_image_map
    )
    == 100
)


assert (
    dev100_alignment[
        "_merge"
    ]
    .eq(
        "both"
    )
    .all()
), (
    "Frozen dev100 subset and B0 image mapping are not identical."
)


# ------------------------------------------------------------
# 0B.9 Per-candidate completeness
# ------------------------------------------------------------

c_counts = (
    c_df
    .groupby(
        "candidate_id"
    )[
        "dataset_index"
    ]
    .nunique()
)


d_counts = (
    d_df
    .groupby(
        "candidate_id"
    )[
        "dataset_index"
    ]
    .nunique()
)


assert (
    c_counts
    == 100
).all(), (
    "At least one C candidate does not contain all 100 images."
)


assert (
    d_counts
    == 100
).all(), (
    "At least one D candidate does not contain all 100 images."
)


# ------------------------------------------------------------
# 0B.10 Density distribution
# ------------------------------------------------------------

DENSITY_ORDER = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]


density_counts = (
    b0_image_map[
        "density_stratum"
    ]
    .value_counts()
    .reindex(
        DENSITY_ORDER
    )
)


expected_density_counts = pd.Series(
    {
        "S1_0_20": 20,
        "S2_20_40": 20,
        "S3_40_60": 20,
        "S4_60_80": 20,
        "S5_80_95": 10,
        "S6_95_100": 10,
    }
)


assert (
    density_counts
    .astype(int)
    .equals(
        expected_density_counts
    )
)


# ------------------------------------------------------------
# 0B.11 Frozen B0 prediction sanity
# ------------------------------------------------------------

assert (
    b0_df[
        "pred_count"
    ]
    >= 0
).all()


assert (
    b0_df[
        "gt_count"
    ]
    > 0
).all()


# Store the authoritative dev100 image table for later B2 steps

b2_dev100_image_df = (
    b0_image_map
    .copy()
)


# ------------------------------------------------------------
# 0B.12 Compact structure summary
# ------------------------------------------------------------

table_summary_df = pd.DataFrame(
    {
        "table": [
            "dev100",
            "B0",
            "C",
            "D",
        ],

        "rows": [
            len(
                dev100_df
            ),
            len(
                b0_df
            ),
            len(
                c_df
            ),
            len(
                d_df
            ),
        ],

        "columns": [
            len(
                dev100_df.columns
            ),
            len(
                b0_df.columns
            ),
            len(
                c_df.columns
            ),
            len(
                d_df.columns
            ),
        ],

        "unique_images": [
            dev100_df[
                "dataset_index"
            ].nunique(),
            b0_df[
                "dataset_index"
            ].nunique(),
            c_df[
                "dataset_index"
            ].nunique(),
            d_df[
                "dataset_index"
            ].nunique(),
        ],
    }
)


# ------------------------------------------------------------
# 0B.13 Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("B2 SETUP 0B - FROZEN TABLE LOAD + ALIGNMENT")
print("=" * 110)

print(
    "dev100 rows                 :",
    len(
        dev100_df
    ),
    "/ 100"
)

print(
    "B0 rows                     :",
    len(
        b0_df
    ),
    "/ 100"
)

print(
    "C rows                      :",
    len(
        c_df
    ),
    "/ 300"
)

print(
    "D rows                      :",
    len(
        d_df
    ),
    "/ 500"
)

print(
    "\nC candidate set             :",
    observed_c_candidates
)

print(
    "D candidate set             :",
    observed_d_candidates
)

print(
    "\nB0 <-> C image mapping      : PASS"
)

print(
    "B0 <-> D image mapping      : PASS"
)

print(
    "dev100 <-> B0 mapping       : PASS"
)

print(
    "Per-candidate completeness  : PASS"
)

print(
    "Density distribution        : PASS"
)

print(
    "Basic B0 numeric sanity     : PASS"
)

print(
    "\nShared dev100 identity cols :",
    dev100_identity_cols
)

print(
    "\nTable structure:"
)

display(
    table_summary_df
)

print(
    "\nFrozen dev100 density distribution:"
)

display(
    density_counts
    .rename(
        "num_images"
    )
    .to_frame()
)

print(
    "\nAuthoritative B2 image table:"
)

display(
    b2_dev100_image_df.head(
        10
    )
)

print(
    "\nNew model inference         : NO"
)

print(
    "Test set accessed           : NO"
)

print("=" * 110)

B2 SETUP 0B - FROZEN TABLE LOAD + ALIGNMENT
dev100 rows                 : 100 / 100
B0 rows                     : 100 / 100
C rows                      : 300 / 300
D rows                      : 500 / 500

C candidate set             : ['C1', 'C2b', 'C2c']
D candidate set             : ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']

B0 <-> C image mapping      : PASS
B0 <-> D image mapping      : PASS
dev100 <-> B0 mapping       : PASS
Per-candidate completeness  : PASS
Density distribution        : PASS
Basic B0 numeric sanity     : PASS

Shared dev100 identity cols : ['dataset_index', 'image_id', 'category', 'density_stratum', 'gt_count']

Table structure:


,table,rows,columns,unique_images
0,dev100,100,5,100
1,B0,100,13,100
2,C,300,27,100
3,D,500,48,100



Frozen dev100 density distribution:


,num_images
density_stratum,
S1_0_20,20
S2_20_40,20
S3_40_60,20
S4_60_80,20
S5_80_95,10
S6_95_100,10



Authoritative B2 image table:


,dataset_index,image_id,category,density_stratum,gt_count
0,9,215.jpg,grapes,S6_95_100,259
1,14,220.jpg,grapes,S4_60_80,59
2,20,228.jpg,grapes,S4_60_80,44
3,31,241.jpg,grapes,S3_40_60,35
4,40,252.jpg,grapes,S3_40_60,22
5,49,556.jpg,flamingos,S2_20_40,14
6,52,559.jpg,flamingos,S2_20_40,20
7,59,567.jpg,flamingos,S1_0_20,10
8,67,575.jpg,flamingos,S1_0_20,12
9,87,596.jpg,seagulls,S2_20_40,14



New model inference         : NO
Test set accessed           : NO


In [6]:
# ============================================================
# Setup 0C - Reproduce Frozen B0 SAM2 Environment / Provenance
# CORRECTED + RE-RUNNABLE VERSION
# ============================================================

from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

import torch


# ------------------------------------------------------------
# 0C.1 Frozen B0 provenance
# ------------------------------------------------------------

EXPECTED_SAM2_COMMIT = (
    "2b90b9f5ceec907a1c18123530e92e794ad901a4"
)

EXPECTED_CHECKPOINT_NAME = (
    "sam2.1_hiera_small.pt"
)

EXPECTED_CHECKPOINT_SHA256 = (
    "6d1aa6f30de5c92224f8172114de081d104bbd23dd9dc5c58996f0cad5dc4d38"
)

SAM2_REPO_URL = (
    "https://github.com/facebookresearch/sam2.git"
)

SAM2_REPO_DIR = Path(
    "/content/sam2_b0_frozen"
)


# This is the Hydra config identifier used by build_sam2().
B0_MODEL_CFG = (
    "configs/sam2.1/sam2.1_hiera_s.yaml"
)


# Physical location of the same config inside the cloned repo.
SAM2_CONFIG_FILE = (
    SAM2_REPO_DIR
    / "sam2"
    / B0_MODEL_CFG
)


# ------------------------------------------------------------
# 0C.2 Frozen checkpoint path
# ------------------------------------------------------------

SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "Checkpoints"
    / "SAM2"
    / EXPECTED_CHECKPOINT_NAME
)

assert (
    SAM2_CHECKPOINT.exists()
), (
    f"Frozen SAM2 checkpoint not found:\n"
    f"{SAM2_CHECKPOINT}"
)


# ------------------------------------------------------------
# 0C.3 CUDA / GPU verification
# ------------------------------------------------------------

assert torch.cuda.is_available(), (
    "CUDA is not available. "
    "Confirm that the Colab runtime is set to T4 GPU."
)

DEVICE = torch.device(
    "cuda"
)

gpu_name = torch.cuda.get_device_name(
    0
)


# ------------------------------------------------------------
# 0C.4 SHA256 helper
# ------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ------------------------------------------------------------
# 0C.5 Verify exact frozen checkpoint
# ------------------------------------------------------------

observed_checkpoint_sha256 = sha256_file(
    SAM2_CHECKPOINT
)


assert (
    observed_checkpoint_sha256
    == EXPECTED_CHECKPOINT_SHA256
), (
    "SAM2 checkpoint SHA256 mismatch.\n\n"
    f"Observed:\n{observed_checkpoint_sha256}\n\n"
    f"Expected:\n{EXPECTED_CHECKPOINT_SHA256}"
)


# ------------------------------------------------------------
# 0C.6 Obtain exact frozen SAM2 source checkout
#
# Re-runnable behaviour:
#
# - If repo does not exist -> clone.
# - If repo exists -> reuse it.
# - Always fetch/check out exact frozen commit.
# - Do NOT unnecessarily delete/reclone on every notebook run.
# ------------------------------------------------------------

if not SAM2_REPO_DIR.exists():

    subprocess.run(
        [
            "git",
            "clone",
            SAM2_REPO_URL,
            str(
                SAM2_REPO_DIR
            ),
        ],
        check=True,
    )


assert (
    (
        SAM2_REPO_DIR
        / ".git"
    ).exists()
), (
    f"{SAM2_REPO_DIR} exists but is not a Git repository."
)


# Make sure the required commit is available.

subprocess.run(
    [
        "git",
        "-C",
        str(
            SAM2_REPO_DIR
        ),
        "fetch",
        "--all",
        "--tags",
    ],
    check=True,
)


# Checkout exact frozen commit.

subprocess.run(
    [
        "git",
        "-C",
        str(
            SAM2_REPO_DIR
        ),
        "checkout",
        "--force",
        EXPECTED_SAM2_COMMIT,
    ],
    check=True,
)


# ------------------------------------------------------------
# 0C.7 Verify exact git commit
# ------------------------------------------------------------

observed_commit = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(
                SAM2_REPO_DIR
            ),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


assert (
    observed_commit
    == EXPECTED_SAM2_COMMIT
), (
    "SAM2 commit mismatch.\n\n"
    f"Observed:\n{observed_commit}\n\n"
    f"Expected:\n{EXPECTED_SAM2_COMMIT}"
)


# ------------------------------------------------------------
# 0C.8 Verify clean source checkout
# ------------------------------------------------------------

git_status = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(
                SAM2_REPO_DIR
            ),
            "status",
            "--porcelain",
        ],
        text=True,
    )
)


sam2_repo_dirty = (
    len(
        git_status.strip()
    )
    > 0
)


assert (
    not sam2_repo_dirty
), (
    "SAM2 source tree is unexpectedly dirty."
)


# ------------------------------------------------------------
# 0C.9 Verify corrected physical config path
# ------------------------------------------------------------

assert (
    SAM2_CONFIG_FILE.exists()
), (
    "SAM2 model config not found.\n\n"
    f"Expected physical path:\n"
    f"{SAM2_CONFIG_FILE}"
)


# ------------------------------------------------------------
# 0C.10 Install exact frozen SAM2 checkout
#
# Editable install keeps the imported package tied directly
# to the exact checked-out source tree.
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(
            SAM2_REPO_DIR
        ),
    ],
    check=True,
)


# Put exact checkout first on Python path.

repo_str = str(
    SAM2_REPO_DIR
)

if repo_str not in sys.path:

    sys.path.insert(
        0,
        repo_str,
    )


# ------------------------------------------------------------
# 0C.11 Import frozen B0 SAM2 components
# ------------------------------------------------------------

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator


# ------------------------------------------------------------
# 0C.12 Verify imported module resolves to this checkout
# ------------------------------------------------------------

import sam2


sam2_import_path = Path(
    sam2.__file__
).resolve()


assert (
    SAM2_REPO_DIR.resolve()
    in sam2_import_path.parents
), (
    "Imported SAM2 does not resolve to the frozen checkout.\n\n"
    f"Imported from:\n{sam2_import_path}"
)


# ------------------------------------------------------------
# 0C.13 Freeze B0 generator protocol
#
# No inference is performed here.
#
# Original frozen B0:
#
#   SAM2AutomaticMaskGenerator
#   no added post-filtering
#   num_selected_masks = num_raw_masks
#   pred_count = num_selected_masks
#
# No semantic guidance is introduced here.
# ------------------------------------------------------------

B0_GENERATOR_PROTOCOL = {
    "generator":
        "SAM2AutomaticMaskGenerator",

    "model":
        "SAM2.1 Hiera-Small",

    "model_cfg":
        B0_MODEL_CFG,

    "model_cfg_physical_path":
        str(
            SAM2_CONFIG_FILE
        ),

    "checkpoint_name":
        EXPECTED_CHECKPOINT_NAME,

    "checkpoint_sha256":
        EXPECTED_CHECKPOINT_SHA256,

    "additional_mask_filtering":
        False,

    "count_definition":
        "pred_count = num_raw_masks",

    "semantic_guidance":
        False,

    "test_set_accessed":
        False,
}


# ------------------------------------------------------------
# 0C.14 Save provenance metadata
# ------------------------------------------------------------

B2_PROVENANCE_FILE = (
    B2_METADATA_DIR
    / "B2_frozen_B0_SAM2_provenance.json"
)


b2_sam2_provenance = {
    "sam2_repo_url":
        SAM2_REPO_URL,

    "sam2_repo_path":
        str(
            SAM2_REPO_DIR
        ),

    "sam2_import_path":
        str(
            sam2_import_path
        ),

    "sam2_commit":
        observed_commit,

    "expected_sam2_commit":
        EXPECTED_SAM2_COMMIT,

    "sam2_repo_dirty":
        sam2_repo_dirty,

    "checkpoint_path":
        str(
            SAM2_CHECKPOINT
        ),

    "checkpoint_name":
        SAM2_CHECKPOINT.name,

    "checkpoint_sha256":
        observed_checkpoint_sha256,

    "expected_checkpoint_sha256":
        EXPECTED_CHECKPOINT_SHA256,

    "model_cfg":
        B0_MODEL_CFG,

    "model_cfg_physical_path":
        str(
            SAM2_CONFIG_FILE
        ),

    "device":
        str(
            DEVICE
        ),

    "gpu_name":
        gpu_name,

    "torch_version":
        torch.__version__,

    "cuda_version":
        torch.version.cuda,

    "b0_generator_protocol":
        B0_GENERATOR_PROTOCOL,

    "new_inference_performed":
        False,

    "test_set_accessed":
        False,
}


with open(
    B2_PROVENANCE_FILE,
    "w",
) as f:

    json.dump(
        b2_sam2_provenance,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 0C.15 Confirmation
# ------------------------------------------------------------

print("=" * 112)
print("B2 SETUP 0C - FROZEN B0 SAM2 ENVIRONMENT / PROVENANCE")
print("=" * 112)

print(
    "CUDA available             : PASS"
)

print(
    "GPU                        :",
    gpu_name
)

print(
    "Torch version              :",
    torch.__version__
)

print(
    "CUDA version               :",
    torch.version.cuda
)


print(
    "\nSAM2 repo                  :",
    SAM2_REPO_DIR
)

print(
    "SAM2 imported from         :",
    sam2_import_path
)

print(
    "SAM2 commit observed       :",
    observed_commit
)

print(
    "SAM2 commit expected       :",
    EXPECTED_SAM2_COMMIT
)

print(
    "SAM2 exact commit          : PASS"
)

print(
    "SAM2 repo dirty            :",
    sam2_repo_dirty
)


print(
    "\nCheckpoint                 :",
    SAM2_CHECKPOINT
)

print(
    "Checkpoint SHA256 observed :",
    observed_checkpoint_sha256
)

print(
    "Checkpoint SHA256 expected :",
    EXPECTED_CHECKPOINT_SHA256
)

print(
    "Checkpoint hash            : PASS"
)


print(
    "\nModel                      : SAM2.1 Hiera-Small"
)

print(
    "Model config identifier    :",
    B0_MODEL_CFG
)

print(
    "Model config physical path :",
    SAM2_CONFIG_FILE
)

print(
    "Model config exists        : PASS"
)

print(
    "Automatic mask generator   : IMPORT PASS"
)


print(
    "\nFrozen B0 protocol:"
)

for key, value in B0_GENERATOR_PROTOCOL.items():

    print(
        f"  {key:28s}: {value}"
    )


print(
    "\nNew image inference        : NO"
)

print(
    "Test set accessed          : NO"
)

print(
    "\nSaved provenance           :",
    B2_PROVENANCE_FILE
)

print("=" * 112)

B2 SETUP 0C - FROZEN B0 SAM2 ENVIRONMENT / PROVENANCE
CUDA available             : PASS
GPU                        : Tesla T4
Torch version              : 2.11.0+cu128
CUDA version               : 12.8

SAM2 repo                  : /content/sam2_b0_frozen
SAM2 imported from         : /content/sam2_b0_frozen/sam2/__init__.py
SAM2 commit observed       : 2b90b9f5ceec907a1c18123530e92e794ad901a4
SAM2 commit expected       : 2b90b9f5ceec907a1c18123530e92e794ad901a4
SAM2 exact commit          : PASS
SAM2 repo dirty            : False

Checkpoint                 : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/Checkpoints/SAM2/sam2.1_hiera_small.pt
Checkpoint SHA256 observed : 6d1aa6f30de5c92224f8172114de081d104bbd23dd9dc5c58996f0cad5dc4d38
Checkpoint SHA256 expected : 6d1aa6f30de5c92224f8172114de081d104bbd23dd9dc5c58996f0cad5dc4d38
Checkpoint hash            : PASS

Model                      : SAM2.1 Hiera-Small
Model config identifier    : configs/sam2.1/sam2.1_hiera_s.yaml
Model co

In [7]:
# ============================================================
# Setup 0D - Frozen B0 Reproduction Sanity Check
# 3 dev100 images only
# ============================================================

from pathlib import Path
import time

import cv2
import pandas as pd
import torch

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator


# ------------------------------------------------------------
# 0D.1 Scope
# ------------------------------------------------------------

SANITY_STRATA = [
    "S1_0_20",   # sparse
    "S4_60_80",  # mid/high
    "S6_95_100", # very dense
]

DATASETS_ROOT = (
    PROJECT_ROOT
    / "datasets"
)


assert DATASETS_ROOT.exists(), (
    f"Datasets directory not found:\n"
    f"{DATASETS_ROOT}"
)


# ------------------------------------------------------------
# 0D.2 Deterministically select one image per stratum
#
# First dataset_index within each stratum.
# ------------------------------------------------------------

sanity_rows = []


for stratum in SANITY_STRATA:

    subset = (
        b0_df[
            b0_df[
                "density_stratum"
            ]
            == stratum
        ]
        .sort_values(
            "dataset_index"
        )
    )

    assert (
        len(
            subset
        )
        > 0
    ), (
        f"No dev100 image found for {stratum}"
    )

    sanity_rows.append(
        subset.iloc[
            0
        ]
    )


sanity_df = pd.DataFrame(
    sanity_rows
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# 0D.3 Locate each FSC-147 image under project datasets
#
# Search is restricted to PROJECT_ROOT/datasets and only
# for the three exact filenames.
# ------------------------------------------------------------

def locate_dataset_image(
    image_id,
):

    matches = list(
        DATASETS_ROOT.rglob(
            image_id
        )
    )

    matches = [
        path
        for path in matches
        if path.is_file()
    ]


    assert (
        len(
            matches
        )
        > 0
    ), (
        f"Could not locate image '{image_id}' "
        f"under:\n{DATASETS_ROOT}"
    )


    # If duplicate copies exist, show them rather than silently
    # assuming they are interchangeable.
    if len(
        matches
    ) > 1:

        print(
            f"\nWARNING: multiple copies found for {image_id}:"
        )

        for path in matches:

            print(
                "  ",
                path
            )


    return matches[
        0
    ]


sanity_df[
    "image_path"
] = [
    str(
        locate_dataset_image(
            image_id
        )
    )
    for image_id in sanity_df[
        "image_id"
    ]
]


# ------------------------------------------------------------
# 0D.4 Build exact frozen B0 SAM2 model
#
# This follows the standard SAM2 automatic-mask setup used
# by the frozen B0 protocol:
#
# build_sam2(..., apply_postprocessing=False)
# +
# native SAM2AutomaticMaskGenerator(model)
#
# No extra filtering is introduced.
# ------------------------------------------------------------

torch.cuda.empty_cache()


sam2_b0_model = build_sam2(
    B0_MODEL_CFG,
    str(
        SAM2_CHECKPOINT
    ),
    device=DEVICE,
    apply_postprocessing=False,
)


b0_mask_generator = SAM2AutomaticMaskGenerator(
    sam2_b0_model
)


# ------------------------------------------------------------
# 0D.5 Run only the three sanity images
# ------------------------------------------------------------

sanity_results = []


print("=" * 118)
print("B2 SETUP 0D - FROZEN B0 REPRODUCTION SANITY CHECK")
print("=" * 118)


for row in sanity_df.itertuples(
    index=False
):

    image_path = Path(
        row.image_path
    )


    image_bgr = cv2.imread(
        str(
            image_path
        ),
        cv2.IMREAD_COLOR,
    )


    assert (
        image_bgr
        is not None
    ), (
        f"Failed to read image:\n"
        f"{image_path}"
    )


    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )


    start = time.time()


    with torch.inference_mode():

        masks = b0_mask_generator.generate(
            image_rgb
        )


    elapsed = time.time() - start


    reproduced_count = int(
        len(
            masks
        )
    )

    frozen_count = int(
        row.pred_count
    )

    count_match = (
        reproduced_count
        == frozen_count
    )


    sanity_results.append(
        {
            "dataset_index":
                int(
                    row.dataset_index
                ),

            "image_id":
                row.image_id,

            "category":
                row.category,

            "density_stratum":
                row.density_stratum,

            "gt_count":
                int(
                    row.gt_count
                ),

            "frozen_b0_count":
                frozen_count,

            "reproduced_b0_count":
                reproduced_count,

            "count_match":
                count_match,

            "runtime_sec":
                elapsed,

            "image_path":
                str(
                    image_path
                ),
        }
    )


    print(
        f"{row.density_stratum:12s} | "
        f"{row.image_id:12s} | "
        f"GT={int(row.gt_count):4d} | "
        f"frozen B0={frozen_count:4d} | "
        f"reproduced={reproduced_count:4d} | "
        f"match={count_match} | "
        f"{elapsed:.2f}s"
    )


# ------------------------------------------------------------
# 0D.6 Results table
# ------------------------------------------------------------

b0_reproduction_sanity_df = pd.DataFrame(
    sanity_results
)


num_matches = int(
    b0_reproduction_sanity_df[
        "count_match"
    ].sum()
)


print("-" * 118)

print(
    "Exact count matches         :",
    f"{num_matches}/3"
)

print(
    "New inference performed     : YES - 3 dev100 images only"
)

print(
    "Masks cached                : NO"
)

print(
    "Semantic processing         : NO"
)

print(
    "Test set accessed           : NO"
)


display(
    b0_reproduction_sanity_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "frozen_b0_count",
            "reproduced_b0_count",
            "count_match",
            "runtime_sec",
        ]
    ]
)


# ------------------------------------------------------------
# 0D.7 Require exact reproduction before continuing
# ------------------------------------------------------------

assert (
    num_matches
    == 3
), (
    "Frozen B0 reproduction failed on at least one sanity image.\n"
    "Do NOT proceed to full dev100 mask caching until the "
    "configuration difference is identified."
)


print(
    "\nFrozen B0 reproduction      : PASS"
)

print(
    "Safe to proceed to dev100 mask caching only after "
    "reviewing this output."
)

print("=" * 118)

B2 SETUP 0D - FROZEN B0 REPRODUCTION SANITY CHECK
S1_0_20      | 567.jpg      | GT=  10 | frozen B0=   0 | reproduced=   0 | match=True | 4.99s
S4_60_80     | 220.jpg      | GT=  59 | frozen B0=  20 | reproduced=  20 | match=True | 4.09s
S6_95_100    | 215.jpg      | GT= 259 | frozen B0= 103 | reproduced= 103 | match=True | 4.01s
----------------------------------------------------------------------------------------------------------------------
Exact count matches         : 3/3
New inference performed     : YES - 3 dev100 images only
Masks cached                : NO
Semantic processing         : NO
Test set accessed           : NO


,dataset_index,image_id,category,density_stratum,gt_count,frozen_b0_count,reproduced_b0_count,count_match,runtime_sec
0,59,567.jpg,flamingos,S1_0_20,10,0,0,True,4.989651
1,14,220.jpg,grapes,S4_60_80,59,20,20,True,4.091525
2,9,215.jpg,grapes,S6_95_100,259,103,103,True,4.014161



Frozen B0 reproduction      : PASS
Safe to proceed to dev100 mask caching only after reviewing this output.


In [9]:
# ============================================================
# Step 1A - Resolve Duplicate dev100 Image Paths Deterministically
# ============================================================

from pathlib import Path
import hashlib
import os

import cv2
import pandas as pd


# ------------------------------------------------------------
# 1A.1 Frozen dev100 image IDs
# ------------------------------------------------------------

required_image_ids = set(
    b2_dev100_image_df[
        "image_id"
    ].tolist()
)


# ------------------------------------------------------------
# 1A.2 Find all copies under datasets/
# ------------------------------------------------------------

image_path_candidates = {
    image_id: []
    for image_id in required_image_ids
}


for root, dirs, files in os.walk(
    DATASETS_ROOT
):

    for filename in files:

        if filename in required_image_ids:

            image_path_candidates[
                filename
            ].append(
                Path(root)
                / filename
            )


# ------------------------------------------------------------
# 1A.3 Basic completeness
# ------------------------------------------------------------

missing_images = [
    image_id
    for image_id, paths
    in image_path_candidates.items()
    if len(
        paths
    )
    == 0
]


assert (
    len(
        missing_images
    )
    == 0
), (
    "Missing frozen dev100 images:\n"
    f"{missing_images}"
)


duplicate_image_ids = sorted(
    [
        image_id
        for image_id, paths
        in image_path_candidates.items()
        if len(
            paths
        )
        > 1
    ]
)


print(
    "Unique-path images          :",
    sum(
        len(
            paths
        )
        == 1
        for paths
        in image_path_candidates.values()
    )
)

print(
    "Duplicate filename images   :",
    len(
        duplicate_image_ids
    )
)

print(
    "Missing images              :",
    len(
        missing_images
    )
)


# ------------------------------------------------------------
# 1A.4 SHA256 helper
# ------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ------------------------------------------------------------
# 1A.5 Inspect all duplicate copies
# ------------------------------------------------------------

duplicate_records = []


for image_id in duplicate_image_ids:

    paths = sorted(
        image_path_candidates[
            image_id
        ],
        key=lambda p: str(
            p
        ),
    )


    for copy_idx, path in enumerate(
        paths,
        start=1,
    ):

        file_hash = sha256_file(
            path
        )


        image = cv2.imread(
            str(
                path
            ),
            cv2.IMREAD_COLOR,
        )


        assert (
            image
            is not None
        ), (
            f"Failed to decode duplicate image:\n{path}"
        )


        height = int(
            image.shape[
                0
            ]
        )

        width = int(
            image.shape[
                1
            ]
        )

        channels = int(
            image.shape[
                2
            ]
        )


        duplicate_records.append(
            {
                "image_id":
                    image_id,

                "copy_idx":
                    copy_idx,

                "path":
                    str(
                        path
                    ),

                "sha256":
                    file_hash,

                "height":
                    height,

                "width":
                    width,

                "channels":
                    channels,

                "file_size_bytes":
                    int(
                        path.stat().st_size
                    ),
            }
        )


duplicate_check_df = pd.DataFrame(
    duplicate_records
)


# ------------------------------------------------------------
# 1A.6 Compare duplicates per image_id
# ------------------------------------------------------------

comparison_records = []


for image_id in duplicate_image_ids:

    group = (
        duplicate_check_df[
            duplicate_check_df[
                "image_id"
            ]
            == image_id
        ]
    )


    num_copies = int(
        len(
            group
        )
    )

    unique_hashes = int(
        group[
            "sha256"
        ]
        .nunique()
    )

    unique_shapes = int(
        group[
            [
                "height",
                "width",
                "channels",
            ]
        ]
        .drop_duplicates()
        .shape[
            0
        ]
    )

    unique_sizes = int(
        group[
            "file_size_bytes"
        ]
        .nunique()
    )


    byte_identical = (
        unique_hashes
        == 1
    )

    same_shape = (
        unique_shapes
        == 1
    )


    comparison_records.append(
        {
            "image_id":
                image_id,

            "num_copies":
                num_copies,

            "unique_hashes":
                unique_hashes,

            "unique_shapes":
                unique_shapes,

            "unique_file_sizes":
                unique_sizes,

            "byte_identical":
                byte_identical,

            "same_shape":
                same_shape,
        }
    )


duplicate_summary_df = pd.DataFrame(
    comparison_records
)


# ------------------------------------------------------------
# 1A.7 Fail if any duplicate copies differ
# ------------------------------------------------------------

non_identical_duplicates = (
    duplicate_summary_df[
        ~duplicate_summary_df[
            "byte_identical"
        ]
    ]
    .copy()
)


if len(
    non_identical_duplicates
) > 0:

    print(
        "\nNON-IDENTICAL DUPLICATE IMAGES FOUND:"
    )

    display(
        non_identical_duplicates
    )

    print(
        "\nDetailed paths:"
    )

    display(
        duplicate_check_df[
            duplicate_check_df[
                "image_id"
            ].isin(
                non_identical_duplicates[
                    "image_id"
                ]
            )
        ]
        .sort_values(
            [
                "image_id",
                "path",
            ]
        )
    )


    raise AssertionError(
        "At least one duplicate dev100 filename has "
        "non-identical file contents. "
        "Do not choose an image path automatically."
    )


# ------------------------------------------------------------
# 1A.8 Build deterministic path map
#
# All duplicate copies are byte-identical at this point.
# For reproducibility, choose lexicographically first path.
# ------------------------------------------------------------

image_path_map = {}


for image_id, paths in image_path_candidates.items():

    sorted_paths = sorted(
        paths,
        key=lambda p: str(
            p
        ),
    )

    image_path_map[
        image_id
    ] = sorted_paths[
        0
    ]


assert (
    len(
        image_path_map
    )
    == 100
)


# ------------------------------------------------------------
# 1A.9 Build resolved path table
# ------------------------------------------------------------

resolved_records = []


for row in b2_dev100_image_df.itertuples(
    index=False
):

    image_id = row.image_id

    selected_path = image_path_map[
        image_id
    ]

    num_copies = len(
        image_path_candidates[
            image_id
        ]
    )


    resolved_records.append(
        {
            "dataset_index":
                int(
                    row.dataset_index
                ),

            "image_id":
                image_id,

            "category":
                row.category,

            "density_stratum":
                row.density_stratum,

            "gt_count":
                int(
                    row.gt_count
                ),

            "num_copies_found":
                num_copies,

            "selected_path":
                str(
                    selected_path
                ),

            "selection_rule":
                (
                    "only_copy"
                    if num_copies == 1
                    else "lexicographically_first_of_byte_identical_copies"
                ),
        }
    )


resolved_image_paths_df = pd.DataFrame(
    resolved_records
)


# ------------------------------------------------------------
# 1A.10 Save audit tables
# ------------------------------------------------------------

DUPLICATE_DETAIL_FILE = (
    B2_TABLES_DIR
    / "B0_dev100_duplicate_image_path_details.csv"
)

DUPLICATE_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_dev100_duplicate_image_path_summary.csv"
)

RESOLVED_PATH_FILE = (
    B2_TABLES_DIR
    / "B0_dev100_resolved_image_paths.csv"
)


duplicate_check_df.to_csv(
    DUPLICATE_DETAIL_FILE,
    index=False,
)

duplicate_summary_df.to_csv(
    DUPLICATE_SUMMARY_FILE,
    index=False,
)

resolved_image_paths_df.to_csv(
    RESOLVED_PATH_FILE,
    index=False,
)


# ------------------------------------------------------------
# 1A.11 Confirmation
# ------------------------------------------------------------

print("=" * 120)
print("STEP 1A - dev100 DUPLICATE IMAGE PATH RESOLUTION")
print("=" * 120)

print(
    "dev100 images               :",
    len(
        resolved_image_paths_df
    )
)

print(
    "Images with one copy        :",
    int(
        (
            resolved_image_paths_df[
                "num_copies_found"
            ]
            == 1
        ).sum()
    )
)

print(
    "Images with duplicate names :",
    int(
        (
            resolved_image_paths_df[
                "num_copies_found"
            ]
            > 1
        ).sum()
    )
)

print(
    "Non-identical duplicates    :",
    len(
        non_identical_duplicates
    )
)

print(
    "All duplicate copies equal  : PASS"
)


print(
    "\nDuplicate summary:"
)

display(
    duplicate_summary_df
)


print(
    "\nDuplicate file details:"
)

display(
    duplicate_check_df
    .sort_values(
        [
            "image_id",
            "path",
        ]
    )
)


print(
    "\nFirst 15 resolved image paths:"
)

display(
    resolved_image_paths_df.head(
        15
    )
)


print(
    "\nSaved duplicate details:"
)

print(
    DUPLICATE_DETAIL_FILE
)

print(
    "\nSaved duplicate summary:"
)

print(
    DUPLICATE_SUMMARY_FILE
)

print(
    "\nSaved resolved path table:"
)

print(
    RESOLVED_PATH_FILE
)

print("=" * 120)

Unique-path images          : 89
Duplicate filename images   : 11
Missing images              : 0
STEP 1A - dev100 DUPLICATE IMAGE PATH RESOLUTION
dev100 images               : 100
Images with one copy        : 89
Images with duplicate names : 11
Non-identical duplicates    : 0
All duplicate copies equal  : PASS

Duplicate summary:


,image_id,num_copies,unique_hashes,unique_shapes,unique_file_sizes,byte_identical,same_shape
0,1890.jpg,2,1,1,1,True,True
1,1935.jpg,2,1,1,1,True,True
2,1964.jpg,2,1,1,1,True,True
3,1973.jpg,2,1,1,1,True,True
4,1976.jpg,2,1,1,1,True,True
5,1992.jpg,2,1,1,1,True,True
6,819.jpg,2,1,1,1,True,True
7,824.jpg,2,1,1,1,True,True
8,826.jpg,2,1,1,1,True,True
9,831.jpg,2,1,1,1,True,True



Duplicate file details:


,image_id,copy_idx,path,sha256,height,width,channels,file_size_bytes
0,1890.jpg,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,e89d276f03536b0a09e126c4fdde8f0a69cfe6a0402d49...,384,408,3,26962
1,1890.jpg,2,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,e89d276f03536b0a09e126c4fdde8f0a69cfe6a0402d49...,384,408,3,26962
2,1935.jpg,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,c8136f2dcd28100500c145fce1dc36f1b367480eae76ae...,384,384,3,36264
3,1935.jpg,2,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,c8136f2dcd28100500c145fce1dc36f1b367480eae76ae...,384,384,3,36264
4,1964.jpg,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,ce1a9cd744ab9209e7077d3c0f2377666536ff486978da...,384,540,3,30269
5,1964.jpg,2,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,ce1a9cd744ab9209e7077d3c0f2377666536ff486978da...,384,540,3,30269
6,1973.jpg,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,490d88ea65217c46c1be07622270cc4d35a7f599320c89...,384,576,3,46987
7,1973.jpg,2,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,490d88ea65217c46c1be07622270cc4d35a7f599320c89...,384,576,3,46987
8,1976.jpg,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,d270f7cc0294e649915b80808742ecd08f89758f5e994b...,384,517,3,25417
9,1976.jpg,2,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,d270f7cc0294e649915b80808742ecd08f89758f5e994b...,384,517,3,25417



First 15 resolved image paths:


,dataset_index,image_id,category,density_stratum,gt_count,num_copies_found,selected_path,selection_rule
0,9,215.jpg,grapes,S6_95_100,259,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
1,14,220.jpg,grapes,S4_60_80,59,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
2,20,228.jpg,grapes,S4_60_80,44,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
3,31,241.jpg,grapes,S3_40_60,35,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
4,40,252.jpg,grapes,S3_40_60,22,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
5,49,556.jpg,flamingos,S2_20_40,14,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
6,52,559.jpg,flamingos,S2_20_40,20,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
7,59,567.jpg,flamingos,S1_0_20,10,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
8,67,575.jpg,flamingos,S1_0_20,12,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy
9,87,596.jpg,seagulls,S2_20_40,14,1,/content/drive/MyDrive/MSc_Project_ZeroShot_Co...,only_copy



Saved duplicate details:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_dev100_duplicate_image_path_details.csv

Saved duplicate summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_dev100_duplicate_image_path_summary.csv

Saved resolved path table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_dev100_resolved_image_paths.csv


In [10]:
# ============================================================
# Step 1 - Reproduce + Cache Frozen B0 Masks for dev100
# REVISED: uses deterministic resolved image paths from Step 1A
# ============================================================

from pathlib import Path
import json
import os
import time
import hashlib

import cv2
import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1.1 Output structure
# ------------------------------------------------------------

B0_MASK_CACHE_DIR = (
    B2_CACHE_DIR
    / "B0_masks"
)

B0_MASK_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


B0_MASK_INDEX_FILE = (
    B2_TABLES_DIR
    / "B0_mask_cache_index.csv"
)

B0_MASK_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_mask_cache_manifest.json"
)


# ------------------------------------------------------------
# 1.2 Require deterministic path resolution from Step 1A
# ------------------------------------------------------------

assert (
    "resolved_image_paths_df"
    in globals()
), (
    "resolved_image_paths_df is not available. "
    "Run Step 1A first."
)

assert (
    "image_path_map"
    in globals()
), (
    "image_path_map is not available. "
    "Run Step 1A first."
)


assert (
    len(
        resolved_image_paths_df
    )
    == 100
)


assert (
    len(
        image_path_map
    )
    == 100
)


assert (
    resolved_image_paths_df[
        "dataset_index"
    ]
    .is_unique
)

assert (
    resolved_image_paths_df[
        "image_id"
    ]
    .is_unique
)


# ------------------------------------------------------------
# 1.3 Verify resolved paths still exist
# ------------------------------------------------------------

missing_resolved_paths = []


for row in resolved_image_paths_df.itertuples(
    index=False
):

    path = Path(
        row.selected_path
    )

    if not path.exists():

        missing_resolved_paths.append(
            {
                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    row.image_id,

                "path":
                    str(
                        path
                    ),
            }
        )


assert (
    len(
        missing_resolved_paths
    )
    == 0
), (
    "Some resolved image paths no longer exist:\n"
    f"{missing_resolved_paths}"
)


# ------------------------------------------------------------
# 1.4 Verify exact alignment of resolved paths with dev100
# ------------------------------------------------------------

resolved_alignment = (
    b2_dev100_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .merge(
        resolved_image_paths_df[
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
                "num_copies_found",
                "selected_path",
                "selection_rule",
            ]
        ],
        on=[
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    len(
        resolved_alignment
    )
    == 100
)


assert (
    resolved_alignment[
        "selected_path"
    ]
    .notna()
    .all()
)


# Use this as the authoritative B2 run table.

b2_run_image_df = (
    resolved_alignment
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 1.5 Frozen B0 lookup
# ------------------------------------------------------------

b0_lookup = (
    b0_df
    .set_index(
        "dataset_index"
    )
)


assert (
    b0_lookup.index.is_unique
)


# ------------------------------------------------------------
# 1.6 Helper: save one image's masks losslessly
# ------------------------------------------------------------

def save_b0_mask_cache(
    cache_path,
    masks,
    image_height,
    image_width,
):

    num_masks = len(
        masks
    )

    num_pixels = (
        image_height
        *
        image_width
    )

    num_packed_bytes = (
        num_pixels
        + 7
    ) // 8


    # --------------------------------------------------------
    # Binary masks
    # --------------------------------------------------------

    if num_masks > 0:

        mask_stack = np.stack(
            [
                np.asarray(
                    item[
                        "segmentation"
                    ],
                    dtype=np.bool_,
                )
                for item in masks
            ],
            axis=0,
        )


        assert (
            mask_stack.shape
            == (
                num_masks,
                image_height,
                image_width,
            )
        )


        packed_masks = np.packbits(
            mask_stack.reshape(
                num_masks,
                -1,
            ),
            axis=1,
            bitorder="little",
        )


        assert (
            packed_masks.shape
            == (
                num_masks,
                num_packed_bytes,
            )
        )


    else:

        packed_masks = np.empty(
            (
                0,
                num_packed_bytes,
            ),
            dtype=np.uint8,
        )


    # --------------------------------------------------------
    # Native SAM2 mask metadata
    # --------------------------------------------------------

    areas = np.asarray(
        [
            item[
                "area"
            ]
            for item in masks
        ],
        dtype=np.int64,
    )


    predicted_iou = np.asarray(
        [
            item[
                "predicted_iou"
            ]
            for item in masks
        ],
        dtype=np.float32,
    )


    stability_score = np.asarray(
        [
            item[
                "stability_score"
            ]
            for item in masks
        ],
        dtype=np.float32,
    )


    if num_masks > 0:

        bbox_xywh = np.asarray(
            [
                item[
                    "bbox"
                ]
                for item in masks
            ],
            dtype=np.float32,
        ).reshape(
            num_masks,
            4,
        )


        crop_box_xywh = np.asarray(
            [
                item[
                    "crop_box"
                ]
                for item in masks
            ],
            dtype=np.float32,
        ).reshape(
            num_masks,
            4,
        )


        point_xy = np.asarray(
            [
                item[
                    "point_coords"
                ][
                    0
                ]
                for item in masks
            ],
            dtype=np.float32,
        ).reshape(
            num_masks,
            2,
        )


    else:

        bbox_xywh = np.empty(
            (
                0,
                4,
            ),
            dtype=np.float32,
        )

        crop_box_xywh = np.empty(
            (
                0,
                4,
            ),
            dtype=np.float32,
        )

        point_xy = np.empty(
            (
                0,
                2,
            ),
            dtype=np.float32,
        )


    # --------------------------------------------------------
    # Save compressed NPZ
    # --------------------------------------------------------

    np.savez_compressed(
        cache_path,

        packed_masks=packed_masks,

        image_height=np.asarray(
            image_height,
            dtype=np.int32,
        ),

        image_width=np.asarray(
            image_width,
            dtype=np.int32,
        ),

        num_masks=np.asarray(
            num_masks,
            dtype=np.int32,
        ),

        bitorder=np.asarray(
            "little"
        ),

        areas=areas,

        predicted_iou=predicted_iou,

        stability_score=stability_score,

        bbox_xywh=bbox_xywh,

        crop_box_xywh=crop_box_xywh,

        point_xy=point_xy,
    )


# ------------------------------------------------------------
# 1.7 Helper: lightweight saved-cache validation
# ------------------------------------------------------------

def validate_saved_cache(
    cache_path,
    expected_num_masks,
    expected_height,
    expected_width,
):

    with np.load(
        cache_path,
        allow_pickle=False,
    ) as data:

        observed_num_masks = int(
            data[
                "num_masks"
            ]
        )

        observed_height = int(
            data[
                "image_height"
            ]
        )

        observed_width = int(
            data[
                "image_width"
            ]
        )

        packed_masks = data[
            "packed_masks"
        ]


        assert (
            observed_num_masks
            == expected_num_masks
        )

        assert (
            observed_height
            == expected_height
        )

        assert (
            observed_width
            == expected_width
        )

        assert (
            packed_masks.shape[
                0
            ]
            == expected_num_masks
        )

        assert (
            len(
                data[
                    "areas"
                ]
            )
            == expected_num_masks
        )

        assert (
            len(
                data[
                    "predicted_iou"
                ]
            )
            == expected_num_masks
        )

        assert (
            len(
                data[
                    "stability_score"
                ]
            )
            == expected_num_masks
        )


# ------------------------------------------------------------
# 1.8 Clear any partial cache from an interrupted prior run
#
# Safe here because Step 1 is a deterministic reconstruction
# of the frozen B0 cache.
# ------------------------------------------------------------

existing_partial_cache_files = list(
    B0_MASK_CACHE_DIR.glob(
        "*_B0_masks.npz"
    )
)


if existing_partial_cache_files:

    print(
        f"Removing {len(existing_partial_cache_files)} "
        "partial B0 cache files from earlier interrupted run..."
    )

    for path in existing_partial_cache_files:

        path.unlink()


# ------------------------------------------------------------
# 1.9 Full dev100 inference
#
# Exact frozen B0:
# - SAM2AutomaticMaskGenerator
# - no semantic processing
# - no additional filtering
# - reproduced count must equal frozen B0 count
# ------------------------------------------------------------

cache_records = []

run_start = time.time()


print("=" * 125)
print("STEP 1 - REPRODUCE + CACHE FROZEN B0 MASKS")
print("=" * 125)

print(
    "Images to process          :",
    len(
        b2_run_image_df
    )
)

print(
    "Resolved image paths       : PASS"
)

print(
    "Duplicate-path ambiguity   : RESOLVED IN STEP 1A"
)

print(
    "Generator                  : SAM2AutomaticMaskGenerator"
)

print(
    "Additional mask filtering  : NO"
)

print(
    "Semantic processing        : NO"
)

print(
    "Test set accessed          : NO"
)

print("-" * 125)


for run_idx, row in enumerate(
    b2_run_image_df.itertuples(
        index=False
    ),
    start=1,
):

    dataset_index = int(
        row.dataset_index
    )

    image_id = row.image_id


    # --------------------------------------------------------
    # Frozen B0 reference
    # --------------------------------------------------------

    frozen_row = b0_lookup.loc[
        dataset_index
    ]


    assert (
        frozen_row[
            "image_id"
        ]
        == image_id
    )


    frozen_b0_count = int(
        frozen_row[
            "pred_count"
        ]
    )


    # --------------------------------------------------------
    # Deterministically resolved image path
    # --------------------------------------------------------

    image_path = Path(
        row.selected_path
    )


    assert (
        image_path.exists()
    )


    # --------------------------------------------------------
    # Read image
    # --------------------------------------------------------

    image_bgr = cv2.imread(
        str(
            image_path
        ),
        cv2.IMREAD_COLOR,
    )


    assert (
        image_bgr
        is not None
    ), (
        f"Failed to read:\n"
        f"{image_path}"
    )


    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )


    image_height = int(
        image_rgb.shape[
            0
        ]
    )

    image_width = int(
        image_rgb.shape[
            1
        ]
    )


    # --------------------------------------------------------
    # Exact frozen B0 inference
    # --------------------------------------------------------

    image_start = time.time()


    with torch.inference_mode():

        masks = b0_mask_generator.generate(
            image_rgb
        )


    runtime_sec = (
        time.time()
        -
        image_start
    )


    reproduced_count = int(
        len(
            masks
        )
    )


    count_match = (
        reproduced_count
        == frozen_b0_count
    )


    # --------------------------------------------------------
    # Do not cache a changed baseline
    # --------------------------------------------------------

    if not count_match:

        raise AssertionError(
            "\nFrozen B0 count mismatch.\n"
            f"dataset_index : {dataset_index}\n"
            f"image_id      : {image_id}\n"
            f"image_path    : {image_path}\n"
            f"frozen count  : {frozen_b0_count}\n"
            f"reproduced    : {reproduced_count}\n\n"
            "Stopping immediately. "
            "Do not continue B2 with a changed B0."
        )


    # --------------------------------------------------------
    # Save masks
    # --------------------------------------------------------

    cache_filename = (
        f"{dataset_index:05d}_"
        f"{Path(image_id).stem}_"
        f"B0_masks.npz"
    )


    cache_path = (
        B0_MASK_CACHE_DIR
        / cache_filename
    )


    save_b0_mask_cache(
        cache_path=cache_path,
        masks=masks,
        image_height=image_height,
        image_width=image_width,
    )


    validate_saved_cache(
        cache_path=cache_path,
        expected_num_masks=reproduced_count,
        expected_height=image_height,
        expected_width=image_width,
    )


    cache_size_bytes = int(
        cache_path.stat().st_size
    )


    # --------------------------------------------------------
    # Record metadata
    # --------------------------------------------------------

    cache_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "category":
                row.category,

            "density_stratum":
                row.density_stratum,

            "gt_count":
                int(
                    row.gt_count
                ),

            "num_copies_found":
                int(
                    row.num_copies_found
                ),

            "path_selection_rule":
                row.selection_rule,

            "image_path":
                str(
                    image_path
                ),

            "frozen_b0_count":
                frozen_b0_count,

            "reproduced_b0_count":
                reproduced_count,

            "count_match":
                count_match,

            "image_height":
                image_height,

            "image_width":
                image_width,

            "runtime_sec":
                runtime_sec,

            "cache_size_bytes":
                cache_size_bytes,

            "cache_file":
                str(
                    cache_path
                ),
        }
    )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"[{run_idx:3d}/100] "
        f"{image_id:12s} | "
        f"{row.density_stratum:12s} | "
        f"GT={int(row.gt_count):4d} | "
        f"B0={reproduced_count:4d} | "
        f"match=YES | "
        f"{runtime_sec:6.2f}s | "
        f"cache={cache_size_bytes / (1024**2):6.2f} MiB"
    )


    del masks
    del image_rgb
    del image_bgr


# ------------------------------------------------------------
# 1.10 Build cache index
# ------------------------------------------------------------

b0_mask_cache_index_df = pd.DataFrame(
    cache_records
)


assert (
    len(
        b0_mask_cache_index_df
    )
    == 100
)


assert (
    b0_mask_cache_index_df[
        "count_match"
    ]
    .all()
)


assert (
    b0_mask_cache_index_df[
        "dataset_index"
    ]
    .is_unique
)


assert (
    b0_mask_cache_index_df[
        "image_id"
    ]
    .is_unique
)


# ------------------------------------------------------------
# 1.11 Exact count identity
# ------------------------------------------------------------

assert np.array_equal(
    b0_mask_cache_index_df[
        "frozen_b0_count"
    ].to_numpy(),
    b0_mask_cache_index_df[
        "reproduced_b0_count"
    ].to_numpy(),
)


total_frozen_masks = int(
    b0_mask_cache_index_df[
        "frozen_b0_count"
    ].sum()
)

total_cached_masks = int(
    b0_mask_cache_index_df[
        "reproduced_b0_count"
    ].sum()
)


assert (
    total_frozen_masks
    == total_cached_masks
)


# ------------------------------------------------------------
# 1.12 Cache-file completeness
# ------------------------------------------------------------

cache_files = sorted(
    B0_MASK_CACHE_DIR.glob(
        "*_B0_masks.npz"
    )
)


assert (
    len(
        cache_files
    )
    == 100
), (
    f"Expected 100 B0 cache files, "
    f"found {len(cache_files)}."
)


# ------------------------------------------------------------
# 1.13 Save cache index
# ------------------------------------------------------------

b0_mask_cache_index_df.to_csv(
    B0_MASK_INDEX_FILE,
    index=False,
)


# ------------------------------------------------------------
# 1.14 Summary statistics
# ------------------------------------------------------------

total_runtime_sec = (
    time.time()
    -
    run_start
)


total_cache_bytes = int(
    b0_mask_cache_index_df[
        "cache_size_bytes"
    ].sum()
)


runtime_summary = (
    b0_mask_cache_index_df[
        "runtime_sec"
    ]
    .describe()
)


count_by_density_df = (
    b0_mask_cache_index_df
    .groupby(
        "density_stratum",
        sort=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        total_b0_masks=(
            "reproduced_b0_count",
            "sum",
        ),

        mean_b0_masks=(
            "reproduced_b0_count",
            "mean",
        ),

        median_b0_masks=(
            "reproduced_b0_count",
            "median",
        ),
    )
    .reindex(
        DENSITY_ORDER
    )
)


# ------------------------------------------------------------
# 1.15 Bundle hash
# ------------------------------------------------------------

bundle_hasher = hashlib.sha256()


for cache_path in cache_files:

    bundle_hasher.update(
        cache_path.name.encode(
            "utf-8"
        )
    )

    with open(
        cache_path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            bundle_hasher.update(
                chunk
            )


cache_bundle_sha256 = (
    bundle_hasher.hexdigest()
)


# ------------------------------------------------------------
# 1.16 Save manifest
# ------------------------------------------------------------

b0_mask_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 1 - frozen B0 mask cache",

    "dataset":
        "FSC-147",

    "subset":
        "frozen dev100",

    "num_images":
        100,

    "image_path_resolution":
        {
            "source":
                str(
                    RESOLVED_PATH_FILE
                ),

            "num_unique_path_images":
                int(
                    (
                        resolved_image_paths_df[
                            "num_copies_found"
                        ]
                        == 1
                    ).sum()
                ),

            "num_duplicate_filename_images":
                int(
                    (
                        resolved_image_paths_df[
                            "num_copies_found"
                        ]
                        > 1
                    ).sum()
                ),

            "duplicate_content_verified_identical":
                True,

            "duplicate_selection_rule":
                "lexicographically_first_of_byte_identical_copies",
        },

    "generator":
        "SAM2AutomaticMaskGenerator",

    "sam2_commit":
        observed_commit,

    "checkpoint_path":
        str(
            SAM2_CHECKPOINT
        ),

    "checkpoint_sha256":
        observed_checkpoint_sha256,

    "model_cfg":
        B0_MODEL_CFG,

    "apply_postprocessing":
        False,

    "additional_mask_filtering":
        False,

    "total_frozen_masks":
        total_frozen_masks,

    "total_cached_masks":
        total_cached_masks,

    "count_identity":
        True,

    "num_exact_count_matches":
        100,

    "num_cache_files":
        len(
            cache_files
        ),

    "cache_format":
        {
            "container":
                "npz_compressed",

            "mask_encoding":
                "np.packbits",

            "bitorder":
                "little",

            "lossless":
                True,

            "stored_mask_metadata":
                [
                    "area",
                    "predicted_iou",
                    "stability_score",
                    "bbox_xywh",
                    "crop_box_xywh",
                    "point_xy",
                ],
        },

    "total_cache_bytes":
        total_cache_bytes,

    "cache_bundle_sha256":
        cache_bundle_sha256,

    "total_runtime_sec":
        total_runtime_sec,

    "semantic_processing":
        False,

    "semantic_recovery":
        False,

    "test_set_accessed":
        False,
}


with open(
    B0_MASK_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        b0_mask_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 1.17 Final confirmation
# ------------------------------------------------------------

print("=" * 125)
print("STEP 1 COMPLETE - FROZEN B0 MASK CACHE")
print("=" * 125)

print(
    "dev100 images processed      :",
    len(
        b0_mask_cache_index_df
    ),
    "/ 100"
)

print(
    "Resolved path mapping        : PASS"
)

print(
    "Duplicate path ambiguity     : RESOLVED"
)

print(
    "Exact B0 count matches       :",
    int(
        b0_mask_cache_index_df[
            "count_match"
        ].sum()
    ),
    "/ 100"
)

print(
    "Frozen B0 masks total        :",
    total_frozen_masks
)

print(
    "Cached B0 masks total        :",
    total_cached_masks
)

print(
    "Cache files                  :",
    len(
        cache_files
    ),
    "/ 100"
)

print(
    "Total cache size             :",
    f"{total_cache_bytes / (1024**2):.2f} MiB"
)

print(
    "Total inference runtime      :",
    f"{total_runtime_sec / 60:.2f} min"
)

print(
    "Mean runtime / image         :",
    f"{runtime_summary['mean']:.2f} sec"
)

print(
    "Cache bundle SHA256          :",
    cache_bundle_sha256
)

print(
    "\nCount identity               : PASS"
)

print(
    "Lossless cache validation    : PASS"
)

print(
    "Semantic processing          : NO"
)

print(
    "Semantic recovery            : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nB0 masks by density:"
)

display(
    count_by_density_df
)


print(
    "\nFirst 10 cache records:"
)

display(
    b0_mask_cache_index_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "num_copies_found",
            "path_selection_rule",
            "frozen_b0_count",
            "reproduced_b0_count",
            "runtime_sec",
            "cache_size_bytes",
        ]
    ]
    .head(
        10
    )
)


print(
    "\nSaved cache index:"
)

print(
    B0_MASK_INDEX_FILE
)

print(
    "\nSaved cache manifest:"
)

print(
    B0_MASK_MANIFEST_FILE
)

print("=" * 125)

Removing 18 partial B0 cache files from earlier interrupted run...
STEP 1 - REPRODUCE + CACHE FROZEN B0 MASKS
Images to process          : 100
Resolved image paths       : PASS
Duplicate-path ambiguity   : RESOLVED IN STEP 1A
Generator                  : SAM2AutomaticMaskGenerator
Additional mask filtering  : NO
Semantic processing        : NO
Test set accessed          : NO
-----------------------------------------------------------------------------------------------------------------------------
[  1/100] 215.jpg      | S6_95_100    | GT= 259 | B0= 103 | match=YES |   5.30s | cache=  0.01 MiB
[  2/100] 220.jpg      | S4_60_80     | GT=  59 | B0=  20 | match=YES |   5.36s | cache=  0.00 MiB
[  3/100] 228.jpg      | S4_60_80     | GT=  44 | B0=   5 | match=YES |   5.47s | cache=  0.00 MiB
[  4/100] 241.jpg      | S3_40_60     | GT=  35 | B0=  25 | match=YES |   5.42s | cache=  0.01 MiB
[  5/100] 252.jpg      | S3_40_60     | GT=  22 | B0=  14 | match=YES |   5.15s | cache=  0.01 MiB
[

,num_images,total_b0_masks,mean_b0_masks,median_b0_masks
density_stratum,,,,
S1_0_20,20,184,9.20,8.0
S2_20_40,20,209,10.45,13.0
S3_40_60,20,266,13.30,11.0
S4_60_80,20,514,25.70,14.0
S5_80_95,10,636,63.60,45.5
S6_95_100,10,489,48.90,9.0



First 10 cache records:


,dataset_index,image_id,category,density_stratum,gt_count,num_copies_found,path_selection_rule,frozen_b0_count,reproduced_b0_count,runtime_sec,cache_size_bytes
0,9,215.jpg,grapes,S6_95_100,259,1,only_copy,103,103,5.304035,13269
1,14,220.jpg,grapes,S4_60_80,59,1,only_copy,20,20,5.363675,4624
2,20,228.jpg,grapes,S4_60_80,44,1,only_copy,5,5,5.467110,3460
3,31,241.jpg,grapes,S3_40_60,35,1,only_copy,25,25,5.417944,10940
4,40,252.jpg,grapes,S3_40_60,22,1,only_copy,14,14,5.153115,6205
5,49,556.jpg,flamingos,S2_20_40,14,1,only_copy,1,1,5.120350,3024
6,52,559.jpg,flamingos,S2_20_40,20,1,only_copy,5,5,4.976386,2863
7,59,567.jpg,flamingos,S1_0_20,10,1,only_copy,0,0,4.860790,2172
8,67,575.jpg,flamingos,S1_0_20,12,1,only_copy,1,1,5.281843,3431
9,87,596.jpg,seagulls,S2_20_40,14,1,only_copy,1,1,4.726476,2343



Saved cache index:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_mask_cache_index.csv

Saved cache manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_mask_cache_manifest.json


In [11]:
# ============================================================
# Step 2A - Decode Frozen B0 Masks + Attach GT Point Support
# ============================================================

from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2A.1 Output files
# ------------------------------------------------------------

B0_MASK_GT_PER_MASK_FILE = (
    B2_TABLES_DIR
    / "B0_mask_GT_support_per_mask.csv"
)

B0_MASK_GT_PER_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_mask_GT_support_per_image.csv"
)

B0_MASK_GT_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_mask_GT_support_manifest.json"
)


# ------------------------------------------------------------
# 2A.2 Required frozen GT-point source
#
# C/D candidate tables already carry the canonical GT point
# coordinates used in the previous diagnostics.
# We use C only as a convenient source of the GT annotations;
# this step does NOT use C semantic predictions.
# ------------------------------------------------------------

GT_REQUIRED = {
    "dataset_index",
    "image_id",
    "gt_count",
    "gt_points_xy",
}


missing_gt_cols = (
    GT_REQUIRED
    - set(
        c_df.columns
    )
)


assert (
    len(
        missing_gt_cols
    )
    == 0
), (
    "C table missing required GT columns: "
    f"{sorted(missing_gt_cols)}"
)


# ------------------------------------------------------------
# 2A.3 Parse stored XY lists
# ------------------------------------------------------------

def parse_xy_list(
    value,
):

    if value is None:

        return np.empty(
            (0, 2),
            dtype=np.float32,
        )


    if isinstance(
        value,
        float,
    ) and np.isnan(
        value
    ):

        return np.empty(
            (0, 2),
            dtype=np.float32,
        )


    if isinstance(
        value,
        str,
    ):

        value = value.strip()

        if (
            value == ""
            or value == "[]"
        ):

            return np.empty(
                (0, 2),
                dtype=np.float32,
            )

        value = ast.literal_eval(
            value
        )


    arr = np.asarray(
        value,
        dtype=np.float32,
    )


    if arr.size == 0:

        return np.empty(
            (0, 2),
            dtype=np.float32,
        )


    return arr.reshape(
        -1,
        2,
    )


# ------------------------------------------------------------
# 2A.4 Build one authoritative GT-point record per dev100 image
#
# GT annotations should be identical across C variants.
# Verify that before using them.
# ------------------------------------------------------------

gt_source_rows = []


for (
    dataset_index,
    image_id,
), group in c_df.groupby(
    [
        "dataset_index",
        "image_id",
    ],
    sort=False,
):

    assert (
        len(
            group
        )
        == len(
            C_CANDIDATES
        )
    )


    parsed_gt_lists = [
        parse_xy_list(
            value
        )
        for value in group[
            "gt_points_xy"
        ].tolist()
    ]


    reference_gt = parsed_gt_lists[
        0
    ]


    for other_gt in parsed_gt_lists[
        1:
    ]:

        assert (
            reference_gt.shape
            == other_gt.shape
        )

        assert np.allclose(
            reference_gt,
            other_gt,
        )


    gt_count = int(
        group[
            "gt_count"
        ].iloc[
            0
        ]
    )


    assert (
        len(
            reference_gt
        )
        == gt_count
    )


    gt_source_rows.append(
        {
            "dataset_index":
                int(
                    dataset_index
                ),

            "image_id":
                image_id,

            "gt_count":
                gt_count,

            "gt_points_xy":
                reference_gt,
        }
    )


gt_source_df = pd.DataFrame(
    gt_source_rows
)


assert (
    len(
        gt_source_df
    )
    == 100
)

assert (
    gt_source_df[
        "dataset_index"
    ]
    .is_unique
)

assert (
    gt_source_df[
        "image_id"
    ]
    .is_unique
)


# ------------------------------------------------------------
# 2A.5 Exact alignment to frozen B0 cache index
# ------------------------------------------------------------

gt_alignment = (
    b0_mask_cache_index_df[
        [
            "dataset_index",
            "image_id",
            "gt_count",
            "image_height",
            "image_width",
            "cache_file",
        ]
    ]
    .merge(
        gt_source_df,
        on=[
            "dataset_index",
            "image_id",
            "gt_count",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    len(
        gt_alignment
    )
    == 100
)


assert (
    gt_alignment[
        "gt_points_xy"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 2A.6 Helper: decode packed B0 masks losslessly
# ------------------------------------------------------------

def load_b0_masks_from_cache(
    cache_path,
):

    with np.load(
        cache_path,
        allow_pickle=False,
    ) as data:

        packed_masks = data[
            "packed_masks"
        ]

        image_height = int(
            data[
                "image_height"
            ]
        )

        image_width = int(
            data[
                "image_width"
            ]
        )

        num_masks = int(
            data[
                "num_masks"
            ]
        )

        areas = data[
            "areas"
        ].astype(
            np.int64
        )

        predicted_iou = data[
            "predicted_iou"
        ].astype(
            np.float32
        )

        stability_score = data[
            "stability_score"
        ].astype(
            np.float32
        )

        bbox_xywh = data[
            "bbox_xywh"
        ].astype(
            np.float32
        )

        crop_box_xywh = data[
            "crop_box_xywh"
        ].astype(
            np.float32
        )

        point_xy = data[
            "point_xy"
        ].astype(
            np.float32
        )


    num_pixels = (
        image_height
        *
        image_width
    )


    if num_masks > 0:

        unpacked = np.unpackbits(
            packed_masks,
            axis=1,
            bitorder="little",
        )


        unpacked = unpacked[
            :,
            :num_pixels
        ]


        masks = (
            unpacked
            .reshape(
                num_masks,
                image_height,
                image_width,
            )
            .astype(
                bool
            )
        )


    else:

        masks = np.empty(
            (
                0,
                image_height,
                image_width,
            ),
            dtype=bool,
        )


    return {
        "masks":
            masks,

        "image_height":
            image_height,

        "image_width":
            image_width,

        "num_masks":
            num_masks,

        "areas":
            areas,

        "predicted_iou":
            predicted_iou,

        "stability_score":
            stability_score,

        "bbox_xywh":
            bbox_xywh,

        "crop_box_xywh":
            crop_box_xywh,

        "point_xy":
            point_xy,
    }


# ------------------------------------------------------------
# 2A.7 Analyse GT support mask-by-mask
# ------------------------------------------------------------

mask_records = []
image_records = []


for row in gt_alignment.itertuples(
    index=False
):

    dataset_index = int(
        row.dataset_index
    )

    image_id = row.image_id

    gt_xy = np.asarray(
        row.gt_points_xy,
        dtype=np.float32,
    )


    cache = load_b0_masks_from_cache(
        row.cache_file
    )


    masks = cache[
        "masks"
    ]

    image_height = int(
        cache[
            "image_height"
        ]
    )

    image_width = int(
        cache[
            "image_width"
        ]
    )

    num_masks = int(
        cache[
            "num_masks"
        ]
    )


    # --------------------------------------------------------
    # Cache-index consistency
    # --------------------------------------------------------

    expected_b0_count = int(
        b0_mask_cache_index_df.loc[
            b0_mask_cache_index_df[
                "dataset_index"
            ]
            == dataset_index,
            "reproduced_b0_count",
        ].iloc[
            0
        ]
    )


    assert (
        num_masks
        == expected_b0_count
    )


    assert (
        image_height
        == int(
            row.image_height
        )
    )

    assert (
        image_width
        == int(
            row.image_width
        )
    )


    # --------------------------------------------------------
    # Canonical discrete GT-point membership
    #
    # Continuous FSC-147 point coordinates are preserved in
    # the source data.
    #
    # For pixel-mask membership only:
    #   np.rint() + clip
    # --------------------------------------------------------

    gt_x_px = np.clip(
        np.rint(
            gt_xy[
                :,
                0
            ]
        ).astype(
            int
        ),
        0,
        image_width - 1,
    )


    gt_y_px = np.clip(
        np.rint(
            gt_xy[
                :,
                1
            ]
        ).astype(
            int
        ),
        0,
        image_height - 1,
    )


    assert (
        len(
            gt_x_px
        )
        == int(
            row.gt_count
        )
    )


    # --------------------------------------------------------
    # Per-image counters
    # --------------------------------------------------------

    zero_gt_masks = 0
    one_gt_masks = 0
    merged_gt_masks = 0

    total_gt_point_memberships = 0


    # --------------------------------------------------------
    # Per-mask GT support
    # --------------------------------------------------------

    for mask_idx in range(
        num_masks
    ):

        mask = masks[
            mask_idx
        ]


        gt_inside_flags = mask[
            gt_y_px,
            gt_x_px,
        ]


        gt_indices_inside = np.flatnonzero(
            gt_inside_flags
        )


        num_gt_inside = int(
            len(
                gt_indices_inside
            )
        )


        total_gt_point_memberships += (
            num_gt_inside
        )


        if num_gt_inside == 0:

            gt_support_class = "zero_gt"
            zero_gt_masks += 1

        elif num_gt_inside == 1:

            gt_support_class = "one_gt"
            one_gt_masks += 1

        else:

            gt_support_class = "merged_gt"
            merged_gt_masks += 1


        bbox = cache[
            "bbox_xywh"
        ][
            mask_idx
        ]

        crop_box = cache[
            "crop_box_xywh"
        ][
            mask_idx
        ]

        prompt_point = cache[
            "point_xy"
        ][
            mask_idx
        ]


        mask_records.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "gt_count":
                    int(
                        row.gt_count
                    ),

                "mask_index":
                    int(
                        mask_idx
                    ),

                "num_gt_points_inside":
                    num_gt_inside,

                "gt_support_class":
                    gt_support_class,

                # List is diagnostic only.
                "gt_indices_inside":
                    ",".join(
                        str(
                            int(
                                idx
                            )
                        )
                        for idx in gt_indices_inside
                    ),

                # Native SAM2 mask metadata
                "mask_area":
                    int(
                        cache[
                            "areas"
                        ][
                            mask_idx
                        ]
                    ),

                "predicted_iou":
                    float(
                        cache[
                            "predicted_iou"
                        ][
                            mask_idx
                        ]
                    ),

                "stability_score":
                    float(
                        cache[
                            "stability_score"
                        ][
                            mask_idx
                        ]
                    ),

                "bbox_x":
                    float(
                        bbox[
                            0
                        ]
                    ),

                "bbox_y":
                    float(
                        bbox[
                            1
                        ]
                    ),

                "bbox_w":
                    float(
                        bbox[
                            2
                        ]
                    ),

                "bbox_h":
                    float(
                        bbox[
                            3
                        ]
                    ),

                "crop_x":
                    float(
                        crop_box[
                            0
                        ]
                    ),

                "crop_y":
                    float(
                        crop_box[
                            1
                        ]
                    ),

                "crop_w":
                    float(
                        crop_box[
                            2
                        ]
                    ),

                "crop_h":
                    float(
                        crop_box[
                            3
                        ]
                    ),

                "prompt_x":
                    float(
                        prompt_point[
                            0
                        ]
                    ),

                "prompt_y":
                    float(
                        prompt_point[
                            1
                        ]
                    ),

                "image_height":
                    image_height,

                "image_width":
                    image_width,
            }
        )


    assert (
        zero_gt_masks
        +
        one_gt_masks
        +
        merged_gt_masks
        ==
        num_masks
    )


    # --------------------------------------------------------
    # GT-side coverage by B0 masks
    #
    # A GT point can belong to multiple overlapping B0 masks.
    # --------------------------------------------------------

    if num_masks > 0:

        gt_masks_per_gt = np.zeros(
            len(
                gt_xy
            ),
            dtype=np.int32,
        )


        for mask_idx in range(
            num_masks
        ):

            gt_masks_per_gt += masks[
                mask_idx,
                gt_y_px,
                gt_x_px,
            ].astype(
                np.int32
            )


    else:

        gt_masks_per_gt = np.zeros(
            len(
                gt_xy
            ),
            dtype=np.int32,
        )


    gt_uncovered = int(
        np.sum(
            gt_masks_per_gt
            == 0
        )
    )

    gt_covered_once = int(
        np.sum(
            gt_masks_per_gt
            == 1
        )
    )

    gt_duplicate_covered = int(
        np.sum(
            gt_masks_per_gt
            >= 2
        )
    )


    image_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "gt_count":
                int(
                    row.gt_count
                ),

            "b0_mask_count":
                num_masks,

            "zero_gt_masks":
                zero_gt_masks,

            "one_gt_masks":
                one_gt_masks,

            "merged_gt_masks":
                merged_gt_masks,

            "supported_masks":
                (
                    one_gt_masks
                    +
                    merged_gt_masks
                ),

            "supported_mask_rate":
                (
                    (
                        one_gt_masks
                        +
                        merged_gt_masks
                    )
                    /
                    num_masks
                    if num_masks > 0
                    else 0.0
                ),

            "total_gt_point_memberships":
                total_gt_point_memberships,

            "gt_uncovered":
                gt_uncovered,

            "gt_covered_once":
                gt_covered_once,

            "gt_duplicate_covered":
                gt_duplicate_covered,

            "gt_coverage_rate":
                (
                    (
                        int(
                            row.gt_count
                        )
                        -
                        gt_uncovered
                    )
                    /
                    int(
                        row.gt_count
                    )
                ),

            "gt_duplicate_coverage_rate":
                (
                    gt_duplicate_covered
                    /
                    int(
                        row.gt_count
                    )
                ),
        }
    )


# ------------------------------------------------------------
# 2A.8 Build tables
# ------------------------------------------------------------

b0_mask_gt_df = pd.DataFrame(
    mask_records
)

b0_mask_gt_image_df = pd.DataFrame(
    image_records
)


# ------------------------------------------------------------
# 2A.9 Strong integrity checks
# ------------------------------------------------------------

assert (
    len(
        b0_mask_gt_df
    )
    == 2298
), (
    f"Expected 2298 B0 mask rows, "
    f"found {len(b0_mask_gt_df)}"
)


assert (
    len(
        b0_mask_gt_image_df
    )
    == 100
)


assert (
    int(
        b0_mask_gt_image_df[
            "b0_mask_count"
        ].sum()
    )
    == 2298
)


assert (
    (
        b0_mask_gt_image_df[
            "zero_gt_masks"
        ]
        +
        b0_mask_gt_image_df[
            "one_gt_masks"
        ]
        +
        b0_mask_gt_image_df[
            "merged_gt_masks"
        ]
    )
    ==
    b0_mask_gt_image_df[
        "b0_mask_count"
    ]
).all()


# ------------------------------------------------------------
# 2A.10 Reattach density/category metadata for summaries
# ------------------------------------------------------------

b0_mask_gt_df = (
    b0_mask_gt_df
    .merge(
        b2_dev100_image_df[
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
            ]
        ],
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


b0_mask_gt_image_df = (
    b0_mask_gt_image_df
    .merge(
        b2_dev100_image_df[
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
            ]
        ],
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 2A.11 Overall diagnostic summary
# ------------------------------------------------------------

overall_zero = int(
    (
        b0_mask_gt_df[
            "gt_support_class"
        ]
        == "zero_gt"
    ).sum()
)

overall_one = int(
    (
        b0_mask_gt_df[
            "gt_support_class"
        ]
        == "one_gt"
    ).sum()
)

overall_merged = int(
    (
        b0_mask_gt_df[
            "gt_support_class"
        ]
        == "merged_gt"
    ).sum()
)


overall_supported = (
    overall_one
    +
    overall_merged
)


total_gt = int(
    b0_mask_gt_image_df[
        "gt_count"
    ].sum()
)


total_gt_uncovered = int(
    b0_mask_gt_image_df[
        "gt_uncovered"
    ].sum()
)


total_gt_duplicate_covered = int(
    b0_mask_gt_image_df[
        "gt_duplicate_covered"
    ].sum()
)


overall_gt_coverage_rate = (
    (
        total_gt
        -
        total_gt_uncovered
    )
    /
    total_gt
)


overall_gt_duplicate_rate = (
    total_gt_duplicate_covered
    /
    total_gt
)


# ------------------------------------------------------------
# 2A.12 Density summary
# ------------------------------------------------------------

density_summary_records = []


for stratum in DENSITY_ORDER:

    mask_group = (
        b0_mask_gt_df[
            b0_mask_gt_df[
                "density_stratum"
            ]
            == stratum
        ]
    )

    image_group = (
        b0_mask_gt_image_df[
            b0_mask_gt_image_df[
                "density_stratum"
            ]
            == stratum
        ]
    )


    num_masks = int(
        len(
            mask_group
        )
    )

    zero_gt = int(
        (
            mask_group[
                "gt_support_class"
            ]
            == "zero_gt"
        ).sum()
    )

    one_gt = int(
        (
            mask_group[
                "gt_support_class"
            ]
            == "one_gt"
        ).sum()
    )

    merged_gt = int(
        (
            mask_group[
                "gt_support_class"
            ]
            == "merged_gt"
        ).sum()
    )


    stratum_gt = int(
        image_group[
            "gt_count"
        ].sum()
    )

    stratum_uncovered = int(
        image_group[
            "gt_uncovered"
        ].sum()
    )

    stratum_duplicate = int(
        image_group[
            "gt_duplicate_covered"
        ].sum()
    )


    density_summary_records.append(
        {
            "density_stratum":
                stratum,

            "num_images":
                int(
                    len(
                        image_group
                    )
                ),

            "total_gt":
                stratum_gt,

            "total_b0_masks":
                num_masks,

            "zero_gt_masks":
                zero_gt,

            "one_gt_masks":
                one_gt,

            "merged_gt_masks":
                merged_gt,

            "supported_mask_rate":
                (
                    (
                        one_gt
                        +
                        merged_gt
                    )
                    /
                    num_masks
                    if num_masks > 0
                    else 0.0
                ),

            "gt_coverage_rate":
                (
                    (
                        stratum_gt
                        -
                        stratum_uncovered
                    )
                    /
                    stratum_gt
                ),

            "gt_duplicate_coverage_rate":
                (
                    stratum_duplicate
                    /
                    stratum_gt
                ),
        }
    )


b0_mask_gt_density_df = pd.DataFrame(
    density_summary_records
)


# ------------------------------------------------------------
# 2A.13 Save outputs
# ------------------------------------------------------------

b0_mask_gt_df.to_csv(
    B0_MASK_GT_PER_MASK_FILE,
    index=False,
)

b0_mask_gt_image_df.to_csv(
    B0_MASK_GT_PER_IMAGE_FILE,
    index=False,
)


b0_mask_gt_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 2A - B0 mask GT support",

    "num_images":
        100,

    "num_b0_masks":
        int(
            len(
                b0_mask_gt_df
            )
        ),

    "gt_membership_rule":
        "np.rint + clip -> mask pixel lookup",

    "mask_support_classes":
        {
            "zero_gt":
                "mask contains 0 FSC-147 GT points",

            "one_gt":
                "mask contains exactly 1 FSC-147 GT point",

            "merged_gt":
                "mask contains >=2 FSC-147 GT points",
        },

    "overall":
        {
            "zero_gt_masks":
                overall_zero,

            "one_gt_masks":
                overall_one,

            "merged_gt_masks":
                overall_merged,

            "supported_masks":
                overall_supported,

            "supported_mask_rate":
                (
                    overall_supported
                    /
                    len(
                        b0_mask_gt_df
                    )
                ),

            "total_gt":
                total_gt,

            "gt_coverage_rate":
                overall_gt_coverage_rate,

            "gt_duplicate_coverage_rate":
                overall_gt_duplicate_rate,
        },

    "semantic_evidence_used":
        False,

    "filtering_performed":
        False,

    "test_set_accessed":
        False,
}


with open(
    B0_MASK_GT_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        b0_mask_gt_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2A.14 Final confirmation
# ------------------------------------------------------------

print("=" * 125)
print("STEP 2A - B0 MASK <-> GT POINT SUPPORT")
print("=" * 125)

print(
    "dev100 images               :",
    len(
        b0_mask_gt_image_df
    ),
    "/ 100"
)

print(
    "B0 masks                    :",
    len(
        b0_mask_gt_df
    ),
    "/ 2298"
)

print(
    "\nMask-side GT support:"
)

print(
    "  zero-GT masks             :",
    overall_zero
)

print(
    "  one-GT masks              :",
    overall_one
)

print(
    "  merged-GT masks           :",
    overall_merged
)

print(
    "  supported mask rate       :",
    f"{overall_supported / len(b0_mask_gt_df):.4f}"
)


print(
    "\nGT-side B0 coverage:"
)

print(
    "  total GT points           :",
    total_gt
)

print(
    "  GT coverage rate          :",
    f"{overall_gt_coverage_rate:.4f}"
)

print(
    "  duplicate coverage rate   :",
    f"{overall_gt_duplicate_rate:.4f}"
)


print(
    "\nBy density:"
)

display(
    b0_mask_gt_density_df.round(
        4
    )
)


print(
    "\nFirst 15 B0 mask records:"
)

display(
    b0_mask_gt_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "mask_index",
            "num_gt_points_inside",
            "gt_support_class",
            "mask_area",
            "predicted_iou",
            "stability_score",
        ]
    ]
    .head(
        15
    )
)


print(
    "\nSemantic evidence used      : NO"
)

print(
    "Mask filtering performed    : NO"
)

print(
    "Test set accessed           : NO"
)

print(
    "\nSaved per-mask table:"
)

print(
    B0_MASK_GT_PER_MASK_FILE
)

print(
    "\nSaved per-image table:"
)

print(
    B0_MASK_GT_PER_IMAGE_FILE
)

print(
    "\nSaved manifest:"
)

print(
    B0_MASK_GT_MANIFEST_FILE
)

print("=" * 125)

STEP 2A - B0 MASK <-> GT POINT SUPPORT
dev100 images               : 100 / 100
B0 masks                    : 2298 / 2298

Mask-side GT support:
  zero-GT masks             : 309
  one-GT masks              : 1960
  merged-GT masks           : 29
  supported mask rate       : 0.8655

GT-side B0 coverage:
  total GT points           : 10045
  GT coverage rate          : 0.4384
  duplicate coverage rate   : 0.0261

By density:


,density_stratum,num_images,total_gt,total_b0_masks,zero_gt_masks,one_gt_masks,merged_gt_masks,supported_mask_rate,gt_coverage_rate,gt_duplicate_coverage_rate
0,S1_0_20,20,195,184,62,121,1,0.6630,0.5487,0.1077
1,S2_20_40,20,320,209,61,148,0,0.7081,0.4188,0.0406
2,S3_40_60,20,537,266,61,201,4,0.7707,0.3799,0.1061
3,S4_60_80,20,1096,514,63,443,8,0.8774,0.5885,0.1040
4,S5_80_95,10,1211,636,26,604,6,0.9591,0.4756,0.0322
5,S6_95_100,10,6686,489,36,443,10,0.9264,0.4095,0.0027



First 15 B0 mask records:


,dataset_index,image_id,category,density_stratum,mask_index,num_gt_points_inside,gt_support_class,mask_area,predicted_iou,stability_score
0,9,215.jpg,grapes,S6_95_100,0,0,zero_gt,2279,0.972012,0.971898
1,9,215.jpg,grapes,S6_95_100,1,1,one_gt,695,0.960591,0.961918
2,9,215.jpg,grapes,S6_95_100,2,0,zero_gt,3581,0.958004,0.960602
3,9,215.jpg,grapes,S6_95_100,3,1,one_gt,690,0.950027,0.961429
4,9,215.jpg,grapes,S6_95_100,4,1,one_gt,526,0.948917,0.968224
5,9,215.jpg,grapes,S6_95_100,5,1,one_gt,556,0.948270,0.971530
6,9,215.jpg,grapes,S6_95_100,6,1,one_gt,432,0.947298,0.970320
7,9,215.jpg,grapes,S6_95_100,7,1,one_gt,691,0.946951,0.954802
8,9,215.jpg,grapes,S6_95_100,8,1,one_gt,814,0.945689,0.960479
9,9,215.jpg,grapes,S6_95_100,9,1,one_gt,589,0.944482,0.969900



Semantic evidence used      : NO
Mask filtering performed    : NO
Test set accessed           : NO

Saved per-mask table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_mask_GT_support_per_mask.csv

Saved per-image table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_mask_GT_support_per_image.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_mask_GT_support_manifest.json


In [13]:
# ============================================================
# Step 2B - Overlay Frozen C/D Semantic Evidence onto B0 Masks
# ============================================================

from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2B.1 Output files
# ------------------------------------------------------------

B0_SEMANTIC_LONG_FILE = (
    B2_TABLES_DIR
    / "B0_mask_semantic_support_long.csv"
)

B0_SEMANTIC_WIDE_FILE = (
    B2_TABLES_DIR
    / "B0_mask_semantic_support_wide.csv"
)

B0_SEMANTIC_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_mask_semantic_support_per_image.csv"
)

B0_SEMANTIC_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_mask_semantic_support_manifest.json"
)


# ------------------------------------------------------------
# 2B.2 Frozen semantic candidate definitions
# ------------------------------------------------------------

SEMANTIC_CANDIDATES = (
    [
        ("C", candidate_id)
        for candidate_id in C_CANDIDATES
    ]
    +
    [
        ("D", candidate_id)
        for candidate_id in D_CANDIDATES
    ]
)


assert (
    len(
        SEMANTIC_CANDIDATES
    )
    == 8
)


# ------------------------------------------------------------
# 2B.3 Required semantic-point columns
# ------------------------------------------------------------

SEMANTIC_REQUIRED = {
    "candidate_id",
    "dataset_index",
    "image_id",
    "pred_count",
    "pred_points_xy",
}


for df, name in [
    (c_df, "C"),
    (d_df, "D"),
]:

    missing = (
        SEMANTIC_REQUIRED
        - set(
            df.columns
        )
    )

    assert (
        len(
            missing
        )
        == 0
    ), (
        f"{name} missing required columns: "
        f"{sorted(missing)}"
    )


# ------------------------------------------------------------
# 2B.4 Parser
#
# Reuse Step 2A parser if already available.
# ------------------------------------------------------------

if "parse_xy_list" not in globals():

    def parse_xy_list(
        value,
    ):

        if value is None:

            return np.empty(
                (0, 2),
                dtype=np.float32,
            )


        if isinstance(
            value,
            float,
        ) and np.isnan(
            value
        ):

            return np.empty(
                (0, 2),
                dtype=np.float32,
            )


        if isinstance(
            value,
            str,
        ):

            value = value.strip()

            if (
                value == ""
                or value == "[]"
            ):

                return np.empty(
                    (0, 2),
                    dtype=np.float32,
                )

            value = ast.literal_eval(
                value
            )


        arr = np.asarray(
            value,
            dtype=np.float32,
        )


        if arr.size == 0:

            return np.empty(
                (0, 2),
                dtype=np.float32,
            )


        return arr.reshape(
            -1,
            2,
        )


# ------------------------------------------------------------
# 2B.5 Build one semantic-point record per
#      image × candidate
# ------------------------------------------------------------

semantic_rows = []


for branch, source_df in [
    ("C", c_df),
    ("D", d_df),
]:

    for row in source_df.itertuples(
        index=False
    ):

        points_xy = parse_xy_list(
            row.pred_points_xy
        )


        assert (
            len(
                points_xy
            )
            == int(
                row.pred_count
            )
        ), (
            f"{branch}/{row.candidate_id}/{row.image_id}: "
            "pred_count does not match pred_points_xy length."
        )


        semantic_rows.append(
            {
                "branch":
                    branch,

                "candidate_id":
                    row.candidate_id,

                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    row.image_id,

                "candidate_pred_count":
                    int(
                        row.pred_count
                    ),

                "pred_points_xy":
                    points_xy,
            }
        )


semantic_source_df = pd.DataFrame(
    semantic_rows
)


# 3 C + 5 D = 8 candidates × 100 images

assert (
    len(
        semantic_source_df
    )
    == 800
)


assert (
    semantic_source_df.groupby(
        [
            "branch",
            "candidate_id",
        ]
    )[
        "dataset_index"
    ]
    .nunique()
    .eq(
        100
    )
    .all()
)


# ------------------------------------------------------------
# 2B.6 Fast lookup by image/candidate
# ------------------------------------------------------------

semantic_lookup = {}


for row in semantic_source_df.itertuples(
    index=False
):

    key = (
        int(
            row.dataset_index
        ),
        row.branch,
        row.candidate_id,
    )

    assert (
        key
        not in semantic_lookup
    )

    semantic_lookup[
        key
    ] = {
        "image_id":
            row.image_id,

        "candidate_pred_count":
            int(
                row.candidate_pred_count
            ),

        "pred_points_xy":
            np.asarray(
                row.pred_points_xy,
                dtype=np.float32,
            ),
    }


# ------------------------------------------------------------
# 2B.7 Cache-file lookup
# ------------------------------------------------------------

cache_file_lookup = (
    b0_mask_cache_index_df
    .set_index(
        "dataset_index"
    )[
        "cache_file"
    ]
    .to_dict()
)


# ------------------------------------------------------------
# 2B.8 Helper to decode masks if Step 2A function unavailable
# ------------------------------------------------------------

if "load_b0_masks_from_cache" not in globals():

    def load_b0_masks_from_cache(
        cache_path,
    ):

        with np.load(
            cache_path,
            allow_pickle=False,
        ) as data:

            packed_masks = data[
                "packed_masks"
            ]

            image_height = int(
                data[
                    "image_height"
                ]
            )

            image_width = int(
                data[
                    "image_width"
                ]
            )

            num_masks = int(
                data[
                    "num_masks"
                ]
            )

            areas = data[
                "areas"
            ].astype(
                np.int64
            )


        num_pixels = (
            image_height
            *
            image_width
        )


        if num_masks > 0:

            masks = (
                np.unpackbits(
                    packed_masks,
                    axis=1,
                    bitorder="little",
                )[
                    :,
                    :num_pixels
                ]
                .reshape(
                    num_masks,
                    image_height,
                    image_width,
                )
                .astype(
                    bool
                )
            )

        else:

            masks = np.empty(
                (
                    0,
                    image_height,
                    image_width,
                ),
                dtype=bool,
            )


        return {
            "masks":
                masks,

            "image_height":
                image_height,

            "image_width":
                image_width,

            "num_masks":
                num_masks,

            "areas":
                areas,
        }


# ------------------------------------------------------------
# 2B.9 Overlay every semantic candidate onto every B0 mask
# ------------------------------------------------------------

semantic_mask_records = []


for image_row in b2_dev100_image_df.itertuples(
    index=False
):

    dataset_index = int(
        image_row.dataset_index
    )

    image_id = image_row.image_id


    # --------------------------------------------------------
    # Load B0 masks
    # --------------------------------------------------------

    cache = load_b0_masks_from_cache(
        cache_file_lookup[
            dataset_index
        ]
    )


    masks = cache[
        "masks"
    ]

    num_masks = int(
        cache[
            "num_masks"
        ]
    )

    image_height = int(
        cache[
            "image_height"
        ]
    )

    image_width = int(
        cache[
            "image_width"
        ]
    )


    # --------------------------------------------------------
    # Compute B0 mask centroids once per image
    # --------------------------------------------------------

    mask_centroids = []


    for mask_idx in range(
        num_masks
    ):

        mask = masks[
            mask_idx
        ]


        yy, xx = np.nonzero(
            mask
        )


        assert (
            len(
                xx
            )
            > 0
        ), (
            f"Empty B0 mask found: "
            f"{image_id}, mask {mask_idx}"
        )


        centroid_x = float(
            xx.mean()
        )

        centroid_y = float(
            yy.mean()
        )


        mask_centroids.append(
            (
                centroid_x,
                centroid_y,
            )
        )


    # --------------------------------------------------------
    # Each frozen semantic candidate
    # --------------------------------------------------------

    for branch, candidate_id in SEMANTIC_CANDIDATES:

        semantic = semantic_lookup[
            (
                dataset_index,
                branch,
                candidate_id,
            )
        ]


        assert (
            semantic[
                "image_id"
            ]
            == image_id
        )


        points_xy = semantic[
            "pred_points_xy"
        ]


        candidate_pred_count = int(
            semantic[
                "candidate_pred_count"
            ]
        )


        assert (
            len(
                points_xy
            )
            == candidate_pred_count
        )


        # ----------------------------------------------------
        # Discrete point membership:
        # same convention as Step 2A
        # ----------------------------------------------------

        if candidate_pred_count > 0:

            point_x_px = np.clip(
                np.rint(
                    points_xy[
                        :,
                        0
                    ]
                ).astype(
                    int
                ),
                0,
                image_width - 1,
            )


            point_y_px = np.clip(
                np.rint(
                    points_xy[
                        :,
                        1
                    ]
                ).astype(
                    int
                ),
                0,
                image_height - 1,
            )


        else:

            point_x_px = np.empty(
                (0,),
                dtype=int,
            )

            point_y_px = np.empty(
                (0,),
                dtype=int,
            )


        # ----------------------------------------------------
        # Evaluate candidate support for each B0 mask
        # ----------------------------------------------------

        for mask_idx in range(
            num_masks
        ):

            mask = masks[
                mask_idx
            ]


            # ----------------------------------------------
            # Number of candidate points inside mask
            # ----------------------------------------------

            if candidate_pred_count > 0:

                inside_flags = mask[
                    point_y_px,
                    point_x_px,
                ]


                semantic_indices_inside = np.flatnonzero(
                    inside_flags
                )


                num_semantic_inside = int(
                    len(
                        semantic_indices_inside
                    )
                )


            else:

                semantic_indices_inside = np.empty(
                    (0,),
                    dtype=int,
                )

                num_semantic_inside = 0


            semantic_supported = (
                num_semantic_inside
                > 0
            )


            # ----------------------------------------------
            # Semantic points per mask pixel
            #
            # Scale to points per 10k pixels for readability.
            # ----------------------------------------------

            mask_area = int(
                cache[
                    "areas"
                ][
                    mask_idx
                ]
            )


            semantic_density_10k = (
                num_semantic_inside
                /
                mask_area
                *
                10000.0
                if mask_area > 0
                else 0.0
            )


            # ----------------------------------------------
            # Fraction of all candidate points in image
            # captured by this B0 mask
            # ----------------------------------------------

            fraction_candidate_points_inside = (
                num_semantic_inside
                /
                candidate_pred_count
                if candidate_pred_count > 0
                else 0.0
            )


            # ----------------------------------------------
            # Distance from B0-mask centroid to nearest
            # candidate semantic point
            #
            # Diagnostic only; no thresholding.
            # ----------------------------------------------

            centroid_x, centroid_y = (
                mask_centroids[
                    mask_idx
                ]
            )


            if candidate_pred_count > 0:

                dx = (
                    points_xy[
                        :,
                        0
                    ]
                    -
                    centroid_x
                )

                dy = (
                    points_xy[
                        :,
                        1
                    ]
                    -
                    centroid_y
                )


                nearest_semantic_distance_px = float(
                    np.sqrt(
                        dx * dx
                        +
                        dy * dy
                    ).min()
                )


            else:

                nearest_semantic_distance_px = np.nan


            # ----------------------------------------------
            # Normalise nearest distance by image diagonal
            # ----------------------------------------------

            image_diagonal = float(
                np.sqrt(
                    image_width**2
                    +
                    image_height**2
                )
            )


            nearest_semantic_distance_norm = (
                nearest_semantic_distance_px
                /
                image_diagonal
                if (
                    candidate_pred_count > 0
                    and image_diagonal > 0
                )
                else np.nan
            )


            semantic_mask_records.append(
                {
                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_id,

                    "category":
                        image_row.category,

                    "density_stratum":
                        image_row.density_stratum,

                    "gt_count":
                        int(
                            image_row.gt_count
                        ),

                    "mask_index":
                        int(
                            mask_idx
                        ),

                    "branch":
                        branch,

                    "candidate_id":
                        candidate_id,

                    "candidate_pred_count":
                        candidate_pred_count,

                    "num_semantic_points_inside":
                        num_semantic_inside,

                    "semantic_supported":
                        semantic_supported,

                    "semantic_density_per_10k_mask_px":
                        semantic_density_10k,

                    "fraction_candidate_points_inside":
                        fraction_candidate_points_inside,

                    "nearest_semantic_distance_px":
                        nearest_semantic_distance_px,

                    "nearest_semantic_distance_norm":
                        nearest_semantic_distance_norm,

                    "mask_centroid_x":
                        centroid_x,

                    "mask_centroid_y":
                        centroid_y,

                    "mask_area":
                        mask_area,
                }
            )


# ------------------------------------------------------------
# 2B.10 Long-format table
# ------------------------------------------------------------

b0_semantic_long_df = pd.DataFrame(
    semantic_mask_records
)


expected_long_rows = (
    len(
        b0_mask_gt_df
    )
    *
    len(
        SEMANTIC_CANDIDATES
    )
)


assert (
    len(
        b0_semantic_long_df
    )
    == expected_long_rows
), (
    f"Expected {expected_long_rows} rows, "
    f"found {len(b0_semantic_long_df)}"
)


# Expected:
# 2298 masks × 8 semantic candidates = 18,384

assert (
    expected_long_rows
    == 18384
)


# ------------------------------------------------------------
# 2B.11 Attach GT-support labels from Step 2A
#
# GT is used only for later diagnosis.
# It does NOT affect semantic support computation.
# ------------------------------------------------------------

b0_semantic_long_df = (
    b0_semantic_long_df
    .merge(
        b0_mask_gt_df[
            [
                "dataset_index",
                "image_id",
                "mask_index",
                "num_gt_points_inside",
                "gt_support_class",
                "predicted_iou",
                "stability_score",
            ]
        ],
        on=[
            "dataset_index",
            "image_id",
            "mask_index",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    b0_semantic_long_df[
        "gt_support_class"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 2B.12 Integrity: each candidate must have exactly
# 2298 B0-mask rows
# ------------------------------------------------------------

candidate_mask_counts = (
    b0_semantic_long_df
    .groupby(
        [
            "branch",
            "candidate_id",
        ]
    )
    .size()
)


assert (
    candidate_mask_counts
    == 2298
).all()


# ------------------------------------------------------------
# 2B.13 Build wide binary/count support table
#
# One row per B0 mask.
# ------------------------------------------------------------

support_count_wide = (
    b0_semantic_long_df
    .pivot(
        index=[
            "dataset_index",
            "image_id",
            "mask_index",
        ],
        columns="candidate_id",
        values="num_semantic_points_inside",
    )
)


support_count_wide.columns = [
    f"{candidate_id}__num_inside"
    for candidate_id in support_count_wide.columns
]


support_binary_wide = (
    b0_semantic_long_df
    .pivot(
        index=[
            "dataset_index",
            "image_id",
            "mask_index",
        ],
        columns="candidate_id",
        values="semantic_supported",
    )
)


support_binary_wide.columns = [
    f"{candidate_id}__supported"
    for candidate_id in support_binary_wide.columns
]


support_density_wide = (
    b0_semantic_long_df
    .pivot(
        index=[
            "dataset_index",
            "image_id",
            "mask_index",
        ],
        columns="candidate_id",
        values="semantic_density_per_10k_mask_px",
    )
)


support_density_wide.columns = [
    f"{candidate_id}__density10k"
    for candidate_id in support_density_wide.columns
]


# ------------------------------------------------------------
# 2B.14 Base B0 mask table
# ------------------------------------------------------------

base_mask_cols = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "mask_index",
    "num_gt_points_inside",
    "gt_support_class",
    "mask_area",
    "predicted_iou",
    "stability_score",
]


b0_semantic_wide_df = (
    b0_mask_gt_df[
        base_mask_cols
    ]
    .set_index(
        [
            "dataset_index",
            "image_id",
            "mask_index",
        ]
    )
    .join(
        support_count_wide
    )
    .join(
        support_binary_wide
    )
    .join(
        support_density_wide
    )
    .reset_index()
)


assert (
    len(
        b0_semantic_wide_df
    )
    == 2298
)


# ------------------------------------------------------------
# 2B.15 Aggregate C/D support indicators
#
# These are inference-available features.
# ------------------------------------------------------------

C_SUPPORT_COLS = [
    f"{candidate_id}__supported"
    for candidate_id in C_CANDIDATES
]

D_SUPPORT_COLS = [
    f"{candidate_id}__supported"
    for candidate_id in D_CANDIDATES
]


b0_semantic_wide_df[
    "num_C_candidates_supporting"
] = (
    b0_semantic_wide_df[
        C_SUPPORT_COLS
    ]
    .sum(
        axis=1
    )
    .astype(
        int
    )
)


b0_semantic_wide_df[
    "num_D_candidates_supporting"
] = (
    b0_semantic_wide_df[
        D_SUPPORT_COLS
    ]
    .sum(
        axis=1
    )
    .astype(
        int
    )
)


b0_semantic_wide_df[
    "any_C_support"
] = (
    b0_semantic_wide_df[
        "num_C_candidates_supporting"
    ]
    > 0
)


b0_semantic_wide_df[
    "any_D_support"
] = (
    b0_semantic_wide_df[
        "num_D_candidates_supporting"
    ]
    > 0
)


b0_semantic_wide_df[
    "any_semantic_support"
] = (
    b0_semantic_wide_df[
        "any_C_support"
    ]
    |
    b0_semantic_wide_df[
        "any_D_support"
    ]
)


b0_semantic_wide_df[
    "C_and_D_support"
] = (
    b0_semantic_wide_df[
        "any_C_support"
    ]
    &
    b0_semantic_wide_df[
        "any_D_support"
    ]
)


b0_semantic_wide_df[
    "total_semantic_candidates_supporting"
] = (
    b0_semantic_wide_df[
        "num_C_candidates_supporting"
    ]
    +
    b0_semantic_wide_df[
        "num_D_candidates_supporting"
    ]
)


# ------------------------------------------------------------
# 2B.16 Candidate-level descriptive support summary
#
# Still no GT comparison beyond descriptive counts.
# Formal discrimination analysis comes in Step 2C.
# ------------------------------------------------------------

candidate_summary_records = []


for (
    branch,
    candidate_id,
), group in b0_semantic_long_df.groupby(
    [
        "branch",
        "candidate_id",
    ],
    sort=False,
):

    num_masks = int(
        len(
            group
        )
    )

    num_supported = int(
        group[
            "semantic_supported"
        ].sum()
    )

    total_inside_points = int(
        group[
            "num_semantic_points_inside"
        ].sum()
    )


    candidate_summary_records.append(
        {
            "branch":
                branch,

            "candidate_id":
                candidate_id,

            "num_b0_masks":
                num_masks,

            "semantically_supported_masks":
                num_supported,

            "semantic_support_rate":
                (
                    num_supported
                    /
                    num_masks
                ),

            "total_semantic_point_memberships":
                total_inside_points,

            "mean_points_inside_per_mask":
                float(
                    group[
                        "num_semantic_points_inside"
                    ].mean()
                ),

            "median_nearest_semantic_distance_norm":
                float(
                    group[
                        "nearest_semantic_distance_norm"
                    ].median(
                        skipna=True
                    )
                ),
        }
    )


candidate_support_summary_df = pd.DataFrame(
    candidate_summary_records
)


# ------------------------------------------------------------
# 2B.17 Per-image semantic-support summary
#
# IMPORTANT:
# Start from all 100 frozen dev100 images.
#
# Images with zero B0 masks do not appear in the mask-level
# table, so a direct groupby(b0_semantic_wide_df) would
# incorrectly drop them.
# ------------------------------------------------------------

mask_level_image_summary_df = (
    b0_semantic_wide_df
    .groupby(
        [
            "dataset_index",
            "image_id",
        ],
        as_index=False,
    )
    .agg(
        b0_mask_count=(
            "mask_index",
            "size",
        ),

        masks_with_any_C_support=(
            "any_C_support",
            "sum",
        ),

        masks_with_any_D_support=(
            "any_D_support",
            "sum",
        ),

        masks_with_C_and_D_support=(
            "C_and_D_support",
            "sum",
        ),

        masks_with_any_semantic_support=(
            "any_semantic_support",
            "sum",
        ),
    )
)


# ------------------------------------------------------------
# Anchor to authoritative 100-image dev100 table
# ------------------------------------------------------------

b0_semantic_image_df = (
    b2_dev100_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .merge(
        mask_level_image_summary_df,
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Images with zero B0 masks legitimately have no mask-level
# semantic-support records. Fill those values with zero.
# ------------------------------------------------------------

ZERO_FILL_COLS = [
    "b0_mask_count",
    "masks_with_any_C_support",
    "masks_with_any_D_support",
    "masks_with_C_and_D_support",
    "masks_with_any_semantic_support",
]


b0_semantic_image_df[
    ZERO_FILL_COLS
] = (
    b0_semantic_image_df[
        ZERO_FILL_COLS
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


# ------------------------------------------------------------
# Strong integrity checks
# ------------------------------------------------------------

assert (
    len(
        b0_semantic_image_df
    )
    == 100
)


assert (
    b0_semantic_image_df[
        "dataset_index"
    ]
    .is_unique
)


assert (
    b0_semantic_image_df[
        "image_id"
    ]
    .is_unique
)


assert (
    int(
        b0_semantic_image_df[
            "b0_mask_count"
        ].sum()
    )
    == 2298
)


# Number of zero-B0-mask images

num_zero_b0_images = int(
    (
        b0_semantic_image_df[
            "b0_mask_count"
        ]
        == 0
    ).sum()
)


print(
    "Per-image table             : 100 / 100"
)

print(
    "Images with zero B0 masks   :",
    num_zero_b0_images
)


# ------------------------------------------------------------
# 2B.18 Save outputs
# ------------------------------------------------------------

b0_semantic_long_df.to_csv(
    B0_SEMANTIC_LONG_FILE,
    index=False,
)

b0_semantic_wide_df.to_csv(
    B0_SEMANTIC_WIDE_FILE,
    index=False,
)

b0_semantic_image_df.to_csv(
    B0_SEMANTIC_IMAGE_FILE,
    index=False,
)


# ------------------------------------------------------------
# 2B.19 Manifest
# ------------------------------------------------------------

b0_semantic_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 2B - semantic support overlay",

    "num_images":
        100,

    "num_b0_masks":
        2298,

    "num_semantic_candidates":
        8,

    "expected_long_rows":
        18384,

    "semantic_candidates":
        {
            "C":
                C_CANDIDATES,

            "D":
                D_CANDIDATES,
        },

    "point_membership_rule":
        "np.rint + clip -> B0 mask pixel lookup",

    "computed_features":
        [
            "num_semantic_points_inside",
            "semantic_supported",
            "semantic_density_per_10k_mask_px",
            "fraction_candidate_points_inside",
            "nearest_semantic_distance_px",
            "nearest_semantic_distance_norm",
            "num_C_candidates_supporting",
            "num_D_candidates_supporting",
            "any_C_support",
            "any_D_support",
            "C_and_D_support",
            "any_semantic_support",
        ],

    "gt_used_for_semantic_support_computation":
        False,

    "mask_filtering_performed":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,
}


with open(
    B0_SEMANTIC_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        b0_semantic_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2B.20 Confirmation
# ------------------------------------------------------------

print("=" * 125)
print("STEP 2B - C/D SEMANTIC SUPPORT OF FROZEN B0 MASKS")
print("=" * 125)

print(
    "B0 masks                    :",
    len(
        b0_semantic_wide_df
    ),
    "/ 2298"
)

print(
    "Semantic candidates          :",
    len(
        SEMANTIC_CANDIDATES
    ),
    "/ 8"
)

print(
    "Long-format rows            :",
    len(
        b0_semantic_long_df
    ),
    "/ 18384"
)

print(
    "Candidate completeness      : PASS"
)

print(
    "GT used to compute support  : NO"
)

print(
    "Mask filtering performed    : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nCandidate-level B0-mask support:"
)

display(
    candidate_support_summary_df.round(
        4
    )
)


print(
    "\nAggregate semantic support across all candidates:"
)

aggregate_support_summary_df = pd.DataFrame(
    {
        "feature":
            [
                "any_C_support",
                "any_D_support",
                "C_and_D_support",
                "any_semantic_support",
            ],

        "num_masks":
            [
                int(
                    b0_semantic_wide_df[
                        "any_C_support"
                    ].sum()
                ),

                int(
                    b0_semantic_wide_df[
                        "any_D_support"
                    ].sum()
                ),

                int(
                    b0_semantic_wide_df[
                        "C_and_D_support"
                    ].sum()
                ),

                int(
                    b0_semantic_wide_df[
                        "any_semantic_support"
                    ].sum()
                ),
            ],
    }
)


aggregate_support_summary_df[
    "rate"
] = (
    aggregate_support_summary_df[
        "num_masks"
    ]
    /
    len(
        b0_semantic_wide_df
    )
)


display(
    aggregate_support_summary_df.round(
        4
    )
)


print(
    "\nFirst 15 B0 masks with semantic-support features:"
)

DISPLAY_COLS = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "mask_index",
    "gt_support_class",
    "predicted_iou",
    "stability_score",
    "num_C_candidates_supporting",
    "num_D_candidates_supporting",
    "any_C_support",
    "any_D_support",
    "C_and_D_support",
]


display(
    b0_semantic_wide_df[
        DISPLAY_COLS
    ]
    .head(
        15
    )
)


print(
    "\nSaved long-format support table:"
)

print(
    B0_SEMANTIC_LONG_FILE
)

print(
    "\nSaved wide support table:"
)

print(
    B0_SEMANTIC_WIDE_FILE
)

print(
    "\nSaved per-image support table:"
)

print(
    B0_SEMANTIC_IMAGE_FILE
)

print(
    "\nSaved manifest:"
)

print(
    B0_SEMANTIC_MANIFEST_FILE
)

print("=" * 125)

Per-image table             : 100 / 100
Images with zero B0 masks   : 6
STEP 2B - C/D SEMANTIC SUPPORT OF FROZEN B0 MASKS
B0 masks                    : 2298 / 2298
Semantic candidates          : 8 / 8
Long-format rows            : 18384 / 18384
Candidate completeness      : PASS
GT used to compute support  : NO
Mask filtering performed    : NO
Threshold tuning            : NO
Test set accessed           : NO

Candidate-level B0-mask support:


,branch,candidate_id,num_b0_masks,semantically_supported_masks,semantic_support_rate,total_semantic_point_memberships,mean_points_inside_per_mask,median_nearest_semantic_distance_norm
0,C,C1,2298,913,0.3973,1530,0.6658,0.0446
1,C,C2b,2298,1082,0.4708,3338,1.4526,0.0353
2,C,C2c,2298,1062,0.4621,2360,1.0270,0.0363
3,D,D1,2298,1745,0.7594,4729,2.0579,0.0177
4,D,D4,2298,1074,0.4674,2452,1.0670,0.0371
5,D,D5-768,2298,1684,0.7328,7305,3.1789,0.0201
6,D,D6B-T-native,2298,1999,0.8699,9079,3.9508,0.0120
7,D,D6B9-strict,2298,1500,0.6527,3092,1.3455,0.0181



Aggregate semantic support across all candidates:


,feature,num_masks,rate
0,any_C_support,1087,0.4730
1,any_D_support,2141,0.9317
2,C_and_D_support,1076,0.4682
3,any_semantic_support,2152,0.9365



First 15 B0 masks with semantic-support features:


,dataset_index,image_id,category,density_stratum,mask_index,gt_support_class,predicted_iou,stability_score,num_C_candidates_supporting,num_D_candidates_supporting,any_C_support,any_D_support,C_and_D_support
0,9,215.jpg,grapes,S6_95_100,0,zero_gt,0.972012,0.971898,3,5,True,True,True
1,9,215.jpg,grapes,S6_95_100,1,one_gt,0.960591,0.961918,0,1,False,True,False
2,9,215.jpg,grapes,S6_95_100,2,zero_gt,0.958004,0.960602,0,4,False,True,False
3,9,215.jpg,grapes,S6_95_100,3,one_gt,0.950027,0.961429,0,2,False,True,False
4,9,215.jpg,grapes,S6_95_100,4,one_gt,0.948917,0.968224,0,1,False,True,False
5,9,215.jpg,grapes,S6_95_100,5,one_gt,0.948270,0.971530,0,3,False,True,False
6,9,215.jpg,grapes,S6_95_100,6,one_gt,0.947298,0.970320,0,4,False,True,False
7,9,215.jpg,grapes,S6_95_100,7,one_gt,0.946951,0.954802,0,3,False,True,False
8,9,215.jpg,grapes,S6_95_100,8,one_gt,0.945689,0.960479,0,3,False,True,False
9,9,215.jpg,grapes,S6_95_100,9,one_gt,0.944482,0.969900,0,2,False,True,False



Saved long-format support table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_mask_semantic_support_long.csv

Saved wide support table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_mask_semantic_support_wide.csv

Saved per-image support table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_mask_semantic_support_per_image.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_mask_semantic_support_manifest.json


In [14]:
# ============================================================
# Step 2C - Semantic Discrimination of B0 Masks
#
# Question:
# Can frozen C/D semantic evidence distinguish B0 masks that
# contain >=1 GT point from B0 masks containing 0 GT points?
#
# IMPORTANT:
# - diagnostic only
# - no mask filtering
# - no threshold tuning
# - no candidate selection yet
# - GT is used only as retrospective diagnostic reference
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 2C.1 Output files
# ------------------------------------------------------------

B0_SEMANTIC_DISCRIMINATION_FILE = (
    B2_TABLES_DIR
    / "B0_semantic_candidate_discrimination.csv"
)

B0_SEMANTIC_AGGREGATE_FILE = (
    B2_TABLES_DIR
    / "B0_semantic_aggregate_discrimination.csv"
)

B0_SEMANTIC_SUPPORT_COUNT_FILE = (
    B2_TABLES_DIR
    / "B0_semantic_support_count_distribution.csv"
)

B0_SEMANTIC_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_semantic_discrimination_by_density.csv"
)

B0_SEMANTIC_CONTINUOUS_FILE = (
    B2_TABLES_DIR
    / "B0_semantic_continuous_feature_discrimination.csv"
)

B0_SEMANTIC_2C_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_semantic_discrimination_manifest.json"
)


# ------------------------------------------------------------
# 2C.2 Integrity checks
# ------------------------------------------------------------

assert (
    len(
        b0_semantic_long_df
    )
    == 18384
)

assert (
    len(
        b0_semantic_wide_df
    )
    == 2298
)


assert (
    b0_semantic_wide_df[
        [
            "dataset_index",
            "image_id",
            "mask_index",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 2C.3 Define diagnostic GT label
#
# Positive:
#   one_gt OR merged_gt
#
# Negative:
#   zero_gt
#
# This is point support, NOT instance-mask ground truth.
# ------------------------------------------------------------

b0_semantic_wide_df = (
    b0_semantic_wide_df
    .copy()
)


b0_semantic_wide_df[
    "gt_supported"
] = (
    b0_semantic_wide_df[
        "num_gt_points_inside"
    ]
    > 0
)


assert (
    set(
        b0_semantic_wide_df[
            "gt_support_class"
        ].unique()
    )
    ==
    {
        "zero_gt",
        "one_gt",
        "merged_gt",
    }
)


num_gt_supported_masks = int(
    b0_semantic_wide_df[
        "gt_supported"
    ].sum()
)

num_zero_gt_masks = int(
    (
        ~b0_semantic_wide_df[
            "gt_supported"
        ]
    ).sum()
)


assert (
    num_gt_supported_masks
    +
    num_zero_gt_masks
    ==
    2298
)


gt_supported_prevalence = (
    num_gt_supported_masks
    /
    2298
)


# ------------------------------------------------------------
# 2C.4 Binary discrimination helper
# ------------------------------------------------------------

def binary_discrimination_metrics(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=bool,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=bool,
    )


    tp = int(
        np.sum(
            y_true
            &
            y_pred
        )
    )

    fp = int(
        np.sum(
            (~y_true)
            &
            y_pred
        )
    )

    fn = int(
        np.sum(
            y_true
            &
            (~y_pred)
        )
    )

    tn = int(
        np.sum(
            (~y_true)
            &
            (~y_pred)
        )
    )


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp
            +
            fn
        )
        > 0
        else np.nan
    )


    false_positive_rate = (
        fp
        /
        (
            fp
            +
            tn
        )
        if (
            fp
            +
            tn
        )
        > 0
        else np.nan
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
        if (
            tn
            +
            fp
        )
        > 0
        else np.nan
    )


    precision_like = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp
            +
            fp
        )
        > 0
        else np.nan
    )


    negative_predictive_value = (
        tn
        /
        (
            tn
            +
            fn
        )
        if (
            tn
            +
            fn
        )
        > 0
        else np.nan
    )


    balanced_accuracy = (
        (
            sensitivity
            +
            specificity
        )
        /
        2.0
    )


    discrimination_gap = (
        sensitivity
        -
        false_positive_rate
    )


    return {
        "tp":
            tp,

        "fp":
            fp,

        "fn":
            fn,

        "tn":
            tn,

        "gt_supported_recall":
            sensitivity,

        "zero_gt_support_rate":
            false_positive_rate,

        "specificity":
            specificity,

        "precision_like":
            precision_like,

        "negative_predictive_value":
            negative_predictive_value,

        "balanced_accuracy":
            balanced_accuracy,

        "discrimination_gap":
            discrimination_gap,
    }


# ------------------------------------------------------------
# 2C.5 Candidate-level binary discrimination
# ------------------------------------------------------------

candidate_discrimination_records = []


y_true = (
    b0_semantic_wide_df[
        "gt_supported"
    ]
    .to_numpy(
        dtype=bool
    )
)


for (
    branch,
    candidate_id
) in SEMANTIC_CANDIDATES:

    candidate_col = (
        f"{candidate_id}__supported"
    )


    assert (
        candidate_col
        in b0_semantic_wide_df.columns
    )


    y_pred = (
        b0_semantic_wide_df[
            candidate_col
        ]
        .to_numpy(
            dtype=bool
        )
    )


    metrics = (
        binary_discrimination_metrics(
            y_true=y_true,
            y_pred=y_pred,
        )
    )


    # Binary ROC-AUC is included only as a threshold-free
    # descriptive summary of this frozen binary support rule.

    binary_auc = float(
        roc_auc_score(
            y_true.astype(
                int
            ),
            y_pred.astype(
                int
            ),
        )
    )


    candidate_discrimination_records.append(
        {
            "branch":
                branch,

            "candidate_id":
                candidate_id,

            "num_b0_masks":
                2298,

            "num_gt_supported_masks":
                num_gt_supported_masks,

            "num_zero_gt_masks":
                num_zero_gt_masks,

            **metrics,

            "binary_support_auc":
                binary_auc,
        }
    )


candidate_discrimination_df = pd.DataFrame(
    candidate_discrimination_records
)


assert (
    len(
        candidate_discrimination_df
    )
    == 8
)


# ------------------------------------------------------------
# 2C.6 Aggregate binary support rules
#
# These were already defined in Step 2B.
# They are NOT new tuned thresholds.
# ------------------------------------------------------------

aggregate_rule_defs = {
    "any_C_support":
        "any_C_support",

    "any_D_support":
        "any_D_support",

    "C_and_D_support":
        "C_and_D_support",

    "any_semantic_support":
        "any_semantic_support",
}


aggregate_records = []


for rule_name, column_name in (
    aggregate_rule_defs.items()
):

    y_pred = (
        b0_semantic_wide_df[
            column_name
        ]
        .to_numpy(
            dtype=bool
        )
    )


    metrics = (
        binary_discrimination_metrics(
            y_true=y_true,
            y_pred=y_pred,
        )
    )


    binary_auc = float(
        roc_auc_score(
            y_true.astype(
                int
            ),
            y_pred.astype(
                int
            ),
        )
    )


    aggregate_records.append(
        {
            "rule":
                rule_name,

            "num_b0_masks":
                2298,

            **metrics,

            "binary_support_auc":
                binary_auc,
        }
    )


aggregate_discrimination_df = pd.DataFrame(
    aggregate_records
)


# ------------------------------------------------------------
# 2C.7 Support-count distributions
#
# Descriptive ONLY:
# no decision threshold is selected.
# ------------------------------------------------------------

support_count_records = []


SUPPORT_COUNT_FEATURES = {
    "num_C_candidates_supporting":
        3,

    "num_D_candidates_supporting":
        5,

    "total_semantic_candidates_supporting":
        8,
}


for feature_name, max_count in (
    SUPPORT_COUNT_FEATURES.items()
):

    for support_count in range(
        max_count
        + 1
    ):

        count_subset = (
            b0_semantic_wide_df[
                b0_semantic_wide_df[
                    feature_name
                ]
                ==
                support_count
            ]
        )


        n_total = int(
            len(
                count_subset
            )
        )


        n_gt_supported = int(
            count_subset[
                "gt_supported"
            ].sum()
        )


        n_zero_gt = (
            n_total
            -
            n_gt_supported
        )


        gt_supported_fraction = (
            n_gt_supported
            /
            n_total
            if n_total > 0
            else np.nan
        )


        support_count_records.append(
            {
                "feature":
                    feature_name,

                "support_count":
                    support_count,

                "num_masks":
                    n_total,

                "gt_supported_masks":
                    n_gt_supported,

                "zero_gt_masks":
                    n_zero_gt,

                "gt_supported_fraction":
                    gt_supported_fraction,
            }
        )


support_count_distribution_df = (
    pd.DataFrame(
        support_count_records
    )
)


# ------------------------------------------------------------
# 2C.8 Continuous semantic-feature discrimination
#
# Features:
# - number of points inside
# - semantic density inside mask
# - nearest semantic-point distance
#
# No threshold is selected.
#
# For distance, smaller distance should imply stronger support,
# so score = -distance.
#
# NaN distances occur when candidate has zero predictions in
# that image; those rows are reported and excluded from the
# distance AUC only.
# ------------------------------------------------------------

continuous_records = []


for (
    branch,
    candidate_id
), group in b0_semantic_long_df.groupby(
    [
        "branch",
        "candidate_id",
    ],
    sort=False,
):

    group = group.copy()


    y = (
        group[
            "num_gt_points_inside"
        ]
        >
        0
    ).astype(
        int
    ).to_numpy()


    # --------------------------------------------------------
    # Feature A: semantic point count inside B0 mask
    # --------------------------------------------------------

    point_count_score = (
        group[
            "num_semantic_points_inside"
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    point_count_auc = float(
        roc_auc_score(
            y,
            point_count_score,
        )
    )


    # --------------------------------------------------------
    # Feature B: semantic point density inside mask
    # --------------------------------------------------------

    density_score = (
        group[
            "semantic_density_per_10k_mask_px"
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    density_auc = float(
        roc_auc_score(
            y,
            density_score,
        )
    )


    # --------------------------------------------------------
    # Feature C: nearest semantic-point distance
    # --------------------------------------------------------

    distance_values = (
        group[
            "nearest_semantic_distance_norm"
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    finite_distance = (
        np.isfinite(
            distance_values
        )
    )


    num_distance_available = int(
        finite_distance.sum()
    )


    num_distance_missing = int(
        (
            ~finite_distance
        ).sum()
    )


    if (
        num_distance_available
        > 0
        and
        len(
            np.unique(
                y[
                    finite_distance
                ]
            )
        )
        == 2
    ):

        distance_auc = float(
            roc_auc_score(
                y[
                    finite_distance
                ],
                -distance_values[
                    finite_distance
                ],
            )
        )


    else:

        distance_auc = np.nan


    # --------------------------------------------------------
    # Group medians by diagnostic GT class
    # --------------------------------------------------------

    zero_group = group[
        group[
            "num_gt_points_inside"
        ]
        ==
        0
    ]

    supported_group = group[
        group[
            "num_gt_points_inside"
        ]
        >
        0
    ]


    continuous_records.append(
        {
            "branch":
                branch,

            "candidate_id":
                candidate_id,

            "num_masks":
                int(
                    len(
                        group
                    )
                ),

            "point_count_auc":
                point_count_auc,

            "semantic_density_auc":
                density_auc,

            "nearest_distance_auc":
                distance_auc,

            "distance_available_masks":
                num_distance_available,

            "distance_missing_masks":
                num_distance_missing,

            "median_points_inside_zero_gt":
                float(
                    zero_group[
                        "num_semantic_points_inside"
                    ].median()
                ),

            "median_points_inside_gt_supported":
                float(
                    supported_group[
                        "num_semantic_points_inside"
                    ].median()
                ),

            "median_density_zero_gt":
                float(
                    zero_group[
                        "semantic_density_per_10k_mask_px"
                    ].median()
                ),

            "median_density_gt_supported":
                float(
                    supported_group[
                        "semantic_density_per_10k_mask_px"
                    ].median()
                ),

            "median_nearest_distance_zero_gt":
                float(
                    zero_group[
                        "nearest_semantic_distance_norm"
                    ].median(
                        skipna=True
                    )
                ),

            "median_nearest_distance_gt_supported":
                float(
                    supported_group[
                        "nearest_semantic_distance_norm"
                    ].median(
                        skipna=True
                    )
                ),
        }
    )


continuous_discrimination_df = (
    pd.DataFrame(
        continuous_records
    )
)


assert (
    len(
        continuous_discrimination_df
    )
    == 8
)


# ------------------------------------------------------------
# 2C.9 Candidate discrimination by density stratum
#
# Diagnostic only.
#
# GT density strata must NOT be used for inference routing.
# ------------------------------------------------------------

density_records = []


for stratum in DENSITY_ORDER:

    stratum_base = (
        b0_semantic_wide_df[
            b0_semantic_wide_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    y_stratum = (
        stratum_base[
            "gt_supported"
        ]
        .to_numpy(
            dtype=bool
        )
    )


    for (
        branch,
        candidate_id
    ) in SEMANTIC_CANDIDATES:

        candidate_col = (
            f"{candidate_id}__supported"
        )


        y_pred_stratum = (
            stratum_base[
                candidate_col
            ]
            .to_numpy(
                dtype=bool
            )
        )


        metrics = (
            binary_discrimination_metrics(
                y_true=y_stratum,
                y_pred=y_pred_stratum,
            )
        )


        density_records.append(
            {
                "density_stratum":
                    stratum,

                "branch":
                    branch,

                "candidate_id":
                    candidate_id,

                "num_b0_masks":
                    int(
                        len(
                            stratum_base
                        )
                    ),

                "num_gt_supported_masks":
                    int(
                        y_stratum.sum()
                    ),

                "num_zero_gt_masks":
                    int(
                        (
                            ~y_stratum
                        ).sum()
                    ),

                **metrics,
            }
        )


density_discrimination_df = pd.DataFrame(
    density_records
)


assert (
    len(
        density_discrimination_df
    )
    ==
    (
        6
        *
        8
    )
)


# ------------------------------------------------------------
# 2C.10 Compare semantic discrimination with SAM2 native scores
#
# This is descriptive context only.
# No score threshold is selected.
# ------------------------------------------------------------

native_score_records = []


for feature_name in [
    "predicted_iou",
    "stability_score",
]:

    score = (
        b0_semantic_wide_df[
            feature_name
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    auc = float(
        roc_auc_score(
            y_true.astype(
                int
            ),
            score,
        )
    )


    zero_values = (
        b0_semantic_wide_df.loc[
            ~b0_semantic_wide_df[
                "gt_supported"
            ],
            feature_name,
        ]
    )


    supported_values = (
        b0_semantic_wide_df.loc[
            b0_semantic_wide_df[
                "gt_supported"
            ],
            feature_name,
        ]
    )


    native_score_records.append(
        {
            "feature":
                feature_name,

            "auc":
                auc,

            "median_zero_gt":
                float(
                    zero_values.median()
                ),

            "median_gt_supported":
                float(
                    supported_values.median()
                ),

            "mean_zero_gt":
                float(
                    zero_values.mean()
                ),

            "mean_gt_supported":
                float(
                    supported_values.mean()
                ),
        }
    )


native_score_discrimination_df = pd.DataFrame(
    native_score_records
)


# ------------------------------------------------------------
# 2C.11 Save tables
# ------------------------------------------------------------

candidate_discrimination_df.to_csv(
    B0_SEMANTIC_DISCRIMINATION_FILE,
    index=False,
)

aggregate_discrimination_df.to_csv(
    B0_SEMANTIC_AGGREGATE_FILE,
    index=False,
)

support_count_distribution_df.to_csv(
    B0_SEMANTIC_SUPPORT_COUNT_FILE,
    index=False,
)

density_discrimination_df.to_csv(
    B0_SEMANTIC_DENSITY_FILE,
    index=False,
)

continuous_discrimination_df.to_csv(
    B0_SEMANTIC_CONTINUOUS_FILE,
    index=False,
)


# ------------------------------------------------------------
# 2C.12 Manifest
# ------------------------------------------------------------

step_2c_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 2C - semantic discrimination analysis",

    "num_images":
        100,

    "num_b0_masks":
        2298,

    "gt_supported_masks":
        num_gt_supported_masks,

    "zero_gt_masks":
        num_zero_gt_masks,

    "gt_supported_prevalence":
        gt_supported_prevalence,

    "diagnostic_positive_definition":
        "B0 mask contains >=1 FSC-147 GT point",

    "diagnostic_negative_definition":
        "B0 mask contains 0 FSC-147 GT points",

    "candidate_metrics":
        [
            "gt_supported_recall",
            "zero_gt_support_rate",
            "specificity",
            "precision_like",
            "negative_predictive_value",
            "balanced_accuracy",
            "discrimination_gap",
            "binary_support_auc",
        ],

    "continuous_metrics":
        [
            "point_count_auc",
            "semantic_density_auc",
            "nearest_distance_auc",
        ],

    "native_sam2_reference_features":
        [
            "predicted_iou",
            "stability_score",
        ],

    "mask_filtering_performed":
        False,

    "candidate_selection_performed":
        False,

    "threshold_tuning":
        False,

    "density_used_for_inference":
        False,

    "test_set_accessed":
        False,

    "caveat":
        (
            "GT support is based on FSC-147 point annotations. "
            "zero_gt is not equivalent to definitive false-positive "
            "instance segmentation."
        ),
}


with open(
    B0_SEMANTIC_2C_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_2c_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2C.13 Final output
# ------------------------------------------------------------

print("=" * 125)
print("STEP 2C - SEMANTIC DISCRIMINATION OF B0 MASKS")
print("=" * 125)

print(
    "B0 masks                    :",
    2298
)

print(
    "GT-supported masks          :",
    num_gt_supported_masks
)

print(
    "Zero-GT masks               :",
    num_zero_gt_masks
)

print(
    "GT-supported prevalence     :",
    f"{gt_supported_prevalence:.4f}"
)

print(
    "\nMask filtering performed    : NO"
)

print(
    "Candidate selected          : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nCandidate binary-support discrimination:"
)

display(
    candidate_discrimination_df[
        [
            "branch",
            "candidate_id",
            "tp",
            "fp",
            "fn",
            "tn",
            "gt_supported_recall",
            "zero_gt_support_rate",
            "precision_like",
            "specificity",
            "balanced_accuracy",
            "discrimination_gap",
            "binary_support_auc",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nAggregate support-rule discrimination:"
)

display(
    aggregate_discrimination_df[
        [
            "rule",
            "tp",
            "fp",
            "fn",
            "tn",
            "gt_supported_recall",
            "zero_gt_support_rate",
            "precision_like",
            "specificity",
            "balanced_accuracy",
            "discrimination_gap",
            "binary_support_auc",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nContinuous semantic-feature discrimination:"
)

display(
    continuous_discrimination_df[
        [
            "branch",
            "candidate_id",
            "point_count_auc",
            "semantic_density_auc",
            "nearest_distance_auc",
            "median_points_inside_zero_gt",
            "median_points_inside_gt_supported",
            "median_density_zero_gt",
            "median_density_gt_supported",
            "median_nearest_distance_zero_gt",
            "median_nearest_distance_gt_supported",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nNative SAM2 score reference:"
)

display(
    native_score_discrimination_df.round(
        4
    )
)


print(
    "\nSupport-count distributions:"
)

display(
    support_count_distribution_df.round(
        4
    )
)


print(
    "\nCandidate discrimination by density stratum:"
)

display(
    density_discrimination_df[
        [
            "density_stratum",
            "branch",
            "candidate_id",
            "num_b0_masks",
            "num_gt_supported_masks",
            "num_zero_gt_masks",
            "gt_supported_recall",
            "zero_gt_support_rate",
            "precision_like",
            "specificity",
            "discrimination_gap",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nSaved candidate discrimination:"
)

print(
    B0_SEMANTIC_DISCRIMINATION_FILE
)

print(
    "\nSaved aggregate discrimination:"
)

print(
    B0_SEMANTIC_AGGREGATE_FILE
)

print(
    "\nSaved support-count distribution:"
)

print(
    B0_SEMANTIC_SUPPORT_COUNT_FILE
)

print(
    "\nSaved density diagnostics:"
)

print(
    B0_SEMANTIC_DENSITY_FILE
)

print(
    "\nSaved continuous-feature diagnostics:"
)

print(
    B0_SEMANTIC_CONTINUOUS_FILE
)

print(
    "\nSaved manifest:"
)

print(
    B0_SEMANTIC_2C_MANIFEST_FILE
)

print("=" * 125)

STEP 2C - SEMANTIC DISCRIMINATION OF B0 MASKS
B0 masks                    : 2298
GT-supported masks          : 1989
Zero-GT masks               : 309
GT-supported prevalence     : 0.8655

Mask filtering performed    : NO
Candidate selected          : NO
Threshold tuning            : NO
Density used for routing    : NO
Test set accessed           : NO

Candidate binary-support discrimination:


,branch,candidate_id,tp,fp,fn,tn,gt_supported_recall,zero_gt_support_rate,precision_like,specificity,balanced_accuracy,discrimination_gap,binary_support_auc
0,C,C1,882,31,1107,278,0.4434,0.1003,0.9660,0.8997,0.6716,0.3431,0.6716
1,C,C2b,1035,47,954,262,0.5204,0.1521,0.9566,0.8479,0.6841,0.3683,0.6841
2,C,C2c,1020,42,969,267,0.5128,0.1359,0.9605,0.8641,0.6884,0.3769,0.6884
3,D,D1,1653,92,336,217,0.8311,0.2977,0.9473,0.7023,0.7667,0.5333,0.7667
4,D,D4,1041,33,948,276,0.5234,0.1068,0.9693,0.8932,0.7083,0.4166,0.7083
5,D,D5-768,1571,113,418,196,0.7898,0.3657,0.9329,0.6343,0.7121,0.4241,0.7121
6,D,D6B-T-native,1860,139,129,170,0.9351,0.4498,0.9305,0.5502,0.7427,0.4853,0.7427
7,D,D6B9-strict,1433,67,556,242,0.7205,0.2168,0.9553,0.7832,0.7518,0.5036,0.7518



Aggregate support-rule discrimination:


,rule,tp,fp,fn,tn,gt_supported_recall,zero_gt_support_rate,precision_like,specificity,balanced_accuracy,discrimination_gap,binary_support_auc
0,any_C_support,1040,47,949,262,0.5229,0.1521,0.9568,0.8479,0.6854,0.3708,0.6854
1,any_D_support,1953,188,36,121,0.9819,0.6084,0.9122,0.3916,0.6867,0.3735,0.6867
2,C_and_D_support,1035,41,954,268,0.5204,0.1327,0.9619,0.8673,0.6938,0.3877,0.6938
3,any_semantic_support,1958,194,31,115,0.9844,0.6278,0.9099,0.3722,0.6783,0.3566,0.6783



Continuous semantic-feature discrimination:


...


Support-count distributions:


,feature,support_count,num_masks,gt_supported_masks,zero_gt_masks,gt_supported_fraction
0,num_C_candidates_supporting,0,1211,949,262,0.7836
1,num_C_candidates_supporting,1,24,20,4,0.8333
2,num_C_candidates_supporting,2,156,143,13,0.9167
3,num_C_candidates_supporting,3,907,877,30,0.9669
4,num_D_candidates_supporting,0,157,36,121,0.2293
5,num_D_candidates_supporting,1,182,116,66,0.6374
6,num_D_candidates_supporting,2,261,218,43,0.8352
7,num_D_candidates_supporting,3,310,275,35,0.8871
8,num_D_candidates_supporting,4,572,539,33,0.9423
9,num_D_candidates_supporting,5,816,805,11,0.9865



Candidate discrimination by density stratum:


,density_stratum,branch,candidate_id,num_b0_masks,num_gt_supported_masks,num_zero_gt_masks,gt_supported_recall,zero_gt_support_rate,precision_like,specificity,discrimination_gap
0,S1_0_20,C,C1,184,122,62,0.6148,0.0968,0.9259,0.9032,0.5180
1,S1_0_20,C,C2b,184,122,62,0.7131,0.1774,0.8878,0.8226,0.5357
2,S1_0_20,C,C2c,184,122,62,0.7049,0.1774,0.8866,0.8226,0.5275
3,S1_0_20,D,D1,184,122,62,0.9754,0.2097,0.9015,0.7903,0.7657
4,S1_0_20,D,D4,184,122,62,0.7869,0.1290,0.9231,0.8710,0.6579
5,S1_0_20,D,D5-768,184,122,62,0.9180,0.4032,0.8175,0.5968,0.5148
6,S1_0_20,D,D6B-T-native,184,122,62,1.0000,0.3226,0.8592,0.6774,0.6774
7,S1_0_20,D,D6B9-strict,184,122,62,0.9098,0.1774,0.9098,0.8226,0.7324
8,S2_20_40,C,C1,209,148,61,0.6824,0.1803,0.9018,0.8197,0.5021
9,S2_20_40,C,C2b,209,148,61,0.8041,0.2131,0.9015,0.7869,0.5909



Saved candidate discrimination:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_semantic_candidate_discrimination.csv

Saved aggregate discrimination:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_semantic_aggregate_discrimination.csv

Saved support-count distribution:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_semantic_support_count_distribution.csv

Saved density diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_semantic_discrimination_by_density.csv

Saved continuous-feature diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_semantic_continuous_feature_discrimination.csv

Saved manifest:
/content

In [15]:
# ============================================================
# Step 2D1 - Define Candidate B0 Validator Scores
#
# Purpose:
# Construct a small, controlled set of inference-available
# mask-validation scores based on Step 2C evidence.
#
# IMPORTANT:
# - NO threshold selection
# - NO mask filtering
# - NO B0 count modification
# - NO GT-derived feature used as validator input
# - GT labels retained only for later diagnostic evaluation
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2D1.1 Output files
# ------------------------------------------------------------

B0_VALIDATOR_SCORE_FILE = (
    B2_TABLES_DIR
    / "B0_validator_candidate_scores.csv"
)

B0_VALIDATOR_DEFINITION_FILE = (
    B2_METADATA_DIR
    / "B0_validator_candidate_definitions.json"
)

B0_VALIDATOR_2D1_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_validator_step_2D1_manifest.json"
)


# ------------------------------------------------------------
# 2D1.2 Required Step 2B / 2C objects
# ------------------------------------------------------------

assert (
    "b0_semantic_wide_df"
    in globals()
), (
    "b0_semantic_wide_df not found. "
    "Run Step 2B first."
)


assert (
    "b0_semantic_long_df"
    in globals()
), (
    "b0_semantic_long_df not found. "
    "Run Step 2B first."
)


assert (
    len(
        b0_semantic_wide_df
    )
    == 2298
)


# ------------------------------------------------------------
# 2D1.3 Required inference-available columns
# ------------------------------------------------------------

required_wide_cols = {
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "mask_index",
    "num_gt_points_inside",
    "gt_support_class",
    "mask_area",
    "predicted_iou",
    "stability_score",
    "D1__supported",
    "D6B9-strict__supported",
    "num_D_candidates_supporting",
    "total_semantic_candidates_supporting",
}


missing_wide_cols = (
    required_wide_cols
    - set(
        b0_semantic_wide_df.columns
    )
)


assert (
    len(
        missing_wide_cols
    )
    == 0
), (
    "Missing required wide-table columns: "
    f"{sorted(missing_wide_cols)}"
)


# ------------------------------------------------------------
# 2D1.4 Extract continuous D6B9-strict features
# ------------------------------------------------------------

d6b9_continuous_df = (
    b0_semantic_long_df[
        b0_semantic_long_df[
            "candidate_id"
        ]
        ==
        "D6B9-strict"
    ][
        [
            "dataset_index",
            "image_id",
            "mask_index",
            "candidate_pred_count",
            "num_semantic_points_inside",
            "semantic_density_per_10k_mask_px",
            "nearest_semantic_distance_px",
            "nearest_semantic_distance_norm",
        ]
    ]
    .copy()
)


assert (
    len(
        d6b9_continuous_df
    )
    == 2298
)


assert (
    d6b9_continuous_df[
        [
            "dataset_index",
            "image_id",
            "mask_index",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)


d6b9_continuous_df = (
    d6b9_continuous_df
    .rename(
        columns={
            "candidate_pred_count":
                "D6B9_candidate_pred_count",

            "num_semantic_points_inside":
                "D6B9_num_inside",

            "semantic_density_per_10k_mask_px":
                "D6B9_density10k",

            "nearest_semantic_distance_px":
                "D6B9_nearest_distance_px",

            "nearest_semantic_distance_norm":
                "D6B9_nearest_distance_norm",
        }
    )
)


# ------------------------------------------------------------
# 2D1.5 Build validator-score table
# ------------------------------------------------------------

validator_df = (
    b0_semantic_wide_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "mask_index",
            "num_gt_points_inside",
            "gt_support_class",
            "mask_area",
            "predicted_iou",
            "stability_score",
            "D1__supported",
            "D6B9-strict__supported",
            "num_D_candidates_supporting",
            "total_semantic_candidates_supporting",
        ]
    ]
    .merge(
        d6b9_continuous_df,
        on=[
            "dataset_index",
            "image_id",
            "mask_index",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    len(
        validator_df
    )
    == 2298
)


# ------------------------------------------------------------
# 2D1.6 Retrospective diagnostic label
#
# NOT an inference feature.
# Used only in Step 2D2 for evaluation.
# ------------------------------------------------------------

validator_df[
    "gt_supported"
] = (
    validator_df[
        "num_gt_points_inside"
    ]
    > 0
)


# ------------------------------------------------------------
# 2D1.7 Define semantic validator scores
#
# Every score follows:
#
#     higher value = stronger evidence that B0 mask
#                    should be retained
#
# No GT is used in score construction.
# ------------------------------------------------------------


# --------------------------------------------------------
# V1 - D1 binary support
# --------------------------------------------------------

validator_df[
    "V1_D1_support"
] = (
    validator_df[
        "D1__supported"
    ]
    .astype(
        float
    )
)


# --------------------------------------------------------
# V2 - D6B9-strict binary support
# --------------------------------------------------------

validator_df[
    "V2_D6B9_support"
] = (
    validator_df[
        "D6B9-strict__supported"
    ]
    .astype(
        float
    )
)


# --------------------------------------------------------
# V3 - D consensus
#
# 5 frozen D candidates:
# score range = [0, 1]
# --------------------------------------------------------

validator_df[
    "V3_D_consensus"
] = (
    validator_df[
        "num_D_candidates_supporting"
    ]
    .astype(
        float
    )
    /
    5.0
)


# --------------------------------------------------------
# V4 - All semantic consensus
#
# 3 C + 5 D = 8 candidates
# score range = [0, 1]
# --------------------------------------------------------

validator_df[
    "V4_CD_consensus"
] = (
    validator_df[
        "total_semantic_candidates_supporting"
    ]
    .astype(
        float
    )
    /
    8.0
)


# --------------------------------------------------------
# V5 - D6B9 nearest-point proximity
#
# nearest_distance_norm is normalised by image diagonal.
#
# Higher score = closer semantic point.
#
# If D6B9 predicts zero points in the entire image:
# nearest distance is NaN -> semantic evidence absent ->
# proximity score is fixed to 0.
#
# No GT-derived imputation is used.
# --------------------------------------------------------

validator_df[
    "V5_D6B9_proximity"
] = (
    1.0
    -
    validator_df[
        "D6B9_nearest_distance_norm"
    ]
)


validator_df[
    "V5_D6B9_proximity"
] = (
    validator_df[
        "V5_D6B9_proximity"
    ]
    .clip(
        lower=0.0,
        upper=1.0,
    )
    .fillna(
        0.0
    )
)


# --------------------------------------------------------
# V6 - D6B9 semantic density inside B0 mask
#
# Keep raw density here.
# No fitted scaling / normalization.
# --------------------------------------------------------

validator_df[
    "V6_D6B9_density10k"
] = (
    validator_df[
        "D6B9_density10k"
    ]
    .astype(
        float
    )
)


# ------------------------------------------------------------
# 2D1.8 Native SAM2 reference scores
#
# These are not semantic validators.
# Included as reference baselines.
# ------------------------------------------------------------

validator_df[
    "R1_SAM2_predicted_iou"
] = (
    validator_df[
        "predicted_iou"
    ]
    .astype(
        float
    )
)


validator_df[
    "R2_SAM2_stability"
] = (
    validator_df[
        "stability_score"
    ]
    .astype(
        float
    )
)


# ------------------------------------------------------------
# 2D1.9 Controlled score registry
# ------------------------------------------------------------

VALIDATOR_SCORE_DEFINITIONS = {
    "V1_D1_support": {
        "family":
            "semantic_binary",

        "source":
            "D1",

        "higher_is_better":
            True,

        "definition":
            "1 if >=1 D1 semantic point lies inside the B0 mask, else 0",

        "range":
            "[0, 1]",

        "gt_used":
            False,
    },

    "V2_D6B9_support": {
        "family":
            "semantic_binary",

        "source":
            "D6B9-strict",

        "higher_is_better":
            True,

        "definition":
            (
                "1 if >=1 D6B9-strict semantic point lies "
                "inside the B0 mask, else 0"
            ),

        "range":
            "[0, 1]",

        "gt_used":
            False,
    },

    "V3_D_consensus": {
        "family":
            "semantic_consensus",

        "source":
            "all 5 frozen D candidates",

        "higher_is_better":
            True,

        "definition":
            "number of supporting D candidates divided by 5",

        "range":
            "[0, 1]",

        "gt_used":
            False,
    },

    "V4_CD_consensus": {
        "family":
            "semantic_consensus",

        "source":
            "all 8 frozen C/D candidates",

        "higher_is_better":
            True,

        "definition":
            "number of supporting C/D candidates divided by 8",

        "range":
            "[0, 1]",

        "gt_used":
            False,
    },

    "V5_D6B9_proximity": {
        "family":
            "semantic_geometry",

        "source":
            "D6B9-strict",

        "higher_is_better":
            True,

        "definition":
            (
                "1 - nearest D6B9-strict point distance "
                "normalised by image diagonal; "
                "0 when candidate predicts no points in image"
            ),

        "range":
            "[0, 1]",

        "gt_used":
            False,
    },

    "V6_D6B9_density10k": {
        "family":
            "semantic_density",

        "source":
            "D6B9-strict",

        "higher_is_better":
            True,

        "definition":
            (
                "number of D6B9-strict semantic points inside "
                "B0 mask per 10,000 mask pixels"
            ),

        "range":
            "[0, +inf)",

        "gt_used":
            False,
    },

    "R1_SAM2_predicted_iou": {
        "family":
            "SAM2_reference",

        "source":
            "SAM2AutomaticMaskGenerator",

        "higher_is_better":
            True,

        "definition":
            "native SAM2 predicted IoU",

        "gt_used":
            False,
    },

    "R2_SAM2_stability": {
        "family":
            "SAM2_reference",

        "source":
            "SAM2AutomaticMaskGenerator",

        "higher_is_better":
            True,

        "definition":
            "native SAM2 stability score",

        "gt_used":
            False,
    },
}


VALIDATOR_SCORE_COLS = list(
    VALIDATOR_SCORE_DEFINITIONS.keys()
)


assert (
    len(
        VALIDATOR_SCORE_COLS
    )
    == 8
)


# ------------------------------------------------------------
# 2D1.10 Integrity checks
# ------------------------------------------------------------

for score_col in VALIDATOR_SCORE_COLS:

    assert (
        validator_df[
            score_col
        ]
        .notna()
        .all()
    ), (
        f"NaN found in validator score: "
        f"{score_col}"
    )


# Consensus score ranges

assert (
    validator_df[
        "V3_D_consensus"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


assert (
    validator_df[
        "V4_CD_consensus"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


assert (
    validator_df[
        "V5_D6B9_proximity"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


# Binary scores

assert set(
    validator_df[
        "V1_D1_support"
    ].unique()
).issubset(
    {
        0.0,
        1.0,
    }
)


assert set(
    validator_df[
        "V2_D6B9_support"
    ].unique()
).issubset(
    {
        0.0,
        1.0,
    }
)


# ------------------------------------------------------------
# 2D1.11 Score descriptive summary
#
# No GT-group comparison here.
# This just verifies score scale/distribution.
# ------------------------------------------------------------

score_summary_records = []


for score_col in VALIDATOR_SCORE_COLS:

    series = (
        validator_df[
            score_col
        ]
        .astype(
            float
        )
    )


    score_summary_records.append(
        {
            "score":
                score_col,

            "family":
                VALIDATOR_SCORE_DEFINITIONS[
                    score_col
                ][
                    "family"
                ],

            "min":
                float(
                    series.min()
                ),

            "p25":
                float(
                    series.quantile(
                        0.25
                    )
                ),

            "median":
                float(
                    series.median()
                ),

            "mean":
                float(
                    series.mean()
                ),

            "p75":
                float(
                    series.quantile(
                        0.75
                    )
                ),

            "max":
                float(
                    series.max()
                ),

            "num_unique":
                int(
                    series.nunique()
                ),
        }
    )


validator_score_summary_df = pd.DataFrame(
    score_summary_records
)


# ------------------------------------------------------------
# 2D1.12 Save score table
# ------------------------------------------------------------

OUTPUT_COLUMNS = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "mask_index",

    # diagnostic GT fields
    "num_gt_points_inside",
    "gt_support_class",
    "gt_supported",

    # contextual inference-available data
    "mask_area",
    "D6B9_candidate_pred_count",
    "D6B9_num_inside",
    "D6B9_nearest_distance_px",
    "D6B9_nearest_distance_norm",

    # candidate validator scores
    *VALIDATOR_SCORE_COLS,
]


validator_df[
    OUTPUT_COLUMNS
].to_csv(
    B0_VALIDATOR_SCORE_FILE,
    index=False,
)


# ------------------------------------------------------------
# 2D1.13 Save definitions
# ------------------------------------------------------------

with open(
    B0_VALIDATOR_DEFINITION_FILE,
    "w",
) as f:

    json.dump(
        VALIDATOR_SCORE_DEFINITIONS,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2D1.14 Manifest
# ------------------------------------------------------------

step_2d1_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 2D1 - define validator candidate scores",

    "num_images_with_masks":
        int(
            validator_df[
                "image_id"
            ].nunique()
        ),

    "num_b0_masks":
        int(
            len(
                validator_df
            )
        ),

    "semantic_validator_scores":
        [
            "V1_D1_support",
            "V2_D6B9_support",
            "V3_D_consensus",
            "V4_CD_consensus",
            "V5_D6B9_proximity",
            "V6_D6B9_density10k",
        ],

    "sam2_reference_scores":
        [
            "R1_SAM2_predicted_iou",
            "R2_SAM2_stability",
        ],

    "score_direction":
        "higher = stronger retain evidence",

    "gt_used_for_score_construction":
        False,

    "gt_retained_for_later_diagnostic_evaluation":
        True,

    "combined_score_defined":
        False,

    "threshold_selected":
        False,

    "mask_filtering_performed":
        False,

    "b0_counts_modified":
        False,

    "density_used_for_routing":
        False,

    "test_set_accessed":
        False,
}


with open(
    B0_VALIDATOR_2D1_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_2d1_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2D1.15 Final confirmation
# ------------------------------------------------------------

print("=" * 125)
print("STEP 2D1 - DEFINE B0 VALIDATOR CANDIDATE SCORES")
print("=" * 125)

print(
    "B0 masks                    :",
    len(
        validator_df
    ),
    "/ 2298"
)

print(
    "Images represented          :",
    validator_df[
        "image_id"
    ].nunique(),
    "/ 94 images with >=1 B0 mask"
)

print(
    "Semantic validator scores   :",
    6
)

print(
    "SAM2 reference scores       :",
    2
)

print(
    "\nGT used to construct scores : NO"
)

print(
    "Combined score defined      : NO"
)

print(
    "Threshold selected          : NO"
)

print(
    "Mask filtering performed    : NO"
)

print(
    "B0 counts modified          : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nValidator score definitions:"
)


definition_display_df = pd.DataFrame(
    [
        {
            "score":
                score,

            "family":
                definition[
                    "family"
                ],

            "source":
                definition[
                    "source"
                ],

            "definition":
                definition[
                    "definition"
                ],
        }

        for score, definition
        in VALIDATOR_SCORE_DEFINITIONS.items()
    ]
)


display(
    definition_display_df
)


print(
    "\nScore distributions:"
)

display(
    validator_score_summary_df.round(
        4
    )
)


print(
    "\nFirst 15 masks:"
)

display(
    validator_df[
        [
            "dataset_index",
            "image_id",
            "mask_index",
            "gt_support_class",
            "V1_D1_support",
            "V2_D6B9_support",
            "V3_D_consensus",
            "V4_CD_consensus",
            "V5_D6B9_proximity",
            "V6_D6B9_density10k",
            "R1_SAM2_predicted_iou",
            "R2_SAM2_stability",
        ]
    ]
    .head(
        15
    )
    .round(
        4
    )
)


print(
    "\nSaved validator score table:"
)

print(
    B0_VALIDATOR_SCORE_FILE
)

print(
    "\nSaved validator definitions:"
)

print(
    B0_VALIDATOR_DEFINITION_FILE
)

print(
    "\nSaved manifest:"
)

print(
    B0_VALIDATOR_2D1_MANIFEST_FILE
)

print("=" * 125)

STEP 2D1 - DEFINE B0 VALIDATOR CANDIDATE SCORES
B0 masks                    : 2298 / 2298
Images represented          : 94 / 94 images with >=1 B0 mask
Semantic validator scores   : 6
SAM2 reference scores       : 2

GT used to construct scores : NO
Combined score defined      : NO
Threshold selected          : NO
Mask filtering performed    : NO
B0 counts modified          : NO
Density used for routing    : NO
Test set accessed           : NO

Validator score definitions:


,score,family,source,definition
0,V1_D1_support,semantic_binary,D1,1 if >=1 D1 semantic point lies inside the B0 ...
1,V2_D6B9_support,semantic_binary,D6B9-strict,1 if >=1 D6B9-strict semantic point lies insid...
2,V3_D_consensus,semantic_consensus,all 5 frozen D candidates,number of supporting D candidates divided by 5
3,V4_CD_consensus,semantic_consensus,all 8 frozen C/D candidates,number of supporting C/D candidates divided by 8
4,V5_D6B9_proximity,semantic_geometry,D6B9-strict,1 - nearest D6B9-strict point distance normali...
5,V6_D6B9_density10k,semantic_density,D6B9-strict,number of D6B9-strict semantic points inside B...
6,R1_SAM2_predicted_iou,SAM2_reference,SAM2AutomaticMaskGenerator,native SAM2 predicted IoU
7,R2_SAM2_stability,SAM2_reference,SAM2AutomaticMaskGenerator,native SAM2 stability score



Score distributions:


,score,family,min,p25,median,mean,p75,max,num_unique
0,V1_D1_support,semantic_binary,0.0000,1.0000,1.0000,0.7594,1.0000,1.0000,2
1,V2_D6B9_support,semantic_binary,0.0000,0.0000,1.0000,0.6527,1.0000,1.0000,2
2,V3_D_consensus,semantic_consensus,0.0000,0.4000,0.8000,0.6964,1.0000,1.0000,6
3,V4_CD_consensus,semantic_consensus,0.0000,0.3750,0.5000,0.6016,1.0000,1.0000,9
4,V5_D6B9_proximity,semantic_geometry,0.0000,0.9625,0.9816,0.9534,0.9906,0.9998,2270
5,V6_D6B9_density10k,semantic_density,0.0000,0.0000,5.4294,6.5753,9.4679,121.9512,1254
6,R1_SAM2_predicted_iou,SAM2_reference,0.8001,0.9186,0.9460,0.9349,0.9633,0.9973,2294
7,R2_SAM2_stability,SAM2_reference,0.9500,0.9558,0.9627,0.9636,0.9704,1.0000,2034



First 15 masks:


,dataset_index,image_id,mask_index,gt_support_class,V1_D1_support,V2_D6B9_support,V3_D_consensus,V4_CD_consensus,V5_D6B9_proximity,V6_D6B9_density10k,R1_SAM2_predicted_iou,R2_SAM2_stability
0,9,215.jpg,0,zero_gt,1.0,1.0,1.0,1.000,0.9792,4.3879,0.9720,0.9719
1,9,215.jpg,1,one_gt,0.0,0.0,0.2,0.125,0.9609,0.0000,0.9606,0.9619
2,9,215.jpg,2,zero_gt,1.0,1.0,0.8,0.500,0.9878,5.5850,0.9580,0.9606
3,9,215.jpg,3,one_gt,0.0,1.0,0.4,0.250,0.9850,14.4928,0.9500,0.9614
4,9,215.jpg,4,one_gt,0.0,0.0,0.2,0.125,0.9503,0.0000,0.9489,0.9682
5,9,215.jpg,5,one_gt,1.0,0.0,0.6,0.375,0.9480,0.0000,0.9483,0.9715
6,9,215.jpg,6,one_gt,1.0,1.0,0.8,0.500,0.9757,23.1481,0.9473,0.9703
7,9,215.jpg,7,one_gt,1.0,1.0,0.6,0.375,0.9855,14.4718,0.9470,0.9548
8,9,215.jpg,8,one_gt,0.0,1.0,0.6,0.375,0.9987,12.2850,0.9457,0.9605
9,9,215.jpg,9,one_gt,0.0,0.0,0.4,0.250,0.9212,0.0000,0.9445,0.9699



Saved validator score table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_candidate_scores.csv

Saved validator definitions:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_validator_candidate_definitions.json

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_validator_step_2D1_manifest.json


In [16]:
# ============================================================
# Step 2D2 - Validator Retention vs Quality Trade-off
#
# Purpose:
# Compare frozen validator scores on equal footing.
#
# IMPORTANT:
# - diagnostic only
# - no final threshold selection
# - no mask filtering applied to B0 outputs
# - no candidate winner frozen yet
# - no density routing
# - no test-set access
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2D2.1 Output files
# ------------------------------------------------------------

B0_VALIDATOR_RETENTION_CURVE_FILE = (
    B2_TABLES_DIR
    / "B0_validator_retention_tradeoff.csv"
)

B0_VALIDATOR_NATURAL_THRESHOLDS_FILE = (
    B2_TABLES_DIR
    / "B0_validator_natural_thresholds.csv"
)

B0_VALIDATOR_RETENTION_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_validator_retention_summary.csv"
)

B0_VALIDATOR_2D2_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_validator_step_2D2_manifest.json"
)


# ------------------------------------------------------------
# 2D2.2 Required objects
# ------------------------------------------------------------

assert (
    "validator_df"
    in globals()
), (
    "validator_df not found. Run Step 2D1 first."
)

assert (
    "VALIDATOR_SCORE_COLS"
    in globals()
)

assert (
    "VALIDATOR_SCORE_DEFINITIONS"
    in globals()
)


assert (
    len(
        validator_df
    )
    == 2298
)


# ------------------------------------------------------------
# 2D2.3 Diagnostic GT label
# ------------------------------------------------------------

assert (
    "gt_supported"
    in validator_df.columns
)


y_true = (
    validator_df[
        "gt_supported"
    ]
    .astype(
        bool
    )
    .to_numpy()
)


TOTAL_MASKS = int(
    len(
        validator_df
    )
)

TOTAL_GT_SUPPORTED = int(
    y_true.sum()
)

TOTAL_ZERO_GT = int(
    (
        ~y_true
    ).sum()
)


assert (
    TOTAL_GT_SUPPORTED
    == 1989
)

assert (
    TOTAL_ZERO_GT
    == 309
)


# ------------------------------------------------------------
# 2D2.4 Trade-off metric helper
# ------------------------------------------------------------

def compute_retention_metrics(
    retain_flags,
):

    retain_flags = np.asarray(
        retain_flags,
        dtype=bool,
    )


    retained_total = int(
        retain_flags.sum()
    )

    removed_total = (
        TOTAL_MASKS
        -
        retained_total
    )


    retained_gt_supported = int(
        np.sum(
            retain_flags
            &
            y_true
        )
    )

    retained_zero_gt = int(
        np.sum(
            retain_flags
            &
            (~y_true)
        )
    )


    removed_gt_supported = (
        TOTAL_GT_SUPPORTED
        -
        retained_gt_supported
    )

    removed_zero_gt = (
        TOTAL_ZERO_GT
        -
        retained_zero_gt
    )


    retention_rate = (
        retained_total
        /
        TOTAL_MASKS
    )


    gt_supported_recall = (
        retained_gt_supported
        /
        TOTAL_GT_SUPPORTED
    )


    zero_gt_retention_rate = (
        retained_zero_gt
        /
        TOTAL_ZERO_GT
    )


    zero_gt_removal_rate = (
        removed_zero_gt
        /
        TOTAL_ZERO_GT
    )


    retained_gt_supported_fraction = (
        retained_gt_supported
        /
        retained_total
        if retained_total > 0
        else np.nan
    )


    removal_precision_zero_gt = (
        removed_zero_gt
        /
        removed_total
        if removed_total > 0
        else np.nan
    )


    gt_supported_loss_rate = (
        removed_gt_supported
        /
        TOTAL_GT_SUPPORTED
    )


    # Useful descriptive utility:
    # positive means zero-GT masks are removed faster than
    # GT-supported masks are lost.
    removal_gap = (
        zero_gt_removal_rate
        -
        gt_supported_loss_rate
    )


    return {
        "retained_masks":
            retained_total,

        "removed_masks":
            removed_total,

        "retention_rate":
            retention_rate,

        "retained_gt_supported":
            retained_gt_supported,

        "removed_gt_supported":
            removed_gt_supported,

        "gt_supported_recall":
            gt_supported_recall,

        "gt_supported_loss_rate":
            gt_supported_loss_rate,

        "retained_zero_gt":
            retained_zero_gt,

        "removed_zero_gt":
            removed_zero_gt,

        "zero_gt_retention_rate":
            zero_gt_retention_rate,

        "zero_gt_removal_rate":
            zero_gt_removal_rate,

        "retained_gt_supported_fraction":
            retained_gt_supported_fraction,

        "removal_precision_zero_gt":
            removal_precision_zero_gt,

        "removal_gap":
            removal_gap,
    }


# ------------------------------------------------------------
# 2D2.5 Fixed retention fractions
#
# This gives fair comparison across score families.
#
# Higher score = stronger retain evidence.
# ------------------------------------------------------------

RETENTION_TARGETS = [
    1.00,
    0.95,
    0.90,
    0.85,
    0.80,
    0.75,
    0.70,
    0.60,
    0.50,
]


retention_curve_records = []


for score_col in VALIDATOR_SCORE_COLS:

    scores = (
        validator_df[
            score_col
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    # Stable deterministic ordering:
    # primary = descending score
    # tie-break = original row order
    order = np.lexsort(
        (
            np.arange(
                TOTAL_MASKS
            ),
            -scores,
        )
    )


    for target_retention in RETENTION_TARGETS:

        target_num_retained = int(
            round(
                TOTAL_MASKS
                *
                target_retention
            )
        )


        target_num_retained = min(
            max(
                target_num_retained,
                0,
            ),
            TOTAL_MASKS,
        )


        retain_flags = np.zeros(
            TOTAL_MASKS,
            dtype=bool,
        )


        if target_num_retained > 0:

            retain_flags[
                order[
                    :target_num_retained
                ]
            ] = True


        metrics = compute_retention_metrics(
            retain_flags
        )


        threshold_min_retained = (
            float(
                scores[
                    order[
                        target_num_retained - 1
                    ]
                ]
            )
            if target_num_retained > 0
            else np.nan
        )


        threshold_max_removed = (
            float(
                scores[
                    order[
                        target_num_retained
                    ]
                ]
            )
            if target_num_retained < TOTAL_MASKS
            else np.nan
        )


        retention_curve_records.append(
            {
                "score":
                    score_col,

                "family":
                    VALIDATOR_SCORE_DEFINITIONS[
                        score_col
                    ][
                        "family"
                    ],

                "evaluation_mode":
                    "fixed_retention",

                "target_retention":
                    target_retention,

                "target_num_retained":
                    target_num_retained,

                "threshold_min_retained":
                    threshold_min_retained,

                "threshold_max_removed":
                    threshold_max_removed,

                **metrics,
            }
        )


retention_tradeoff_df = pd.DataFrame(
    retention_curve_records
)


# ------------------------------------------------------------
# 2D2.6 Natural score thresholds
#
# Evaluate all natural levels only for low-cardinality scores.
#
# No threshold is selected.
# ------------------------------------------------------------

NATURAL_THRESHOLD_SCORES = [
    "V1_D1_support",
    "V2_D6B9_support",
    "V3_D_consensus",
    "V4_CD_consensus",
]


natural_threshold_records = []


for score_col in NATURAL_THRESHOLD_SCORES:

    scores = (
        validator_df[
            score_col
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    unique_thresholds = sorted(
        np.unique(
            scores
        ),
        reverse=True,
    )


    # Include a retain-all operating point
    # just below minimum score.

    retain_all_threshold = (
        float(
            np.min(
                scores
            )
        )
        -
        1e-12
    )


    thresholds_to_test = (
        unique_thresholds
        +
        [
            retain_all_threshold
        ]
    )


    for threshold in thresholds_to_test:

        retain_flags = (
            scores
            >= threshold
        )


        metrics = compute_retention_metrics(
            retain_flags
        )


        natural_threshold_records.append(
            {
                "score":
                    score_col,

                "family":
                    VALIDATOR_SCORE_DEFINITIONS[
                        score_col
                    ][
                        "family"
                    ],

                "threshold":
                    float(
                        threshold
                    ),

                **metrics,
            }
        )


natural_threshold_df = pd.DataFrame(
    natural_threshold_records
)


# ------------------------------------------------------------
# 2D2.7 Compact summary at common retention levels
#
# Focus on 90%, 80%, 70% for easy comparison.
# ------------------------------------------------------------

SUMMARY_RETENTIONS = [
    0.90,
    0.80,
    0.70,
]


summary_df = (
    retention_tradeoff_df[
        retention_tradeoff_df[
            "target_retention"
        ].isin(
            SUMMARY_RETENTIONS
        )
    ]
    .copy()
)


summary_df = summary_df[
    [
        "score",
        "family",
        "target_retention",
        "retained_masks",
        "removed_masks",
        "gt_supported_recall",
        "gt_supported_loss_rate",
        "zero_gt_removal_rate",
        "retained_gt_supported_fraction",
        "removal_precision_zero_gt",
        "removal_gap",
    ]
]


# ------------------------------------------------------------
# 2D2.8 Sanity checks
# ------------------------------------------------------------

assert (
    len(
        retention_tradeoff_df
    )
    ==
    len(
        VALIDATOR_SCORE_COLS
    )
    *
    len(
        RETENTION_TARGETS
    )
)


assert (
    retention_tradeoff_df[
        "retained_masks"
    ]
    .between(
        0,
        TOTAL_MASKS
    )
    .all()
)


assert (
    retention_tradeoff_df[
        "gt_supported_recall"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


assert (
    retention_tradeoff_df[
        "zero_gt_removal_rate"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


# Retain-all rows must recover baseline composition.

retain_all_rows = (
    retention_tradeoff_df[
        retention_tradeoff_df[
            "target_retention"
        ]
        ==
        1.00
    ]
)


assert (
    retain_all_rows[
        "retained_masks"
    ]
    .eq(
        TOTAL_MASKS
    )
    .all()
)


assert np.allclose(
    retain_all_rows[
        "gt_supported_recall"
    ],
    1.0,
)


assert np.allclose(
    retain_all_rows[
        "zero_gt_removal_rate"
    ],
    0.0,
)


# ------------------------------------------------------------
# 2D2.9 Save tables
# ------------------------------------------------------------

retention_tradeoff_df.to_csv(
    B0_VALIDATOR_RETENTION_CURVE_FILE,
    index=False,
)


natural_threshold_df.to_csv(
    B0_VALIDATOR_NATURAL_THRESHOLDS_FILE,
    index=False,
)


summary_df.to_csv(
    B0_VALIDATOR_RETENTION_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 2D2.10 Manifest
# ------------------------------------------------------------

step_2d2_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 2D2 - validator retention-quality trade-off",

    "num_b0_masks":
        TOTAL_MASKS,

    "gt_supported_masks":
        TOTAL_GT_SUPPORTED,

    "zero_gt_masks":
        TOTAL_ZERO_GT,

    "scores_evaluated":
        VALIDATOR_SCORE_COLS,

    "fixed_retention_targets":
        RETENTION_TARGETS,

    "natural_threshold_scores":
        NATURAL_THRESHOLD_SCORES,

    "metrics":
        [
            "retention_rate",
            "gt_supported_recall",
            "gt_supported_loss_rate",
            "zero_gt_removal_rate",
            "retained_gt_supported_fraction",
            "removal_precision_zero_gt",
            "removal_gap",
        ],

    "fixed_retention_tie_break":
        "stable original row order",

    "threshold_selected":
        False,

    "validator_selected":
        False,

    "mask_filtering_applied":
        False,

    "b0_counts_modified":
        False,

    "density_used_for_routing":
        False,

    "test_set_accessed":
        False,

    "caveat":
        (
            "GT support is based on FSC-147 point annotations. "
            "Trade-off analysis is developmental diagnostic evidence, "
            "not independent validation."
        ),
}


with open(
    B0_VALIDATOR_2D2_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_2d2_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2D2.11 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 2D2 - VALIDATOR RETENTION vs QUALITY TRADE-OFF")
print("=" * 125)

print(
    "B0 masks                    :",
    TOTAL_MASKS
)

print(
    "GT-supported masks          :",
    TOTAL_GT_SUPPORTED
)

print(
    "Zero-GT masks               :",
    TOTAL_ZERO_GT
)

print(
    "\nScores evaluated            :",
    len(
        VALIDATOR_SCORE_COLS
    )
)

print(
    "Fixed retention levels      :",
    len(
        RETENTION_TARGETS
    )
)

print(
    "Natural-threshold scores    :",
    len(
        NATURAL_THRESHOLD_SCORES
    )
)

print(
    "\nThreshold selected          : NO"
)

print(
    "Validator selected          : NO"
)

print(
    "Mask filtering applied      : NO"
)

print(
    "B0 counts modified          : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nCommon retention levels (90%, 80%, 70%):"
)

display(
    summary_df.sort_values(
        [
            "target_retention",
            "removal_gap",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .round(
        4
    )
)


print(
    "\nNatural thresholds:"
)

display(
    natural_threshold_df[
        [
            "score",
            "threshold",
            "retention_rate",
            "gt_supported_recall",
            "gt_supported_loss_rate",
            "zero_gt_removal_rate",
            "retained_gt_supported_fraction",
            "removal_precision_zero_gt",
            "removal_gap",
        ]
    ]
    .sort_values(
        [
            "score",
            "threshold",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        4
    )
)


print(
    "\nFull fixed-retention trade-off table:"
)

display(
    retention_tradeoff_df[
        [
            "score",
            "family",
            "target_retention",
            "retained_masks",
            "removed_masks",
            "gt_supported_recall",
            "gt_supported_loss_rate",
            "zero_gt_removal_rate",
            "retained_gt_supported_fraction",
            "removal_precision_zero_gt",
            "removal_gap",
        ]
    ]
    .sort_values(
        [
            "score",
            "target_retention",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        4
    )
)


print(
    "\nSaved fixed-retention trade-off:"
)

print(
    B0_VALIDATOR_RETENTION_CURVE_FILE
)

print(
    "\nSaved natural-threshold table:"
)

print(
    B0_VALIDATOR_NATURAL_THRESHOLDS_FILE
)

print(
    "\nSaved compact summary:"
)

print(
    B0_VALIDATOR_RETENTION_SUMMARY_FILE
)

print(
    "\nSaved manifest:"
)

print(
    B0_VALIDATOR_2D2_MANIFEST_FILE
)

print("=" * 125)

STEP 2D2 - VALIDATOR RETENTION vs QUALITY TRADE-OFF
B0 masks                    : 2298
GT-supported masks          : 1989
Zero-GT masks               : 309

Scores evaluated            : 8
Fixed retention levels      : 9
Natural-threshold scores    : 4

Threshold selected          : NO
Validator selected          : NO
Mask filtering applied      : NO
B0 counts modified          : NO
Density used for routing    : NO
Test set accessed           : NO

Common retention levels (90%, 80%, 70%):


,score,family,target_retention,retained_masks,removed_masks,gt_supported_recall,gt_supported_loss_rate,zero_gt_removal_rate,retained_gt_supported_fraction,removal_precision_zero_gt,removal_gap
38,V5_D6B9_proximity,semantic_geometry,0.9,2068,230,0.9618,0.0382,0.4984,0.9250,0.6696,0.4602
29,V4_CD_consensus,semantic_consensus,0.9,2068,230,0.9583,0.0417,0.4757,0.9217,0.6391,0.4340
20,V3_D_consensus,semantic_consensus,0.9,2068,230,0.9548,0.0452,0.4531,0.9183,0.6087,0.4078
56,R1_SAM2_predicted_iou,SAM2_reference,0.9,2068,230,0.9281,0.0719,0.2816,0.8926,0.3783,0.2097
2,V1_D1_support,semantic_binary,0.9,2068,230,0.9211,0.0789,0.2362,0.8859,0.3174,0.1573
65,R2_SAM2_stability,SAM2_reference,0.9,2068,230,0.9065,0.0935,0.1424,0.8719,0.1913,0.0489
11,V2_D6B9_support,semantic_binary,0.9,2068,230,0.9010,0.0990,0.1068,0.8665,0.1435,0.0078
47,V6_D6B9_density10k,semantic_density,0.9,2068,230,0.9010,0.0990,0.1068,0.8665,0.1435,0.0078
40,V5_D6B9_proximity,semantic_geometry,0.8,1838,460,0.8768,0.1232,0.6958,0.9489,0.4674,0.5726
4,V1_D1_support,semantic_binary,0.8,1838,460,0.8753,0.1247,0.6861,0.9472,0.4609,0.5614



Natural thresholds:


,score,threshold,retention_rate,gt_supported_recall,gt_supported_loss_rate,zero_gt_removal_rate,retained_gt_supported_fraction,removal_precision_zero_gt,removal_gap
0,V1_D1_support,1.000,0.7594,0.8311,0.1689,0.7023,0.9473,0.3924,0.5333
1,V1_D1_support,0.000,1.0000,1.0000,0.0000,0.0000,0.8655,NaN,0.0000
2,V1_D1_support,-0.000,1.0000,1.0000,0.0000,0.0000,0.8655,NaN,0.0000
3,V2_D6B9_support,1.000,0.6527,0.7205,0.2795,0.7832,0.9553,0.3033,0.5036
4,V2_D6B9_support,0.000,1.0000,1.0000,0.0000,0.0000,0.8655,NaN,0.0000
5,V2_D6B9_support,-0.000,1.0000,1.0000,0.0000,0.0000,0.8655,NaN,0.0000
6,V3_D_consensus,1.000,0.3551,0.4047,0.5953,0.9644,0.9865,0.2011,0.3691
7,V3_D_consensus,0.800,0.6040,0.6757,0.3243,0.8576,0.9683,0.2912,0.5333
8,V3_D_consensus,0.600,0.7389,0.8140,0.1860,0.7443,0.9535,0.3833,0.5583
9,V3_D_consensus,0.400,0.8525,0.9236,0.0764,0.6052,0.9377,0.5516,0.5288



Full fixed-retention trade-off table:


,score,family,target_retention,retained_masks,removed_masks,gt_supported_recall,gt_supported_loss_rate,zero_gt_removal_rate,retained_gt_supported_fraction,removal_precision_zero_gt,removal_gap
54,R1_SAM2_predicted_iou,SAM2_reference,1.00,2298,0,1.0000,0.0000,0.0000,0.8655,NaN,0.0000
55,R1_SAM2_predicted_iou,SAM2_reference,0.95,2183,115,0.9623,0.0377,0.1294,0.8768,0.3478,0.0917
56,R1_SAM2_predicted_iou,SAM2_reference,0.90,2068,230,0.9281,0.0719,0.2816,0.8926,0.3783,0.2097
57,R1_SAM2_predicted_iou,SAM2_reference,0.85,1953,345,0.8864,0.1136,0.3851,0.9027,0.3449,0.2715
58,R1_SAM2_predicted_iou,SAM2_reference,0.80,1838,460,0.8371,0.1629,0.4401,0.9059,0.2957,0.2772
...,...,...,...,...,...,...,...,...,...,...,...
49,V6_D6B9_density10k,semantic_density,0.80,1838,460,0.8276,0.1724,0.3786,0.8955,0.2543,0.2062
50,V6_D6B9_density10k,semantic_density,0.75,1724,574,0.7929,0.2071,0.5243,0.9147,0.2822,0.3171
51,V6_D6B9_density10k,semantic_density,0.70,1609,689,0.7642,0.2358,0.7120,0.9447,0.3193,0.4762
52,V6_D6B9_density10k,semantic_density,0.60,1379,919,0.6687,0.3313,0.8414,0.9645,0.2829,0.5101



Saved fixed-retention trade-off:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_retention_tradeoff.csv

Saved natural-threshold table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_natural_thresholds.csv

Saved compact summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_retention_summary.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_validator_step_2D2_manifest.json


In [17]:
# ============================================================
# Step 2D3 - Apply Selected Validator Operating Points
#            and Measure Image-Level Count Impact
#
# Purpose:
# Test whether mask-level semantic validation actually improves
# counting performance relative to frozen B0.
#
# IMPORTANT:
# - dev100 only
# - no test access
# - no semantic recovery yet
# - no density routing
# - no C candidates used in selected validators
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2D3.1 Output files
# ------------------------------------------------------------

B0_VALIDATOR_COUNT_RESULTS_FILE = (
    B2_TABLES_DIR
    / "B0_validator_count_results_per_image.csv"
)

B0_VALIDATOR_COUNT_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_validator_count_summary.csv"
)

B0_VALIDATOR_COUNT_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_validator_count_summary_by_density.csv"
)

B0_VALIDATOR_2D3_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_validator_step_2D3_manifest.json"
)


# ------------------------------------------------------------
# 2D3.2 Preconditions
# ------------------------------------------------------------

assert (
    "validator_df"
    in globals()
), (
    "validator_df not found. Run Step 2D1 first."
)


assert (
    len(
        validator_df
    )
    == 2298
)


# ------------------------------------------------------------
# 2D3.3 Define selected operating points
#
# V5:
#   D6B9-strict proximity ranking
#
# V3:
#   D-consensus across all 5 frozen D candidates
#
# No C evidence is used here.
# ------------------------------------------------------------

OPERATING_POINTS = {
    "B0_unfiltered": {
        "type":
            "baseline",

        "description":
            "Frozen B0 count with no semantic filtering",
    },

    "V5_top95": {
        "type":
            "global_rank",

        "score":
            "V5_D6B9_proximity",

        "retention_fraction":
            0.95,

        "description":
            "Retain global top 95% of B0 masks by D6B9-strict proximity",
    },

    "V5_top90": {
        "type":
            "global_rank",

        "score":
            "V5_D6B9_proximity",

        "retention_fraction":
            0.90,

        "description":
            "Retain global top 90% of B0 masks by D6B9-strict proximity",
    },

    "V3_Dcons_ge_0p2": {
        "type":
            "threshold",

        "score":
            "V3_D_consensus",

        "threshold":
            0.20,

        "description":
            "Retain B0 mask if at least 1 of 5 D candidates supports it",
    },

    "V3_Dcons_ge_0p4": {
        "type":
            "threshold",

        "score":
            "V3_D_consensus",

        "threshold":
            0.40,

        "description":
            "Retain B0 mask if at least 2 of 5 D candidates support it",
    },
}


# ------------------------------------------------------------
# 2D3.4 Build global rank masks for V5
#
# IMPORTANT:
# This is developmental diagnostic analysis.
# Global top-X% retention is not yet a deployable inference
# rule because it depends on the evaluated batch.
#
# It is used here only to measure whether V5 ranking quality
# translates into count improvement.
# ------------------------------------------------------------

global_rank_keep_flags = {}


for op_name, op_def in OPERATING_POINTS.items():

    if op_def[
        "type"
    ] != "global_rank":

        continue


    score_col = op_def[
        "score"
    ]


    retention_fraction = float(
        op_def[
            "retention_fraction"
        ]
    )


    scores = (
        validator_df[
            score_col
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    total_masks = len(
        scores
    )


    num_keep = int(
        round(
            total_masks
            *
            retention_fraction
        )
    )


    order = np.lexsort(
        (
            np.arange(
                total_masks
            ),
            -scores,
        )
    )


    keep = np.zeros(
        total_masks,
        dtype=bool,
    )


    keep[
        order[
            :num_keep
        ]
    ] = True


    global_rank_keep_flags[
        op_name
    ] = keep


# ------------------------------------------------------------
# 2D3.5 Attach operating-point keep flags to mask table
# ------------------------------------------------------------

eval_mask_df = (
    validator_df
    .copy()
)


eval_mask_df[
    "keep_B0_unfiltered"
] = True


# Global ranking rules

for op_name, keep_flags in (
    global_rank_keep_flags.items()
):

    eval_mask_df[
        f"keep_{op_name}"
    ] = keep_flags


# Natural D-consensus thresholds

eval_mask_df[
    "keep_V3_Dcons_ge_0p2"
] = (
    eval_mask_df[
        "V3_D_consensus"
    ]
    >= 0.20
)


eval_mask_df[
    "keep_V3_Dcons_ge_0p4"
] = (
    eval_mask_df[
        "V3_D_consensus"
    ]
    >= 0.40
)


# ------------------------------------------------------------
# 2D3.6 Per-image filtered counts
#
# Start from all 100 dev100 images so the six images with
# zero B0 masks remain explicitly represented.
# ------------------------------------------------------------

per_image_records = []


for image_row in b2_dev100_image_df.itertuples(
    index=False
):

    dataset_index = int(
        image_row.dataset_index
    )

    image_id = image_row.image_id

    gt_count = int(
        image_row.gt_count
    )


    image_masks = (
        eval_mask_df[
            eval_mask_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
    )


    frozen_b0_count = int(
        len(
            image_masks
        )
    )


    record = {
        "dataset_index":
            dataset_index,

        "image_id":
            image_id,

        "category":
            image_row.category,

        "density_stratum":
            image_row.density_stratum,

        "gt_count":
            gt_count,

        "B0_unfiltered":
            frozen_b0_count,
    }


    for op_name in OPERATING_POINTS:

        if op_name == "B0_unfiltered":

            continue


        keep_col = (
            f"keep_{op_name}"
        )


        filtered_count = int(
            image_masks[
                keep_col
            ].sum()
        )


        record[
            op_name
        ] = filtered_count


    per_image_records.append(
        record
    )


validator_count_per_image_df = pd.DataFrame(
    per_image_records
)


assert (
    len(
        validator_count_per_image_df
    )
    == 100
)


# ------------------------------------------------------------
# 2D3.7 Verify frozen B0 count identity
# ------------------------------------------------------------

b0_reference_counts = (
    b0_mask_cache_index_df[
        [
            "dataset_index",
            "reproduced_b0_count",
        ]
    ]
    .set_index(
        "dataset_index"
    )
)


for row in (
    validator_count_per_image_df
    .itertuples(
        index=False
    )
):

    assert (
        int(
            row.B0_unfiltered
        )
        ==
        int(
            b0_reference_counts.loc[
                row.dataset_index,
                "reproduced_b0_count",
            ]
        )
    )


# ------------------------------------------------------------
# 2D3.8 Error metrics helper
# ------------------------------------------------------------

def count_metrics(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )


    error = (
        pred
        -
        gt
    )


    abs_error = np.abs(
        error
    )


    mae = float(
        abs_error.mean()
    )


    rmse = float(
        np.sqrt(
            np.mean(
                error**2
            )
        )
    )


    signed_error = float(
        error.mean()
    )


    under = int(
        np.sum(
            error
            <
            0
        )
    )

    exact = int(
        np.sum(
            error
            ==
            0
        )
    )

    over = int(
        np.sum(
            error
            >
            0
        )
    )


    return {
        "MAE":
            mae,

        "RMSE":
            rmse,

        "mean_signed_error":
            signed_error,

        "under_count_images":
            under,

        "exact_count_images":
            exact,

        "over_count_images":
            over,
    }


# ------------------------------------------------------------
# 2D3.9 Overall count performance
# ------------------------------------------------------------

summary_records = []


gt_all = (
    validator_count_per_image_df[
        "gt_count"
    ]
    .to_numpy()
)


for op_name in OPERATING_POINTS:

    pred = (
        validator_count_per_image_df[
            op_name
        ]
        .to_numpy()
    )


    metrics = count_metrics(
        gt=gt_all,
        pred=pred,
    )


    total_pred_masks = int(
        pred.sum()
    )


    summary_records.append(
        {
            "operating_point":
                op_name,

            "description":
                OPERATING_POINTS[
                    op_name
                ][
                    "description"
                ],

            "total_predicted_count":
                total_pred_masks,

            "mean_predicted_count":
                float(
                    pred.mean()
                ),

            **metrics,
        }
    )


validator_count_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 2D3.10 Add change vs frozen B0
# ------------------------------------------------------------

b0_summary_row = (
    validator_count_summary_df[
        validator_count_summary_df[
            "operating_point"
        ]
        ==
        "B0_unfiltered"
    ]
    .iloc[
        0
    ]
)


b0_mae = float(
    b0_summary_row[
        "MAE"
    ]
)

b0_rmse = float(
    b0_summary_row[
        "RMSE"
    ]
)


validator_count_summary_df[
    "delta_MAE_vs_B0"
] = (
    validator_count_summary_df[
        "MAE"
    ]
    -
    b0_mae
)


validator_count_summary_df[
    "delta_RMSE_vs_B0"
] = (
    validator_count_summary_df[
        "RMSE"
    ]
    -
    b0_rmse
)


# ------------------------------------------------------------
# 2D3.11 Per-density diagnostic summary
#
# Diagnostic only.
# GT strata must NOT be used for inference routing.
# ------------------------------------------------------------

density_records = []


for stratum in DENSITY_ORDER:

    subset = (
        validator_count_per_image_df[
            validator_count_per_image_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    gt = (
        subset[
            "gt_count"
        ]
        .to_numpy()
    )


    for op_name in OPERATING_POINTS:

        pred = (
            subset[
                op_name
            ]
            .to_numpy()
        )


        metrics = count_metrics(
            gt=gt,
            pred=pred,
        )


        density_records.append(
            {
                "density_stratum":
                    stratum,

                "operating_point":
                    op_name,

                "num_images":
                    int(
                        len(
                            subset
                        )
                    ),

                **metrics,
            }
        )


validator_count_density_df = pd.DataFrame(
    density_records
)


# ------------------------------------------------------------
# 2D3.12 Per-image changes relative to B0
# ------------------------------------------------------------

for op_name in OPERATING_POINTS:

    if op_name == "B0_unfiltered":

        continue


    validator_count_per_image_df[
        f"{op_name}__delta_count_vs_B0"
    ] = (
        validator_count_per_image_df[
            op_name
        ]
        -
        validator_count_per_image_df[
            "B0_unfiltered"
        ]
    )


    validator_count_per_image_df[
        f"{op_name}__abs_error"
    ] = np.abs(
        validator_count_per_image_df[
            op_name
        ]
        -
        validator_count_per_image_df[
            "gt_count"
        ]
    )


validator_count_per_image_df[
    "B0_unfiltered__abs_error"
] = np.abs(
    validator_count_per_image_df[
        "B0_unfiltered"
    ]
    -
    validator_count_per_image_df[
        "gt_count"
    ]
)


# ------------------------------------------------------------
# 2D3.13 Help / hurt counts
# ------------------------------------------------------------

help_hurt_records = []


for op_name in OPERATING_POINTS:

    if op_name == "B0_unfiltered":

        continue


    new_abs = (
        validator_count_per_image_df[
            f"{op_name}__abs_error"
        ]
    )


    b0_abs = (
        validator_count_per_image_df[
            "B0_unfiltered__abs_error"
        ]
    )


    improved = int(
        (
            new_abs
            <
            b0_abs
        ).sum()
    )

    unchanged = int(
        (
            new_abs
            ==
            b0_abs
        ).sum()
    )

    worsened = int(
        (
            new_abs
            >
            b0_abs
        ).sum()
    )


    help_hurt_records.append(
        {
            "operating_point":
                op_name,

            "improved_images":
                improved,

            "unchanged_images":
                unchanged,

            "worsened_images":
                worsened,
        }
    )


help_hurt_df = pd.DataFrame(
    help_hurt_records
)


# ------------------------------------------------------------
# 2D3.14 Save outputs
# ------------------------------------------------------------

validator_count_per_image_df.to_csv(
    B0_VALIDATOR_COUNT_RESULTS_FILE,
    index=False,
)


validator_count_summary_df.to_csv(
    B0_VALIDATOR_COUNT_SUMMARY_FILE,
    index=False,
)


validator_count_density_df.to_csv(
    B0_VALIDATOR_COUNT_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 2D3.15 Manifest
# ------------------------------------------------------------

step_2d3_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 2D3 - image-level count impact",

    "num_images":
        100,

    "num_b0_masks":
        2298,

    "operating_points":
        OPERATING_POINTS,

    "semantic_sources_used":
        {
            "V5":
                "D6B9-strict only",

            "V3":
                [
                    "D1",
                    "D4",
                    "D5-768",
                    "D6B-T-native",
                    "D6B9-strict",
                ],
        },

    "C_candidates_used":
        False,

    "semantic_recovery":
        False,

    "density_used_for_routing":
        False,

    "global_rank_warning":
        (
            "V5 top-95/top-90 are developmental batch-level "
            "diagnostics, not deployable inference thresholds."
        ),

    "test_set_accessed":
        False,
}


with open(
    B0_VALIDATOR_2D3_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_2d3_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2D3.16 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 2D3 - IMAGE-LEVEL COUNT IMPACT OF B0 VALIDATORS")
print("=" * 125)

print(
    "Images                      :",
    len(
        validator_count_per_image_df
    ),
    "/ 100"
)

print(
    "Frozen B0 masks             :",
    int(
        validator_count_per_image_df[
            "B0_unfiltered"
        ].sum()
    )
)

print(
    "\nC candidates used           : NO"
)

print(
    "V5 source                   : D6B9-strict"
)

print(
    "V3 source                   : all 5 D candidates"
)

print(
    "Semantic recovery           : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nOverall count performance:"
)

display(
    validator_count_summary_df[
        [
            "operating_point",
            "total_predicted_count",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_B0",
            "delta_RMSE_vs_B0",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nHelp / hurt relative to B0:"
)

display(
    help_hurt_df
)


print(
    "\nBy density stratum:"
)

display(
    validator_count_density_df[
        [
            "density_stratum",
            "operating_point",
            "num_images",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nSaved per-image count results:"
)

print(
    B0_VALIDATOR_COUNT_RESULTS_FILE
)

print(
    "\nSaved overall count summary:"
)

print(
    B0_VALIDATOR_COUNT_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    B0_VALIDATOR_COUNT_DENSITY_FILE
)

print(
    "\nSaved manifest:"
)

print(
    B0_VALIDATOR_2D3_MANIFEST_FILE
)

print("=" * 125)

STEP 2D3 - IMAGE-LEVEL COUNT IMPACT OF B0 VALIDATORS
Images                      : 100 / 100
Frozen B0 masks             : 2298

C candidates used           : NO
V5 source                   : D6B9-strict
V3 source                   : all 5 D candidates
Semantic recovery           : NO
Density used for routing    : NO
Test set accessed           : NO

Overall count performance:


,operating_point,total_predicted_count,MAE,RMSE,mean_signed_error,delta_MAE_vs_B0,delta_RMSE_vs_B0,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,2298,80.69,270.2339,-77.47,0.00,0.0000,76,4,20
1,V5_top95,2183,81.68,270.9700,-78.62,0.99,0.7361,77,3,20
2,V5_top90,2068,82.27,271.1140,-79.77,1.58,0.8801,80,4,16
3,V3_Dcons_ge_0p2,2141,81.68,270.8055,-79.04,0.99,0.5716,78,4,18
4,V3_Dcons_ge_0p4,1959,82.80,271.4501,-80.86,2.11,1.2162,81,3,16



Help / hurt relative to B0:


,operating_point,improved_images,unchanged_images,worsened_images
0,V5_top95,4,67,29
1,V5_top90,9,39,52
2,V3_Dcons_ge_0p2,7,45,48
3,V3_Dcons_ge_0p4,11,29,60



By density stratum:


,density_stratum,operating_point,num_images,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,S1_0_20,B0_unfiltered,20,5.15,6.1115,-0.55,10,1,9
1,S1_0_20,V5_top95,20,5.15,6.1441,-1.25,10,1,9
2,S1_0_20,V5_top90,20,4.85,6.1847,-2.75,12,1,7
3,S1_0_20,V3_Dcons_ge_0p2,20,5.00,6.2610,-1.90,10,2,8
4,S1_0_20,V3_Dcons_ge_0p4,20,4.65,6.1196,-2.75,11,1,8
5,S2_20_40,B0_unfiltered,20,6.55,8.1086,-5.55,16,2,2
6,S2_20_40,V5_top95,20,7.65,8.9191,-6.75,17,1,2
7,S2_20_40,V5_top90,20,8.30,9.5760,-7.50,17,1,2
8,S2_20_40,V3_Dcons_ge_0p2,20,7.95,9.2168,-7.15,17,1,2
9,S2_20_40,V3_Dcons_ge_0p4,20,8.20,9.4446,-7.50,17,1,2



Saved per-image count results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_count_results_per_image.csv

Saved overall count summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_count_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_validator_count_summary_by_density.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_validator_step_2D3_manifest.json


In [19]:
# ============================================================
# Step 3A - B0 Coverage of Semantic Points
#
# Purpose:
# For selected semantic sources, classify each semantic point as:
#
#   1. covered_by_B0
#      -> point lies inside >=1 frozen B0 mask
#
#   2. recovery_candidate
#      -> point lies outside all frozen B0 masks
#
# IMPORTANT:
# - no recovery counts added yet
# - no SAM2 prompting
# - no threshold tuning
# - no candidate selection
# - no test-set access
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3A.1 Selected recovery-source candidates
# ------------------------------------------------------------

RECOVERY_SOURCES = [
    ("C", "C1"),
    ("D", "D6B9-strict"),
    ("D", "D6B-T-native"),
]


# ------------------------------------------------------------
# 3A.2 Output files
# ------------------------------------------------------------

RECOVERY_POINT_LONG_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_semantic_points_long.csv"
)

RECOVERY_POINT_CANDIDATE_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_candidate_summary.csv"
)

RECOVERY_POINT_IMAGE_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_candidate_summary_per_image.csv"
)

RECOVERY_POINT_DENSITY_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_candidate_summary_by_density.csv"
)

RECOVERY_POINT_3A_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3A_manifest.json"
)


# ------------------------------------------------------------
# 3A.3 Preconditions
# ------------------------------------------------------------

assert (
    "b0_semantic_long_df"
    in globals()
), (
    "b0_semantic_long_df not found. "
    "Run Step 2B first."
)

assert (
    "b0_mask_cache_index_df"
    in globals()
), (
    "b0_mask_cache_index_df not found. "
    "Run Step 1 first."
)

assert (
    "load_b0_masks_from_cache"
    in globals()
), (
    "load_b0_masks_from_cache helper not found. "
    "Run Step 2A first."
)

assert (
    "parse_xy_list"
    in globals()
), (
    "parse_xy_list helper not found."
)


# ------------------------------------------------------------
# 3A.3b Reload authoritative C/D candidate tables
# ------------------------------------------------------------

C_CANDIDATE_FILE = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "Group_C"
    / "B1_Group_C_E_candidates_per_image.csv"
)

D_CANDIDATE_FILE = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "Group_D"
    / "B1_Group_D_E_candidates_per_image.csv"
)


assert C_CANDIDATE_FILE.exists(), C_CANDIDATE_FILE
assert D_CANDIDATE_FILE.exists(), D_CANDIDATE_FILE


C_candidates_df = pd.read_csv(
    C_CANDIDATE_FILE
)

D_candidates_df = pd.read_csv(
    D_CANDIDATE_FILE
)


print(
    "C candidate rows:",
    len(C_candidates_df)
)

print(
    "D candidate rows:",
    len(D_candidates_df)
)

print(
    "C candidates:",
    sorted(
        C_candidates_df[
            "candidate_id"
        ].unique()
    )
)

print(
    "D candidates:",
    sorted(
        D_candidates_df[
            "candidate_id"
        ].unique()
    )
)


# ------------------------------------------------------------
# 3A.4 Build authoritative semantic-point source table
#
# We use the original candidate prediction rows, not the
# mask-level long table, because Step 3A is point-centric.
# ------------------------------------------------------------

semantic_source_rows = []


for branch, candidate_id in RECOVERY_SOURCES:

    source_df = (
        C_candidates_df
        if branch == "C"
        else D_candidates_df
    )

    candidate_rows = (
        source_df[
            source_df[
                "candidate_id"
            ]
            ==
            candidate_id
        ]
        .copy()
    )


    assert (
        len(
            candidate_rows
        )
        == 100
    ), (
        f"{candidate_id}: expected 100 image rows, "
        f"got {len(candidate_rows)}"
    )


    for row in candidate_rows.itertuples(
        index=False
    ):

        points_xy = parse_xy_list(
            row.pred_points_xy
        )


        semantic_source_rows.append(
            {
                "branch":
                    branch,

                "candidate_id":
                    candidate_id,

                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    row.image_id,

                "points_xy":
                    points_xy,

                "candidate_pred_count":
                    int(
                        len(
                            points_xy
                        )
                    ),
            }
        )


semantic_source_df = pd.DataFrame(
    semantic_source_rows
)


assert (
    len(
        semantic_source_df
    )
    ==
    300
)


# ------------------------------------------------------------
# 3A.5 Attach authoritative image metadata
# ------------------------------------------------------------

semantic_source_df = (
    semantic_source_df
    .merge(
        b2_dev100_image_df[
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
            ]
        ],
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    semantic_source_df[
        "gt_count"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 3A.6 Build B0 cache lookup
# ------------------------------------------------------------

b0_cache_lookup = (
    b0_mask_cache_index_df[
        [
            "dataset_index",
            "image_id",
            "cache_file",
        ]
    ]
    .set_index(
        "dataset_index"
    )
)


assert (
    len(
        b0_cache_lookup
    )
    ==
    100
)


# ------------------------------------------------------------
# 3A.7 Point membership helper
#
# Match Step 2A convention:
# np.rint + clip -> mask pixel.
# ------------------------------------------------------------

def point_inside_mask(
    x,
    y,
    mask,
):

    h, w = mask.shape

    px = int(
        np.clip(
            np.rint(
                x
            ),
            0,
            w - 1,
        )
    )

    py = int(
        np.clip(
            np.rint(
                y
            ),
            0,
            h - 1,
        )
    )

    return bool(
        mask[
            py,
            px,
        ]
    )


# ------------------------------------------------------------
# 3A.8 Classify every semantic point against frozen B0 masks
# ------------------------------------------------------------

point_records = []


for source_row in semantic_source_df.itertuples(
    index=False
):

    dataset_index = int(
        source_row.dataset_index
    )

    image_id = source_row.image_id


    cache_file = Path(
        b0_cache_lookup.loc[
            dataset_index,
            "cache_file",
        ]
    )


    masks_bundle = load_b0_masks_from_cache(
        cache_file
    )


    masks = masks_bundle[
        "masks"
    ]


    num_b0_masks = int(
        len(
            masks
        )
    )


    points_xy = source_row.points_xy


    for point_index, (
        x,
        y
    ) in enumerate(
        points_xy
    ):

        covering_mask_indices = []


        for mask_index, mask in enumerate(
            masks
        ):

            if point_inside_mask(
                x=x,
                y=y,
                mask=mask,
            ):

                covering_mask_indices.append(
                    int(
                        mask_index
                    )
                )


        num_covering_masks = int(
            len(
                covering_mask_indices
            )
        )


        covered_by_b0 = (
            num_covering_masks
            >
            0
        )


        recovery_candidate = (
            not covered_by_b0
        )


        point_records.append(
            {
                "branch":
                    source_row.branch,

                "candidate_id":
                    source_row.candidate_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "category":
                    source_row.category,

                "density_stratum":
                    source_row.density_stratum,

                "gt_count":
                    int(
                        source_row.gt_count
                    ),

                "candidate_pred_count":
                    int(
                        source_row.candidate_pred_count
                    ),

                "point_index":
                    int(
                        point_index
                    ),

                "x":
                    float(
                        x
                    ),

                "y":
                    float(
                        y
                    ),

                "num_b0_masks_in_image":
                    num_b0_masks,

                "num_covering_b0_masks":
                    num_covering_masks,

                "covered_by_b0":
                    bool(
                        covered_by_b0
                    ),

                "recovery_candidate":
                    bool(
                        recovery_candidate
                    ),

                "covering_b0_mask_indices":
                    covering_mask_indices,
            }
        )


recovery_point_long_df = pd.DataFrame(
    point_records
)


# ------------------------------------------------------------
# 3A.9 Integrity checks
# ------------------------------------------------------------

expected_total_points = int(
    semantic_source_df[
        "candidate_pred_count"
    ].sum()
)


assert (
    len(
        recovery_point_long_df
    )
    ==
    expected_total_points
)


assert (
    (
        recovery_point_long_df[
            "covered_by_b0"
        ]
        ^
        recovery_point_long_df[
            "recovery_candidate"
        ]
    )
    .all()
)


assert (
    (
        recovery_point_long_df[
            "num_covering_b0_masks"
        ]
        >
        0
    )
    .eq(
        recovery_point_long_df[
            "covered_by_b0"
        ]
    )
    .all()
)


# ------------------------------------------------------------
# 3A.10 Candidate-level summary
# ------------------------------------------------------------

candidate_summary_df = (
    recovery_point_long_df
    .groupby(
        [
            "branch",
            "candidate_id",
        ],
        as_index=False,
    )
    .agg(
        total_semantic_points=(
            "point_index",
            "size",
        ),

        covered_points=(
            "covered_by_b0",
            "sum",
        ),

        recovery_candidate_points=(
            "recovery_candidate",
            "sum",
        ),

        mean_num_covering_b0_masks=(
            "num_covering_b0_masks",
            "mean",
        ),
    )
)


candidate_summary_df[
    "covered_fraction"
] = (
    candidate_summary_df[
        "covered_points"
    ]
    /
    candidate_summary_df[
        "total_semantic_points"
    ]
)


candidate_summary_df[
    "recovery_candidate_fraction"
] = (
    candidate_summary_df[
        "recovery_candidate_points"
    ]
    /
    candidate_summary_df[
        "total_semantic_points"
    ]
)


# ------------------------------------------------------------
# 3A.11 Per-image summary
#
# Start from all 100 images × 3 sources so that images with
# zero candidate points remain explicitly represented.
# ------------------------------------------------------------

image_grid_records = []


for image_row in b2_dev100_image_df.itertuples(
    index=False
):

    for branch, candidate_id in RECOVERY_SOURCES:

        image_grid_records.append(
            {
                "dataset_index":
                    int(
                        image_row.dataset_index
                    ),

                "image_id":
                    image_row.image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "gt_count":
                    int(
                        image_row.gt_count
                    ),

                "branch":
                    branch,

                "candidate_id":
                    candidate_id,
            }
        )


image_grid_df = pd.DataFrame(
    image_grid_records
)


assert (
    len(
        image_grid_df
    )
    ==
    300
)


point_image_summary_df = (
    recovery_point_long_df
    .groupby(
        [
            "dataset_index",
            "image_id",
            "branch",
            "candidate_id",
        ],
        as_index=False,
    )
    .agg(
        total_semantic_points=(
            "point_index",
            "size",
        ),

        covered_points=(
            "covered_by_b0",
            "sum",
        ),

        recovery_candidate_points=(
            "recovery_candidate",
            "sum",
        ),
    )
)


recovery_point_image_df = (
    image_grid_df
    .merge(
        point_image_summary_df,
        on=[
            "dataset_index",
            "image_id",
            "branch",
            "candidate_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


COUNT_COLS = [
    "total_semantic_points",
    "covered_points",
    "recovery_candidate_points",
]


recovery_point_image_df[
    COUNT_COLS
] = (
    recovery_point_image_df[
        COUNT_COLS
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


recovery_point_image_df[
    "covered_fraction"
] = np.where(
    recovery_point_image_df[
        "total_semantic_points"
    ]
    >
    0,

    recovery_point_image_df[
        "covered_points"
    ]
    /
    recovery_point_image_df[
        "total_semantic_points"
    ],

    np.nan,
)


recovery_point_image_df[
    "recovery_candidate_fraction"
] = np.where(
    recovery_point_image_df[
        "total_semantic_points"
    ]
    >
    0,

    recovery_point_image_df[
        "recovery_candidate_points"
    ]
    /
    recovery_point_image_df[
        "total_semantic_points"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3A.12 Density-stratum summary
#
# Diagnostic only.
# No density-based inference routing.
# ------------------------------------------------------------

density_summary_df = (
    recovery_point_image_df
    .groupby(
        [
            "density_stratum",
            "branch",
            "candidate_id",
        ],
        as_index=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        total_semantic_points=(
            "total_semantic_points",
            "sum",
        ),

        covered_points=(
            "covered_points",
            "sum",
        ),

        recovery_candidate_points=(
            "recovery_candidate_points",
            "sum",
        ),
    )
)


density_summary_df[
    "covered_fraction"
] = np.where(
    density_summary_df[
        "total_semantic_points"
    ]
    >
    0,

    density_summary_df[
        "covered_points"
    ]
    /
    density_summary_df[
        "total_semantic_points"
    ],

    np.nan,
)


density_summary_df[
    "recovery_candidate_fraction"
] = np.where(
    density_summary_df[
        "total_semantic_points"
    ]
    >
    0,

    density_summary_df[
        "recovery_candidate_points"
    ]
    /
    density_summary_df[
        "total_semantic_points"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3A.13 Extra diagnostic:
# points covered by multiple B0 masks
# ------------------------------------------------------------

multi_cover_summary_df = (
    recovery_point_long_df
    .groupby(
        [
            "branch",
            "candidate_id",
        ],
        as_index=False,
    )
    .agg(
        points_covered_by_multiple_B0_masks=(
            "num_covering_b0_masks",
            lambda s: int(
                (
                    s
                    >
                    1
                ).sum()
            ),
        )
    )
)


candidate_summary_df = (
    candidate_summary_df
    .merge(
        multi_cover_summary_df,
        on=[
            "branch",
            "candidate_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


candidate_summary_df[
    "multi_cover_fraction"
] = (
    candidate_summary_df[
        "points_covered_by_multiple_B0_masks"
    ]
    /
    candidate_summary_df[
        "total_semantic_points"
    ]
)


# ------------------------------------------------------------
# 3A.14 Save outputs
# ------------------------------------------------------------

recovery_point_long_df.to_csv(
    RECOVERY_POINT_LONG_FILE,
    index=False,
)


candidate_summary_df.to_csv(
    RECOVERY_POINT_CANDIDATE_SUMMARY_FILE,
    index=False,
)


recovery_point_image_df.to_csv(
    RECOVERY_POINT_IMAGE_SUMMARY_FILE,
    index=False,
)


density_summary_df.to_csv(
    RECOVERY_POINT_DENSITY_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3A.15 Manifest
# ------------------------------------------------------------

step_3a_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3A - B0 coverage of semantic recovery points",

    "recovery_sources":
        [
            {
                "branch":
                    branch,

                "candidate_id":
                    candidate_id,
            }

            for branch, candidate_id
            in RECOVERY_SOURCES
        ],

    "recovery_candidate_definition":
        (
            "semantic point lies outside every frozen B0 mask"
        ),

    "point_membership_rule":
        "np.rint + clip -> mask pixel",

    "semantic_points_added_to_count":
        False,

    "sam2_reprompting":
        False,

    "candidate_selection":
        False,

    "threshold_tuning":
        False,

    "density_used_for_routing":
        False,

    "test_set_accessed":
        False,
}


with open(
    RECOVERY_POINT_3A_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_3a_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3A.16 Final output
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3A - B0 COVERAGE OF SEMANTIC RECOVERY POINTS")
print("=" * 125)

print(
    "Recovery sources            :",
    len(
        RECOVERY_SOURCES
    )
)

print(
    "Source-image rows           :",
    len(
        semantic_source_df
    ),
    "/ 300"
)

print(
    "Total semantic points       :",
    len(
        recovery_point_long_df
    )
)

print(
    "\nSemantic points added       : NO"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Candidate selected          : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nCandidate-level B0 coverage:"
)

display(
    candidate_summary_df[
        [
            "branch",
            "candidate_id",
            "total_semantic_points",
            "covered_points",
            "recovery_candidate_points",
            "covered_fraction",
            "recovery_candidate_fraction",
            "points_covered_by_multiple_B0_masks",
            "multi_cover_fraction",
            "mean_num_covering_b0_masks",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nRecovery candidates by density stratum:"
)

display(
    density_summary_df[
        [
            "density_stratum",
            "branch",
            "candidate_id",
            "num_images",
            "total_semantic_points",
            "covered_points",
            "recovery_candidate_points",
            "covered_fraction",
            "recovery_candidate_fraction",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nImages with largest number of recovery candidates:"
)

display(
    recovery_point_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "branch",
            "candidate_id",
            "total_semantic_points",
            "covered_points",
            "recovery_candidate_points",
            "recovery_candidate_fraction",
        ]
    ]
    .sort_values(
        [
            "recovery_candidate_points",
            "total_semantic_points",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        30
    )
    .round(
        4
    )
)


print(
    "\nSaved semantic-point table:"
)

print(
    RECOVERY_POINT_LONG_FILE
)

print(
    "\nSaved candidate summary:"
)

print(
    RECOVERY_POINT_CANDIDATE_SUMMARY_FILE
)

print(
    "\nSaved per-image summary:"
)

print(
    RECOVERY_POINT_IMAGE_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    RECOVERY_POINT_DENSITY_SUMMARY_FILE
)

print(
    "\nSaved manifest:"
)

print(
    RECOVERY_POINT_3A_MANIFEST_FILE
)

print("=" * 125)

C candidate rows: 300
D candidate rows: 500
C candidates: ['C1', 'C2b', 'C2c']
D candidates: ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']
STEP 3A - B0 COVERAGE OF SEMANTIC RECOVERY POINTS
Recovery sources            : 3
Source-image rows           : 300 / 300
Total semantic points       : 23018

Semantic points added       : NO
SAM2 re-prompting           : NO
Candidate selected          : NO
Threshold tuning            : NO
Density used for routing    : NO
Test set accessed           : NO

Candidate-level B0 coverage:


,branch,candidate_id,total_semantic_points,covered_points,recovery_candidate_points,covered_fraction,recovery_candidate_fraction,points_covered_by_multiple_B0_masks,multi_cover_fraction,mean_num_covering_b0_masks
0,C,C1,2554,1339,1215,0.5243,0.4757,177,0.0693,0.5991
1,D,D6B-T-native,15913,7866,8047,0.4943,0.5057,1059,0.0665,0.5705
2,D,D6B9-strict,4551,2640,1911,0.5801,0.4199,387,0.0850,0.6794



Recovery candidates by density stratum:


,density_stratum,branch,candidate_id,num_images,total_semantic_points,covered_points,recovery_candidate_points,covered_fraction,recovery_candidate_fraction
0,S1_0_20,C,C1,20,240,148,92,0.6167,0.3833
1,S1_0_20,D,D6B-T-native,20,2024,1083,941,0.5351,0.4649
2,S1_0_20,D,D6B9-strict,20,529,342,187,0.6465,0.3535
3,S2_20_40,C,C1,20,392,209,183,0.5332,0.4668
4,S2_20_40,D,D6B-T-native,20,2850,1278,1572,0.4484,0.5516
5,S2_20_40,D,D6B9-strict,20,852,402,450,0.4718,0.5282
6,S3_40_60,C,C1,20,401,203,198,0.5062,0.4938
7,S3_40_60,D,D6B-T-native,20,2972,1396,1576,0.4697,0.5303
8,S3_40_60,D,D6B9-strict,20,1017,521,496,0.5123,0.4877
9,S4_60_80,C,C1,20,566,351,215,0.6201,0.3799



Images with largest number of recovery candidates:


,dataset_index,image_id,category,density_stratum,gt_count,branch,candidate_id,total_semantic_points,covered_points,recovery_candidate_points,recovery_candidate_fraction
233,938,5866.jpg,grapes,S6_95_100,267,D,D6B-T-native,579,81,498,0.8601
5,14,220.jpg,grapes,S4_60_80,59,D,D6B-T-native,387,25,362,0.9354
74,183,838.jpg,books,S5_80_95,113,D,D6B-T-native,395,134,261,0.6608
131,396,2846.jpg,chairs,S4_60_80,38,D,D6B-T-native,281,21,260,0.9253
77,184,840.jpg,books,S6_95_100,637,D,D6B-T-native,260,3,257,0.9885
50,148,769.jpg,books,S2_20_40,13,D,D6B-T-native,224,7,217,0.9688
296,1260,7100.jpg,milk cartons,S3_40_60,27,D,D6B-T-native,443,227,216,0.4876
299,1269,7060.jpg,pills,S4_60_80,65,D,D6B-T-native,287,89,198,0.6899
137,425,2884.jpg,chairs,S5_80_95,165,D,D6B-T-native,299,103,196,0.6555
8,20,228.jpg,grapes,S4_60_80,44,D,D6B-T-native,203,12,191,0.9409



Saved semantic-point table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_semantic_points_long.csv

Saved candidate summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_candidate_summary.csv

Saved per-image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_candidate_summary_per_image.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_candidate_summary_by_density.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_3A_manifest.json


In [23]:
# ============================================================
# Verify exact Step 2A GT coverage totals
# ============================================================

step2a_total_gt = int(
    b0_mask_gt_image_df[
        "gt_count"
    ].sum()
)

step2a_uncovered_gt = int(
    b0_mask_gt_image_df[
        "gt_uncovered"
    ].sum()
)

step2a_covered_once = int(
    b0_mask_gt_image_df[
        "gt_covered_once"
    ].sum()
)

step2a_duplicate_covered = int(
    b0_mask_gt_image_df[
        "gt_duplicate_covered"
    ].sum()
)

step2a_unique_covered = (
    step2a_covered_once
    +
    step2a_duplicate_covered
)


print("=" * 90)
print("FROZEN STEP 2A GT COVERAGE TOTALS")
print("=" * 90)

print("Total GT points        :", step2a_total_gt)
print("GT uncovered           :", step2a_uncovered_gt)
print("GT covered exactly once:", step2a_covered_once)
print("GT duplicate-covered   :", step2a_duplicate_covered)
print("Unique GT covered      :", step2a_unique_covered)

print(
    "\nIdentity check         :",
    step2a_unique_covered
    +
    step2a_uncovered_gt
)

print(
    "Coverage rate          :",
    step2a_unique_covered
    /
    step2a_total_gt
)

print("=" * 90)

FROZEN STEP 2A GT COVERAGE TOTALS
Total GT points        : 10045
GT uncovered           : 5641
GT covered exactly once: 4142
GT duplicate-covered   : 262
Unique GT covered      : 4404

Identity check         : 10045
Coverage rate          : 0.4384270781483325


In [24]:
# ============================================================
# Step 3B - GT Quality of B0 Recovery Candidates
#
# Purpose:
# Evaluate semantic points that lie outside all frozen B0 masks.
#
# A recovery TP requires:
#   1. semantic point is outside every B0 mask; AND
#   2. it one-to-one matches a GT point that is itself NOT
#      covered by any B0 mask; AND
#   3. match distance <= the same frozen matching radius R
#      used in the earlier C/D diagnostics.
#
# IMPORTANT:
# - GT used diagnostically only
# - no points added to final count
# - no SAM2 re-prompting
# - no recovery rule selected
# - no threshold tuning
# - no test-set access
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3B.1 Output files
# ------------------------------------------------------------

RECOVERY_GT_MATCH_LONG_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_GT_matching_long.csv"
)

RECOVERY_GT_SOURCE_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_GT_matching_summary.csv"
)

RECOVERY_GT_IMAGE_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_GT_matching_per_image.csv"
)

RECOVERY_GT_DENSITY_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_GT_matching_by_density.csv"
)

RECOVERY_GT_3B_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3B_manifest.json"
)


# ------------------------------------------------------------
# 3B.2 Preconditions
# ------------------------------------------------------------

assert (
    "recovery_point_long_df"
    in globals()
), (
    "recovery_point_long_df not found. "
    "Run Step 3A first."
)

assert (
    "b2_dev100_image_df"
    in globals()
)

assert (
    "b0_mask_cache_index_df"
    in globals()
)

assert (
    "b0_mask_gt_image_df"
    in globals()
), (
    "b0_mask_gt_image_df not found. "
    "Run Step 2A first."
)

assert (
    "load_b0_masks_from_cache"
    in globals()
)

assert (
    "parse_xy_list"
    in globals()
)

assert (
    len(
        recovery_point_long_df
    )
    ==
    23018
)


# ------------------------------------------------------------
# 3B.3 Frozen Step 2A GT coverage references
#
# Derive these from the frozen Step 2A table.
# Do NOT hard-code them.
# ------------------------------------------------------------

STEP2A_TOTAL_GT = int(
    b0_mask_gt_image_df[
        "gt_count"
    ].sum()
)

STEP2A_B0_MISSED_GT = int(
    b0_mask_gt_image_df[
        "gt_uncovered"
    ].sum()
)

STEP2A_B0_COVERED_GT = int(
    (
        b0_mask_gt_image_df[
            "gt_covered_once"
        ]
        +
        b0_mask_gt_image_df[
            "gt_duplicate_covered"
        ]
    ).sum()
)


assert (
    STEP2A_TOTAL_GT
    ==
    (
        STEP2A_B0_COVERED_GT
        +
        STEP2A_B0_MISSED_GT
    )
)


# ------------------------------------------------------------
# 3B.4 Resolve authoritative GT-point and matching-radius fields
#
# IMPORTANT:
# We do NOT create a new matching-radius definition here.
# We reuse the frozen fields already present in existing
# authoritative candidate tables.
# ------------------------------------------------------------

SEARCH_TABLES = {
    "b2_dev100_image_df":
        b2_dev100_image_df,

    "C_candidates_df":
        C_candidates_df,

    "D_candidates_df":
        D_candidates_df,
}


GT_POINT_COLUMN_ALIASES = [
    "gt_points_xy",
    "gt_points",
    "points_xy_gt",
    "annotation_points_xy",
    "ann_points_xy",
]


MATCH_RADIUS_COLUMN_ALIASES = [
    "matching_radius",
    "match_radius",
    "matching_radius_px",
    "match_radius_px",
    "radius_px",
]


def find_existing_column(
    tables,
    aliases,
    label,
):

    matches = []

    for table_name, df in tables.items():

        for col in aliases:

            if col in df.columns:

                matches.append(
                    (
                        table_name,
                        col,
                    )
                )


    if len(
        matches
    ) == 0:

        print(
            f"\nCould not find existing {label}."
        )

        print(
            "\nAvailable potentially relevant columns:"
        )

        for table_name, df in tables.items():

            relevant = [
                c
                for c in df.columns
                if (
                    "point" in c.lower()
                    or
                    "radius" in c.lower()
                    or
                    "match" in c.lower()
                )
            ]

            print(
                f"\n{table_name}:"
            )

            print(
                relevant
            )

        raise RuntimeError(
            f"Frozen {label} could not be resolved. "
            "Do not invent a replacement."
        )


    priority = {
        "b2_dev100_image_df":
            0,

        "C_candidates_df":
            1,

        "D_candidates_df":
            2,
    }


    matches = sorted(
        matches,
        key=lambda x:
            priority[
                x[
                    0
                ]
            ],
    )


    return matches[
        0
    ]


GT_SOURCE_TABLE_NAME, GT_POINT_COL = (
    find_existing_column(
        SEARCH_TABLES,
        GT_POINT_COLUMN_ALIASES,
        "GT-point column",
    )
)


R_SOURCE_TABLE_NAME, MATCH_RADIUS_COL = (
    find_existing_column(
        SEARCH_TABLES,
        MATCH_RADIUS_COLUMN_ALIASES,
        "matching-radius column",
    )
)


GT_SOURCE_DF = SEARCH_TABLES[
    GT_SOURCE_TABLE_NAME
]

R_SOURCE_DF = SEARCH_TABLES[
    R_SOURCE_TABLE_NAME
]


print(
    "Resolved GT points from    :",
    GT_SOURCE_TABLE_NAME,
    "/",
    GT_POINT_COL,
)

print(
    "Resolved frozen radius from:",
    R_SOURCE_TABLE_NAME,
    "/",
    MATCH_RADIUS_COL,
)


# ------------------------------------------------------------
# 3B.5 Build authoritative per-image GT/radius table
# ------------------------------------------------------------

gt_image_df = (
    GT_SOURCE_DF[
        [
            "dataset_index",
            "image_id",
            GT_POINT_COL,
        ]
    ]
    .drop_duplicates(
        subset=[
            "dataset_index",
            "image_id",
        ]
    )
    .copy()
)


radius_image_df = (
    R_SOURCE_DF[
        [
            "dataset_index",
            "image_id",
            MATCH_RADIUS_COL,
        ]
    ]
    .drop_duplicates(
        subset=[
            "dataset_index",
            "image_id",
        ]
    )
    .copy()
)


gt_image_df = (
    b2_dev100_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .merge(
        gt_image_df,
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        radius_image_df,
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    len(
        gt_image_df
    )
    == 100
)

assert (
    gt_image_df[
        GT_POINT_COL
    ]
    .notna()
    .all()
)

assert (
    gt_image_df[
        MATCH_RADIUS_COL
    ]
    .notna()
    .all()
)


gt_image_df[
    "gt_points_xy"
] = (
    gt_image_df[
        GT_POINT_COL
    ]
    .apply(
        parse_xy_list
    )
)


gt_image_df[
    "matching_radius_px"
] = (
    gt_image_df[
        MATCH_RADIUS_COL
    ]
    .astype(
        float
    )
)


# GT-count identity

assert (
    gt_image_df.apply(
        lambda r:
        len(
            r[
                "gt_points_xy"
            ]
        )
        ==
        int(
            r[
                "gt_count"
            ]
        ),
        axis=1,
    )
    .all()
)


# ------------------------------------------------------------
# 3B.6 B0 cache lookup
# ------------------------------------------------------------

b0_cache_lookup = (
    b0_mask_cache_index_df[
        [
            "dataset_index",
            "image_id",
            "cache_file",
        ]
    ]
    .set_index(
        "dataset_index"
    )
)


# ------------------------------------------------------------
# 3B.7 Same point->pixel convention as Step 2A / 3A
# ------------------------------------------------------------

def point_inside_any_mask(
    x,
    y,
    masks,
):

    if len(
        masks
    ) == 0:

        return False


    h, w = masks[
        0
    ].shape


    px = int(
        np.clip(
            np.rint(
                x
            ),
            0,
            w - 1,
        )
    )

    py = int(
        np.clip(
            np.rint(
                y
            ),
            0,
            h - 1,
        )
    )


    for mask in masks:

        if bool(
            mask[
                py,
                px,
            ]
        ):

            return True


    return False


# ------------------------------------------------------------
# 3B.8 Identify GT points already covered / missed by B0
# ------------------------------------------------------------

missed_gt_records = []

gt_coverage_image_records = []


for row in gt_image_df.itertuples(
    index=False
):

    dataset_index = int(
        row.dataset_index
    )


    cache_file = Path(
        b0_cache_lookup.loc[
            dataset_index,
            "cache_file",
        ]
    )


    masks_bundle = load_b0_masks_from_cache(
        cache_file
    )


    masks = masks_bundle[
        "masks"
    ]


    gt_points = row.gt_points_xy


    num_covered_gt = 0
    num_missed_gt = 0


    for gt_index, (
        gx,
        gy
    ) in enumerate(
        gt_points
    ):

        covered = point_inside_any_mask(
            x=gx,
            y=gy,
            masks=masks,
        )


        missed = (
            not covered
        )


        if covered:

            num_covered_gt += 1

        else:

            num_missed_gt += 1


        missed_gt_records.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    row.image_id,

                "category":
                    row.category,

                "density_stratum":
                    row.density_stratum,

                "gt_count":
                    int(
                        row.gt_count
                    ),

                "gt_index":
                    int(
                        gt_index
                    ),

                "gt_x":
                    float(
                        gx
                    ),

                "gt_y":
                    float(
                        gy
                    ),

                "covered_by_B0":
                    bool(
                        covered
                    ),

                "missed_by_B0":
                    bool(
                        missed
                    ),

                "matching_radius_px":
                    float(
                        row.matching_radius_px
                    ),
            }
        )


    gt_coverage_image_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                row.image_id,

            "gt_count":
                int(
                    row.gt_count
                ),

            "b0_covered_gt":
                int(
                    num_covered_gt
                ),

            "b0_missed_gt":
                int(
                    num_missed_gt
                ),
        }
    )


gt_point_status_df = pd.DataFrame(
    missed_gt_records
)


gt_coverage_image_df = pd.DataFrame(
    gt_coverage_image_records
)


assert (
    len(
        gt_point_status_df
    )
    ==
    STEP2A_TOTAL_GT
)


# ------------------------------------------------------------
# 3B.9 Reproduce frozen Step 2A GT coverage exactly
# ------------------------------------------------------------

STEP3B_B0_COVERED_GT = int(
    gt_point_status_df[
        "covered_by_B0"
    ].sum()
)

STEP3B_B0_MISSED_GT = int(
    gt_point_status_df[
        "missed_by_B0"
    ].sum()
)


assert (
    STEP3B_B0_COVERED_GT
    ==
    STEP2A_B0_COVERED_GT
), (
    "Step 3B does not reproduce frozen Step 2A "
    "GT-covered total."
)


assert (
    STEP3B_B0_MISSED_GT
    ==
    STEP2A_B0_MISSED_GT
), (
    "Step 3B does not reproduce frozen Step 2A "
    "GT-missed total."
)


# ------------------------------------------------------------
# 3B.10 One-to-one spatial matching helper
#
# Matching:
#   recovery semantic point <-> B0-missed GT point
#
# Candidate pair allowed only when distance <= frozen R.
#
# Greedy globally shortest valid pair matching:
# - no prediction receives multiple TP credit
# - no GT receives multiple TP credit
# ------------------------------------------------------------

def one_to_one_radius_match(
    pred_xy,
    gt_xy,
    radius_px,
):

    pred_xy = np.asarray(
        pred_xy,
        dtype=float,
    )

    gt_xy = np.asarray(
        gt_xy,
        dtype=float,
    )


    num_pred = int(
        len(
            pred_xy
        )
    )

    num_gt = int(
        len(
            gt_xy
        )
    )


    if (
        num_pred == 0
        or
        num_gt == 0
    ):

        return []


    candidate_pairs = []


    for pred_i in range(
        num_pred
    ):

        px, py = pred_xy[
            pred_i
        ]


        for gt_i in range(
            num_gt
        ):

            gx, gy = gt_xy[
                gt_i
            ]


            distance = float(
                np.hypot(
                    px - gx,
                    py - gy,
                )
            )


            if distance <= radius_px:

                candidate_pairs.append(
                    (
                        distance,
                        pred_i,
                        gt_i,
                    )
                )


    candidate_pairs.sort(
        key=lambda z: (
            z[
                0
            ],
            z[
                1
            ],
            z[
                2
            ],
        )
    )


    used_pred = set()
    used_gt = set()

    matches = []


    for (
        distance,
        pred_i,
        gt_i,
    ) in candidate_pairs:

        if pred_i in used_pred:

            continue

        if gt_i in used_gt:

            continue


        used_pred.add(
            pred_i
        )

        used_gt.add(
            gt_i
        )


        matches.append(
            {
                "pred_local_index":
                    int(
                        pred_i
                    ),

                "gt_local_index":
                    int(
                        gt_i
                    ),

                "distance_px":
                    float(
                        distance
                    ),
            }
        )


    return matches


# ------------------------------------------------------------
# 3B.11 Match each recovery source image-by-image
# ------------------------------------------------------------

match_point_records = []

source_image_records = []


for (
    branch,
    candidate_id
) in RECOVERY_SOURCES:


    for image_row in gt_image_df.itertuples(
        index=False
    ):

        dataset_index = int(
            image_row.dataset_index
        )


        # ----------------------------------------------------
        # Step 3A recovery points for this source/image
        # ----------------------------------------------------

        recovery_subset = (
            recovery_point_long_df[
                (
                    recovery_point_long_df[
                        "branch"
                    ]
                    ==
                    branch
                )
                &
                (
                    recovery_point_long_df[
                        "candidate_id"
                    ]
                    ==
                    candidate_id
                )
                &
                (
                    recovery_point_long_df[
                        "dataset_index"
                    ]
                    ==
                    dataset_index
                )
                &
                (
                    recovery_point_long_df[
                        "recovery_candidate"
                    ]
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # GT instances genuinely missed by B0
        # ----------------------------------------------------

        missed_gt_subset = (
            gt_point_status_df[
                (
                    gt_point_status_df[
                        "dataset_index"
                    ]
                    ==
                    dataset_index
                )
                &
                (
                    gt_point_status_df[
                        "missed_by_B0"
                    ]
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        pred_xy = (
            recovery_subset[
                [
                    "x",
                    "y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


        missed_gt_xy = (
            missed_gt_subset[
                [
                    "gt_x",
                    "gt_y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


        radius_px = float(
            image_row.matching_radius_px
        )


        matches = one_to_one_radius_match(
            pred_xy=pred_xy,
            gt_xy=missed_gt_xy,
            radius_px=radius_px,
        )


        matched_pred_local_indices = {
            m[
                "pred_local_index"
            ]
            for m in matches
        }


        match_by_pred = {
            m[
                "pred_local_index"
            ]:
            m

            for m in matches
        }


        # ----------------------------------------------------
        # Store every recovery point
        # ----------------------------------------------------

        for pred_local_index, pred_row in (
            recovery_subset
            .iterrows()
        ):

            matched = (
                pred_local_index
                in matched_pred_local_indices
            )


            if matched:

                match_info = match_by_pred[
                    pred_local_index
                ]

                gt_local_index = (
                    match_info[
                        "gt_local_index"
                    ]
                )

                matched_gt_row = (
                    missed_gt_subset.iloc[
                        gt_local_index
                    ]
                )

                matched_gt_index = int(
                    matched_gt_row[
                        "gt_index"
                    ]
                )

                match_distance_px = float(
                    match_info[
                        "distance_px"
                    ]
                )


            else:

                matched_gt_index = np.nan
                match_distance_px = np.nan


            match_point_records.append(
                {
                    "branch":
                        branch,

                    "candidate_id":
                        candidate_id,

                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_row.image_id,

                    "category":
                        image_row.category,

                    "density_stratum":
                        image_row.density_stratum,

                    "gt_count":
                        int(
                            image_row.gt_count
                        ),

                    "b0_missed_gt_count":
                        int(
                            len(
                                missed_gt_subset
                            )
                        ),

                    "matching_radius_px":
                        radius_px,

                    "point_index":
                        int(
                            pred_row[
                                "point_index"
                            ]
                        ),

                    "x":
                        float(
                            pred_row[
                                "x"
                            ]
                        ),

                    "y":
                        float(
                            pred_row[
                                "y"
                            ]
                        ),

                    "matched_missed_gt":
                        bool(
                            matched
                        ),

                    "matched_gt_index":
                        matched_gt_index,

                    "match_distance_px":
                        match_distance_px,

                    "unsupported_recovery_point":
                        bool(
                            not matched
                        ),
                }
            )


        # ----------------------------------------------------
        # Per-source-image summary
        # ----------------------------------------------------

        num_recovery_points = int(
            len(
                recovery_subset
            )
        )

        num_missed_gt = int(
            len(
                missed_gt_subset
            )
        )

        recovered_gt = int(
            len(
                matches
            )
        )

        unsupported_points = (
            num_recovery_points
            -
            recovered_gt
        )

        unrecovered_missed_gt = (
            num_missed_gt
            -
            recovered_gt
        )


        source_image_records.append(
            {
                "branch":
                    branch,

                "candidate_id":
                    candidate_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_row.image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "gt_count":
                    int(
                        image_row.gt_count
                    ),

                "matching_radius_px":
                    radius_px,

                "b0_missed_gt":
                    num_missed_gt,

                "recovery_candidate_points":
                    num_recovery_points,

                "recovered_missed_gt":
                    recovered_gt,

                "unsupported_recovery_points":
                    unsupported_points,

                "unrecovered_missed_gt":
                    unrecovered_missed_gt,
            }
        )


recovery_gt_match_long_df = pd.DataFrame(
    match_point_records
)


recovery_gt_image_df = pd.DataFrame(
    source_image_records
)


assert (
    len(
        recovery_gt_image_df
    )
    ==
    300
)


# ------------------------------------------------------------
# 3B.12 Candidate-level metrics
#
# precision_like:
#   recovered missed GT / recovery candidate points
#
# missed_GT_recall:
#   recovered missed GT / all GT instances missed by B0
#
# FP_per_TP:
#   unsupported recovery points / recovered missed GT
#
# These are diagnostic point-matching quantities only.
# ------------------------------------------------------------

source_summary_df = (
    recovery_gt_image_df
    .groupby(
        [
            "branch",
            "candidate_id",
        ],
        as_index=False,
    )
    .agg(
        total_b0_missed_gt=(
            "b0_missed_gt",
            "sum",
        ),

        recovery_candidate_points=(
            "recovery_candidate_points",
            "sum",
        ),

        recovered_missed_gt=(
            "recovered_missed_gt",
            "sum",
        ),

        unsupported_recovery_points=(
            "unsupported_recovery_points",
            "sum",
        ),

        unrecovered_missed_gt=(
            "unrecovered_missed_gt",
            "sum",
        ),
    )
)


source_summary_df[
    "recovery_precision_like"
] = np.where(
    source_summary_df[
        "recovery_candidate_points"
    ]
    >
    0,

    source_summary_df[
        "recovered_missed_gt"
    ]
    /
    source_summary_df[
        "recovery_candidate_points"
    ],

    np.nan,
)


source_summary_df[
    "missed_GT_recall"
] = np.where(
    source_summary_df[
        "total_b0_missed_gt"
    ]
    >
    0,

    source_summary_df[
        "recovered_missed_gt"
    ]
    /
    source_summary_df[
        "total_b0_missed_gt"
    ],

    np.nan,
)


source_summary_df[
    "FP_per_TP"
] = np.where(
    source_summary_df[
        "recovered_missed_gt"
    ]
    >
    0,

    source_summary_df[
        "unsupported_recovery_points"
    ]
    /
    source_summary_df[
        "recovered_missed_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3B.13 Density-stratum summary
#
# Diagnostic only.
# Never use GT density stratum for inference routing.
# ------------------------------------------------------------

density_summary_df_3b = (
    recovery_gt_image_df
    .groupby(
        [
            "density_stratum",
            "branch",
            "candidate_id",
        ],
        as_index=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        total_b0_missed_gt=(
            "b0_missed_gt",
            "sum",
        ),

        recovery_candidate_points=(
            "recovery_candidate_points",
            "sum",
        ),

        recovered_missed_gt=(
            "recovered_missed_gt",
            "sum",
        ),

        unsupported_recovery_points=(
            "unsupported_recovery_points",
            "sum",
        ),
    )
)


density_summary_df_3b[
    "recovery_precision_like"
] = np.where(
    density_summary_df_3b[
        "recovery_candidate_points"
    ]
    >
    0,

    density_summary_df_3b[
        "recovered_missed_gt"
    ]
    /
    density_summary_df_3b[
        "recovery_candidate_points"
    ],

    np.nan,
)


density_summary_df_3b[
    "missed_GT_recall"
] = np.where(
    density_summary_df_3b[
        "total_b0_missed_gt"
    ]
    >
    0,

    density_summary_df_3b[
        "recovered_missed_gt"
    ]
    /
    density_summary_df_3b[
        "total_b0_missed_gt"
    ],

    np.nan,
)


density_summary_df_3b[
    "FP_per_TP"
] = np.where(
    density_summary_df_3b[
        "recovered_missed_gt"
    ]
    >
    0,

    density_summary_df_3b[
        "unsupported_recovery_points"
    ]
    /
    density_summary_df_3b[
        "recovered_missed_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3B.14 Integrity checks
# ------------------------------------------------------------

# Every source must see the same B0-missed GT population.

missed_gt_totals = (
    source_summary_df[
        "total_b0_missed_gt"
    ]
    .unique()
)


assert (
    len(
        missed_gt_totals
    )
    ==
    1
)


TOTAL_B0_MISSED_GT = int(
    missed_gt_totals[
        0
    ]
)


assert (
    TOTAL_B0_MISSED_GT
    ==
    STEP2A_B0_MISSED_GT
), (
    "B0-missed GT total does not agree with frozen Step 2A."
)


# Recovery-candidate totals must reproduce Step 3A.

step3a_totals = (
    candidate_summary_df[
        [
            "branch",
            "candidate_id",
            "recovery_candidate_points",
        ]
    ]
    .copy()
)


check_df = (
    source_summary_df[
        [
            "branch",
            "candidate_id",
            "recovery_candidate_points",
        ]
    ]
    .merge(
        step3a_totals,
        on=[
            "branch",
            "candidate_id",
        ],
        suffixes=(
            "_3B",
            "_3A",
        ),
        validate="one_to_one",
    )
)


assert (
    check_df[
        "recovery_candidate_points_3B"
    ]
    .eq(
        check_df[
            "recovery_candidate_points_3A"
        ]
    )
    .all()
)


# ------------------------------------------------------------
# 3B.15 Save outputs
# ------------------------------------------------------------

recovery_gt_match_long_df.to_csv(
    RECOVERY_GT_MATCH_LONG_FILE,
    index=False,
)


source_summary_df.to_csv(
    RECOVERY_GT_SOURCE_SUMMARY_FILE,
    index=False,
)


recovery_gt_image_df.to_csv(
    RECOVERY_GT_IMAGE_SUMMARY_FILE,
    index=False,
)


density_summary_df_3b.to_csv(
    RECOVERY_GT_DENSITY_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3B.16 Manifest
# ------------------------------------------------------------

step_3b_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3B - GT quality of recovery candidates",

    "recovery_sources":
        [
            {
                "branch":
                    branch,

                "candidate_id":
                    candidate_id,
            }

            for branch, candidate_id
            in RECOVERY_SOURCES
        ],

    "frozen_step2a_gt_totals": {
        "total_gt":
            STEP2A_TOTAL_GT,

        "b0_covered_gt":
            STEP2A_B0_COVERED_GT,

        "b0_missed_gt":
            STEP2A_B0_MISSED_GT,
    },

    "GT_match_population":
        (
            "Only GT points not covered by any frozen B0 mask"
        ),

    "matching_rule":
        (
            "one-to-one greedy globally shortest valid pairs "
            "within frozen per-image matching radius"
        ),

    "matching_radius_source_table":
        R_SOURCE_TABLE_NAME,

    "matching_radius_source_column":
        MATCH_RADIUS_COL,

    "gt_point_source_table":
        GT_SOURCE_TABLE_NAME,

    "gt_point_source_column":
        GT_POINT_COL,

    "point_to_mask_rule":
        "np.rint + clip -> mask pixel",

    "semantic_points_added_to_count":
        False,

    "sam2_reprompting":
        False,

    "recovery_rule_selected":
        False,

    "threshold_tuning":
        False,

    "density_used_for_routing":
        False,

    "test_set_accessed":
        False,

    "caveat":
        (
            "FSC-147 supplies point annotations. "
            "Matching indicates spatial support relative to the "
            "frozen radius, not full instance-segmentation correctness."
        ),
}


with open(
    RECOVERY_GT_3B_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_3b_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3B.17 Final display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3B - GT QUALITY OF B0 RECOVERY CANDIDATES")
print("=" * 125)

print(
    "Total GT points             :",
    len(
        gt_point_status_df
    )
)

print(
    "GT covered by B0            :",
    STEP3B_B0_COVERED_GT
)

print(
    "GT missed by B0             :",
    STEP3B_B0_MISSED_GT
)

print(
    "\nStep 2A reproduction        : PASS"
)

print(
    "\nGT source                   :",
    GT_SOURCE_TABLE_NAME,
    "/",
    GT_POINT_COL,
)

print(
    "Frozen radius source        :",
    R_SOURCE_TABLE_NAME,
    "/",
    MATCH_RADIUS_COL,
)

print(
    "\nPoints added to count       : NO"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Recovery rule selected      : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nRecovery-source GT matching:"
)

display(
    source_summary_df[
        [
            "branch",
            "candidate_id",
            "total_b0_missed_gt",
            "recovery_candidate_points",
            "recovered_missed_gt",
            "unsupported_recovery_points",
            "unrecovered_missed_gt",
            "recovery_precision_like",
            "missed_GT_recall",
            "FP_per_TP",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nRecovery quality by density stratum:"
)

display(
    density_summary_df_3b[
        [
            "density_stratum",
            "branch",
            "candidate_id",
            "num_images",
            "total_b0_missed_gt",
            "recovery_candidate_points",
            "recovered_missed_gt",
            "unsupported_recovery_points",
            "recovery_precision_like",
            "missed_GT_recall",
            "FP_per_TP",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nImages with most genuinely recovered B0-missed GT:"
)

display(
    recovery_gt_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "branch",
            "candidate_id",
            "b0_missed_gt",
            "recovery_candidate_points",
            "recovered_missed_gt",
            "unsupported_recovery_points",
            "unrecovered_missed_gt",
        ]
    ]
    .sort_values(
        [
            "recovered_missed_gt",
            "recovery_candidate_points",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        30
    )
)


print(
    "\nSaved point-level GT matching:"
)

print(
    RECOVERY_GT_MATCH_LONG_FILE
)

print(
    "\nSaved source summary:"
)

print(
    RECOVERY_GT_SOURCE_SUMMARY_FILE
)

print(
    "\nSaved per-image summary:"
)

print(
    RECOVERY_GT_IMAGE_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    RECOVERY_GT_DENSITY_SUMMARY_FILE
)

print(
    "\nSaved manifest:"
)

print(
    RECOVERY_GT_3B_MANIFEST_FILE
)

print("=" * 125)

Resolved GT points from    : C_candidates_df / gt_points_xy
Resolved frozen radius from: C_candidates_df / radius_px
STEP 3B - GT QUALITY OF B0 RECOVERY CANDIDATES
Total GT points             : 10045
GT covered by B0            : 4404
GT missed by B0             : 5641

Step 2A reproduction        : PASS

GT source                   : C_candidates_df / gt_points_xy
Frozen radius source        : C_candidates_df / radius_px

Points added to count       : NO
SAM2 re-prompting           : NO
Recovery rule selected      : NO
Threshold tuning            : NO
Density used for routing    : NO
Test set accessed           : NO

Recovery-source GT matching:


,branch,candidate_id,total_b0_missed_gt,recovery_candidate_points,recovered_missed_gt,unsupported_recovery_points,unrecovered_missed_gt,recovery_precision_like,missed_GT_recall,FP_per_TP
0,C,C1,5641,1215,623,592,5018,0.5128,0.1104,0.9502
1,D,D6B-T-native,5641,8047,1453,6594,4188,0.1806,0.2576,4.5382
2,D,D6B9-strict,5641,1911,550,1361,5091,0.2878,0.0975,2.4745



Recovery quality by density stratum:


,density_stratum,branch,candidate_id,num_images,total_b0_missed_gt,recovery_candidate_points,recovered_missed_gt,unsupported_recovery_points,recovery_precision_like,missed_GT_recall,FP_per_TP
0,S1_0_20,C,C1,20,88,92,24,68,0.2609,0.2727,2.8333
1,S1_0_20,D,D6B-T-native,20,88,941,59,882,0.0627,0.6705,14.9492
2,S1_0_20,D,D6B9-strict,20,88,187,41,146,0.2193,0.4659,3.5610
3,S2_20_40,C,C1,20,186,183,55,128,0.3005,0.2957,2.3273
4,S2_20_40,D,D6B-T-native,20,186,1572,131,1441,0.0833,0.7043,11.0000
5,S2_20_40,D,D6B9-strict,20,186,450,99,351,0.2200,0.5323,3.5455
6,S3_40_60,C,C1,20,333,198,48,150,0.2424,0.1441,3.1250
7,S3_40_60,D,D6B-T-native,20,333,1576,203,1373,0.1288,0.6096,6.7635
8,S3_40_60,D,D6B9-strict,20,333,496,104,392,0.2097,0.3123,3.7692
9,S4_60_80,C,C1,20,451,215,110,105,0.5116,0.2439,0.9545



Images with most genuinely recovered B0-missed GT:


,dataset_index,image_id,category,density_stratum,gt_count,branch,candidate_id,b0_missed_gt,recovery_candidate_points,recovered_missed_gt,unsupported_recovery_points,unrecovered_missed_gt
277,938,5866.jpg,grapes,S6_95_100,267,D,D6B-T-native,221,498,159,339,62
232,253,975.jpg,birds,S6_95_100,718,D,D6B-T-native,718,148,108,40,610
32,253,975.jpg,birds,S6_95_100,718,C,C1,718,107,100,7,618
92,1187,6931.jpg,birds,S6_95_100,355,C,C1,352,96,93,3,259
225,184,840.jpg,books,S6_95_100,637,D,D6B-T-native,626,257,78,179,548
245,425,2884.jpg,chairs,S5_80_95,165,D,D6B-T-native,130,196,77,119,53
200,9,215.jpg,grapes,S6_95_100,259,D,D6B-T-native,159,186,67,119,92
234,298,1935.jpg,bottle caps,S5_80_95,175,D,D6B-T-native,125,132,65,67,60
293,1192,6969.jpg,birds,S6_95_100,949,D,D6B-T-native,949,62,56,6,893
292,1187,6931.jpg,birds,S6_95_100,355,D,D6B-T-native,352,87,53,34,299



Saved point-level GT matching:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_GT_matching_long.csv

Saved source summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_GT_matching_summary.csv

Saved per-image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_GT_matching_per_image.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_GT_matching_by_density.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_3B_manifest.json


In [25]:
# ============================================================
# Step 3C - B0 + Raw Semantic Recovery Count Ablation
#
# Purpose:
# Measure whether adding uncovered semantic recovery points
# improves image-level counting relative to frozen B0.
#
# Compared systems:
#   1. B0_unfiltered
#   2. B0 + C1 recovery candidates
#   3. B0 + D6B9-strict recovery candidates
#   4. B0 + D6B-T-native recovery candidates
#
# IMPORTANT:
# - every uncovered semantic point contributes +1
# - NO GT is used to decide which point is added
# - NO SAM2 re-prompting
# - NO recovery filtering
# - NO cross-source fusion
# - NO density routing
# - test set untouched
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3C.1 Output files
# ------------------------------------------------------------

RECOVERY_COUNT_PER_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_plus_raw_recovery_counts_per_image.csv"
)

RECOVERY_COUNT_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_plus_raw_recovery_count_summary.csv"
)

RECOVERY_COUNT_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_plus_raw_recovery_count_summary_by_density.csv"
)

RECOVERY_COUNT_HELP_HURT_FILE = (
    B2_TABLES_DIR
    / "B0_plus_raw_recovery_help_hurt.csv"
)

RECOVERY_COUNT_3C_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3C_manifest.json"
)


# ------------------------------------------------------------
# 3C.2 Preconditions
# ------------------------------------------------------------

assert (
    "recovery_point_image_df"
    in globals()
), (
    "recovery_point_image_df not found. "
    "Run Step 3A first."
)

assert (
    "recovery_gt_image_df"
    in globals()
), (
    "recovery_gt_image_df not found. "
    "Run Step 3B first."
)

assert (
    "b2_dev100_image_df"
    in globals()
)

assert (
    "b0_mask_cache_index_df"
    in globals()
)


assert (
    len(
        recovery_point_image_df
    )
    ==
    300
)


# ------------------------------------------------------------
# 3C.3 Recovery sources
# ------------------------------------------------------------

RECOVERY_ABLATION_SOURCES = [
    ("C", "C1"),
    ("D", "D6B9-strict"),
    ("D", "D6B-T-native"),
]


RECOVERY_SYSTEM_NAMES = {
    ("C", "C1"):
        "B0_plus_C1_raw",

    ("D", "D6B9-strict"):
        "B0_plus_D6B9_raw",

    ("D", "D6B-T-native"):
        "B0_plus_D6BT_raw",
}


# ------------------------------------------------------------
# 3C.4 Build authoritative frozen B0 per-image table
# ------------------------------------------------------------

b0_count_df = (
    b0_mask_cache_index_df[
        [
            "dataset_index",
            "image_id",
            "reproduced_b0_count",
        ]
    ]
    .copy()
    .rename(
        columns={
            "reproduced_b0_count":
                "B0_unfiltered"
        }
    )
)


assert (
    len(
        b0_count_df
    )
    ==
    100
)


# ------------------------------------------------------------
# 3C.5 Start from all 100 dev images
# ------------------------------------------------------------

recovery_count_df = (
    b2_dev100_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .merge(
        b0_count_df,
        on=[
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    len(
        recovery_count_df
    )
    ==
    100
)

assert (
    recovery_count_df[
        "B0_unfiltered"
    ]
    .notna()
    .all()
)


recovery_count_df[
    "B0_unfiltered"
] = (
    recovery_count_df[
        "B0_unfiltered"
    ]
    .astype(
        int
    )
)


# ------------------------------------------------------------
# 3C.6 Attach raw recovery candidate counts
#
# IMPORTANT:
# Only Step 3A recovery_candidate_points are used here.
#
# Step 3B GT matching is NOT used to decide which points
# contribute to the count.
# ------------------------------------------------------------

for (
    branch,
    candidate_id
) in RECOVERY_ABLATION_SOURCES:

    system_name = (
        RECOVERY_SYSTEM_NAMES[
            (
                branch,
                candidate_id,
            )
        ]
    )


    source_recovery_df = (
        recovery_point_image_df[
            (
                recovery_point_image_df[
                    "branch"
                ]
                ==
                branch
            )
            &
            (
                recovery_point_image_df[
                    "candidate_id"
                ]
                ==
                candidate_id
            )
        ][
            [
                "dataset_index",
                "image_id",
                "recovery_candidate_points",
            ]
        ]
        .copy()
        .rename(
            columns={
                "recovery_candidate_points":
                    f"{system_name}__added_points"
            }
        )
    )


    assert (
        len(
            source_recovery_df
        )
        ==
        100
    )


    recovery_count_df = (
        recovery_count_df
        .merge(
            source_recovery_df,
            on=[
                "dataset_index",
                "image_id",
            ],
            how="left",
            validate="one_to_one",
        )
    )


    added_col = (
        f"{system_name}__added_points"
    )


    recovery_count_df[
        added_col
    ] = (
        recovery_count_df[
            added_col
        ]
        .fillna(
            0
        )
        .astype(
            int
        )
    )


    recovery_count_df[
        system_name
    ] = (
        recovery_count_df[
            "B0_unfiltered"
        ]
        +
        recovery_count_df[
            added_col
        ]
    )


# ------------------------------------------------------------
# 3C.7 Integrity checks against Step 3A totals
# ------------------------------------------------------------

EXPECTED_RECOVERY_TOTALS = {
    "B0_plus_C1_raw":
        1215,

    "B0_plus_D6B9_raw":
        1911,

    "B0_plus_D6BT_raw":
        8047,
}


for system_name, expected_total in (
    EXPECTED_RECOVERY_TOTALS.items()
):

    added_total = int(
        recovery_count_df[
            f"{system_name}__added_points"
        ].sum()
    )


    assert (
        added_total
        ==
        expected_total
    ), (
        f"{system_name}: recovery total mismatch. "
        f"Expected {expected_total}, got {added_total}."
    )


# Frozen B0 total

assert (
    int(
        recovery_count_df[
            "B0_unfiltered"
        ].sum()
    )
    ==
    2298
)


# ------------------------------------------------------------
# 3C.8 Count metric helper
# ------------------------------------------------------------

def count_metrics_3c(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )


    error = (
        pred
        -
        gt
    )


    abs_error = np.abs(
        error
    )


    return {
        "MAE":
            float(
                abs_error.mean()
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error**2
                    )
                )
            ),

        "mean_signed_error":
            float(
                error.mean()
            ),

        "under_count_images":
            int(
                np.sum(
                    error
                    <
                    0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    error
                    ==
                    0
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    error
                    >
                    0
                )
            ),
    }


# ------------------------------------------------------------
# 3C.9 Overall count performance
# ------------------------------------------------------------

SYSTEMS = [
    "B0_unfiltered",
    "B0_plus_C1_raw",
    "B0_plus_D6B9_raw",
    "B0_plus_D6BT_raw",
]


gt_all = (
    recovery_count_df[
        "gt_count"
    ]
    .to_numpy()
)


summary_records = []


for system_name in SYSTEMS:

    pred = (
        recovery_count_df[
            system_name
        ]
        .to_numpy()
    )


    metrics = count_metrics_3c(
        gt=gt_all,
        pred=pred,
    )


    if system_name == "B0_unfiltered":

        added_points = 0

    else:

        added_points = int(
            recovery_count_df[
                f"{system_name}__added_points"
            ].sum()
        )


    summary_records.append(
        {
            "system":
                system_name,

            "total_added_recovery_points":
                added_points,

            "total_predicted_count":
                int(
                    pred.sum()
                ),

            "mean_predicted_count":
                float(
                    pred.mean()
                ),

            **metrics,
        }
    )


recovery_count_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 3C.10 Change versus frozen B0
# ------------------------------------------------------------

b0_row = (
    recovery_count_summary_df[
        recovery_count_summary_df[
            "system"
        ]
        ==
        "B0_unfiltered"
    ]
    .iloc[
        0
    ]
)


B0_MAE = float(
    b0_row[
        "MAE"
    ]
)

B0_RMSE = float(
    b0_row[
        "RMSE"
    ]
)


recovery_count_summary_df[
    "delta_MAE_vs_B0"
] = (
    recovery_count_summary_df[
        "MAE"
    ]
    -
    B0_MAE
)


recovery_count_summary_df[
    "delta_RMSE_vs_B0"
] = (
    recovery_count_summary_df[
        "RMSE"
    ]
    -
    B0_RMSE
)


# ------------------------------------------------------------
# 3C.11 Per-image absolute errors
# ------------------------------------------------------------

for system_name in SYSTEMS:

    recovery_count_df[
        f"{system_name}__signed_error"
    ] = (
        recovery_count_df[
            system_name
        ]
        -
        recovery_count_df[
            "gt_count"
        ]
    )


    recovery_count_df[
        f"{system_name}__abs_error"
    ] = np.abs(
        recovery_count_df[
            f"{system_name}__signed_error"
        ]
    )


# ------------------------------------------------------------
# 3C.12 Help / hurt analysis
# ------------------------------------------------------------

help_hurt_records = []


b0_abs = (
    recovery_count_df[
        "B0_unfiltered__abs_error"
    ]
)


for system_name in SYSTEMS:

    if system_name == "B0_unfiltered":

        continue


    new_abs = (
        recovery_count_df[
            f"{system_name}__abs_error"
        ]
    )


    improved = int(
        (
            new_abs
            <
            b0_abs
        ).sum()
    )

    unchanged = int(
        (
            new_abs
            ==
            b0_abs
        ).sum()
    )

    worsened = int(
        (
            new_abs
            >
            b0_abs
        ).sum()
    )


    help_hurt_records.append(
        {
            "system":
                system_name,

            "improved_images":
                improved,

            "unchanged_images":
                unchanged,

            "worsened_images":
                worsened,
        }
    )


recovery_help_hurt_df = pd.DataFrame(
    help_hurt_records
)


# ------------------------------------------------------------
# 3C.13 Diagnostic summary by GT density stratum
#
# Diagnostic only.
# Density is NOT used for routing.
# ------------------------------------------------------------

density_records = []


for stratum in DENSITY_ORDER:

    stratum_df = (
        recovery_count_df[
            recovery_count_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    gt = (
        stratum_df[
            "gt_count"
        ]
        .to_numpy()
    )


    for system_name in SYSTEMS:

        pred = (
            stratum_df[
                system_name
            ]
            .to_numpy()
        )


        metrics = count_metrics_3c(
            gt=gt,
            pred=pred,
        )


        if system_name == "B0_unfiltered":

            added_points = 0

        else:

            added_points = int(
                stratum_df[
                    f"{system_name}__added_points"
                ].sum()
            )


        density_records.append(
            {
                "density_stratum":
                    stratum,

                "system":
                    system_name,

                "num_images":
                    int(
                        len(
                            stratum_df
                        )
                    ),

                "added_recovery_points":
                    added_points,

                **metrics,
            }
        )


recovery_count_density_df = pd.DataFrame(
    density_records
)


# ------------------------------------------------------------
# 3C.14 Extra comparison with Step 3B diagnostic TP/FP
#
# This is descriptive only.
# GT matching did NOT determine the actual counts above.
# ------------------------------------------------------------

recovery_quality_context_df = (
    source_summary_df[
        [
            "branch",
            "candidate_id",
            "recovery_candidate_points",
            "recovered_missed_gt",
            "unsupported_recovery_points",
            "recovery_precision_like",
            "missed_GT_recall",
            "FP_per_TP",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 3C.15 Save outputs
# ------------------------------------------------------------

recovery_count_df.to_csv(
    RECOVERY_COUNT_PER_IMAGE_FILE,
    index=False,
)


recovery_count_summary_df.to_csv(
    RECOVERY_COUNT_SUMMARY_FILE,
    index=False,
)


recovery_count_density_df.to_csv(
    RECOVERY_COUNT_DENSITY_FILE,
    index=False,
)


recovery_help_hurt_df.to_csv(
    RECOVERY_COUNT_HELP_HURT_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3C.16 Manifest
# ------------------------------------------------------------

step_3c_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3C - B0 plus raw semantic recovery count ablation",

    "num_images":
        100,

    "frozen_B0_total_count":
        int(
            recovery_count_df[
                "B0_unfiltered"
            ].sum()
        ),

    "systems":
        SYSTEMS,

    "recovery_sources":
        {
            "B0_plus_C1_raw":
                "C1",

            "B0_plus_D6B9_raw":
                "D6B9-strict",

            "B0_plus_D6BT_raw":
                "D6B-T-native",
        },

    "recovery_rule":
        (
            "Add +1 for every semantic point lying outside "
            "all frozen B0 masks."
        ),

    "GT_used_to_select_recovery_points":
        False,

    "GT_used_for_evaluation":
        True,

    "cross_source_fusion":
        False,

    "within_source_deduplication":
        False,

    "sam2_reprompting":
        False,

    "recovery_filtering":
        False,

    "threshold_tuning":
        False,

    "density_used_for_routing":
        False,

    "test_set_accessed":
        False,

    "caveat":
        (
            "This is a raw additive point-count ablation. "
            "Recovery candidates may contain duplicate or unsupported "
            "semantic points; Step 3C intentionally does not remove them."
        ),
}


with open(
    RECOVERY_COUNT_3C_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step_3c_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3C.17 Final display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3C - B0 + RAW SEMANTIC RECOVERY COUNT ABLATION")
print("=" * 125)

print(
    "Images                      :",
    len(
        recovery_count_df
    ),
    "/ 100"
)

print(
    "Frozen B0 total count       :",
    int(
        recovery_count_df[
            "B0_unfiltered"
        ].sum()
    )
)

print(
    "\nC1 points added             :",
    int(
        recovery_count_df[
            "B0_plus_C1_raw__added_points"
        ].sum()
    )
)

print(
    "D6B9 points added           :",
    int(
        recovery_count_df[
            "B0_plus_D6B9_raw__added_points"
        ].sum()
    )
)

print(
    "D6B-T points added          :",
    int(
        recovery_count_df[
            "B0_plus_D6BT_raw__added_points"
        ].sum()
    )
)

print(
    "\nGT used to select points    : NO"
)

print(
    "Cross-source fusion         : NO"
)

print(
    "Recovery filtering          : NO"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nOverall count performance:"
)

display(
    recovery_count_summary_df[
        [
            "system",
            "total_added_recovery_points",
            "total_predicted_count",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_B0",
            "delta_RMSE_vs_B0",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nHelp / hurt relative to frozen B0:"
)

display(
    recovery_help_hurt_df
)


print(
    "\nStep 3B recovery-quality context:"
)

display(
    recovery_quality_context_df.round(
        4
    )
)


print(
    "\nCount performance by density stratum:"
)

display(
    recovery_count_density_df[
        [
            "density_stratum",
            "system",
            "num_images",
            "added_recovery_points",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nImages with largest raw recovery additions:"
)

added_display_cols = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "B0_unfiltered",
    "B0_plus_C1_raw__added_points",
    "B0_plus_D6B9_raw__added_points",
    "B0_plus_D6BT_raw__added_points",
    "B0_plus_C1_raw",
    "B0_plus_D6B9_raw",
    "B0_plus_D6BT_raw",
]


display(
    recovery_count_df[
        added_display_cols
    ]
    .sort_values(
        "B0_plus_D6BT_raw__added_points",
        ascending=False,
    )
    .head(
        30
    )
)


print(
    "\nSaved per-image count results:"
)

print(
    RECOVERY_COUNT_PER_IMAGE_FILE
)

print(
    "\nSaved overall summary:"
)

print(
    RECOVERY_COUNT_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    RECOVERY_COUNT_DENSITY_FILE
)

print(
    "\nSaved help/hurt summary:"
)

print(
    RECOVERY_COUNT_HELP_HURT_FILE
)

print(
    "\nSaved manifest:"
)

print(
    RECOVERY_COUNT_3C_MANIFEST_FILE
)

print("=" * 125)

STEP 3C - B0 + RAW SEMANTIC RECOVERY COUNT ABLATION
Images                      : 100 / 100
Frozen B0 total count       : 2298

C1 points added             : 1215
D6B9 points added           : 1911
D6B-T points added          : 8047

GT used to select points    : NO
Cross-source fusion         : NO
Recovery filtering          : NO
SAM2 re-prompting           : NO
Density used for routing    : NO
Test set accessed           : NO

Overall count performance:


,system,total_added_recovery_points,total_predicted_count,MAE,RMSE,mean_signed_error,delta_MAE_vs_B0,delta_RMSE_vs_B0,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,0,2298,80.69,270.2339,-77.47,0.00,0.0000,76,4,20
1,B0_plus_C1_raw,1215,3513,72.00,262.2353,-65.32,-8.69,-7.9986,52,5,43
2,B0_plus_D6B9_raw,1911,4209,80.64,267.6874,-58.36,-0.05,-2.5465,45,4,51
3,B0_plus_D6BT_raw,8047,10345,117.96,270.8270,3.00,37.27,0.5931,27,0,73



Help / hurt relative to frozen B0:


,system,improved_images,unchanged_images,worsened_images
0,B0_plus_C1_raw,55,20,25
1,B0_plus_D6B9_raw,44,15,41
2,B0_plus_D6BT_raw,31,7,62



Step 3B recovery-quality context:


...


Count performance by density stratum:


,density_stratum,system,num_images,added_recovery_points,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,S1_0_20,B0_unfiltered,20,0,5.15,6.1115,-0.55,10,1,9
1,S1_0_20,B0_plus_C1_raw,20,92,4.55,6.1685,4.05,3,3,14
2,S1_0_20,B0_plus_D6B9_raw,20,187,9.70,12.3085,8.80,3,0,17
3,S1_0_20,B0_plus_D6BT_raw,20,941,46.50,58.7256,46.50,0,0,20
4,S2_20_40,B0_unfiltered,20,0,6.55,8.1086,-5.55,16,2,2
5,S2_20_40,B0_plus_C1_raw,20,183,5.60,7.0569,3.60,4,1,15
6,S2_20_40,B0_plus_D6B9_raw,20,450,18.55,24.5102,16.95,5,2,13
7,S2_20_40,B0_plus_D6BT_raw,20,1572,73.45,95.6998,73.05,2,0,18
8,S3_40_60,B0_unfiltered,20,0,14.95,17.6423,-13.55,17,1,2
9,S3_40_60,B0_plus_C1_raw,20,198,8.95,11.3468,-3.65,12,1,7



Images with largest raw recovery additions:


,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw__added_points,B0_plus_D6B9_raw__added_points,B0_plus_D6BT_raw__added_points,B0_plus_C1_raw,B0_plus_D6B9_raw,B0_plus_D6BT_raw
77,938,5866.jpg,grapes,S6_95_100,267,47,18,100,498,65,147,545
1,14,220.jpg,grapes,S4_60_80,59,20,32,74,362,52,94,382
24,183,838.jpg,books,S5_80_95,113,43,27,75,261,70,118,304
43,396,2846.jpg,chairs,S4_60_80,38,7,23,63,260,30,70,267
25,184,840.jpg,books,S6_95_100,637,7,54,6,257,61,13,264
16,148,769.jpg,books,S2_20_40,13,3,28,43,217,31,46,220
98,1260,7100.jpg,milk cartons,S3_40_60,27,40,8,83,216,48,123,256
99,1269,7060.jpg,pills,S4_60_80,65,11,14,70,198,25,81,209
45,425,2884.jpg,chairs,S5_80_95,165,38,8,60,196,46,98,234
2,20,228.jpg,grapes,S4_60_80,44,5,22,38,191,27,43,196



Saved per-image count results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_plus_raw_recovery_counts_per_image.csv

Saved overall summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_plus_raw_recovery_count_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_plus_raw_recovery_count_summary_by_density.csv

Saved help/hurt summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_plus_raw_recovery_help_hurt.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_3C_manifest.json


In [26]:
# ============================================================
# Step 3D - C1-Anchored Selective D Recovery
#
# Purpose:
# Test whether D recovery evidence can add useful counts on top
# of the successful C1 raw recovery baseline, while suppressing
# unsupported D proposals.
#
# Rule:
#   - keep every C1 recovery candidate
#   - add a D recovery candidate only if it lies within the
#     frozen per-image radius R of at least one C1 recovery point
#
# Compared systems:
#   1. B0
#   2. B0 + C1 raw
#   3. B0 + C1 raw + C1-supported D6B9
#   4. B0 + C1 raw + C1-supported D6B-T
#
# IMPORTANT:
# - GT is NOT used to decide which points are added
# - no SAM2 re-prompting
# - no density routing
# - no test-set access
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3D.1 Output files
# ------------------------------------------------------------

STEP3D_POINT_FILE = (
    B2_TABLES_DIR
    / "B0_C1_anchored_selective_D_points.csv"
)

STEP3D_PER_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_C1_anchored_selective_D_counts_per_image.csv"
)

STEP3D_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_anchored_selective_D_summary.csv"
)

STEP3D_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_anchored_selective_D_by_density.csv"
)

STEP3D_HELP_HURT_FILE = (
    B2_TABLES_DIR
    / "B0_C1_anchored_selective_D_help_hurt.csv"
)

STEP3D_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3D_manifest.json"
)


# ------------------------------------------------------------
# 3D.2 Preconditions
# ------------------------------------------------------------

assert (
    "recovery_point_long_df"
    in globals()
), "Run Step 3A first."

assert (
    "gt_image_df"
    in globals()
), "Run Step 3B first."

assert (
    "recovery_count_df"
    in globals()
), "Run Step 3C first."

assert (
    len(
        recovery_count_df
    )
    == 100
)


# ------------------------------------------------------------
# 3D.3 Extract recovery points for selected sources
# ------------------------------------------------------------

c1_recovery_df = (
    recovery_point_long_df[
        (
            recovery_point_long_df[
                "candidate_id"
            ]
            ==
            "C1"
        )
        &
        (
            recovery_point_long_df[
                "recovery_candidate"
            ]
        )
    ]
    .copy()
)


d6b9_recovery_df = (
    recovery_point_long_df[
        (
            recovery_point_long_df[
                "candidate_id"
            ]
            ==
            "D6B9-strict"
        )
        &
        (
            recovery_point_long_df[
                "recovery_candidate"
            ]
        )
    ]
    .copy()
)


d6bt_recovery_df = (
    recovery_point_long_df[
        (
            recovery_point_long_df[
                "candidate_id"
            ]
            ==
            "D6B-T-native"
        )
        &
        (
            recovery_point_long_df[
                "recovery_candidate"
            ]
        )
    ]
    .copy()
)


assert len(c1_recovery_df) == 1215
assert len(d6b9_recovery_df) == 1911
assert len(d6bt_recovery_df) == 8047


# ------------------------------------------------------------
# 3D.4 Radius lookup
# ------------------------------------------------------------

radius_lookup = (
    gt_image_df[
        [
            "dataset_index",
            "matching_radius_px",
        ]
    ]
    .set_index(
        "dataset_index"
    )
)


# ------------------------------------------------------------
# 3D.5 Helper:
# nearest C1 recovery-point distance
# ------------------------------------------------------------

def nearest_distance_to_points(
    x,
    y,
    ref_xy,
):

    if len(
        ref_xy
    ) == 0:

        return np.nan


    ref_xy = np.asarray(
        ref_xy,
        dtype=float,
    )


    distances = np.hypot(
        ref_xy[:, 0] - float(x),
        ref_xy[:, 1] - float(y),
    )


    return float(
        distances.min()
    )


# ------------------------------------------------------------
# 3D.6 Build per-image C1 recovery-point lookup
# ------------------------------------------------------------

c1_xy_lookup = {}


for dataset_index, group in (
    c1_recovery_df
    .groupby(
        "dataset_index"
    )
):

    c1_xy_lookup[
        int(
            dataset_index
        )
    ] = (
        group[
            [
                "x",
                "y",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )


# ------------------------------------------------------------
# 3D.7 Annotate D points with C1 support
#
# D point is accepted if:
# nearest distance to any C1 recovery point <= R
# ------------------------------------------------------------

def annotate_D_with_C1_support(
    source_df,
    source_name,
):

    records = []


    for row in source_df.itertuples(
        index=False
    ):

        dataset_index = int(
            row.dataset_index
        )


        c1_xy = c1_xy_lookup.get(
            dataset_index,
            np.empty(
                (
                    0,
                    2,
                ),
                dtype=float,
            ),
        )


        radius_px = float(
            radius_lookup.loc[
                dataset_index,
                "matching_radius_px",
            ]
        )


        nearest_dist = (
            nearest_distance_to_points(
                x=row.x,
                y=row.y,
                ref_xy=c1_xy,
            )
        )


        c1_supported = bool(
            np.isfinite(
                nearest_dist
            )
            and
            (
                nearest_dist
                <=
                radius_px
            )
        )


        records.append(
            {
                "source":
                    source_name,

                "dataset_index":
                    dataset_index,

                "image_id":
                    row.image_id,

                "category":
                    row.category,

                "density_stratum":
                    row.density_stratum,

                "gt_count":
                    int(
                        row.gt_count
                    ),

                "point_index":
                    int(
                        row.point_index
                    ),

                "x":
                    float(
                        row.x
                    ),

                "y":
                    float(
                        row.y
                    ),

                "matching_radius_px":
                    radius_px,

                "nearest_C1_recovery_distance_px":
                    (
                        float(
                            nearest_dist
                        )
                        if np.isfinite(
                            nearest_dist
                        )
                        else np.nan
                    ),

                "supported_by_C1_recovery":
                    c1_supported,
            }
        )


    return pd.DataFrame(
        records
    )


d6b9_c1_support_df = annotate_D_with_C1_support(
    d6b9_recovery_df,
    "D6B9-strict",
)


d6bt_c1_support_df = annotate_D_with_C1_support(
    d6bt_recovery_df,
    "D6B-T-native",
)


step3d_point_df = pd.concat(
    [
        d6b9_c1_support_df,
        d6bt_c1_support_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 3D.8 Candidate-level support summary
# ------------------------------------------------------------

step3d_support_summary_df = (
    step3d_point_df
    .groupby(
        "source",
        as_index=False,
    )
    .agg(
        total_D_recovery_points=(
            "point_index",
            "size",
        ),

        C1_supported_D_points=(
            "supported_by_C1_recovery",
            "sum",
        ),
    )
)


step3d_support_summary_df[
    "C1_supported_fraction"
] = (
    step3d_support_summary_df[
        "C1_supported_D_points"
    ]
    /
    step3d_support_summary_df[
        "total_D_recovery_points"
    ]
)


# ------------------------------------------------------------
# 3D.9 Per-image accepted D counts
# ------------------------------------------------------------

accepted_D_image_df = (
    step3d_point_df[
        step3d_point_df[
            "supported_by_C1_recovery"
        ]
    ]
    .groupby(
        [
            "dataset_index",
            "source",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "accepted_D_points"
        }
    )
)


# ------------------------------------------------------------
# 3D.10 Start from Step 3C per-image table
# ------------------------------------------------------------

step3d_count_df = (
    recovery_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "B0_plus_C1_raw__added_points",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 3D.11 Add accepted D6B9 counts
# ------------------------------------------------------------

d6b9_added_df = (
    accepted_D_image_df[
        accepted_D_image_df[
            "source"
        ]
        ==
        "D6B9-strict"
    ][
        [
            "dataset_index",
            "accepted_D_points",
        ]
    ]
    .rename(
        columns={
            "accepted_D_points":
                "D6B9_C1_supported_added"
        }
    )
)


step3d_count_df = (
    step3d_count_df
    .merge(
        d6b9_added_df,
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


step3d_count_df[
    "D6B9_C1_supported_added"
] = (
    step3d_count_df[
        "D6B9_C1_supported_added"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


# ------------------------------------------------------------
# 3D.12 Add accepted D6B-T counts
# ------------------------------------------------------------

d6bt_added_df = (
    accepted_D_image_df[
        accepted_D_image_df[
            "source"
        ]
        ==
        "D6B-T-native"
    ][
        [
            "dataset_index",
            "accepted_D_points",
        ]
    ]
    .rename(
        columns={
            "accepted_D_points":
                "D6BT_C1_supported_added"
        }
    )
)


step3d_count_df = (
    step3d_count_df
    .merge(
        d6bt_added_df,
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


step3d_count_df[
    "D6BT_C1_supported_added"
] = (
    step3d_count_df[
        "D6BT_C1_supported_added"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


# ------------------------------------------------------------
# 3D.13 Construct fused counts
# ------------------------------------------------------------

step3d_count_df[
    "B0_plus_C1_plus_D6B9_C1supported"
] = (
    step3d_count_df[
        "B0_plus_C1_raw"
    ]
    +
    step3d_count_df[
        "D6B9_C1_supported_added"
    ]
)


step3d_count_df[
    "B0_plus_C1_plus_D6BT_C1supported"
] = (
    step3d_count_df[
        "B0_plus_C1_raw"
    ]
    +
    step3d_count_df[
        "D6BT_C1_supported_added"
    ]
)


# ------------------------------------------------------------
# 3D.14 Metric helper
# ------------------------------------------------------------

def count_metrics_3d(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )


    error = (
        pred
        -
        gt
    )


    return {
        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error**2
                    )
                )
            ),

        "mean_signed_error":
            float(
                error.mean()
            ),

        "under_count_images":
            int(
                np.sum(
                    error
                    <
                    0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    error
                    ==
                    0
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    error
                    >
                    0
                )
            ),
    }


# ------------------------------------------------------------
# 3D.15 Overall performance
# ------------------------------------------------------------

STEP3D_SYSTEMS = [
    "B0_unfiltered",
    "B0_plus_C1_raw",
    "B0_plus_C1_plus_D6B9_C1supported",
    "B0_plus_C1_plus_D6BT_C1supported",
]


summary_records = []


gt_all = (
    step3d_count_df[
        "gt_count"
    ]
    .to_numpy()
)


for system_name in STEP3D_SYSTEMS:

    pred = (
        step3d_count_df[
            system_name
        ]
        .to_numpy()
    )


    metrics = count_metrics_3d(
        gt_all,
        pred,
    )


    if system_name == "B0_unfiltered":

        added_D = 0

    elif system_name == "B0_plus_C1_raw":

        added_D = 0

    elif system_name == "B0_plus_C1_plus_D6B9_C1supported":

        added_D = int(
            step3d_count_df[
                "D6B9_C1_supported_added"
            ].sum()
        )

    elif system_name == "B0_plus_C1_plus_D6BT_C1supported":

        added_D = int(
            step3d_count_df[
                "D6BT_C1_supported_added"
            ].sum()
        )


    summary_records.append(
        {
            "system":
                system_name,

            "additional_D_points":
                added_D,

            "total_predicted_count":
                int(
                    pred.sum()
                ),

            **metrics,
        }
    )


step3d_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 3D.16 Delta vs B0 and C1
# ------------------------------------------------------------

b0_metrics = (
    step3d_summary_df[
        step3d_summary_df[
            "system"
        ]
        ==
        "B0_unfiltered"
    ]
    .iloc[
        0
    ]
)


c1_metrics = (
    step3d_summary_df[
        step3d_summary_df[
            "system"
        ]
        ==
        "B0_plus_C1_raw"
    ]
    .iloc[
        0
    ]
)


step3d_summary_df[
    "delta_MAE_vs_B0"
] = (
    step3d_summary_df[
        "MAE"
    ]
    -
    float(
        b0_metrics[
            "MAE"
        ]
    )
)


step3d_summary_df[
    "delta_RMSE_vs_B0"
] = (
    step3d_summary_df[
        "RMSE"
    ]
    -
    float(
        b0_metrics[
            "RMSE"
        ]
    )
)


step3d_summary_df[
    "delta_MAE_vs_C1"
] = (
    step3d_summary_df[
        "MAE"
    ]
    -
    float(
        c1_metrics[
            "MAE"
        ]
    )
)


step3d_summary_df[
    "delta_RMSE_vs_C1"
] = (
    step3d_summary_df[
        "RMSE"
    ]
    -
    float(
        c1_metrics[
            "RMSE"
        ]
    )
)


# ------------------------------------------------------------
# 3D.17 Per-image absolute errors
# ------------------------------------------------------------

for system_name in STEP3D_SYSTEMS:

    step3d_count_df[
        f"{system_name}__abs_error"
    ] = np.abs(
        step3d_count_df[
            system_name
        ]
        -
        step3d_count_df[
            "gt_count"
        ]
    )


# ------------------------------------------------------------
# 3D.18 Help/hurt relative to C1
# ------------------------------------------------------------

help_hurt_records = []


c1_abs = (
    step3d_count_df[
        "B0_plus_C1_raw__abs_error"
    ]
)


for system_name in [
    "B0_plus_C1_plus_D6B9_C1supported",
    "B0_plus_C1_plus_D6BT_C1supported",
]:

    new_abs = (
        step3d_count_df[
            f"{system_name}__abs_error"
        ]
    )


    help_hurt_records.append(
        {
            "system":
                system_name,

            "improved_vs_C1":
                int(
                    (
                        new_abs
                        <
                        c1_abs
                    ).sum()
                ),

            "unchanged_vs_C1":
                int(
                    (
                        new_abs
                        ==
                        c1_abs
                    ).sum()
                ),

            "worsened_vs_C1":
                int(
                    (
                        new_abs
                        >
                        c1_abs
                    ).sum()
                ),
        }
    )


step3d_help_hurt_df = pd.DataFrame(
    help_hurt_records
)


# ------------------------------------------------------------
# 3D.19 Density-stratum diagnostics
# ------------------------------------------------------------

density_records = []


for stratum in DENSITY_ORDER:

    subset = (
        step3d_count_df[
            step3d_count_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    gt = subset[
        "gt_count"
    ].to_numpy()


    for system_name in STEP3D_SYSTEMS:

        pred = subset[
            system_name
        ].to_numpy()


        metrics = count_metrics_3d(
            gt,
            pred,
        )


        density_records.append(
            {
                "density_stratum":
                    stratum,

                "system":
                    system_name,

                "num_images":
                    int(
                        len(
                            subset
                        )
                    ),

                **metrics,
            }
        )


step3d_density_df = pd.DataFrame(
    density_records
)


# ------------------------------------------------------------
# 3D.20 Save outputs
# ------------------------------------------------------------

step3d_point_df.to_csv(
    STEP3D_POINT_FILE,
    index=False,
)


step3d_count_df.to_csv(
    STEP3D_PER_IMAGE_FILE,
    index=False,
)


step3d_summary_df.to_csv(
    STEP3D_SUMMARY_FILE,
    index=False,
)


step3d_density_df.to_csv(
    STEP3D_DENSITY_FILE,
    index=False,
)


step3d_help_hurt_df.to_csv(
    STEP3D_HELP_HURT_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3D.21 Manifest
# ------------------------------------------------------------

step3d_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3D - C1 anchored selective D recovery",

    "anchor_recovery_source":
        "C1",

    "D_sources":
        [
            "D6B9-strict",
            "D6B-T-native",
        ],

    "D_acceptance_rule":
        (
            "D recovery point accepted if nearest C1 recovery "
            "point distance <= frozen per-image radius R"
        ),

    "GT_used_for_point_selection":
        False,

    "GT_used_for_evaluation":
        True,

    "sam2_reprompting":
        False,

    "density_used_for_routing":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,
}


with open(
    STEP3D_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step3d_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3D.22 Final display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3D - C1-ANCHORED SELECTIVE D RECOVERY")
print("=" * 125)

print(
    "C1 raw recovery points      :",
    len(
        c1_recovery_df
    )
)

print(
    "D6B9 raw recovery points    :",
    len(
        d6b9_recovery_df
    )
)

print(
    "D6B-T raw recovery points   :",
    len(
        d6bt_recovery_df
    )
)

print(
    "\nGT used for point selection : NO"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nD points surviving C1 support:"
)

display(
    step3d_support_summary_df.round(
        4
    )
)


print(
    "\nOverall count performance:"
)

display(
    step3d_summary_df[
        [
            "system",
            "additional_D_points",
            "total_predicted_count",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_B0",
            "delta_RMSE_vs_B0",
            "delta_MAE_vs_C1",
            "delta_RMSE_vs_C1",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nHelp / hurt relative to C1 baseline:"
)

display(
    step3d_help_hurt_df
)


print(
    "\nBy density stratum:"
)

display(
    step3d_density_df[
        [
            "density_stratum",
            "system",
            "num_images",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nSaved D-point support table:"
)

print(
    STEP3D_POINT_FILE
)

print(
    "\nSaved per-image counts:"
)

print(
    STEP3D_PER_IMAGE_FILE
)

print(
    "\nSaved overall summary:"
)

print(
    STEP3D_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    STEP3D_DENSITY_FILE
)

print(
    "\nSaved help/hurt summary:"
)

print(
    STEP3D_HELP_HURT_FILE
)

print(
    "\nSaved manifest:"
)

print(
    STEP3D_MANIFEST_FILE
)

print("=" * 125)

STEP 3D - C1-ANCHORED SELECTIVE D RECOVERY
C1 raw recovery points      : 1215
D6B9 raw recovery points    : 1911
D6B-T raw recovery points   : 8047

GT used for point selection : NO
SAM2 re-prompting           : NO
Density used for routing    : NO
Test set accessed           : NO

D points surviving C1 support:


,source,total_D_recovery_points,C1_supported_D_points,C1_supported_fraction
0,D6B-T-native,8047,805,0.1000
1,D6B9-strict,1911,288,0.1507



Overall count performance:


,system,additional_D_points,total_predicted_count,MAE,RMSE,mean_signed_error,delta_MAE_vs_B0,delta_RMSE_vs_B0,delta_MAE_vs_C1,delta_RMSE_vs_C1,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,0,2298,80.69,270.2339,-77.47,0.00,0.0000,8.69,7.9986,76,4,20
1,B0_plus_C1_raw,0,3513,72.00,262.2353,-65.32,-8.69,-7.9986,0.00,0.0000,52,5,43
2,B0_plus_C1_plus_D6B9_C1supported,288,3801,71.80,261.3904,-62.44,-8.89,-8.8434,-0.20,-0.8449,45,6,49
3,B0_plus_C1_plus_D6BT_C1supported,805,4318,71.29,258.4460,-57.27,-9.40,-11.7879,-0.71,-3.7893,37,2,61



Help / hurt relative to C1 baseline:


,system,improved_vs_C1,unchanged_vs_C1,worsened_vs_C1
0,B0_plus_C1_plus_D6B9_C1supported,31,37,32
1,B0_plus_C1_plus_D6BT_C1supported,31,28,41



By density stratum:


,density_stratum,system,num_images,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,S1_0_20,B0_unfiltered,20,5.15,6.1115,-0.55,10,1,9
1,S1_0_20,B0_plus_C1_raw,20,4.55,6.1685,4.05,3,3,14
2,S1_0_20,B0_plus_C1_plus_D6B9_C1supported,20,6.05,8.8119,5.85,2,2,16
3,S1_0_20,B0_plus_C1_plus_D6BT_C1supported,20,7.95,12.2413,7.85,1,1,18
4,S2_20_40,B0_unfiltered,20,6.55,8.1086,-5.55,16,2,2
5,S2_20_40,B0_plus_C1_raw,20,5.60,7.0569,3.60,4,1,15
6,S2_20_40,B0_plus_C1_plus_D6B9_C1supported,20,8.50,10.7191,7.50,2,1,17
7,S2_20_40,B0_plus_C1_plus_D6BT_C1supported,20,12.00,15.8871,11.10,2,0,18
8,S3_40_60,B0_unfiltered,20,14.95,17.6423,-13.55,17,1,2
9,S3_40_60,B0_plus_C1_raw,20,8.95,11.3468,-3.65,12,1,7



Saved D-point support table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_anchored_selective_D_points.csv

Saved per-image counts:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_anchored_selective_D_counts_per_image.csv

Saved overall summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_anchored_selective_D_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_anchored_selective_D_by_density.csv

Saved help/hurt summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_anchored_selective_D_help_hurt.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/res

In [27]:
# ============================================================
# Step 3E - C1 Anchor + Complementary D-D Consensus Recovery
#
# Purpose:
# Test whether D6B9-strict and D6B-T-native can provide
# complementary recovery on top of C1 WITHOUT simply adding
# D points that overlap C1 evidence.
#
# Core rule:
#
#   1. Keep all C1 raw recovery points.
#
#   2. Consider D6B9-strict and D6B-T-native recovery points
#      that are outside all B0 masks.
#
#   3. One-to-one match D6B9 <-> D6B-T within frozen radius R.
#
#   4. A D-D pair is "complementary" only if BOTH D points are
#      farther than R from every C1 recovery point.
#
#   5. Each accepted D-D pair contributes exactly +1 count.
#
# Compared systems:
#   - B0
#   - B0 + C1 raw
#   - Step 3D best reference:
#       B0 + C1 + C1-supported D6B-T
#   - B0 + C1 + complementary D-D consensus
#
# IMPORTANT:
# - GT is NOT used for selecting recovery points
# - one D-D pair contributes +1, not +2
# - no SAM2 re-prompting
# - no density routing
# - no threshold tuning
# - test set untouched
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3E.1 Output files
# ------------------------------------------------------------

STEP3E_PAIR_FILE = (
    B2_TABLES_DIR
    / "B0_C1_complementary_DD_consensus_pairs.csv"
)

STEP3E_PER_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_C1_complementary_DD_consensus_counts_per_image.csv"
)

STEP3E_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_complementary_DD_consensus_summary.csv"
)

STEP3E_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_complementary_DD_consensus_by_density.csv"
)

STEP3E_HELP_HURT_FILE = (
    B2_TABLES_DIR
    / "B0_C1_complementary_DD_consensus_help_hurt.csv"
)

STEP3E_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3E_manifest.json"
)


# ------------------------------------------------------------
# 3E.2 Preconditions
# ------------------------------------------------------------

assert (
    "recovery_point_long_df"
    in globals()
), "Run Step 3A first."

assert (
    "gt_image_df"
    in globals()
), "Run Step 3B first."

assert (
    "recovery_count_df"
    in globals()
), "Run Step 3C first."

assert (
    "step3d_count_df"
    in globals()
), "Run Step 3D first."

assert (
    len(
        recovery_count_df
    )
    ==
    100
)


# ------------------------------------------------------------
# 3E.3 Extract recovery points
# ------------------------------------------------------------

def get_recovery_source(
    candidate_id
):

    return (
        recovery_point_long_df[
            (
                recovery_point_long_df[
                    "candidate_id"
                ]
                ==
                candidate_id
            )
            &
            (
                recovery_point_long_df[
                    "recovery_candidate"
                ]
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


c1_recovery_df = get_recovery_source(
    "C1"
)

d6b9_recovery_df = get_recovery_source(
    "D6B9-strict"
)

d6bt_recovery_df = get_recovery_source(
    "D6B-T-native"
)


assert len(c1_recovery_df) == 1215
assert len(d6b9_recovery_df) == 1911
assert len(d6bt_recovery_df) == 8047


# ------------------------------------------------------------
# 3E.4 Per-image frozen radius lookup
# ------------------------------------------------------------

radius_lookup = (
    gt_image_df[
        [
            "dataset_index",
            "matching_radius_px",
        ]
    ]
    .drop_duplicates(
        subset=[
            "dataset_index"
        ]
    )
    .set_index(
        "dataset_index"
    )
)


assert (
    len(
        radius_lookup
    )
    ==
    100
)


# ------------------------------------------------------------
# 3E.5 Per-image point lookups
# ------------------------------------------------------------

def make_xy_lookup(
    df
):

    lookup = {}


    for dataset_index, group in (
        df.groupby(
            "dataset_index"
        )
    ):

        lookup[
            int(
                dataset_index
            )
        ] = (
            group[
                [
                    "x",
                    "y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


    return lookup


c1_xy_lookup = make_xy_lookup(
    c1_recovery_df
)

d6b9_xy_lookup = make_xy_lookup(
    d6b9_recovery_df
)

d6bt_xy_lookup = make_xy_lookup(
    d6bt_recovery_df
)


EMPTY_XY = np.empty(
    (
        0,
        2,
    ),
    dtype=float,
)


# ------------------------------------------------------------
# 3E.6 Helper:
# nearest distance to a point set
# ------------------------------------------------------------

def nearest_distance_to_set(
    point_xy,
    ref_xy
):

    if len(
        ref_xy
    ) == 0:

        return np.inf


    point_xy = np.asarray(
        point_xy,
        dtype=float,
    )

    ref_xy = np.asarray(
        ref_xy,
        dtype=float,
    )


    distances = np.hypot(
        ref_xy[:, 0]
        -
        point_xy[
            0
        ],

        ref_xy[:, 1]
        -
        point_xy[
            1
        ],
    )


    return float(
        distances.min()
    )


# ------------------------------------------------------------
# 3E.7 One-to-one D6B9 <-> D6B-T matching
#
# Greedy globally-shortest valid pairing within R.
#
# This is the same style of one-to-one spatial matching used
# in Step 3B, but here it is entirely inference-side:
# D source <-> D source.
# ------------------------------------------------------------

def one_to_one_DD_match(
    d6b9_xy,
    d6bt_xy,
    radius_px,
):

    d6b9_xy = np.asarray(
        d6b9_xy,
        dtype=float,
    )

    d6bt_xy = np.asarray(
        d6bt_xy,
        dtype=float,
    )


    if (
        len(
            d6b9_xy
        )
        ==
        0
        or
        len(
            d6bt_xy
        )
        ==
        0
    ):

        return []


    candidate_pairs = []


    for i in range(
        len(
            d6b9_xy
        )
    ):

        x1, y1 = d6b9_xy[
            i
        ]


        # vectorised distances from this D6B9 point
        # to all D6B-T points

        distances = np.hypot(
            d6bt_xy[
                :,
                0
            ]
            -
            x1,

            d6bt_xy[
                :,
                1
            ]
            -
            y1,
        )


        valid_js = np.where(
            distances
            <=
            radius_px
        )[
            0
        ]


        for j in valid_js:

            candidate_pairs.append(
                (
                    float(
                        distances[
                            j
                        ]
                    ),
                    int(
                        i
                    ),
                    int(
                        j
                    ),
                )
            )


    # globally shortest valid pair first

    candidate_pairs.sort(
        key=lambda z: (
            z[
                0
            ],
            z[
                1
            ],
            z[
                2
            ],
        )
    )


    used_d6b9 = set()
    used_d6bt = set()

    matches = []


    for (
        distance,
        i,
        j
    ) in candidate_pairs:

        if i in used_d6b9:

            continue

        if j in used_d6bt:

            continue


        used_d6b9.add(
            i
        )

        used_d6bt.add(
            j
        )


        matches.append(
            {
                "d6b9_local_index":
                    i,

                "d6bt_local_index":
                    j,

                "DD_distance_px":
                    distance,
            }
        )


    return matches


# ------------------------------------------------------------
# 3E.8 Construct D-D pairs and test complementarity to C1
# ------------------------------------------------------------

pair_records = []

image_pair_records = []


for image_row in (
    b2_dev100_image_df
    .itertuples(
        index=False
    )
):

    dataset_index = int(
        image_row.dataset_index
    )


    radius_px = float(
        radius_lookup.loc[
            dataset_index,
            "matching_radius_px",
        ]
    )


    c1_xy = c1_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )

    d6b9_xy = d6b9_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )

    d6bt_xy = d6bt_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )


    matches = one_to_one_DD_match(
        d6b9_xy=d6b9_xy,
        d6bt_xy=d6bt_xy,
        radius_px=radius_px,
    )


    num_all_pairs = int(
        len(
            matches
        )
    )

    num_complementary_pairs = 0


    for pair_index, match in enumerate(
        matches
    ):

        d6b9_idx = int(
            match[
                "d6b9_local_index"
            ]
        )

        d6bt_idx = int(
            match[
                "d6bt_local_index"
            ]
        )


        d6b9_point = d6b9_xy[
            d6b9_idx
        ]

        d6bt_point = d6bt_xy[
            d6bt_idx
        ]


        d6b9_to_c1 = nearest_distance_to_set(
            d6b9_point,
            c1_xy,
        )

        d6bt_to_c1 = nearest_distance_to_set(
            d6bt_point,
            c1_xy,
        )


        # Strict complementary rule:
        # BOTH D endpoints must be farther than R from C1.

        complementary_to_C1 = bool(
            (
                d6b9_to_c1
                >
                radius_px
            )
            and
            (
                d6bt_to_c1
                >
                radius_px
            )
        )


        if complementary_to_C1:

            num_complementary_pairs += 1


        # midpoint is stored for diagnostics / possible later SAM2 prompt
        midpoint_x = float(
            (
                d6b9_point[
                    0
                ]
                +
                d6bt_point[
                    0
                ]
            )
            /
            2.0
        )

        midpoint_y = float(
            (
                d6b9_point[
                    1
                ]
                +
                d6bt_point[
                    1
                ]
            )
            /
            2.0
        )


        pair_records.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_row.image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "gt_count":
                    int(
                        image_row.gt_count
                    ),

                "pair_index":
                    int(
                        pair_index
                    ),

                "matching_radius_px":
                    radius_px,

                "d6b9_x":
                    float(
                        d6b9_point[
                            0
                        ]
                    ),

                "d6b9_y":
                    float(
                        d6b9_point[
                            1
                        ]
                    ),

                "d6bt_x":
                    float(
                        d6bt_point[
                            0
                        ]
                    ),

                "d6bt_y":
                    float(
                        d6bt_point[
                            1
                        ]
                    ),

                "DD_distance_px":
                    float(
                        match[
                            "DD_distance_px"
                        ]
                    ),

                "d6b9_nearest_C1_distance_px":
                    (
                        float(
                            d6b9_to_c1
                        )
                        if np.isfinite(
                            d6b9_to_c1
                        )
                        else np.nan
                    ),

                "d6bt_nearest_C1_distance_px":
                    (
                        float(
                            d6bt_to_c1
                        )
                        if np.isfinite(
                            d6bt_to_c1
                        )
                        else np.nan
                    ),

                "complementary_to_C1":
                    complementary_to_C1,

                "representative_midpoint_x":
                    midpoint_x,

                "representative_midpoint_y":
                    midpoint_y,
            }
        )


    image_pair_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_row.image_id,

            "all_DD_pairs":
                num_all_pairs,

            "complementary_DD_pairs":
                int(
                    num_complementary_pairs
                ),
        }
    )


step3e_pair_df = pd.DataFrame(
    pair_records
)

step3e_image_pair_df = pd.DataFrame(
    image_pair_records
)


assert (
    len(
        step3e_image_pair_df
    )
    ==
    100
)


# ------------------------------------------------------------
# 3E.9 Global pair summary
# ------------------------------------------------------------

TOTAL_DD_PAIRS = int(
    len(
        step3e_pair_df
    )
)

TOTAL_COMPLEMENTARY_DD_PAIRS = int(
    step3e_pair_df[
        "complementary_to_C1"
    ].sum()
)


if TOTAL_DD_PAIRS > 0:

    COMPLEMENTARY_DD_FRACTION = (
        TOTAL_COMPLEMENTARY_DD_PAIRS
        /
        TOTAL_DD_PAIRS
    )

else:

    COMPLEMENTARY_DD_FRACTION = np.nan


# ------------------------------------------------------------
# 3E.10 Construct per-image count table
# ------------------------------------------------------------

step3e_count_df = (
    step3d_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "B0_plus_C1_plus_D6BT_C1supported",
        ]
    ]
    .copy()
)


step3e_count_df = (
    step3e_count_df
    .merge(
        step3e_image_pair_df[
            [
                "dataset_index",
                "all_DD_pairs",
                "complementary_DD_pairs",
            ]
        ],
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


step3e_count_df[
    "all_DD_pairs"
] = (
    step3e_count_df[
        "all_DD_pairs"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


step3e_count_df[
    "complementary_DD_pairs"
] = (
    step3e_count_df[
        "complementary_DD_pairs"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


# Each accepted D-D pair contributes exactly +1.

step3e_count_df[
    "B0_plus_C1_plus_complementary_DD"
] = (
    step3e_count_df[
        "B0_plus_C1_raw"
    ]
    +
    step3e_count_df[
        "complementary_DD_pairs"
    ]
)


# ------------------------------------------------------------
# 3E.11 Metric helper
# ------------------------------------------------------------

def count_metrics_3e(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )


    error = (
        pred
        -
        gt
    )


    return {
        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error**2
                    )
                )
            ),

        "mean_signed_error":
            float(
                np.mean(
                    error
                )
            ),

        "under_count_images":
            int(
                np.sum(
                    error
                    <
                    0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    error
                    ==
                    0
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    error
                    >
                    0
                )
            ),
    }


# ------------------------------------------------------------
# 3E.12 Overall performance
# ------------------------------------------------------------

STEP3E_SYSTEMS = [
    "B0_unfiltered",
    "B0_plus_C1_raw",
    "B0_plus_C1_plus_D6BT_C1supported",
    "B0_plus_C1_plus_complementary_DD",
]


gt_all = (
    step3e_count_df[
        "gt_count"
    ]
    .to_numpy()
)


summary_records = []


for system_name in STEP3E_SYSTEMS:

    pred = (
        step3e_count_df[
            system_name
        ]
        .to_numpy()
    )


    metrics = count_metrics_3e(
        gt_all,
        pred,
    )


    if (
        system_name
        ==
        "B0_plus_C1_plus_complementary_DD"
    ):

        additional_recovery = (
            TOTAL_COMPLEMENTARY_DD_PAIRS
        )

    elif (
        system_name
        ==
        "B0_plus_C1_plus_D6BT_C1supported"
    ):

        additional_recovery = int(
            step3d_count_df[
                "D6BT_C1_supported_added"
            ].sum()
        )

    else:

        additional_recovery = 0


    summary_records.append(
        {
            "system":
                system_name,

            "additional_recovery_beyond_C1":
                int(
                    additional_recovery
                ),

            "total_predicted_count":
                int(
                    pred.sum()
                ),

            **metrics,
        }
    )


step3e_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 3E.13 Deltas vs B0 and C1
# ------------------------------------------------------------

b0_row = (
    step3e_summary_df[
        step3e_summary_df[
            "system"
        ]
        ==
        "B0_unfiltered"
    ]
    .iloc[
        0
    ]
)


c1_row = (
    step3e_summary_df[
        step3e_summary_df[
            "system"
        ]
        ==
        "B0_plus_C1_raw"
    ]
    .iloc[
        0
    ]
)


for metric in [
    "MAE",
    "RMSE",
]:

    step3e_summary_df[
        f"delta_{metric}_vs_B0"
    ] = (
        step3e_summary_df[
            metric
        ]
        -
        float(
            b0_row[
                metric
            ]
        )
    )


    step3e_summary_df[
        f"delta_{metric}_vs_C1"
    ] = (
        step3e_summary_df[
            metric
        ]
        -
        float(
            c1_row[
                metric
            ]
        )
    )


# ------------------------------------------------------------
# 3E.14 Per-image absolute errors
# ------------------------------------------------------------

for system_name in STEP3E_SYSTEMS:

    step3e_count_df[
        f"{system_name}__abs_error"
    ] = np.abs(
        step3e_count_df[
            system_name
        ]
        -
        step3e_count_df[
            "gt_count"
        ]
    )


# ------------------------------------------------------------
# 3E.15 Help / hurt relative to C1
# ------------------------------------------------------------

c1_abs = (
    step3e_count_df[
        "B0_plus_C1_raw__abs_error"
    ]
)


help_hurt_records = []


for system_name in [
    "B0_plus_C1_plus_D6BT_C1supported",
    "B0_plus_C1_plus_complementary_DD",
]:

    new_abs = (
        step3e_count_df[
            f"{system_name}__abs_error"
        ]
    )


    help_hurt_records.append(
        {
            "system":
                system_name,

            "improved_vs_C1":
                int(
                    (
                        new_abs
                        <
                        c1_abs
                    ).sum()
                ),

            "unchanged_vs_C1":
                int(
                    (
                        new_abs
                        ==
                        c1_abs
                    ).sum()
                ),

            "worsened_vs_C1":
                int(
                    (
                        new_abs
                        >
                        c1_abs
                    ).sum()
                ),
        }
    )


step3e_help_hurt_df = pd.DataFrame(
    help_hurt_records
)


# ------------------------------------------------------------
# 3E.16 Density-stratum diagnostics
#
# Diagnostic only.
# Density is NOT used in inference.
# ------------------------------------------------------------

density_records = []


for stratum in DENSITY_ORDER:

    subset = (
        step3e_count_df[
            step3e_count_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    gt = (
        subset[
            "gt_count"
        ]
        .to_numpy()
    )


    for system_name in STEP3E_SYSTEMS:

        pred = (
            subset[
                system_name
            ]
            .to_numpy()
        )


        metrics = count_metrics_3e(
            gt,
            pred,
        )


        if (
            system_name
            ==
            "B0_plus_C1_plus_complementary_DD"
        ):

            added = int(
                subset[
                    "complementary_DD_pairs"
                ].sum()
            )

        elif (
            system_name
            ==
            "B0_plus_C1_plus_D6BT_C1supported"
        ):

            # obtain Step 3D additions for these images

            image_indices = set(
                subset[
                    "dataset_index"
                ].tolist()
            )

            added = int(
                step3d_count_df[
                    step3d_count_df[
                        "dataset_index"
                    ]
                    .isin(
                        image_indices
                    )
                ][
                    "D6BT_C1_supported_added"
                ]
                .sum()
            )

        else:

            added = 0


        density_records.append(
            {
                "density_stratum":
                    stratum,

                "system":
                    system_name,

                "num_images":
                    int(
                        len(
                            subset
                        )
                    ),

                "additional_recovery_beyond_C1":
                    added,

                **metrics,
            }
        )


step3e_density_df = pd.DataFrame(
    density_records
)


# ------------------------------------------------------------
# 3E.17 Save outputs
# ------------------------------------------------------------

step3e_pair_df.to_csv(
    STEP3E_PAIR_FILE,
    index=False,
)


step3e_count_df.to_csv(
    STEP3E_PER_IMAGE_FILE,
    index=False,
)


step3e_summary_df.to_csv(
    STEP3E_SUMMARY_FILE,
    index=False,
)


step3e_density_df.to_csv(
    STEP3E_DENSITY_FILE,
    index=False,
)


step3e_help_hurt_df.to_csv(
    STEP3E_HELP_HURT_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3E.18 Manifest
# ------------------------------------------------------------

step3e_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3E - C1 anchor plus complementary D-D consensus",

    "anchor_recovery_source":
        "C1",

    "D_sources":
        [
            "D6B9-strict",
            "D6B-T-native",
        ],

    "D_pairing_rule":
        (
            "One-to-one greedy globally shortest D6B9-to-D6B-T "
            "pairing within frozen per-image radius R."
        ),

    "complementary_rule":
        (
            "Both endpoints of the D-D pair must be farther "
            "than R from every C1 recovery point."
        ),

    "count_contribution_per_DD_pair":
        1,

    "GT_used_for_point_selection":
        False,

    "GT_used_for_evaluation":
        True,

    "sam2_reprompting":
        False,

    "density_used_for_routing":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,

    "notes":
        (
            "Step 3E explicitly tests complementary D evidence "
            "rather than adding D points that spatially overlap "
            "the C1 recovery anchor."
        ),
}


with open(
    STEP3E_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step3e_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3E.19 Final display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3E - C1 ANCHOR + COMPLEMENTARY D-D CONSENSUS")
print("=" * 125)

print(
    "C1 raw recovery points      :",
    len(
        c1_recovery_df
    )
)

print(
    "D6B9 raw recovery points    :",
    len(
        d6b9_recovery_df
    )
)

print(
    "D6B-T raw recovery points   :",
    len(
        d6bt_recovery_df
    )
)

print(
    "\nAll one-to-one D-D pairs    :",
    TOTAL_DD_PAIRS
)

print(
    "Complementary D-D pairs     :",
    TOTAL_COMPLEMENTARY_DD_PAIRS
)

print(
    "Complementary pair fraction :",
    (
        round(
            COMPLEMENTARY_DD_FRACTION,
            4,
        )
        if np.isfinite(
            COMPLEMENTARY_DD_FRACTION
        )
        else np.nan
    )
)


print(
    "\nGT used for point selection : NO"
)

print(
    "Each accepted D-D pair      : +1 count"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nOverall count performance:"
)

display(
    step3e_summary_df[
        [
            "system",
            "additional_recovery_beyond_C1",
            "total_predicted_count",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_B0",
            "delta_RMSE_vs_B0",
            "delta_MAE_vs_C1",
            "delta_RMSE_vs_C1",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nHelp / hurt relative to C1:"
)

display(
    step3e_help_hurt_df
)


print(
    "\nPerformance by density stratum:"
)

display(
    step3e_density_df[
        [
            "density_stratum",
            "system",
            "num_images",
            "additional_recovery_beyond_C1",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nImages with most complementary D-D recovery:"
)

display(
    step3e_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "all_DD_pairs",
            "complementary_DD_pairs",
            "B0_plus_C1_plus_complementary_DD",
        ]
    ]
    .sort_values(
        "complementary_DD_pairs",
        ascending=False,
    )
    .head(
        30
    )
)


print(
    "\nSaved D-D pair diagnostics:"
)

print(
    STEP3E_PAIR_FILE
)

print(
    "\nSaved per-image count results:"
)

print(
    STEP3E_PER_IMAGE_FILE
)

print(
    "\nSaved overall summary:"
)

print(
    STEP3E_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    STEP3E_DENSITY_FILE
)

print(
    "\nSaved help/hurt summary:"
)

print(
    STEP3E_HELP_HURT_FILE
)

print(
    "\nSaved manifest:"
)

print(
    STEP3E_MANIFEST_FILE
)

print("=" * 125)

STEP 3E - C1 ANCHOR + COMPLEMENTARY D-D CONSENSUS
C1 raw recovery points      : 1215
D6B9 raw recovery points    : 1911
D6B-T raw recovery points   : 8047

All one-to-one D-D pairs    : 1658
Complementary D-D pairs     : 1355
Complementary pair fraction : 0.8172

GT used for point selection : NO
Each accepted D-D pair      : +1 count
SAM2 re-prompting           : NO
Density used for routing    : NO
Threshold tuning            : NO
Test set accessed           : NO

Overall count performance:


,system,additional_recovery_beyond_C1,total_predicted_count,MAE,RMSE,mean_signed_error,delta_MAE_vs_B0,delta_RMSE_vs_B0,delta_MAE_vs_C1,delta_RMSE_vs_C1,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,0,2298,80.69,270.2339,-77.47,0.00,0.0000,8.69,7.9986,76,4,20
1,B0_plus_C1_raw,0,3513,72.00,262.2353,-65.32,-8.69,-7.9986,0.00,0.0000,52,5,43
2,B0_plus_C1_plus_D6BT_C1supported,805,4318,71.29,258.4460,-57.27,-9.40,-11.7879,-0.71,-3.7893,37,2,61
3,B0_plus_C1_plus_complementary_DD,1355,4868,75.99,261.2695,-51.77,-4.70,-8.9644,3.99,-0.9658,34,4,62



Help / hurt relative to C1:


,system,improved_vs_C1,unchanged_vs_C1,worsened_vs_C1
0,B0_plus_C1_plus_D6BT_C1supported,31,28,41
1,B0_plus_C1_plus_complementary_DD,32,22,46



Performance by density stratum:


,density_stratum,system,num_images,additional_recovery_beyond_C1,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,S1_0_20,B0_unfiltered,20,0,5.15,6.1115,-0.55,10,1,9
1,S1_0_20,B0_plus_C1_raw,20,0,4.55,6.1685,4.05,3,3,14
2,S1_0_20,B0_plus_C1_plus_D6BT_C1supported,20,76,7.95,12.2413,7.85,1,1,18
3,S1_0_20,B0_plus_C1_plus_complementary_DD,20,124,10.25,13.3098,10.25,0,1,19
4,S2_20_40,B0_unfiltered,20,0,6.55,8.1086,-5.55,16,2,2
5,S2_20_40,B0_plus_C1_raw,20,0,5.60,7.0569,3.60,4,1,15
6,S2_20_40,B0_plus_C1_plus_D6BT_C1supported,20,150,12.00,15.8871,11.10,2,0,18
7,S2_20_40,B0_plus_C1_plus_complementary_DD,20,301,18.85,24.4264,18.65,1,2,17
8,S3_40_60,B0_unfiltered,20,0,14.95,17.6423,-13.55,17,1,2
9,S3_40_60,B0_plus_C1_raw,20,0,8.95,11.3468,-3.65,12,1,7



Images with most complementary D-D recovery:


,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw,all_DD_pairs,complementary_DD_pairs,B0_plus_C1_plus_complementary_DD
77,938,5866.jpg,grapes,S6_95_100,267,47,65,94,90,155
98,1260,7100.jpg,milk cartons,S3_40_60,27,40,48,73,69,117
1,14,220.jpg,grapes,S4_60_80,59,20,52,67,66,118
24,183,838.jpg,books,S5_80_95,113,43,70,65,52,122
45,425,2884.jpg,chairs,S5_80_95,165,38,46,55,51,97
99,1269,7060.jpg,pills,S4_60_80,65,11,25,61,50,75
26,217,919.jpg,skateboard,S3_40_60,32,11,18,53,50,68
69,845,5101.jpg,chicken wings,S3_40_60,23,24,30,51,49,79
43,396,2846.jpg,chairs,S4_60_80,38,7,30,58,49,79
48,488,3543.jpg,chicken wings,S3_40_60,30,28,40,44,40,80



Saved D-D pair diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_complementary_DD_consensus_pairs.csv

Saved per-image count results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_complementary_DD_consensus_counts_per_image.csv

Saved overall summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_complementary_DD_consensus_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_complementary_DD_consensus_by_density.csv

Saved help/hurt summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_complementary_DD_consensus_help_hurt.csv

Saved manifest:
/content/drive/MyDriv

In [28]:
# ============================================================
# Step 3F - Incremental GT Quality After B0 + C1
#
# Purpose:
# Explain why Step 3D (C1-supported D6B-T) performed better
# than Step 3E (complementary D-D consensus).
#
# Diagnostic target:
#
#   GT missed by B0
#       MINUS
#   GT already recovered by C1
#       =
#   residual GT after B0 + C1
#
# We then evaluate:
#
#   A. Step 3D accepted D6B-T-native points
#   B. Step 3E accepted complementary D-D pair representatives
#
# against ONLY this residual GT population.
#
# IMPORTANT:
# - GT is diagnostic only
# - no count rule is changed
# - no threshold selected
# - no density routing
# - no SAM2 re-prompting
# - test set untouched
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3F.1 Output files
# ------------------------------------------------------------

STEP3F_POINT_FILE = (
    B2_TABLES_DIR
    / "B0_C1_incremental_D_quality_long.csv"
)

STEP3F_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_incremental_D_quality_summary.csv"
)

STEP3F_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_incremental_D_quality_by_density.csv"
)

STEP3F_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_C1_incremental_D_quality_per_image.csv"
)

STEP3F_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3F_manifest.json"
)


# ------------------------------------------------------------
# 3F.2 Preconditions
# ------------------------------------------------------------

assert (
    "gt_point_status_df"
    in globals()
), "Run Step 3B first."

assert (
    "recovery_gt_match_long_df"
    in globals()
), "Run Step 3B first."

assert (
    "step3d_point_df"
    in globals()
), "Run Step 3D first."

assert (
    "step3e_pair_df"
    in globals()
), "Run Step 3E first."

assert (
    "gt_image_df"
    in globals()
), "Run Step 3B first."

assert (
    len(
        gt_point_status_df
    )
    ==
    10045
)


# ------------------------------------------------------------
# 3F.3 Identify GT already recovered by C1
#
# Step 3B matched C1 only against GT missed by B0.
# Therefore these are genuine incremental C1 recoveries.
# ------------------------------------------------------------

c1_matched_df = (
    recovery_gt_match_long_df[
        (
            recovery_gt_match_long_df[
                "candidate_id"
            ]
            ==
            "C1"
        )
        &
        (
            recovery_gt_match_long_df[
                "matched_missed_gt"
            ]
        )
    ]
    .copy()
)


assert (
    len(
        c1_matched_df
    )
    ==
    623
), (
    "Expected 623 C1-recovered B0-missed GT from Step 3B."
)


c1_recovered_gt_keys = set(
    zip(
        c1_matched_df[
            "dataset_index"
        ].astype(
            int
        ),

        c1_matched_df[
            "matched_gt_index"
        ].astype(
            int
        ),
    )
)


# ------------------------------------------------------------
# 3F.4 Build residual GT population after B0 + C1
# ------------------------------------------------------------

residual_gt_df = (
    gt_point_status_df[
        gt_point_status_df[
            "missed_by_B0"
        ]
    ]
    .copy()
)


residual_gt_df[
    "recovered_by_C1"
] = [
    (
        int(
            row.dataset_index
        ),
        int(
            row.gt_index
        ),
    )
    in
    c1_recovered_gt_keys

    for row in residual_gt_df.itertuples(
        index=False
    )
]


residual_gt_df = (
    residual_gt_df[
        ~residual_gt_df[
            "recovered_by_C1"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


TOTAL_B0_MISSED_GT = int(
    gt_point_status_df[
        "missed_by_B0"
    ].sum()
)


TOTAL_C1_RECOVERED_GT = int(
    len(
        c1_recovered_gt_keys
    )
)


TOTAL_RESIDUAL_GT = int(
    len(
        residual_gt_df
    )
)


assert (
    TOTAL_B0_MISSED_GT
    ==
    5641
)

assert (
    TOTAL_C1_RECOVERED_GT
    ==
    623
)

assert (
    TOTAL_RESIDUAL_GT
    ==
    (
        5641
        -
        623
    )
)


# ------------------------------------------------------------
# 3F.5 Prepare Step 3D accepted additions
#
# Step 3D best system:
# C1-supported D6B-T-native points.
# ------------------------------------------------------------

step3d_eval_df = (
    step3d_point_df[
        (
            step3d_point_df[
                "source"
            ]
            ==
            "D6B-T-native"
        )
        &
        (
            step3d_point_df[
                "supported_by_C1_recovery"
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert (
    len(
        step3d_eval_df
    )
    ==
    805
)


step3d_eval_df[
    "eval_source"
] = (
    "Step3D_C1supported_D6BT"
)


step3d_eval_df[
    "eval_x"
] = (
    step3d_eval_df[
        "x"
    ].astype(
        float
    )
)


step3d_eval_df[
    "eval_y"
] = (
    step3d_eval_df[
        "y"
    ].astype(
        float
    )
)


# ------------------------------------------------------------
# 3F.6 Prepare Step 3E accepted additions
#
# One complementary D-D pair contributed +1 count.
# Therefore evaluate ONE representative point per pair.
#
# Use the midpoint already stored in Step 3E.
# ------------------------------------------------------------

step3e_eval_df = (
    step3e_pair_df[
        step3e_pair_df[
            "complementary_to_C1"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert (
    len(
        step3e_eval_df
    )
    ==
    1355
)


step3e_eval_df[
    "eval_source"
] = (
    "Step3E_complementary_DD"
)


step3e_eval_df[
    "eval_x"
] = (
    step3e_eval_df[
        "representative_midpoint_x"
    ].astype(
        float
    )
)


step3e_eval_df[
    "eval_y"
] = (
    step3e_eval_df[
        "representative_midpoint_y"
    ].astype(
        float
    )
)


# ------------------------------------------------------------
# 3F.7 Standardise evaluation candidate tables
# ------------------------------------------------------------

COMMON_COLS = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "eval_source",
    "eval_x",
    "eval_y",
]


step3f_candidate_df = pd.concat(
    [
        step3d_eval_df[
            COMMON_COLS
        ],

        step3e_eval_df[
            COMMON_COLS
        ],
    ],
    ignore_index=True,
)


assert (
    len(
        step3f_candidate_df
    )
    ==
    (
        805
        +
        1355
    )
)


# ------------------------------------------------------------
# 3F.8 Frozen radius lookup
# ------------------------------------------------------------

radius_lookup_3f = (
    gt_image_df[
        [
            "dataset_index",
            "matching_radius_px",
        ]
    ]
    .drop_duplicates(
        subset=[
            "dataset_index"
        ]
    )
    .set_index(
        "dataset_index"
    )
)


assert (
    len(
        radius_lookup_3f
    )
    ==
    100
)


# ------------------------------------------------------------
# 3F.9 One-to-one radius matching helper
#
# Same matching principle as Step 3B:
# globally shortest valid pairs within R.
# ------------------------------------------------------------

def one_to_one_match_3f(
    pred_xy,
    gt_xy,
    radius_px,
):

    pred_xy = np.asarray(
        pred_xy,
        dtype=float,
    )

    gt_xy = np.asarray(
        gt_xy,
        dtype=float,
    )


    if (
        len(
            pred_xy
        )
        ==
        0
        or
        len(
            gt_xy
        )
        ==
        0
    ):

        return []


    candidate_pairs = []


    for pred_i in range(
        len(
            pred_xy
        )
    ):

        px, py = pred_xy[
            pred_i
        ]


        distances = np.hypot(
            gt_xy[
                :,
                0
            ]
            -
            px,

            gt_xy[
                :,
                1
            ]
            -
            py,
        )


        valid_gt = np.where(
            distances
            <=
            radius_px
        )[
            0
        ]


        for gt_i in valid_gt:

            candidate_pairs.append(
                (
                    float(
                        distances[
                            gt_i
                        ]
                    ),
                    int(
                        pred_i
                    ),
                    int(
                        gt_i
                    ),
                )
            )


    candidate_pairs.sort(
        key=lambda z: (
            z[
                0
            ],
            z[
                1
            ],
            z[
                2
            ],
        )
    )


    used_pred = set()
    used_gt = set()

    matches = []


    for (
        distance,
        pred_i,
        gt_i
    ) in candidate_pairs:

        if pred_i in used_pred:

            continue

        if gt_i in used_gt:

            continue


        used_pred.add(
            pred_i
        )

        used_gt.add(
            gt_i
        )


        matches.append(
            {
                "pred_local_index":
                    int(
                        pred_i
                    ),

                "gt_local_index":
                    int(
                        gt_i
                    ),

                "distance_px":
                    float(
                        distance
                    ),
            }
        )


    return matches


# ------------------------------------------------------------
# 3F.10 Match each accepted recovery set against
#       residual GT after B0 + C1
# ------------------------------------------------------------

point_records = []

image_records = []


EVAL_SOURCES = [
    "Step3D_C1supported_D6BT",
    "Step3E_complementary_DD",
]


for eval_source in EVAL_SOURCES:

    source_df = (
        step3f_candidate_df[
            step3f_candidate_df[
                "eval_source"
            ]
            ==
            eval_source
        ]
    )


    for image_row in (
        b2_dev100_image_df
        .itertuples(
            index=False
        )
    ):

        dataset_index = int(
            image_row.dataset_index
        )


        pred_subset = (
            source_df[
                source_df[
                    "dataset_index"
                ]
                ==
                dataset_index
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        residual_subset = (
            residual_gt_df[
                residual_gt_df[
                    "dataset_index"
                ]
                ==
                dataset_index
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        pred_xy = (
            pred_subset[
                [
                    "eval_x",
                    "eval_y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


        gt_xy = (
            residual_subset[
                [
                    "gt_x",
                    "gt_y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


        radius_px = float(
            radius_lookup_3f.loc[
                dataset_index,
                "matching_radius_px",
            ]
        )


        matches = one_to_one_match_3f(
            pred_xy=pred_xy,
            gt_xy=gt_xy,
            radius_px=radius_px,
        )


        match_by_pred = {
            int(
                m[
                    "pred_local_index"
                ]
            ):
            m

            for m in matches
        }


        num_candidates = int(
            len(
                pred_subset
            )
        )

        num_residual_gt = int(
            len(
                residual_subset
            )
        )

        num_tp = int(
            len(
                matches
            )
        )

        num_unsupported = (
            num_candidates
            -
            num_tp
        )


        # Point-level records

        for pred_i, pred_row in (
            pred_subset.iterrows()
        ):

            matched = (
                int(
                    pred_i
                )
                in
                match_by_pred
            )


            if matched:

                match_info = match_by_pred[
                    int(
                        pred_i
                    )
                ]

                gt_local_index = int(
                    match_info[
                        "gt_local_index"
                    ]
                )

                matched_gt_row = (
                    residual_subset.iloc[
                        gt_local_index
                    ]
                )


                matched_gt_index = int(
                    matched_gt_row[
                        "gt_index"
                    ]
                )

                distance_px = float(
                    match_info[
                        "distance_px"
                    ]
                )

            else:

                matched_gt_index = np.nan
                distance_px = np.nan


            point_records.append(
                {
                    "eval_source":
                        eval_source,

                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_row.image_id,

                    "category":
                        image_row.category,

                    "density_stratum":
                        image_row.density_stratum,

                    "gt_count":
                        int(
                            image_row.gt_count
                        ),

                    "residual_gt_after_B0_C1":
                        num_residual_gt,

                    "matching_radius_px":
                        radius_px,

                    "eval_x":
                        float(
                            pred_row[
                                "eval_x"
                            ]
                        ),

                    "eval_y":
                        float(
                            pred_row[
                                "eval_y"
                            ]
                        ),

                    "matched_residual_gt":
                        bool(
                            matched
                        ),

                    "matched_gt_index":
                        matched_gt_index,

                    "match_distance_px":
                        distance_px,

                    "unsupported_incremental":
                        bool(
                            not matched
                        ),
                }
            )


        image_records.append(
            {
                "eval_source":
                    eval_source,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_row.image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "gt_count":
                    int(
                        image_row.gt_count
                    ),

                "residual_gt_after_B0_C1":
                    num_residual_gt,

                "accepted_recovery_candidates":
                    num_candidates,

                "incremental_recovered_gt":
                    num_tp,

                "unsupported_incremental":
                    num_unsupported,

                "unrecovered_residual_gt":
                    (
                        num_residual_gt
                        -
                        num_tp
                    ),
            }
        )


step3f_point_df = pd.DataFrame(
    point_records
)


step3f_image_df = pd.DataFrame(
    image_records
)


assert (
    len(
        step3f_image_df
    )
    ==
    200
)


# ------------------------------------------------------------
# 3F.11 Overall incremental quality summary
# ------------------------------------------------------------

step3f_summary_df = (
    step3f_image_df
    .groupby(
        "eval_source",
        as_index=False,
    )
    .agg(
        total_residual_gt_after_B0_C1=(
            "residual_gt_after_B0_C1",
            "sum",
        ),

        accepted_recovery_candidates=(
            "accepted_recovery_candidates",
            "sum",
        ),

        incremental_recovered_gt=(
            "incremental_recovered_gt",
            "sum",
        ),

        unsupported_incremental=(
            "unsupported_incremental",
            "sum",
        ),

        unrecovered_residual_gt=(
            "unrecovered_residual_gt",
            "sum",
        ),
    )
)


step3f_summary_df[
    "incremental_precision_like"
] = np.where(
    step3f_summary_df[
        "accepted_recovery_candidates"
    ]
    >
    0,

    step3f_summary_df[
        "incremental_recovered_gt"
    ]
    /
    step3f_summary_df[
        "accepted_recovery_candidates"
    ],

    np.nan,
)


step3f_summary_df[
    "residual_GT_recall"
] = np.where(
    step3f_summary_df[
        "total_residual_gt_after_B0_C1"
    ]
    >
    0,

    step3f_summary_df[
        "incremental_recovered_gt"
    ]
    /
    step3f_summary_df[
        "total_residual_gt_after_B0_C1"
    ],

    np.nan,
)


step3f_summary_df[
    "FP_per_incremental_TP"
] = np.where(
    step3f_summary_df[
        "incremental_recovered_gt"
    ]
    >
    0,

    step3f_summary_df[
        "unsupported_incremental"
    ]
    /
    step3f_summary_df[
        "incremental_recovered_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3F.12 Density diagnostic
# ------------------------------------------------------------

step3f_density_df = (
    step3f_image_df
    .groupby(
        [
            "density_stratum",
            "eval_source",
        ],
        as_index=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        residual_gt_after_B0_C1=(
            "residual_gt_after_B0_C1",
            "sum",
        ),

        accepted_recovery_candidates=(
            "accepted_recovery_candidates",
            "sum",
        ),

        incremental_recovered_gt=(
            "incremental_recovered_gt",
            "sum",
        ),

        unsupported_incremental=(
            "unsupported_incremental",
            "sum",
        ),
    )
)


step3f_density_df[
    "incremental_precision_like"
] = np.where(
    step3f_density_df[
        "accepted_recovery_candidates"
    ]
    >
    0,

    step3f_density_df[
        "incremental_recovered_gt"
    ]
    /
    step3f_density_df[
        "accepted_recovery_candidates"
    ],

    np.nan,
)


step3f_density_df[
    "residual_GT_recall"
] = np.where(
    step3f_density_df[
        "residual_gt_after_B0_C1"
    ]
    >
    0,

    step3f_density_df[
        "incremental_recovered_gt"
    ]
    /
    step3f_density_df[
        "residual_gt_after_B0_C1"
    ],

    np.nan,
)


step3f_density_df[
    "FP_per_incremental_TP"
] = np.where(
    step3f_density_df[
        "incremental_recovered_gt"
    ]
    >
    0,

    step3f_density_df[
        "unsupported_incremental"
    ]
    /
    step3f_density_df[
        "incremental_recovered_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3F.13 Integrity checks
# ------------------------------------------------------------

residual_totals = (
    step3f_summary_df[
        "total_residual_gt_after_B0_C1"
    ]
    .unique()
)


assert (
    len(
        residual_totals
    )
    ==
    1
)


assert (
    int(
        residual_totals[
            0
        ]
    )
    ==
    TOTAL_RESIDUAL_GT
)


candidate_total_lookup = dict(
    zip(
        step3f_summary_df[
            "eval_source"
        ],

        step3f_summary_df[
            "accepted_recovery_candidates"
        ],
    )
)


assert (
    int(
        candidate_total_lookup[
            "Step3D_C1supported_D6BT"
        ]
    )
    ==
    805
)


assert (
    int(
        candidate_total_lookup[
            "Step3E_complementary_DD"
        ]
    )
    ==
    1355
)


# ------------------------------------------------------------
# 3F.14 Save outputs
# ------------------------------------------------------------

step3f_point_df.to_csv(
    STEP3F_POINT_FILE,
    index=False,
)


step3f_summary_df.to_csv(
    STEP3F_SUMMARY_FILE,
    index=False,
)


step3f_density_df.to_csv(
    STEP3F_DENSITY_FILE,
    index=False,
)


step3f_image_df.to_csv(
    STEP3F_IMAGE_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3F.15 Manifest
# ------------------------------------------------------------

step3f_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3F - incremental D quality after B0 plus C1",

    "B0_missed_GT":
        TOTAL_B0_MISSED_GT,

    "C1_recovered_B0_missed_GT":
        TOTAL_C1_RECOVERED_GT,

    "residual_GT_after_B0_plus_C1":
        TOTAL_RESIDUAL_GT,

    "evaluated_recovery_sets": {
        "Step3D_C1supported_D6BT":
            805,

        "Step3E_complementary_DD":
            1355,
    },

    "step3e_pair_representative":
        "midpoint of accepted D6B9 / D6B-T pair",

    "matching_rule":
        (
            "one-to-one greedy globally shortest spatial "
            "matching to residual GT within frozen per-image R"
        ),

    "GT_used_for_selection":
        False,

    "GT_used_for_diagnostic_evaluation":
        True,

    "count_rule_changed":
        False,

    "sam2_reprompting":
        False,

    "density_used_for_routing":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,
}


with open(
    STEP3F_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step3f_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3F.16 Final display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3F - INCREMENTAL D QUALITY AFTER B0 + C1")
print("=" * 125)

print(
    "B0-missed GT                :",
    TOTAL_B0_MISSED_GT
)

print(
    "Recovered by C1             :",
    TOTAL_C1_RECOVERED_GT
)

print(
    "Residual GT after B0 + C1   :",
    TOTAL_RESIDUAL_GT
)

print(
    "\nStep 3D candidates evaluated:",
    len(
        step3d_eval_df
    )
)

print(
    "Step 3E candidates evaluated:",
    len(
        step3e_eval_df
    )
)

print(
    "\nGT used for selection       : NO"
)

print(
    "Count rule changed          : NO"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nOverall incremental recovery quality:"
)

display(
    step3f_summary_df[
        [
            "eval_source",
            "total_residual_gt_after_B0_C1",
            "accepted_recovery_candidates",
            "incremental_recovered_gt",
            "unsupported_incremental",
            "unrecovered_residual_gt",
            "incremental_precision_like",
            "residual_GT_recall",
            "FP_per_incremental_TP",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nIncremental recovery quality by density stratum:"
)

display(
    step3f_density_df[
        [
            "density_stratum",
            "eval_source",
            "num_images",
            "residual_gt_after_B0_C1",
            "accepted_recovery_candidates",
            "incremental_recovered_gt",
            "unsupported_incremental",
            "incremental_precision_like",
            "residual_GT_recall",
            "FP_per_incremental_TP",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nImages with most incremental recovered GT:"
)

display(
    step3f_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "eval_source",
            "residual_gt_after_B0_C1",
            "accepted_recovery_candidates",
            "incremental_recovered_gt",
            "unsupported_incremental",
        ]
    ]
    .sort_values(
        [
            "incremental_recovered_gt",
            "accepted_recovery_candidates",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        30
    )
)


print(
    "\nSaved point-level diagnostics:"
)

print(
    STEP3F_POINT_FILE
)

print(
    "\nSaved overall summary:"
)

print(
    STEP3F_SUMMARY_FILE
)

print(
    "\nSaved density summary:"
)

print(
    STEP3F_DENSITY_FILE
)

print(
    "\nSaved per-image summary:"
)

print(
    STEP3F_IMAGE_FILE
)

print(
    "\nSaved manifest:"
)

print(
    STEP3F_MANIFEST_FILE
)

print("=" * 125)

STEP 3F - INCREMENTAL D QUALITY AFTER B0 + C1
B0-missed GT                : 5641
Recovered by C1             : 623
Residual GT after B0 + C1   : 5018

Step 3D candidates evaluated: 805
Step 3E candidates evaluated: 1355

GT used for selection       : NO
Count rule changed          : NO
SAM2 re-prompting           : NO
Density used for routing    : NO
Threshold tuning            : NO
Test set accessed           : NO

Overall incremental recovery quality:


,eval_source,total_residual_gt_after_B0_C1,accepted_recovery_candidates,incremental_recovered_gt,unsupported_incremental,unrecovered_residual_gt,incremental_precision_like,residual_GT_recall,FP_per_incremental_TP
0,Step3D_C1supported_D6BT,5018,805,147,658,4871,0.1826,0.0293,4.4762
1,Step3E_complementary_DD,5018,1355,358,997,4660,0.2642,0.0713,2.7849



Incremental recovery quality by density stratum:


,density_stratum,eval_source,num_images,residual_gt_after_B0_C1,accepted_recovery_candidates,incremental_recovered_gt,unsupported_incremental,incremental_precision_like,residual_GT_recall,FP_per_incremental_TP
0,S1_0_20,Step3D_C1supported_D6BT,20,64,76,1,75,0.0132,0.0156,75.0000
1,S1_0_20,Step3E_complementary_DD,20,64,124,24,100,0.1935,0.3750,4.1667
2,S2_20_40,Step3D_C1supported_D6BT,20,131,150,4,146,0.0267,0.0305,36.5000
3,S2_20_40,Step3E_complementary_DD,20,131,301,55,246,0.1827,0.4198,4.4727
4,S3_40_60,Step3D_C1supported_D6BT,20,285,119,7,112,0.0588,0.0246,16.0000
5,S3_40_60,Step3E_complementary_DD,20,285,376,82,294,0.2181,0.2877,3.5854
6,S4_60_80,Step3D_C1supported_D6BT,20,341,167,17,150,0.1018,0.0499,8.8235
7,S4_60_80,Step3E_complementary_DD,20,341,262,62,200,0.2366,0.1818,3.2258
8,S5_80_95,Step3D_C1supported_D6BT,10,534,93,12,81,0.1290,0.0225,6.7500
9,S5_80_95,Step3E_complementary_DD,10,534,163,84,79,0.5153,0.1573,0.9405



Images with most incremental recovered GT:


,dataset_index,image_id,category,density_stratum,gt_count,eval_source,residual_gt_after_B0_C1,accepted_recovery_candidates,incremental_recovered_gt,unsupported_incremental
145,425,2884.jpg,chairs,S5_80_95,165,Step3E_complementary_DD,127,51,42,9
32,253,975.jpg,birds,S6_95_100,718,Step3D_C1supported_D6BT,618,61,37,24
177,938,5866.jpg,grapes,S6_95_100,267,Step3E_complementary_DD,221,90,34,56
93,1192,6969.jpg,birds,S6_95_100,949,Step3D_C1supported_D6BT,897,27,25,2
92,1187,6931.jpg,birds,S6_95_100,355,Step3D_C1supported_D6BT,259,46,21,25
25,184,840.jpg,books,S6_95_100,637,Step3D_C1supported_D6BT,595,41,21,20
134,298,1935.jpg,bottle caps,S5_80_95,175,Step3E_complementary_DD,114,29,21,8
199,1269,7060.jpg,pills,S4_60_80,65,Step3E_complementary_DD,50,50,14,36
102,20,228.jpg,grapes,S4_60_80,44,Step3E_complementary_DD,35,28,11,17
124,183,838.jpg,books,S5_80_95,113,Step3E_complementary_DD,76,52,10,42



Saved point-level diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_incremental_D_quality_long.csv

Saved overall summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_incremental_D_quality_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_incremental_D_quality_by_density.csv

Saved per-image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_incremental_D_quality_per_image.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_3F_manifest.json


In [31]:
# ============================================================
# Step 3G-A — Construct D5-768 B0-Outside Recovery Pool
# CORRECTED CACHE-LOADING VERSION
#
# Purpose:
# Bring D5-768 into the same B0-anchored recovery framework.
#
# Recovery candidate:
# frozen D5-768 point lying outside ALL frozen B0 masks.
#
# No GT used for selection.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3G-A.1 Output files
# ------------------------------------------------------------

STEP3GA_D5_LONG_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_D5_768_points_long.csv"
)

STEP3GA_D5_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_D5_768_per_image.csv"
)

STEP3GA_D5_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_recovery_D5_768_by_density.csv"
)

STEP3GA_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3GA_D5_768_manifest.json"
)


# ------------------------------------------------------------
# 3G-A.2 Preconditions
# ------------------------------------------------------------

required_vars = [
    "D_candidates_df",
    "b2_dev100_image_df",
    "b0_mask_cache_index_df",
    "load_b0_masks_from_cache",
    "parse_xy_list",
]

for var_name in required_vars:

    assert var_name in globals(), (
        f"Required variable not found: {var_name}"
    )


assert len(b2_dev100_image_df) == 100
assert len(b0_mask_cache_index_df) == 100


# ------------------------------------------------------------
# 3G-A.3 Extract frozen D5-768
# ------------------------------------------------------------

d5_source_df = (
    D_candidates_df[
        D_candidates_df[
            "candidate_id"
        ]
        ==
        "D5-768"
    ]
    .copy()
)


assert len(d5_source_df) == 100

assert (
    d5_source_df[
        "dataset_index"
    ].nunique()
    ==
    100
)

assert (
    "pred_points_xy"
    in
    d5_source_df.columns
)

assert (
    "pred_count"
    in
    d5_source_df.columns
)


# ------------------------------------------------------------
# 3G-A.4 Helper:
# test one point against B0 mask stack
#
# Membership convention preserved:
# np.rint -> clip -> mask pixel lookup
# ------------------------------------------------------------

def point_inside_any_b0_mask_3ga(
    x,
    y,
    masks,
):

    masks = np.asarray(
        masks,
        dtype=bool,
    )


    if masks.shape[0] == 0:

        return False


    _, h, w = masks.shape


    xi = int(
        np.clip(
            np.rint(
                float(x)
            ),
            0,
            w - 1,
        )
    )


    yi = int(
        np.clip(
            np.rint(
                float(y)
            ),
            0,
            h - 1,
        )
    )


    return bool(
        masks[
            :,
            yi,
            xi,
        ].any()
    )


# ------------------------------------------------------------
# 3G-A.5 Process all 100 images
# ------------------------------------------------------------

d5_point_records = []

cache_integrity_records = []


for image_row in (
    b2_dev100_image_df
    .itertuples(
        index=False
    )
):

    dataset_index = int(
        image_row.dataset_index
    )

    image_id = image_row.image_id


    # --------------------------------------------------------
    # Frozen D5 row
    # --------------------------------------------------------

    d5_rows = (
        d5_source_df[
            d5_source_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
    )


    assert len(d5_rows) == 1


    d5_row = d5_rows.iloc[0]


    assert (
        d5_row[
            "image_id"
        ]
        ==
        image_id
    )


    d5_points_xy = np.asarray(
        parse_xy_list(
            d5_row[
                "pred_points_xy"
            ]
        ),
        dtype=float,
    ).reshape(
        -1,
        2,
    )


    assert (
        len(
            d5_points_xy
        )
        ==
        int(
            d5_row[
                "pred_count"
            ]
        )
    )


    # --------------------------------------------------------
    # Frozen B0 cache row
    #
    # IMPORTANT:
    # exactly ONE cache-index row per image.
    # --------------------------------------------------------

    cache_rows = (
        b0_mask_cache_index_df[
            b0_mask_cache_index_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
    )


    assert (
        len(
            cache_rows
        )
        ==
        1
    )


    cache_row = cache_rows.iloc[0]


    assert (
        cache_row[
            "image_id"
        ]
        ==
        image_id
    )


    cache_path = Path(
        cache_row[
            "cache_file"
        ]
    )


    assert cache_path.exists(), (
        f"Missing B0 cache file: {cache_path}"
    )


    # --------------------------------------------------------
    # Correct cache-loader call
    # --------------------------------------------------------

    cache_bundle = (
        load_b0_masks_from_cache(
            cache_path
        )
    )


    b0_masks = cache_bundle[
        "masks"
    ]


    # --------------------------------------------------------
    # Cache integrity checks
    # --------------------------------------------------------

    assert (
        int(
            cache_bundle[
                "num_masks"
            ]
        )
        ==
        int(
            cache_row[
                "reproduced_b0_count"
            ]
        )
    )


    assert (
        b0_masks.shape[0]
        ==
        int(
            cache_row[
                "reproduced_b0_count"
            ]
        )
    )


    assert (
        int(
            cache_bundle[
                "image_height"
            ]
        )
        ==
        int(
            cache_row[
                "image_height"
            ]
        )
    )


    assert (
        int(
            cache_bundle[
                "image_width"
            ]
        )
        ==
        int(
            cache_row[
                "image_width"
            ]
        )
    )


    cache_integrity_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "num_B0_masks":
                int(
                    b0_masks.shape[0]
                ),

            "cache_match":
                True,
        }
    )


    # --------------------------------------------------------
    # Classify every D5 point
    # --------------------------------------------------------

    for point_index, (
        x,
        y,
    ) in enumerate(
        d5_points_xy
    ):

        covered_by_B0 = (
            point_inside_any_b0_mask_3ga(
                x=x,
                y=y,
                masks=b0_masks,
            )
        )


        d5_point_records.append(
            {
                "branch":
                    "D",

                "candidate_id":
                    "D5-768",

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "gt_count":
                    int(
                        image_row.gt_count
                    ),

                "point_index":
                    int(
                        point_index
                    ),

                "x":
                    float(
                        x
                    ),

                "y":
                    float(
                        y
                    ),

                "covered_by_B0":
                    bool(
                        covered_by_B0
                    ),

                "recovery_candidate":
                    bool(
                        not covered_by_B0
                    ),
            }
        )


d5_recovery_point_long_df = pd.DataFrame(
    d5_point_records
)


cache_integrity_df = pd.DataFrame(
    cache_integrity_records
)


assert len(cache_integrity_df) == 100
assert cache_integrity_df["cache_match"].all()


# ------------------------------------------------------------
# 3G-A.6 Per-image summary
# ------------------------------------------------------------

if len(
    d5_recovery_point_long_df
) > 0:

    d5_point_summary_df = (
        d5_recovery_point_long_df
        .groupby(
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
            ],
            as_index=False,
        )
        .agg(
            total_D5_points=(
                "point_index",
                "size",
            ),

            B0_covered_D5_points=(
                "covered_by_B0",
                "sum",
            ),

            recovery_candidate_points=(
                "recovery_candidate",
                "sum",
            ),
        )
    )

else:

    d5_point_summary_df = pd.DataFrame(
        columns=[
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "total_D5_points",
            "B0_covered_D5_points",
            "recovery_candidate_points",
        ]
    )


d5_recovery_image_df = (
    b2_dev100_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .merge(
        d5_point_summary_df,
        on=[
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ],
        how="left",
        validate="one_to_one",
    )
)


for col in [
    "total_D5_points",
    "B0_covered_D5_points",
    "recovery_candidate_points",
]:

    d5_recovery_image_df[
        col
    ] = (
        d5_recovery_image_df[
            col
        ]
        .fillna(
            0
        )
        .astype(
            int
        )
    )


d5_recovery_image_df[
    "B0_covered_fraction"
] = np.where(
    d5_recovery_image_df[
        "total_D5_points"
    ]
    >
    0,

    d5_recovery_image_df[
        "B0_covered_D5_points"
    ]
    /
    d5_recovery_image_df[
        "total_D5_points"
    ],

    np.nan,
)


d5_recovery_image_df[
    "recovery_fraction"
] = np.where(
    d5_recovery_image_df[
        "total_D5_points"
    ]
    >
    0,

    d5_recovery_image_df[
        "recovery_candidate_points"
    ]
    /
    d5_recovery_image_df[
        "total_D5_points"
    ],

    np.nan,
)


assert len(d5_recovery_image_df) == 100


# ------------------------------------------------------------
# 3G-A.7 Density summary
# ------------------------------------------------------------

d5_recovery_density_df = (
    d5_recovery_image_df
    .groupby(
        "density_stratum",
        as_index=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        total_D5_points=(
            "total_D5_points",
            "sum",
        ),

        B0_covered_D5_points=(
            "B0_covered_D5_points",
            "sum",
        ),

        recovery_candidate_points=(
            "recovery_candidate_points",
            "sum",
        ),
    )
)


d5_recovery_density_df[
    "recovery_fraction"
] = np.where(
    d5_recovery_density_df[
        "total_D5_points"
    ]
    >
    0,

    d5_recovery_density_df[
        "recovery_candidate_points"
    ]
    /
    d5_recovery_density_df[
        "total_D5_points"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3G-A.8 Global integrity
# ------------------------------------------------------------

TOTAL_D5_POINTS = int(
    d5_recovery_image_df[
        "total_D5_points"
    ].sum()
)


TOTAL_D5_B0_COVERED = int(
    d5_recovery_image_df[
        "B0_covered_D5_points"
    ].sum()
)


TOTAL_D5_RECOVERY = int(
    d5_recovery_image_df[
        "recovery_candidate_points"
    ].sum()
)


assert (
    TOTAL_D5_POINTS
    ==
    TOTAL_D5_B0_COVERED
    +
    TOTAL_D5_RECOVERY
)


assert (
    TOTAL_D5_POINTS
    ==
    int(
        d5_source_df[
            "pred_count"
        ].sum()
    )
)


# ------------------------------------------------------------
# 3G-A.9 Save
# ------------------------------------------------------------

d5_recovery_point_long_df.to_csv(
    STEP3GA_D5_LONG_FILE,
    index=False,
)


d5_recovery_image_df.to_csv(
    STEP3GA_D5_IMAGE_FILE,
    index=False,
)


d5_recovery_density_df.to_csv(
    STEP3GA_D5_DENSITY_FILE,
    index=False,
)


step3ga_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3G-A - D5-768 B0-outside recovery pool",

    "candidate":
        "D5-768",

    "candidate_family":
        "CLIP-Surgery",

    "recovery_definition":
        (
            "Frozen D5-768 point lies outside every frozen "
            "B0 SAM2 mask."
        ),

    "B0_cache_loader":
        (
            "load_b0_masks_from_cache(cache_file)"
        ),

    "point_membership_rule":
        "np.rint + clip + any-mask lookup",

    "GT_used_for_recovery_selection":
        False,

    "count_modified":
        False,

    "sam2_reprompting":
        False,

    "threshold_tuning":
        False,

    "density_used_for_routing":
        False,

    "test_set_accessed":
        False,

    "total_D5_points":
        TOTAL_D5_POINTS,

    "B0_covered_D5_points":
        TOTAL_D5_B0_COVERED,

    "D5_recovery_candidate_points":
        TOTAL_D5_RECOVERY,
}


with open(
    STEP3GA_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step3ga_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3G-A.10 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3G-A - D5-768 B0-OUTSIDE RECOVERY POOL")
print("=" * 125)

print(
    "Images                     :",
    len(
        d5_recovery_image_df
    ),
    "/ 100"
)

print(
    "B0 cache integrity          :",
    int(
        cache_integrity_df[
            "cache_match"
        ].sum()
    ),
    "/ 100 PASS"
)

print(
    "Total frozen D5-768 points :",
    TOTAL_D5_POINTS
)

print(
    "D5 points covered by B0    :",
    TOTAL_D5_B0_COVERED
)

print(
    "D5 recovery candidates     :",
    TOTAL_D5_RECOVERY
)

print(
    "Recovery fraction          :",
    round(
        (
            TOTAL_D5_RECOVERY
            /
            TOTAL_D5_POINTS
        )
        if TOTAL_D5_POINTS > 0
        else np.nan,
        4,
    )
)

print("\nGT used for selection      : NO")
print("Count modified             : NO")
print("SAM2 re-prompting          : NO")
print("Density used for routing   : NO")
print("Threshold tuning           : NO")
print("Test set accessed          : NO")


print(
    "\nD5 recovery pool by density stratum:"
)

display(
    d5_recovery_density_df.round(
        4
    )
)


print(
    "\nImages with most D5 recovery candidates:"
)

display(
    d5_recovery_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "total_D5_points",
            "B0_covered_D5_points",
            "recovery_candidate_points",
            "recovery_fraction",
        ]
    ]
    .sort_values(
        "recovery_candidate_points",
        ascending=False,
    )
    .head(
        30
    )
)


print(
    "\nSaved D5 point-level recovery pool:"
)

print(
    STEP3GA_D5_LONG_FILE
)

print(
    "\nSaved D5 per-image summary:"
)

print(
    STEP3GA_D5_IMAGE_FILE
)

print(
    "\nSaved D5 density summary:"
)

print(
    STEP3GA_D5_DENSITY_FILE
)

print(
    "\nSaved manifest:"
)

print(
    STEP3GA_MANIFEST_FILE
)

print("=" * 125)

STEP 3G-A - D5-768 B0-OUTSIDE RECOVERY POOL
Images                     : 100 / 100
B0 cache integrity          : 100 / 100 PASS
Total frozen D5-768 points : 14125
D5 points covered by B0    : 6349
D5 recovery candidates     : 7776
Recovery fraction          : 0.5505

GT used for selection      : NO
Count modified             : NO
SAM2 re-prompting          : NO
Density used for routing   : NO
Threshold tuning           : NO
Test set accessed          : NO

D5 recovery pool by density stratum:


,density_stratum,num_images,total_D5_points,B0_covered_D5_points,recovery_candidate_points,recovery_fraction
0,S1_0_20,20,2000,821,1179,0.5895
1,S2_20_40,20,2545,1163,1382,0.5430
2,S3_40_60,20,2441,1007,1434,0.5875
3,S4_60_80,20,2962,1565,1397,0.4716
4,S5_80_95,10,2112,1190,922,0.4366
5,S6_95_100,10,2065,603,1462,0.7080



Images with most D5 recovery candidates:


,dataset_index,image_id,category,density_stratum,gt_count,total_D5_points,B0_covered_D5_points,recovery_candidate_points,recovery_fraction
25,184,840.jpg,books,S6_95_100,637,267,2,265,0.992509
77,938,5866.jpg,grapes,S6_95_100,267,296,39,257,0.868243
1,14,220.jpg,grapes,S4_60_80,59,265,14,251,0.947170
32,253,975.jpg,birds,S6_95_100,718,236,0,236,1.000000
92,1187,6931.jpg,birds,S6_95_100,355,224,1,223,0.995536
45,425,2884.jpg,chairs,S5_80_95,165,279,81,198,0.709677
26,217,919.jpg,skateboard,S3_40_60,32,258,65,193,0.748062
93,1192,6969.jpg,birds,S6_95_100,949,185,0,185,1.000000
58,564,3777.jpg,donuts tray,S2_20_40,14,176,0,176,1.000000
70,855,5112.jpg,chicken wings,S3_40_60,22,241,67,174,0.721992



Saved D5 point-level recovery pool:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_D5_768_points_long.csv

Saved D5 per-image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_D5_768_per_image.csv

Saved D5 density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_recovery_D5_768_by_density.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_3GA_D5_768_manifest.json


In [32]:
# ============================================================
# Step 3G-B — Heterogeneous D Consensus Recovery
#
# Compare:
#
#   H1: D5-768 <-> D6B-T-native
#   H2: D5-768 <-> D6B9-strict
#
# Reference:
#   Step 3E: D6B-T-native <-> D6B9-strict
#
# Rule:
#   1. Both points must already be outside B0.
#   2. One-to-one pair within frozen per-image radius R.
#   3. BOTH endpoints must be farther than R from every
#      C1 recovery point.
#   4. Each accepted pair contributes exactly +1 count.
#
# Also evaluates accepted pair midpoints against residual GT
# after B0 + C1, exactly in the spirit of Step 3F.
#
# IMPORTANT:
# - GT NOT used for pair selection
# - no SAM2 re-prompting
# - no density routing
# - no threshold tuning
# - test set untouched
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3G-B.1 Output files
# ------------------------------------------------------------

STEP3GB_PAIR_FILE = (
    B2_TABLES_DIR
    / "B0_C1_D5_heterogeneous_consensus_pairs.csv"
)

STEP3GB_COUNT_FILE = (
    B2_TABLES_DIR
    / "B0_C1_D5_heterogeneous_consensus_counts_per_image.csv"
)

STEP3GB_COUNT_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_D5_heterogeneous_consensus_count_summary.csv"
)

STEP3GB_GT_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_D5_heterogeneous_consensus_GT_quality_summary.csv"
)

STEP3GB_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_D5_heterogeneous_consensus_by_density.csv"
)

STEP3GB_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3GB_D5_heterogeneous_consensus_manifest.json"
)


# ------------------------------------------------------------
# 3G-B.2 Preconditions
# ------------------------------------------------------------

required_vars = [
    "d5_recovery_point_long_df",
    "recovery_point_long_df",
    "b2_dev100_image_df",
    "gt_image_df",
    "recovery_count_df",
    "step3e_summary_df",
    "step3f_summary_df",
    "residual_gt_df",
]

for var_name in required_vars:

    assert var_name in globals(), (
        f"Required variable not found: {var_name}"
    )


# ------------------------------------------------------------
# 3G-B.3 Extract recovery pools
# ------------------------------------------------------------

d5_recovery_df = (
    d5_recovery_point_long_df[
        d5_recovery_point_long_df[
            "recovery_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


def get_existing_recovery_source(
    candidate_id
):

    return (
        recovery_point_long_df[
            (
                recovery_point_long_df[
                    "candidate_id"
                ]
                ==
                candidate_id
            )
            &
            (
                recovery_point_long_df[
                    "recovery_candidate"
                ]
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


c1_recovery_df = get_existing_recovery_source(
    "C1"
)

d6bt_recovery_df = get_existing_recovery_source(
    "D6B-T-native"
)

d6b9_recovery_df = get_existing_recovery_source(
    "D6B9-strict"
)


assert len(d5_recovery_df) == 7776
assert len(c1_recovery_df) == 1215
assert len(d6bt_recovery_df) == 8047
assert len(d6b9_recovery_df) == 1911


# ------------------------------------------------------------
# 3G-B.4 Frozen radius lookup
# ------------------------------------------------------------

radius_lookup_3gb = (
    gt_image_df[
        [
            "dataset_index",
            "matching_radius_px",
        ]
    ]
    .drop_duplicates(
        subset=[
            "dataset_index"
        ]
    )
    .set_index(
        "dataset_index"
    )
)


assert len(radius_lookup_3gb) == 100


# ------------------------------------------------------------
# 3G-B.5 XY lookup helper
# ------------------------------------------------------------

def make_xy_lookup_3gb(
    df
):

    lookup = {}


    for dataset_index, group in (
        df.groupby(
            "dataset_index"
        )
    ):

        lookup[
            int(
                dataset_index
            )
        ] = (
            group[
                [
                    "x",
                    "y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


    return lookup


EMPTY_XY = np.empty(
    (
        0,
        2,
    ),
    dtype=float,
)


c1_xy_lookup = make_xy_lookup_3gb(
    c1_recovery_df
)

d5_xy_lookup = make_xy_lookup_3gb(
    d5_recovery_df
)

d6bt_xy_lookup = make_xy_lookup_3gb(
    d6bt_recovery_df
)

d6b9_xy_lookup = make_xy_lookup_3gb(
    d6b9_recovery_df
)


# ------------------------------------------------------------
# 3G-B.6 Distance helper
# ------------------------------------------------------------

def nearest_distance_to_set_3gb(
    point_xy,
    ref_xy
):

    if len(
        ref_xy
    ) == 0:

        return np.inf


    point_xy = np.asarray(
        point_xy,
        dtype=float,
    )

    ref_xy = np.asarray(
        ref_xy,
        dtype=float,
    )


    return float(
        np.hypot(
            ref_xy[
                :,
                0
            ]
            -
            point_xy[
                0
            ],

            ref_xy[
                :,
                1
            ]
            -
            point_xy[
                1
            ],
        ).min()
    )


# ------------------------------------------------------------
# 3G-B.7 Generic one-to-one matching helper
#
# Same globally-shortest greedy convention used in 3E / 3F.
# ------------------------------------------------------------

def one_to_one_pair_3gb(
    a_xy,
    b_xy,
    radius_px,
):

    a_xy = np.asarray(
        a_xy,
        dtype=float,
    )

    b_xy = np.asarray(
        b_xy,
        dtype=float,
    )


    if (
        len(
            a_xy
        )
        ==
        0
        or
        len(
            b_xy
        )
        ==
        0
    ):

        return []


    candidate_pairs = []


    for i in range(
        len(
            a_xy
        )
    ):

        ax, ay = a_xy[
            i
        ]


        distances = np.hypot(
            b_xy[
                :,
                0
            ]
            -
            ax,

            b_xy[
                :,
                1
            ]
            -
            ay,
        )


        valid_js = np.where(
            distances
            <=
            radius_px
        )[
            0
        ]


        for j in valid_js:

            candidate_pairs.append(
                (
                    float(
                        distances[
                            j
                        ]
                    ),
                    int(
                        i
                    ),
                    int(
                        j
                    ),
                )
            )


    candidate_pairs.sort(
        key=lambda z: (
            z[
                0
            ],
            z[
                1
            ],
            z[
                2
            ],
        )
    )


    used_a = set()
    used_b = set()

    matches = []


    for (
        distance,
        i,
        j
    ) in candidate_pairs:

        if i in used_a:

            continue

        if j in used_b:

            continue


        used_a.add(
            i
        )

        used_b.add(
            j
        )


        matches.append(
            {
                "a_index":
                    int(
                        i
                    ),

                "b_index":
                    int(
                        j
                    ),

                "pair_distance_px":
                    float(
                        distance
                    ),
            }
        )


    return matches


# ------------------------------------------------------------
# 3G-B.8 Pair definitions
# ------------------------------------------------------------

PAIR_CONFIGS = [
    {
        "pair_id":
            "D5xD6BT",

        "source_a":
            "D5-768",

        "source_b":
            "D6B-T-native",

        "a_lookup":
            d5_xy_lookup,

        "b_lookup":
            d6bt_xy_lookup,
    },

    {
        "pair_id":
            "D5xD6B9",

        "source_a":
            "D5-768",

        "source_b":
            "D6B9-strict",

        "a_lookup":
            d5_xy_lookup,

        "b_lookup":
            d6b9_xy_lookup,
    },
]


# ------------------------------------------------------------
# 3G-B.9 Build heterogeneous pairs
# ------------------------------------------------------------

pair_records = []

pair_image_records = []


for config in PAIR_CONFIGS:

    pair_id = config[
        "pair_id"
    ]

    source_a = config[
        "source_a"
    ]

    source_b = config[
        "source_b"
    ]


    for image_row in (
        b2_dev100_image_df
        .itertuples(
            index=False
        )
    ):

        dataset_index = int(
            image_row.dataset_index
        )


        radius_px = float(
            radius_lookup_3gb.loc[
                dataset_index,
                "matching_radius_px",
            ]
        )


        c1_xy = c1_xy_lookup.get(
            dataset_index,
            EMPTY_XY,
        )

        a_xy = (
            config[
                "a_lookup"
            ]
            .get(
                dataset_index,
                EMPTY_XY,
            )
        )

        b_xy = (
            config[
                "b_lookup"
            ]
            .get(
                dataset_index,
                EMPTY_XY,
            )
        )


        matches = one_to_one_pair_3gb(
            a_xy=a_xy,
            b_xy=b_xy,
            radius_px=radius_px,
        )


        complementary_count = 0


        for pair_index, match in enumerate(
            matches
        ):

            a_point = a_xy[
                match[
                    "a_index"
                ]
            ]

            b_point = b_xy[
                match[
                    "b_index"
                ]
            ]


            a_to_c1 = (
                nearest_distance_to_set_3gb(
                    a_point,
                    c1_xy,
                )
            )

            b_to_c1 = (
                nearest_distance_to_set_3gb(
                    b_point,
                    c1_xy,
                )
            )


            complementary = bool(
                (
                    a_to_c1
                    >
                    radius_px
                )
                and
                (
                    b_to_c1
                    >
                    radius_px
                )
            )


            if complementary:

                complementary_count += 1


            midpoint_x = float(
                (
                    a_point[
                        0
                    ]
                    +
                    b_point[
                        0
                    ]
                )
                /
                2.0
            )

            midpoint_y = float(
                (
                    a_point[
                        1
                    ]
                    +
                    b_point[
                        1
                    ]
                )
                /
                2.0
            )


            pair_records.append(
                {
                    "pair_id":
                        pair_id,

                    "source_a":
                        source_a,

                    "source_b":
                        source_b,

                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_row.image_id,

                    "category":
                        image_row.category,

                    "density_stratum":
                        image_row.density_stratum,

                    "gt_count":
                        int(
                            image_row.gt_count
                        ),

                    "pair_index":
                        int(
                            pair_index
                        ),

                    "matching_radius_px":
                        radius_px,

                    "a_x":
                        float(
                            a_point[
                                0
                            ]
                        ),

                    "a_y":
                        float(
                            a_point[
                                1
                            ]
                        ),

                    "b_x":
                        float(
                            b_point[
                                0
                            ]
                        ),

                    "b_y":
                        float(
                            b_point[
                                1
                            ]
                        ),

                    "pair_distance_px":
                        float(
                            match[
                                "pair_distance_px"
                            ]
                        ),

                    "a_nearest_C1_distance_px":
                        (
                            float(
                                a_to_c1
                            )
                            if np.isfinite(
                                a_to_c1
                            )
                            else np.nan
                        ),

                    "b_nearest_C1_distance_px":
                        (
                            float(
                                b_to_c1
                            )
                            if np.isfinite(
                                b_to_c1
                            )
                            else np.nan
                        ),

                    "complementary_to_C1":
                        complementary,

                    "representative_midpoint_x":
                        midpoint_x,

                    "representative_midpoint_y":
                        midpoint_y,
                }
            )


        pair_image_records.append(
            {
                "pair_id":
                    pair_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_row.image_id,

                "all_pairs":
                    int(
                        len(
                            matches
                        )
                    ),

                "complementary_pairs":
                    int(
                        complementary_count
                    ),
            }
        )


step3gb_pair_df = pd.DataFrame(
    pair_records
)


step3gb_pair_image_df = pd.DataFrame(
    pair_image_records
)


assert len(step3gb_pair_image_df) == 200


# ------------------------------------------------------------
# 3G-B.10 Pair-level summary
# ------------------------------------------------------------

step3gb_pair_summary_df = (
    step3gb_pair_image_df
    .groupby(
        "pair_id",
        as_index=False,
    )
    .agg(
        all_pairs=(
            "all_pairs",
            "sum",
        ),

        complementary_pairs=(
            "complementary_pairs",
            "sum",
        ),
    )
)


step3gb_pair_summary_df[
    "complementary_fraction"
] = (
    step3gb_pair_summary_df[
        "complementary_pairs"
    ]
    /
    step3gb_pair_summary_df[
        "all_pairs"
    ]
)


# ------------------------------------------------------------
# 3G-B.11 Construct count systems
# ------------------------------------------------------------

step3gb_count_df = (
    recovery_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
        ]
    ]
    .copy()
)


for pair_id in [
    "D5xD6BT",
    "D5xD6B9",
]:

    pair_counts = (
        step3gb_pair_image_df[
            step3gb_pair_image_df[
                "pair_id"
            ]
            ==
            pair_id
        ][
            [
                "dataset_index",
                "complementary_pairs",
            ]
        ]
        .rename(
            columns={
                "complementary_pairs":
                    f"{pair_id}__added_pairs"
            }
        )
    )


    step3gb_count_df = (
        step3gb_count_df
        .merge(
            pair_counts,
            on="dataset_index",
            how="left",
            validate="one_to_one",
        )
    )


    step3gb_count_df[
        f"{pair_id}__added_pairs"
    ] = (
        step3gb_count_df[
            f"{pair_id}__added_pairs"
        ]
        .fillna(
            0
        )
        .astype(
            int
        )
    )


    step3gb_count_df[
        f"B0_plus_C1_plus_{pair_id}"
    ] = (
        step3gb_count_df[
            "B0_plus_C1_raw"
        ]
        +
        step3gb_count_df[
            f"{pair_id}__added_pairs"
        ]
    )


# ------------------------------------------------------------
# 3G-B.12 Count metric helper
# ------------------------------------------------------------

def count_metrics_3gb(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )


    error = pred - gt


    return {
        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "mean_signed_error":
            float(
                np.mean(
                    error
                )
            ),

        "under_count_images":
            int(
                np.sum(
                    error < 0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    error == 0
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    error > 0
                )
            ),
    }


# ------------------------------------------------------------
# 3G-B.13 Overall count performance
# ------------------------------------------------------------

COUNT_SYSTEMS = [
    "B0_unfiltered",
    "B0_plus_C1_raw",
    "B0_plus_C1_plus_D5xD6BT",
    "B0_plus_C1_plus_D5xD6B9",
]


gt_all = (
    step3gb_count_df[
        "gt_count"
    ]
    .to_numpy()
)


count_summary_records = []


for system_name in COUNT_SYSTEMS:

    pred = (
        step3gb_count_df[
            system_name
        ]
        .to_numpy()
    )


    metrics = count_metrics_3gb(
        gt_all,
        pred,
    )


    if system_name == "B0_plus_C1_plus_D5xD6BT":

        added = int(
            step3gb_count_df[
                "D5xD6BT__added_pairs"
            ].sum()
        )

    elif system_name == "B0_plus_C1_plus_D5xD6B9":

        added = int(
            step3gb_count_df[
                "D5xD6B9__added_pairs"
            ].sum()
        )

    else:

        added = 0


    count_summary_records.append(
        {
            "system":
                system_name,

            "additional_pairs_beyond_C1":
                added,

            "total_predicted_count":
                int(
                    pred.sum()
                ),

            **metrics,
        }
    )


step3gb_count_summary_df = pd.DataFrame(
    count_summary_records
)


# Add Step 3E same-family reference

step3e_ref = (
    step3e_summary_df[
        step3e_summary_df[
            "system"
        ]
        ==
        "B0_plus_C1_plus_complementary_DD"
    ]
    .iloc[
        0
    ]
)


step3gb_reference_count_df = pd.DataFrame(
    [
        {
            "system":
                "REFERENCE_D6BTxD6B9_Step3E",

            "additional_pairs_beyond_C1":
                int(
                    step3e_ref[
                        "additional_recovery_beyond_C1"
                    ]
                ),

            "total_predicted_count":
                int(
                    step3e_ref[
                        "total_predicted_count"
                    ]
                ),

            "MAE":
                float(
                    step3e_ref[
                        "MAE"
                    ]
                ),

            "RMSE":
                float(
                    step3e_ref[
                        "RMSE"
                    ]
                ),

            "mean_signed_error":
                float(
                    step3e_ref[
                        "mean_signed_error"
                    ]
                ),

            "under_count_images":
                int(
                    step3e_ref[
                        "under_count_images"
                    ]
                ),

            "exact_count_images":
                int(
                    step3e_ref[
                        "exact_count_images"
                    ]
                ),

            "over_count_images":
                int(
                    step3e_ref[
                        "over_count_images"
                    ]
                ),
        }
    ]
)


step3gb_count_summary_with_ref_df = pd.concat(
    [
        step3gb_count_summary_df,
        step3gb_reference_count_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 3G-B.14 Prepare accepted heterogeneous pair midpoints
# ------------------------------------------------------------

accepted_pair_df = (
    step3gb_pair_df[
        step3gb_pair_df[
            "complementary_to_C1"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


accepted_pair_df[
    "eval_x"
] = (
    accepted_pair_df[
        "representative_midpoint_x"
    ]
)


accepted_pair_df[
    "eval_y"
] = (
    accepted_pair_df[
        "representative_midpoint_y"
    ]
)


# ------------------------------------------------------------
# 3G-B.15 Match accepted midpoints against residual GT
# ------------------------------------------------------------

gt_quality_records = []

gt_image_records = []


for pair_id in [
    "D5xD6BT",
    "D5xD6B9",
]:

    source_df = (
        accepted_pair_df[
            accepted_pair_df[
                "pair_id"
            ]
            ==
            pair_id
        ]
    )


    for image_row in (
        b2_dev100_image_df
        .itertuples(
            index=False
        )
    ):

        dataset_index = int(
            image_row.dataset_index
        )


        pred_subset = (
            source_df[
                source_df[
                    "dataset_index"
                ]
                ==
                dataset_index
            ]
            .reset_index(
                drop=True
            )
        )


        gt_subset = (
            residual_gt_df[
                residual_gt_df[
                    "dataset_index"
                ]
                ==
                dataset_index
            ]
            .reset_index(
                drop=True
            )
        )


        pred_xy = (
            pred_subset[
                [
                    "eval_x",
                    "eval_y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


        gt_xy = (
            gt_subset[
                [
                    "gt_x",
                    "gt_y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )


        radius_px = float(
            radius_lookup_3gb.loc[
                dataset_index,
                "matching_radius_px",
            ]
        )


        matches = one_to_one_pair_3gb(
            a_xy=pred_xy,
            b_xy=gt_xy,
            radius_px=radius_px,
        )


        tp = int(
            len(
                matches
            )
        )

        candidates = int(
            len(
                pred_subset
            )
        )

        residual_gt_count = int(
            len(
                gt_subset
            )
        )


        gt_image_records.append(
            {
                "pair_id":
                    pair_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_row.image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "residual_gt_after_B0_C1":
                    residual_gt_count,

                "accepted_consensus_pairs":
                    candidates,

                "incremental_recovered_gt":
                    tp,

                "unsupported_incremental":
                    (
                        candidates
                        -
                        tp
                    ),
            }
        )


step3gb_gt_image_df = pd.DataFrame(
    gt_image_records
)


assert len(step3gb_gt_image_df) == 200


# ------------------------------------------------------------
# 3G-B.16 GT quality summary
# ------------------------------------------------------------

step3gb_gt_summary_df = (
    step3gb_gt_image_df
    .groupby(
        "pair_id",
        as_index=False,
    )
    .agg(
        total_residual_gt_after_B0_C1=(
            "residual_gt_after_B0_C1",
            "sum",
        ),

        accepted_consensus_pairs=(
            "accepted_consensus_pairs",
            "sum",
        ),

        incremental_recovered_gt=(
            "incremental_recovered_gt",
            "sum",
        ),

        unsupported_incremental=(
            "unsupported_incremental",
            "sum",
        ),
    )
)


step3gb_gt_summary_df[
    "incremental_precision_like"
] = (
    step3gb_gt_summary_df[
        "incremental_recovered_gt"
    ]
    /
    step3gb_gt_summary_df[
        "accepted_consensus_pairs"
    ]
)


step3gb_gt_summary_df[
    "residual_GT_recall"
] = (
    step3gb_gt_summary_df[
        "incremental_recovered_gt"
    ]
    /
    step3gb_gt_summary_df[
        "total_residual_gt_after_B0_C1"
    ]
)


step3gb_gt_summary_df[
    "FP_per_incremental_TP"
] = (
    step3gb_gt_summary_df[
        "unsupported_incremental"
    ]
    /
    step3gb_gt_summary_df[
        "incremental_recovered_gt"
    ]
)


# Add Step 3F same-family reference

step3f_ref = (
    step3f_summary_df[
        step3f_summary_df[
            "eval_source"
        ]
        ==
        "Step3E_complementary_DD"
    ]
    .iloc[
        0
    ]
)


step3gb_gt_reference_df = pd.DataFrame(
    [
        {
            "pair_id":
                "REFERENCE_D6BTxD6B9_Step3E",

            "total_residual_gt_after_B0_C1":
                int(
                    step3f_ref[
                        "total_residual_gt_after_B0_C1"
                    ]
                ),

            "accepted_consensus_pairs":
                int(
                    step3f_ref[
                        "accepted_recovery_candidates"
                    ]
                ),

            "incremental_recovered_gt":
                int(
                    step3f_ref[
                        "incremental_recovered_gt"
                    ]
                ),

            "unsupported_incremental":
                int(
                    step3f_ref[
                        "unsupported_incremental"
                    ]
                ),

            "incremental_precision_like":
                float(
                    step3f_ref[
                        "incremental_precision_like"
                    ]
                ),

            "residual_GT_recall":
                float(
                    step3f_ref[
                        "residual_GT_recall"
                    ]
                ),

            "FP_per_incremental_TP":
                float(
                    step3f_ref[
                        "FP_per_incremental_TP"
                    ]
                ),
        }
    ]
)


step3gb_gt_summary_with_ref_df = pd.concat(
    [
        step3gb_gt_summary_df,
        step3gb_gt_reference_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 3G-B.17 Density GT-quality summary
# ------------------------------------------------------------

step3gb_density_df = (
    step3gb_gt_image_df
    .groupby(
        [
            "density_stratum",
            "pair_id",
        ],
        as_index=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        residual_gt_after_B0_C1=(
            "residual_gt_after_B0_C1",
            "sum",
        ),

        accepted_consensus_pairs=(
            "accepted_consensus_pairs",
            "sum",
        ),

        incremental_recovered_gt=(
            "incremental_recovered_gt",
            "sum",
        ),

        unsupported_incremental=(
            "unsupported_incremental",
            "sum",
        ),
    )
)


step3gb_density_df[
    "incremental_precision_like"
] = np.where(
    step3gb_density_df[
        "accepted_consensus_pairs"
    ]
    >
    0,

    step3gb_density_df[
        "incremental_recovered_gt"
    ]
    /
    step3gb_density_df[
        "accepted_consensus_pairs"
    ],

    np.nan,
)


step3gb_density_df[
    "residual_GT_recall"
] = np.where(
    step3gb_density_df[
        "residual_gt_after_B0_C1"
    ]
    >
    0,

    step3gb_density_df[
        "incremental_recovered_gt"
    ]
    /
    step3gb_density_df[
        "residual_gt_after_B0_C1"
    ],

    np.nan,
)


step3gb_density_df[
    "FP_per_incremental_TP"
] = np.where(
    step3gb_density_df[
        "incremental_recovered_gt"
    ]
    >
    0,

    step3gb_density_df[
        "unsupported_incremental"
    ]
    /
    step3gb_density_df[
        "incremental_recovered_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3G-B.18 Save
# ------------------------------------------------------------

step3gb_pair_df.to_csv(
    STEP3GB_PAIR_FILE,
    index=False,
)


step3gb_count_df.to_csv(
    STEP3GB_COUNT_FILE,
    index=False,
)


step3gb_count_summary_with_ref_df.to_csv(
    STEP3GB_COUNT_SUMMARY_FILE,
    index=False,
)


step3gb_gt_summary_with_ref_df.to_csv(
    STEP3GB_GT_SUMMARY_FILE,
    index=False,
)


step3gb_density_df.to_csv(
    STEP3GB_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3G-B.19 Manifest
# ------------------------------------------------------------

step3gb_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3G-B - heterogeneous D consensus",

    "heterogeneous_pairs": [
        "D5-768 x D6B-T-native",
        "D5-768 x D6B9-strict",
    ],

    "reference_pair":
        "D6B-T-native x D6B9-strict from Step 3E",

    "pair_rule":
        (
            "one-to-one globally-shortest greedy pairing "
            "within frozen per-image R"
        ),

    "complementary_rule":
        (
            "both pair endpoints farther than R from every "
            "C1 recovery point"
        ),

    "count_contribution_per_pair":
        1,

    "GT_used_for_pair_selection":
        False,

    "GT_used_for_diagnostic_evaluation":
        True,

    "sam2_reprompting":
        False,

    "density_used_for_routing":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,
}


with open(
    STEP3GB_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step3gb_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3G-B.20 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3G-B - D5 HETEROGENEOUS CONSENSUS")
print("=" * 125)

print("D5 recovery pool           :", len(d5_recovery_df))
print("D6B-T recovery pool        :", len(d6bt_recovery_df))
print("D6B9 recovery pool         :", len(d6b9_recovery_df))

print("\nGT used for pair selection : NO")
print("Each accepted pair         : +1 count")
print("SAM2 re-prompting          : NO")
print("Density used for routing   : NO")
print("Threshold tuning           : NO")
print("Test set accessed          : NO")


print(
    "\nHeterogeneous pair survival:"
)

display(
    step3gb_pair_summary_df.round(
        4
    )
)


print(
    "\nOverall count performance:"
)

display(
    step3gb_count_summary_with_ref_df[
        [
            "system",
            "additional_pairs_beyond_C1",
            "total_predicted_count",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nIncremental GT quality after B0 + C1:"
)

display(
    step3gb_gt_summary_with_ref_df[
        [
            "pair_id",
            "total_residual_gt_after_B0_C1",
            "accepted_consensus_pairs",
            "incremental_recovered_gt",
            "unsupported_incremental",
            "incremental_precision_like",
            "residual_GT_recall",
            "FP_per_incremental_TP",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nHeterogeneous GT quality by density:"
)

display(
    step3gb_density_df[
        [
            "density_stratum",
            "pair_id",
            "num_images",
            "residual_gt_after_B0_C1",
            "accepted_consensus_pairs",
            "incremental_recovered_gt",
            "unsupported_incremental",
            "incremental_precision_like",
            "residual_GT_recall",
            "FP_per_incremental_TP",
        ]
    ]
    .round(
        4
    )
)


print(
    "\nImages with most accepted heterogeneous consensus pairs:"
)

display(
    step3gb_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "D5xD6BT__added_pairs",
            "D5xD6B9__added_pairs",
            "B0_plus_C1_plus_D5xD6BT",
            "B0_plus_C1_plus_D5xD6B9",
        ]
    ]
    .sort_values(
        "D5xD6BT__added_pairs",
        ascending=False,
    )
    .head(
        30
    )
)


print("\nSaved pair diagnostics:")
print(STEP3GB_PAIR_FILE)

print("\nSaved per-image counts:")
print(STEP3GB_COUNT_FILE)

print("\nSaved count summary:")
print(STEP3GB_COUNT_SUMMARY_FILE)

print("\nSaved GT-quality summary:")
print(STEP3GB_GT_SUMMARY_FILE)

print("\nSaved density summary:")
print(STEP3GB_DENSITY_FILE)

print("\nSaved manifest:")
print(STEP3GB_MANIFEST_FILE)

print("=" * 125)

STEP 3G-B - D5 HETEROGENEOUS CONSENSUS
D5 recovery pool           : 7776
D6B-T recovery pool        : 8047
D6B9 recovery pool         : 1911

GT used for pair selection : NO
Each accepted pair         : +1 count
SAM2 re-prompting          : NO
Density used for routing   : NO
Threshold tuning           : NO
Test set accessed          : NO

Heterogeneous pair survival:


,pair_id,all_pairs,complementary_pairs,complementary_fraction
0,D5xD6B9,825,656,0.7952
1,D5xD6BT,2924,2446,0.8365



Overall count performance:


,system,additional_pairs_beyond_C1,total_predicted_count,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,0,2298,80.69,270.2339,-77.47,76,4,20
1,B0_plus_C1_raw,0,3513,72.00,262.2353,-65.32,52,5,43
2,B0_plus_C1_plus_D5xD6BT,2446,5959,79.24,258.9390,-40.86,29,3,68
3,B0_plus_C1_plus_D5xD6B9,656,4169,72.82,261.4989,-58.76,41,3,56
4,REFERENCE_D6BTxD6B9_Step3E,1355,4868,75.99,261.2695,-51.77,34,4,62



Incremental GT quality after B0 + C1:


...


Heterogeneous GT quality by density:


,density_stratum,pair_id,num_images,residual_gt_after_B0_C1,accepted_consensus_pairs,incremental_recovered_gt,unsupported_incremental,incremental_precision_like,residual_GT_recall,FP_per_incremental_TP
0,S1_0_20,D5xD6B9,20,64,63,8,55,0.1270,0.1250,6.8750
1,S1_0_20,D5xD6BT,20,64,281,22,259,0.0783,0.3438,11.7727
2,S2_20_40,D5xD6B9,20,131,152,32,120,0.2105,0.2443,3.7500
3,S2_20_40,D5xD6BT,20,131,461,58,403,0.1258,0.4427,6.9483
4,S3_40_60,D5xD6B9,20,285,177,46,131,0.2599,0.1614,2.8478
5,S3_40_60,D5xD6BT,20,285,519,96,423,0.1850,0.3368,4.4062
6,S4_60_80,D5xD6B9,20,341,125,43,82,0.3440,0.1261,1.9070
7,S4_60_80,D5xD6BT,20,341,493,93,400,0.1886,0.2727,4.3011
8,S5_80_95,D5xD6B9,10,534,77,39,38,0.5065,0.0730,0.9744
9,S5_80_95,D5xD6BT,10,534,279,103,176,0.3692,0.1929,1.7087



Images with most accepted heterogeneous consensus pairs:


,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw,D5xD6BT__added_pairs,D5xD6B9__added_pairs,B0_plus_C1_plus_D5xD6BT,B0_plus_C1_plus_D5xD6B9
77,938,5866.jpg,grapes,S6_95_100,267,47,65,177,39,242,104
1,14,220.jpg,grapes,S4_60_80,59,20,52,126,22,178,74
25,184,840.jpg,books,S6_95_100,637,7,61,93,1,154,62
45,425,2884.jpg,chairs,S5_80_95,165,38,46,90,25,136,71
0,9,215.jpg,grapes,S6_95_100,259,103,108,84,14,192,122
43,396,2846.jpg,chairs,S4_60_80,38,7,30,80,27,110,57
26,217,919.jpg,skateboard,S3_40_60,32,11,18,78,25,96,43
24,183,838.jpg,books,S5_80_95,113,43,70,77,23,147,93
98,1260,7100.jpg,milk cartons,S3_40_60,27,40,48,69,31,117,79
2,20,228.jpg,grapes,S4_60_80,44,5,27,63,21,90,48



Saved pair diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_D5_heterogeneous_consensus_pairs.csv

Saved per-image counts:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_D5_heterogeneous_consensus_counts_per_image.csv

Saved count summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_D5_heterogeneous_consensus_count_summary.csv

Saved GT-quality summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_D5_heterogeneous_consensus_GT_quality_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_D5_heterogeneous_consensus_by_density.csv

Saved manifest:
/content/drive/MyD

In [33]:
# ============================================================
# Step 3G-C — Triple Consensus Recovery
#
# Sources:
#   D5-768
#   D6B-T-native
#   D6B9-strict
#
# Triple construction:
#
#   1. Start from B0-outside recovery pools.
#   2. Use D6B9-strict as the anchor.
#   3. One-to-one match D6B9 <-> D5 within frozen radius R.
#   4. One-to-one match D6B9 <-> D6B-T within frozen radius R.
#   5. Keep only D6B9 anchor points that participate in BOTH
#      pairings.
#   6. Require all three endpoints to be farther than R from
#      every C1 recovery point.
#   7. Each accepted triple contributes exactly +1 count.
#
# Evaluate:
#   - count performance
#   - incremental GT quality after B0 + C1
#
# References retained:
#   - C1 baseline
#   - Step 3D best
#   - Step 3E D6BT x D6B9
#   - Step 3G-B D5 x D6BT
#   - Step 3G-B D5 x D6B9
#
# IMPORTANT:
# - GT NOT used for triple selection
# - no SAM2 re-prompting
# - no density routing
# - no threshold tuning
# - test set untouched
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3G-C.1 Output files
# ------------------------------------------------------------

STEP3GC_TRIPLE_FILE = (
    B2_TABLES_DIR
    / "B0_C1_D5_D6BT_D6B9_triple_consensus.csv"
)

STEP3GC_COUNT_FILE = (
    B2_TABLES_DIR
    / "B0_C1_triple_consensus_counts_per_image.csv"
)

STEP3GC_COUNT_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_triple_consensus_count_summary.csv"
)

STEP3GC_GT_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_triple_consensus_GT_quality_summary.csv"
)

STEP3GC_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_triple_consensus_by_density.csv"
)

STEP3GC_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_3GC_triple_consensus_manifest.json"
)


# ------------------------------------------------------------
# 3G-C.2 Preconditions
# ------------------------------------------------------------

required_vars = [
    "d5_recovery_point_long_df",
    "recovery_point_long_df",
    "b2_dev100_image_df",
    "gt_image_df",
    "recovery_count_df",
    "step3d_summary_df",
    "step3e_summary_df",
    "step3gb_count_summary_with_ref_df",
    "step3gb_gt_summary_with_ref_df",
    "residual_gt_df",
]

for var_name in required_vars:

    assert var_name in globals(), (
        f"Required variable not found: {var_name}"
    )


# ------------------------------------------------------------
# 3G-C.3 Recovery pools
# ------------------------------------------------------------

d5_recovery_df = (
    d5_recovery_point_long_df[
        d5_recovery_point_long_df[
            "recovery_candidate"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


def get_recovery_source_3gc(
    candidate_id
):

    return (
        recovery_point_long_df[
            (
                recovery_point_long_df[
                    "candidate_id"
                ]
                ==
                candidate_id
            )
            &
            (
                recovery_point_long_df[
                    "recovery_candidate"
                ]
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


c1_recovery_df = get_recovery_source_3gc(
    "C1"
)

d6bt_recovery_df = get_recovery_source_3gc(
    "D6B-T-native"
)

d6b9_recovery_df = get_recovery_source_3gc(
    "D6B9-strict"
)


assert len(d5_recovery_df) == 7776
assert len(c1_recovery_df) == 1215
assert len(d6bt_recovery_df) == 8047
assert len(d6b9_recovery_df) == 1911


# ------------------------------------------------------------
# 3G-C.4 Radius lookup
# ------------------------------------------------------------

radius_lookup_3gc = (
    gt_image_df[
        [
            "dataset_index",
            "matching_radius_px",
        ]
    ]
    .drop_duplicates(
        subset=["dataset_index"]
    )
    .set_index("dataset_index")
)


assert len(radius_lookup_3gc) == 100


# ------------------------------------------------------------
# 3G-C.5 XY lookup helper
# ------------------------------------------------------------

def make_xy_lookup_3gc(
    df
):

    lookup = {}

    for dataset_index, group in (
        df.groupby("dataset_index")
    ):

        lookup[
            int(dataset_index)
        ] = (
            group[
                ["x", "y"]
            ]
            .to_numpy(dtype=float)
        )

    return lookup


EMPTY_XY = np.empty(
    (0, 2),
    dtype=float,
)


c1_xy_lookup = make_xy_lookup_3gc(
    c1_recovery_df
)

d5_xy_lookup = make_xy_lookup_3gc(
    d5_recovery_df
)

d6bt_xy_lookup = make_xy_lookup_3gc(
    d6bt_recovery_df
)

d6b9_xy_lookup = make_xy_lookup_3gc(
    d6b9_recovery_df
)


# ------------------------------------------------------------
# 3G-C.6 Nearest-distance helper
# ------------------------------------------------------------

def nearest_distance_to_set_3gc(
    point_xy,
    ref_xy
):

    if len(ref_xy) == 0:
        return np.inf

    point_xy = np.asarray(
        point_xy,
        dtype=float,
    )

    ref_xy = np.asarray(
        ref_xy,
        dtype=float,
    )

    distances = np.hypot(
        ref_xy[:, 0] - point_xy[0],
        ref_xy[:, 1] - point_xy[1],
    )

    return float(
        distances.min()
    )


# ------------------------------------------------------------
# 3G-C.7 Generic one-to-one matching
#
# Same globally-shortest greedy convention used previously.
# ------------------------------------------------------------

def one_to_one_match_3gc(
    a_xy,
    b_xy,
    radius_px,
):

    a_xy = np.asarray(
        a_xy,
        dtype=float,
    )

    b_xy = np.asarray(
        b_xy,
        dtype=float,
    )

    if (
        len(a_xy) == 0
        or
        len(b_xy) == 0
    ):
        return []


    candidate_pairs = []


    for i in range(len(a_xy)):

        ax, ay = a_xy[i]

        distances = np.hypot(
            b_xy[:, 0] - ax,
            b_xy[:, 1] - ay,
        )

        valid_js = np.where(
            distances <= radius_px
        )[0]

        for j in valid_js:

            candidate_pairs.append(
                (
                    float(distances[j]),
                    int(i),
                    int(j),
                )
            )


    candidate_pairs.sort(
        key=lambda z: (
            z[0],
            z[1],
            z[2],
        )
    )


    used_a = set()
    used_b = set()

    matches = []


    for distance, i, j in candidate_pairs:

        if i in used_a:
            continue

        if j in used_b:
            continue

        used_a.add(i)
        used_b.add(j)

        matches.append(
            {
                "a_index": int(i),
                "b_index": int(j),
                "distance_px": float(distance),
            }
        )


    return matches


# ------------------------------------------------------------
# 3G-C.8 Construct triples
#
# D6B9 is anchor:
#   anchor <-> D5
#   anchor <-> D6BT
#
# Triple exists only if same D6B9 anchor participates in both.
# ------------------------------------------------------------

triple_records = []
triple_image_records = []


for image_row in (
    b2_dev100_image_df
    .itertuples(index=False)
):

    dataset_index = int(
        image_row.dataset_index
    )

    radius_px = float(
        radius_lookup_3gc.loc[
            dataset_index,
            "matching_radius_px",
        ]
    )


    c1_xy = c1_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )

    d5_xy = d5_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )

    d6bt_xy = d6bt_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )

    d6b9_xy = d6b9_xy_lookup.get(
        dataset_index,
        EMPTY_XY,
    )


    # --------------------------------------------------------
    # Anchor D6B9 -> D5
    # --------------------------------------------------------

    match_b9_d5 = one_to_one_match_3gc(
        a_xy=d6b9_xy,
        b_xy=d5_xy,
        radius_px=radius_px,
    )


    # --------------------------------------------------------
    # Anchor D6B9 -> D6BT
    # --------------------------------------------------------

    match_b9_bt = one_to_one_match_3gc(
        a_xy=d6b9_xy,
        b_xy=d6bt_xy,
        radius_px=radius_px,
    )


    d5_by_anchor = {
        int(m["a_index"]):
            m
        for m in match_b9_d5
    }

    bt_by_anchor = {
        int(m["a_index"]):
            m
        for m in match_b9_bt
    }


    common_anchor_indices = sorted(
        set(
            d5_by_anchor.keys()
        )
        &
        set(
            bt_by_anchor.keys()
        )
    )


    complementary_triples = 0


    for triple_index, anchor_idx in enumerate(
        common_anchor_indices
    ):

        d6b9_point = d6b9_xy[
            anchor_idx
        ]


        d5_match = d5_by_anchor[
            anchor_idx
        ]

        bt_match = bt_by_anchor[
            anchor_idx
        ]


        d5_idx = int(
            d5_match[
                "b_index"
            ]
        )

        d6bt_idx = int(
            bt_match[
                "b_index"
            ]
        )


        d5_point = d5_xy[
            d5_idx
        ]

        d6bt_point = d6bt_xy[
            d6bt_idx
        ]


        d5_to_c1 = (
            nearest_distance_to_set_3gc(
                d5_point,
                c1_xy,
            )
        )

        d6bt_to_c1 = (
            nearest_distance_to_set_3gc(
                d6bt_point,
                c1_xy,
            )
        )

        d6b9_to_c1 = (
            nearest_distance_to_set_3gc(
                d6b9_point,
                c1_xy,
            )
        )


        complementary = bool(
            (
                d5_to_c1 > radius_px
            )
            and
            (
                d6bt_to_c1 > radius_px
            )
            and
            (
                d6b9_to_c1 > radius_px
            )
        )


        if complementary:
            complementary_triples += 1


        centroid_x = float(
            (
                d5_point[0]
                +
                d6bt_point[0]
                +
                d6b9_point[0]
            )
            /
            3.0
        )

        centroid_y = float(
            (
                d5_point[1]
                +
                d6bt_point[1]
                +
                d6b9_point[1]
            )
            /
            3.0
        )


        triple_records.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_row.image_id,

                "category":
                    image_row.category,

                "density_stratum":
                    image_row.density_stratum,

                "gt_count":
                    int(
                        image_row.gt_count
                    ),

                "triple_index":
                    int(
                        triple_index
                    ),

                "matching_radius_px":
                    radius_px,

                "d5_x":
                    float(
                        d5_point[0]
                    ),

                "d5_y":
                    float(
                        d5_point[1]
                    ),

                "d6bt_x":
                    float(
                        d6bt_point[0]
                    ),

                "d6bt_y":
                    float(
                        d6bt_point[1]
                    ),

                "d6b9_x":
                    float(
                        d6b9_point[0]
                    ),

                "d6b9_y":
                    float(
                        d6b9_point[1]
                    ),

                "D6B9_to_D5_distance_px":
                    float(
                        d5_match[
                            "distance_px"
                        ]
                    ),

                "D6B9_to_D6BT_distance_px":
                    float(
                        bt_match[
                            "distance_px"
                        ]
                    ),

                "d5_nearest_C1_distance_px":
                    (
                        float(d5_to_c1)
                        if np.isfinite(d5_to_c1)
                        else np.nan
                    ),

                "d6bt_nearest_C1_distance_px":
                    (
                        float(d6bt_to_c1)
                        if np.isfinite(d6bt_to_c1)
                        else np.nan
                    ),

                "d6b9_nearest_C1_distance_px":
                    (
                        float(d6b9_to_c1)
                        if np.isfinite(d6b9_to_c1)
                        else np.nan
                    ),

                "complementary_to_C1":
                    complementary,

                "representative_centroid_x":
                    centroid_x,

                "representative_centroid_y":
                    centroid_y,
            }
        )


    triple_image_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_row.image_id,

            "all_triples":
                int(
                    len(
                        common_anchor_indices
                    )
                ),

            "complementary_triples":
                int(
                    complementary_triples
                ),
        }
    )


step3gc_triple_df = pd.DataFrame(
    triple_records
)


step3gc_triple_image_df = pd.DataFrame(
    triple_image_records
)


assert len(step3gc_triple_image_df) == 100


# ------------------------------------------------------------
# 3G-C.9 Triple survival summary
# ------------------------------------------------------------

TOTAL_TRIPLES = int(
    len(
        step3gc_triple_df
    )
)


TOTAL_COMPLEMENTARY_TRIPLES = int(
    step3gc_triple_df[
        "complementary_to_C1"
    ].sum()
)


TRIPLE_COMPLEMENTARY_FRACTION = (
    TOTAL_COMPLEMENTARY_TRIPLES
    /
    TOTAL_TRIPLES
    if TOTAL_TRIPLES > 0
    else np.nan
)


# ------------------------------------------------------------
# 3G-C.10 Construct count system
# ------------------------------------------------------------

step3gc_count_df = (
    recovery_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
        ]
    ]
    .copy()
)


step3gc_count_df = (
    step3gc_count_df
    .merge(
        step3gc_triple_image_df[
            [
                "dataset_index",
                "all_triples",
                "complementary_triples",
            ]
        ],
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


step3gc_count_df[
    "all_triples"
] = (
    step3gc_count_df[
        "all_triples"
    ]
    .fillna(0)
    .astype(int)
)


step3gc_count_df[
    "complementary_triples"
] = (
    step3gc_count_df[
        "complementary_triples"
    ]
    .fillna(0)
    .astype(int)
)


step3gc_count_df[
    "B0_plus_C1_plus_triple"
] = (
    step3gc_count_df[
        "B0_plus_C1_raw"
    ]
    +
    step3gc_count_df[
        "complementary_triples"
    ]
)


# ------------------------------------------------------------
# 3G-C.11 Count metric helper
# ------------------------------------------------------------

def count_metrics_3gc(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )

    error = pred - gt


    return {
        "MAE":
            float(
                np.mean(
                    np.abs(error)
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "mean_signed_error":
            float(
                np.mean(error)
            ),

        "under_count_images":
            int(
                np.sum(
                    error < 0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    error == 0
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    error > 0
                )
            ),
    }


# ------------------------------------------------------------
# 3G-C.12 Count summary
# ------------------------------------------------------------

count_rows = []


for system_name in [
    "B0_unfiltered",
    "B0_plus_C1_raw",
    "B0_plus_C1_plus_triple",
]:

    pred = (
        step3gc_count_df[
            system_name
        ]
        .to_numpy()
    )

    gt = (
        step3gc_count_df[
            "gt_count"
        ]
        .to_numpy()
    )


    metrics = count_metrics_3gc(
        gt,
        pred,
    )


    added = (
        TOTAL_COMPLEMENTARY_TRIPLES
        if system_name
        ==
        "B0_plus_C1_plus_triple"
        else 0
    )


    count_rows.append(
        {
            "system":
                system_name,

            "additional_recovery_beyond_C1":
                int(
                    added
                ),

            "total_predicted_count":
                int(
                    pred.sum()
                ),

            **metrics,
        }
    )


step3gc_count_summary_df = pd.DataFrame(
    count_rows
)


# ------------------------------------------------------------
# 3G-C.13 Add references
# ------------------------------------------------------------

reference_rows = []


# Step 3D best

step3d_ref = (
    step3d_summary_df[
        step3d_summary_df[
            "system"
        ]
        ==
        "B0_plus_C1_plus_D6BT_C1supported"
    ]
    .iloc[0]
)


reference_rows.append(
    {
        "system":
            "REFERENCE_Step3D_C1supported_D6BT",

        "additional_recovery_beyond_C1":
            int(
                step3d_ref[
                    "additional_D_points"
                ]
            ),

        "total_predicted_count":
            int(
                step3d_ref[
                    "total_predicted_count"
                ]
            ),

        "MAE":
            float(
                step3d_ref[
                    "MAE"
                ]
            ),

        "RMSE":
            float(
                step3d_ref[
                    "RMSE"
                ]
            ),

        "mean_signed_error":
            float(
                step3d_ref[
                    "mean_signed_error"
                ]
            ),

        "under_count_images":
            int(
                step3d_ref[
                    "under_count_images"
                ]
            ),

        "exact_count_images":
            int(
                step3d_ref[
                    "exact_count_images"
                ]
            ),

        "over_count_images":
            int(
                step3d_ref[
                    "over_count_images"
                ]
            ),
    }
)


# Step 3E same-family

step3e_ref = (
    step3e_summary_df[
        step3e_summary_df[
            "system"
        ]
        ==
        "B0_plus_C1_plus_complementary_DD"
    ]
    .iloc[0]
)


reference_rows.append(
    {
        "system":
            "REFERENCE_Step3E_D6BTxD6B9",

        "additional_recovery_beyond_C1":
            int(
                step3e_ref[
                    "additional_recovery_beyond_C1"
                ]
            ),

        "total_predicted_count":
            int(
                step3e_ref[
                    "total_predicted_count"
                ]
            ),

        "MAE":
            float(
                step3e_ref[
                    "MAE"
                ]
            ),

        "RMSE":
            float(
                step3e_ref[
                    "RMSE"
                ]
            ),

        "mean_signed_error":
            float(
                step3e_ref[
                    "mean_signed_error"
                ]
            ),

        "under_count_images":
            int(
                step3e_ref[
                    "under_count_images"
                ]
            ),

        "exact_count_images":
            int(
                step3e_ref[
                    "exact_count_images"
                ]
            ),

        "over_count_images":
            int(
                step3e_ref[
                    "over_count_images"
                ]
            ),
    }
)


# Step 3G-B pairwise references

for ref_system in [
    "B0_plus_C1_plus_D5xD6BT",
    "B0_plus_C1_plus_D5xD6B9",
]:

    ref_row = (
        step3gb_count_summary_with_ref_df[
            step3gb_count_summary_with_ref_df[
                "system"
            ]
            ==
            ref_system
        ]
        .iloc[0]
    )


    reference_rows.append(
        {
            "system":
                f"REFERENCE_{ref_system}",

            "additional_recovery_beyond_C1":
                int(
                    ref_row[
                        "additional_pairs_beyond_C1"
                    ]
                ),

            "total_predicted_count":
                int(
                    ref_row[
                        "total_predicted_count"
                    ]
                ),

            "MAE":
                float(
                    ref_row[
                        "MAE"
                    ]
                ),

            "RMSE":
                float(
                    ref_row[
                        "RMSE"
                    ]
                ),

            "mean_signed_error":
                float(
                    ref_row[
                        "mean_signed_error"
                    ]
                ),

            "under_count_images":
                int(
                    ref_row[
                        "under_count_images"
                    ]
                ),

            "exact_count_images":
                int(
                    ref_row[
                        "exact_count_images"
                    ]
                ),

            "over_count_images":
                int(
                    ref_row[
                        "over_count_images"
                    ]
                ),
        }
    )


step3gc_count_summary_with_refs_df = pd.concat(
    [
        step3gc_count_summary_df,
        pd.DataFrame(
            reference_rows
        ),
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 3G-C.14 Accepted triple representatives
# ------------------------------------------------------------

accepted_triple_df = (
    step3gc_triple_df[
        step3gc_triple_df[
            "complementary_to_C1"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


accepted_triple_df[
    "eval_x"
] = (
    accepted_triple_df[
        "representative_centroid_x"
    ]
)


accepted_triple_df[
    "eval_y"
] = (
    accepted_triple_df[
        "representative_centroid_y"
    ]
)


# ------------------------------------------------------------
# 3G-C.15 Evaluate triple against residual GT
# ------------------------------------------------------------

triple_gt_image_records = []


for image_row in (
    b2_dev100_image_df
    .itertuples(index=False)
):

    dataset_index = int(
        image_row.dataset_index
    )


    pred_subset = (
        accepted_triple_df[
            accepted_triple_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
        .reset_index(drop=True)
    )


    gt_subset = (
        residual_gt_df[
            residual_gt_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
        .reset_index(drop=True)
    )


    pred_xy = (
        pred_subset[
            [
                "eval_x",
                "eval_y",
            ]
        ]
        .to_numpy(dtype=float)
    )


    gt_xy = (
        gt_subset[
            [
                "gt_x",
                "gt_y",
            ]
        ]
        .to_numpy(dtype=float)
    )


    radius_px = float(
        radius_lookup_3gc.loc[
            dataset_index,
            "matching_radius_px",
        ]
    )


    matches = one_to_one_match_3gc(
        a_xy=pred_xy,
        b_xy=gt_xy,
        radius_px=radius_px,
    )


    tp = int(
        len(matches)
    )

    candidates = int(
        len(pred_subset)
    )

    residual_gt_count = int(
        len(gt_subset)
    )


    triple_gt_image_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_row.image_id,

            "category":
                image_row.category,

            "density_stratum":
                image_row.density_stratum,

            "residual_gt_after_B0_C1":
                residual_gt_count,

            "accepted_triples":
                candidates,

            "incremental_recovered_gt":
                tp,

            "unsupported_incremental":
                (
                    candidates
                    -
                    tp
                ),
        }
    )


step3gc_gt_image_df = pd.DataFrame(
    triple_gt_image_records
)


assert len(step3gc_gt_image_df) == 100


# ------------------------------------------------------------
# 3G-C.16 Overall GT quality
# ------------------------------------------------------------

TOTAL_RESIDUAL_GT = int(
    step3gc_gt_image_df[
        "residual_gt_after_B0_C1"
    ].sum()
)


TOTAL_ACCEPTED_TRIPLES = int(
    step3gc_gt_image_df[
        "accepted_triples"
    ].sum()
)


TOTAL_INCREMENTAL_TP = int(
    step3gc_gt_image_df[
        "incremental_recovered_gt"
    ].sum()
)


TOTAL_UNSUPPORTED = int(
    step3gc_gt_image_df[
        "unsupported_incremental"
    ].sum()
)


TRIPLE_PRECISION_LIKE = (
    TOTAL_INCREMENTAL_TP
    /
    TOTAL_ACCEPTED_TRIPLES
    if TOTAL_ACCEPTED_TRIPLES > 0
    else np.nan
)


TRIPLE_RESIDUAL_RECALL = (
    TOTAL_INCREMENTAL_TP
    /
    TOTAL_RESIDUAL_GT
    if TOTAL_RESIDUAL_GT > 0
    else np.nan
)


TRIPLE_FP_PER_TP = (
    TOTAL_UNSUPPORTED
    /
    TOTAL_INCREMENTAL_TP
    if TOTAL_INCREMENTAL_TP > 0
    else np.nan
)


step3gc_gt_summary_df = pd.DataFrame(
    [
        {
            "consensus":
                "D5xD6BTxD6B9",

            "total_residual_gt_after_B0_C1":
                TOTAL_RESIDUAL_GT,

            "accepted_consensus_units":
                TOTAL_ACCEPTED_TRIPLES,

            "incremental_recovered_gt":
                TOTAL_INCREMENTAL_TP,

            "unsupported_incremental":
                TOTAL_UNSUPPORTED,

            "incremental_precision_like":
                TRIPLE_PRECISION_LIKE,

            "residual_GT_recall":
                TRIPLE_RESIDUAL_RECALL,

            "FP_per_incremental_TP":
                TRIPLE_FP_PER_TP,
        }
    ]
)


# ------------------------------------------------------------
# 3G-C.17 Add GT-quality references from 3G-B
# ------------------------------------------------------------

gt_reference_rows = []


for pair_id in [
    "D5xD6BT",
    "D5xD6B9",
    "REFERENCE_D6BTxD6B9_Step3E",
]:

    ref_row = (
        step3gb_gt_summary_with_ref_df[
            step3gb_gt_summary_with_ref_df[
                "pair_id"
            ]
            ==
            pair_id
        ]
        .iloc[0]
    )


    gt_reference_rows.append(
        {
            "consensus":
                f"REFERENCE_{pair_id}",

            "total_residual_gt_after_B0_C1":
                int(
                    ref_row[
                        "total_residual_gt_after_B0_C1"
                    ]
                ),

            "accepted_consensus_units":
                int(
                    ref_row[
                        "accepted_consensus_pairs"
                    ]
                ),

            "incremental_recovered_gt":
                int(
                    ref_row[
                        "incremental_recovered_gt"
                    ]
                ),

            "unsupported_incremental":
                int(
                    ref_row[
                        "unsupported_incremental"
                    ]
                ),

            "incremental_precision_like":
                float(
                    ref_row[
                        "incremental_precision_like"
                    ]
                ),

            "residual_GT_recall":
                float(
                    ref_row[
                        "residual_GT_recall"
                    ]
                ),

            "FP_per_incremental_TP":
                float(
                    ref_row[
                        "FP_per_incremental_TP"
                    ]
                ),
        }
    )


step3gc_gt_summary_with_refs_df = pd.concat(
    [
        step3gc_gt_summary_df,
        pd.DataFrame(
            gt_reference_rows
        ),
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 3G-C.18 Density GT quality
# ------------------------------------------------------------

step3gc_density_df = (
    step3gc_gt_image_df
    .groupby(
        "density_stratum",
        as_index=False,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        residual_gt_after_B0_C1=(
            "residual_gt_after_B0_C1",
            "sum",
        ),

        accepted_triples=(
            "accepted_triples",
            "sum",
        ),

        incremental_recovered_gt=(
            "incremental_recovered_gt",
            "sum",
        ),

        unsupported_incremental=(
            "unsupported_incremental",
            "sum",
        ),
    )
)


step3gc_density_df[
    "incremental_precision_like"
] = np.where(
    step3gc_density_df[
        "accepted_triples"
    ]
    >
    0,

    step3gc_density_df[
        "incremental_recovered_gt"
    ]
    /
    step3gc_density_df[
        "accepted_triples"
    ],

    np.nan,
)


step3gc_density_df[
    "residual_GT_recall"
] = np.where(
    step3gc_density_df[
        "residual_gt_after_B0_C1"
    ]
    >
    0,

    step3gc_density_df[
        "incremental_recovered_gt"
    ]
    /
    step3gc_density_df[
        "residual_gt_after_B0_C1"
    ],

    np.nan,
)


step3gc_density_df[
    "FP_per_incremental_TP"
] = np.where(
    step3gc_density_df[
        "incremental_recovered_gt"
    ]
    >
    0,

    step3gc_density_df[
        "unsupported_incremental"
    ]
    /
    step3gc_density_df[
        "incremental_recovered_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3G-C.19 Save
# ------------------------------------------------------------

step3gc_triple_df.to_csv(
    STEP3GC_TRIPLE_FILE,
    index=False,
)


step3gc_count_df.to_csv(
    STEP3GC_COUNT_FILE,
    index=False,
)


step3gc_count_summary_with_refs_df.to_csv(
    STEP3GC_COUNT_SUMMARY_FILE,
    index=False,
)


step3gc_gt_summary_with_refs_df.to_csv(
    STEP3GC_GT_SUMMARY_FILE,
    index=False,
)


step3gc_density_df.to_csv(
    STEP3GC_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3G-C.20 Manifest
# ------------------------------------------------------------

step3gc_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 3G-C - triple consensus recovery",

    "sources": [
        "D5-768",
        "D6B-T-native",
        "D6B9-strict",
    ],

    "anchor_source":
        "D6B9-strict",

    "triple_construction":
        (
            "D6B9 anchor independently matched one-to-one "
            "to D5-768 and D6B-T-native within frozen R; "
            "anchor retained only if both matches exist."
        ),

    "complementary_rule":
        (
            "all three endpoints farther than R from every "
            "C1 recovery point"
        ),

    "count_contribution_per_triple":
        1,

    "GT_used_for_selection":
        False,

    "GT_used_for_diagnostic_evaluation":
        True,

    "sam2_reprompting":
        False,

    "density_used_for_routing":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,

    "total_triples":
        TOTAL_TRIPLES,

    "complementary_triples":
        TOTAL_COMPLEMENTARY_TRIPLES,
}


with open(
    STEP3GC_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step3gc_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 3G-C.21 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3G-C - TRIPLE CONSENSUS")
print("=" * 125)

print(
    "D5 recovery pool            :",
    len(d5_recovery_df)
)

print(
    "D6B-T recovery pool         :",
    len(d6bt_recovery_df)
)

print(
    "D6B9 recovery pool          :",
    len(d6b9_recovery_df)
)

print(
    "\nAll triple anchors          :",
    TOTAL_TRIPLES
)

print(
    "Complementary triples       :",
    TOTAL_COMPLEMENTARY_TRIPLES
)

print(
    "Complementary fraction      :",
    round(
        TRIPLE_COMPLEMENTARY_FRACTION,
        4,
    )
)

print(
    "\nGT used for selection       : NO"
)

print(
    "Each accepted triple        : +1 count"
)

print(
    "SAM2 re-prompting           : NO"
)

print(
    "Density used for routing    : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Test set accessed           : NO"
)


print(
    "\nOverall count performance:"
)

display(
    step3gc_count_summary_with_refs_df[
        [
            "system",
            "additional_recovery_beyond_C1",
            "total_predicted_count",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .round(4)
)


print(
    "\nIncremental GT quality after B0 + C1:"
)

display(
    step3gc_gt_summary_with_refs_df[
        [
            "consensus",
            "total_residual_gt_after_B0_C1",
            "accepted_consensus_units",
            "incremental_recovered_gt",
            "unsupported_incremental",
            "incremental_precision_like",
            "residual_GT_recall",
            "FP_per_incremental_TP",
        ]
    ]
    .round(4)
)


print(
    "\nTriple GT quality by density:"
)

display(
    step3gc_density_df[
        [
            "density_stratum",
            "num_images",
            "residual_gt_after_B0_C1",
            "accepted_triples",
            "incremental_recovered_gt",
            "unsupported_incremental",
            "incremental_precision_like",
            "residual_GT_recall",
            "FP_per_incremental_TP",
        ]
    ]
    .round(4)
)


print(
    "\nImages with most accepted triples:"
)

display(
    step3gc_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "complementary_triples",
            "B0_plus_C1_plus_triple",
        ]
    ]
    .sort_values(
        "complementary_triples",
        ascending=False,
    )
    .head(30)
)


print("\nSaved triple diagnostics:")
print(STEP3GC_TRIPLE_FILE)

print("\nSaved per-image counts:")
print(STEP3GC_COUNT_FILE)

print("\nSaved count summary:")
print(STEP3GC_COUNT_SUMMARY_FILE)

print("\nSaved GT-quality summary:")
print(STEP3GC_GT_SUMMARY_FILE)

print("\nSaved density summary:")
print(STEP3GC_DENSITY_FILE)

print("\nSaved manifest:")
print(STEP3GC_MANIFEST_FILE)

print("=" * 125)

STEP 3G-C - TRIPLE CONSENSUS
D5 recovery pool            : 7776
D6B-T recovery pool         : 8047
D6B9 recovery pool          : 1911

All triple anchors          : 735
Complementary triples       : 567
Complementary fraction      : 0.7714

GT used for selection       : NO
Each accepted triple        : +1 count
SAM2 re-prompting           : NO
Density used for routing    : NO
Threshold tuning            : NO
Test set accessed           : NO

Overall count performance:


,system,additional_recovery_beyond_C1,total_predicted_count,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,0,2298,80.69,270.2339,-77.47,76,4,20
1,B0_plus_C1_raw,0,3513,72.00,262.2353,-65.32,52,5,43
2,B0_plus_C1_plus_triple,567,4080,72.39,261.5693,-59.65,41,3,56
3,REFERENCE_Step3D_C1supported_D6BT,805,4318,71.29,258.4460,-57.27,37,2,61
4,REFERENCE_Step3E_D6BTxD6B9,1355,4868,75.99,261.2695,-51.77,34,4,62
5,REFERENCE_B0_plus_C1_plus_D5xD6BT,2446,5959,79.24,258.9390,-40.86,29,3,68
6,REFERENCE_B0_plus_C1_plus_D5xD6B9,656,4169,72.82,261.4989,-58.76,41,3,56



Incremental GT quality after B0 + C1:


,consensus,total_residual_gt_after_B0_C1,accepted_consensus_units,incremental_recovered_gt,unsupported_incremental,incremental_precision_like,residual_GT_recall,FP_per_incremental_TP
0,D5xD6BTxD6B9,5018,567,187,380,0.3298,0.0373,2.0321
1,REFERENCE_D5xD6BT,5018,2446,586,1860,0.2396,0.1168,3.1741
2,REFERENCE_D5xD6B9,5018,656,202,454,0.3079,0.0403,2.2475
3,REFERENCE_REFERENCE_D6BTxD6B9_Step3E,5018,1355,358,997,0.2642,0.0713,2.7849



Triple GT quality by density:


,density_stratum,num_images,residual_gt_after_B0_C1,accepted_triples,incremental_recovered_gt,unsupported_incremental,incremental_precision_like,residual_GT_recall,FP_per_incremental_TP
0,S1_0_20,20,64,50,9,41,0.1800,0.1406,4.5556
1,S2_20_40,20,131,130,33,97,0.2538,0.2519,2.9394
2,S3_40_60,20,285,152,41,111,0.2697,0.1439,2.7073
3,S4_60_80,20,341,110,36,74,0.3273,0.1056,2.0556
4,S5_80_95,10,534,66,38,28,0.5758,0.0712,0.7368
5,S6_95_100,10,3663,59,30,29,0.5085,0.0082,0.9667



Images with most accepted triples:


,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw,complementary_triples,B0_plus_C1_plus_triple
77,938,5866.jpg,grapes,S6_95_100,267,47,65,38,103
98,1260,7100.jpg,milk cartons,S3_40_60,27,40,48,26,74
43,396,2846.jpg,chairs,S4_60_80,38,7,30,24,54
26,217,919.jpg,skateboard,S3_40_60,32,11,18,23,41
45,425,2884.jpg,chairs,S5_80_95,165,38,46,23,69
99,1269,7060.jpg,pills,S4_60_80,65,11,25,22,47
86,1122,6896.jpg,books,S2_20_40,15,14,19,20,39
24,183,838.jpg,books,S5_80_95,113,43,70,20,90
1,14,220.jpg,grapes,S4_60_80,59,20,52,19,71
2,20,228.jpg,grapes,S4_60_80,44,5,27,19,46



Saved triple diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_D5_D6BT_D6B9_triple_consensus.csv

Saved per-image counts:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_triple_consensus_counts_per_image.csv

Saved count summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_triple_consensus_count_summary.csv

Saved GT-quality summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_triple_consensus_GT_quality_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_triple_consensus_by_density.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/result

## Step 4 — Recovery-to-Count Integration

In [34]:
# ============================================================
# Step 4A PRE-CHECK
# Identify the exact per-image count/recovery tables currently
# available before implementing global contribution scaling.
#
# Diagnostic only:
# - no experiment
# - no files changed
# - no GT-based selection
# ============================================================

candidate_var_names = [
    "recovery_count_df",

    "step3d_count_df",
    "step3d_image_df",
    "step3d_per_image_df",
    "step3d_summary_df",

    "step3e_count_df",
    "step3e_image_df",
    "step3e_per_image_df",
    "step3e_summary_df",

    "step3gb_count_df",
    "step3gb_pair_image_df",
    "step3gb_count_summary_with_ref_df",

    "step3gc_count_df",
    "step3gc_triple_image_df",
    "step3gc_count_summary_with_refs_df",
]

print("=" * 110)
print("STEP 4A PRE-CHECK — AVAILABLE VARIABLES")
print("=" * 110)

for var_name in candidate_var_names:

    if var_name not in globals():

        print(
            f"\n{var_name}: NOT FOUND"
        )

        continue


    obj = globals()[var_name]

    print(
        f"\n{var_name}: FOUND"
    )

    print(
        "Type:",
        type(obj).__name__,
    )


    if hasattr(
        obj,
        "shape",
    ):

        print(
            "Shape:",
            obj.shape,
        )


    if hasattr(
        obj,
        "columns",
    ):

        print(
            "Columns:"
        )

        print(
            obj.columns.tolist()
        )


        print(
            "First 2 rows:"
        )

        display(
            obj.head(2)
        )

print("\n" + "=" * 110)

STEP 4A PRE-CHECK — AVAILABLE VARIABLES

recovery_count_df: FOUND
Type: DataFrame
Shape: (100, 20)
Columns:
['dataset_index', 'image_id', 'category', 'density_stratum', 'gt_count', 'B0_unfiltered', 'B0_plus_C1_raw__added_points', 'B0_plus_C1_raw', 'B0_plus_D6B9_raw__added_points', 'B0_plus_D6B9_raw', 'B0_plus_D6BT_raw__added_points', 'B0_plus_D6BT_raw', 'B0_unfiltered__signed_error', 'B0_unfiltered__abs_error', 'B0_plus_C1_raw__signed_error', 'B0_plus_C1_raw__abs_error', 'B0_plus_D6B9_raw__signed_error', 'B0_plus_D6B9_raw__abs_error', 'B0_plus_D6BT_raw__signed_error', 'B0_plus_D6BT_raw__abs_error']
First 2 rows:


,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw__added_points,B0_plus_C1_raw,B0_plus_D6B9_raw__added_points,B0_plus_D6B9_raw,B0_plus_D6BT_raw__added_points,B0_plus_D6BT_raw,B0_unfiltered__signed_error,B0_unfiltered__abs_error,B0_plus_C1_raw__signed_error,B0_plus_C1_raw__abs_error,B0_plus_D6B9_raw__signed_error,B0_plus_D6B9_raw__abs_error,B0_plus_D6BT_raw__signed_error,B0_plus_D6BT_raw__abs_error
0,9,215.jpg,grapes,S6_95_100,259,103,5,108,26,129,186,289,-156,156,-151,151,-130,130,30,30
1,14,220.jpg,grapes,S4_60_80,59,20,32,52,74,94,362,382,-39,39,-7,7,35,35,323,323



step3d_count_df: FOUND
Type: DataFrame
Shape: (100, 16)
Columns:
['dataset_index', 'image_id', 'category', 'density_stratum', 'gt_count', 'B0_unfiltered', 'B0_plus_C1_raw', 'B0_plus_C1_raw__added_points', 'D6B9_C1_supported_added', 'D6BT_C1_supported_added', 'B0_plus_C1_plus_D6B9_C1supported', 'B0_plus_C1_plus_D6BT_C1supported', 'B0_unfiltered__abs_error', 'B0_plus_C1_raw__abs_error', 'B0_plus_C1_plus_D6B9_C1supported__abs_error', 'B0_plus_C1_plus_D6BT_C1supported__abs_error']
First 2 rows:


,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw,B0_plus_C1_raw__added_points,D6B9_C1_supported_added,D6BT_C1_supported_added,B0_plus_C1_plus_D6B9_C1supported,B0_plus_C1_plus_D6BT_C1supported,B0_unfiltered__abs_error,B0_plus_C1_raw__abs_error,B0_plus_C1_plus_D6B9_C1supported__abs_error,B0_plus_C1_plus_D6BT_C1supported__abs_error
0,9,215.jpg,grapes,S6_95_100,259,103,108,5,0,2,108,110,156,151,151,149
1,14,220.jpg,grapes,S4_60_80,59,20,52,32,0,28,52,80,39,7,7,21



step3d_image_df: NOT FOUND

step3d_per_image_df: NOT FOUND

step3d_summary_df: FOUND
Type: DataFrame
Shape: (4, 13)
Columns:
['system', 'additional_D_points', 'total_predicted_count', 'MAE', 'RMSE', 'mean_signed_error', 'under_count_images', 'exact_count_images', 'over_count_images', 'delta_MAE_vs_B0', 'delta_RMSE_vs_B0', 'delta_MAE_vs_C1', 'delta_RMSE_vs_C1']
First 2 rows:


...

,dataset_index,image_id,category,density_stratum,gt_count,B0_unfiltered,B0_plus_C1_raw,all_triples,complementary_triples,B0_plus_C1_plus_triple
0,9,215.jpg,grapes,S6_95_100,259,103,108,14,14,122
1,14,220.jpg,grapes,S4_60_80,59,20,52,21,19,71



step3gc_triple_image_df: FOUND
Type: DataFrame
Shape: (100, 4)
Columns:
['dataset_index', 'image_id', 'all_triples', 'complementary_triples']
First 2 rows:


,dataset_index,image_id,all_triples,complementary_triples
0,9,215.jpg,14,14
1,14,220.jpg,21,19



step3gc_count_summary_with_refs_df: FOUND
Type: DataFrame
Shape: (7, 9)
Columns:
['system', 'additional_recovery_beyond_C1', 'total_predicted_count', 'MAE', 'RMSE', 'mean_signed_error', 'under_count_images', 'exact_count_images', 'over_count_images']
First 2 rows:


,system,additional_recovery_beyond_C1,total_predicted_count,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
0,B0_unfiltered,0,2298,80.69,270.233880,-77.47,76,4,20
1,B0_plus_C1_raw,0,3513,72.00,262.235314,-65.32,52,5,43


In [35]:
# ============================================================
# Step 4A — Global Recovery Contribution Scaling
#
# Question:
# Does semantic recovery help more when its count contribution
# is down-weighted rather than assuming every accepted recovery
# unit contributes exactly +1?
#
# Formula:
#
#   predicted_count =
#       B0_plus_C1_raw + alpha * recovery_units
#
# alpha grid:
#   0.00, 0.25, 0.50, 0.75, 1.00
#
# Recovery rules:
#   R1 = Step 3D C1-supported D6B-T-native
#   R2 = Step 3E D6B-T x D6B9 complementary consensus
#   R3 = Step 3G-B D5 x D6B-T
#   R4 = Step 3G-B D5 x D6B9
#   R5 = Step 3G-C triple consensus
#
# Developmental sensitivity experiment only.
#
# IMPORTANT:
# - semantic evidence unchanged
# - no GT-based routing
# - no density-based routing
# - no SAM2 re-prompting
# - no threshold changes
# - test set untouched
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 4A.1 Output files
# ------------------------------------------------------------

STEP4A_LONG_FILE = (
    B2_TABLES_DIR
    / "B0_C1_global_recovery_scaling_long.csv"
)

STEP4A_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_global_recovery_scaling_summary.csv"
)

STEP4A_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_global_recovery_scaling_by_density.csv"
)

STEP4A_PER_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_C1_global_recovery_scaling_per_image.csv"
)

STEP4A_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_4A_global_scaling_manifest.json"
)


# ------------------------------------------------------------
# 4A.2 Preconditions
# ------------------------------------------------------------

required_vars = [
    "step3d_count_df",
    "step3e_count_df",
    "step3gb_count_df",
    "step3gc_count_df",
]

for var_name in required_vars:

    assert var_name in globals(), (
        f"Required variable not found: {var_name}"
    )


for df_name in required_vars:

    df = globals()[df_name]

    assert len(df) == 100, (
        f"{df_name} does not contain 100 rows"
    )

    assert (
        df["dataset_index"].nunique()
        ==
        100
    ), (
        f"{df_name} has duplicated dataset_index"
    )


# ------------------------------------------------------------
# 4A.3 Build one authoritative scaling table
# ------------------------------------------------------------

step4a_df = (
    step3d_count_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "D6BT_C1_supported_added",
        ]
    ]
    .copy()
    .rename(
        columns={
            "D6BT_C1_supported_added":
                "R_Step3D_D6BT_C1supported"
        }
    )
)


# Step 3E

step4a_df = (
    step4a_df
    .merge(
        step3e_count_df[
            [
                "dataset_index",
                "complementary_DD_pairs",
            ]
        ]
        .rename(
            columns={
                "complementary_DD_pairs":
                    "R_Step3E_D6BTxD6B9"
            }
        ),
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


# Step 3G-B

step4a_df = (
    step4a_df
    .merge(
        step3gb_count_df[
            [
                "dataset_index",
                "D5xD6BT__added_pairs",
                "D5xD6B9__added_pairs",
            ]
        ]
        .rename(
            columns={
                "D5xD6BT__added_pairs":
                    "R_Step3GB_D5xD6BT",

                "D5xD6B9__added_pairs":
                    "R_Step3GB_D5xD6B9",
            }
        ),
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


# Step 3G-C

step4a_df = (
    step4a_df
    .merge(
        step3gc_count_df[
            [
                "dataset_index",
                "complementary_triples",
            ]
        ]
        .rename(
            columns={
                "complementary_triples":
                    "R_Step3GC_Triple"
            }
        ),
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


assert len(step4a_df) == 100


# ------------------------------------------------------------
# 4A.4 Verify baseline agreement across source tables
# ------------------------------------------------------------

baseline_checks = [
    step3e_count_df,
    step3gb_count_df,
    step3gc_count_df,
]


for other_df in baseline_checks:

    merged_check = (
        step4a_df[
            [
                "dataset_index",
                "B0_plus_C1_raw",
            ]
        ]
        .merge(
            other_df[
                [
                    "dataset_index",
                    "B0_plus_C1_raw",
                ]
            ],
            on="dataset_index",
            suffixes=(
                "_base",
                "_other",
            ),
            validate="one_to_one",
        )
    )


    assert np.array_equal(
        merged_check[
            "B0_plus_C1_raw_base"
        ].to_numpy(),
        merged_check[
            "B0_plus_C1_raw_other"
        ].to_numpy(),
    ), (
        "B0+C1 baseline mismatch detected"
    )


# ------------------------------------------------------------
# 4A.5 Recovery configurations
# ------------------------------------------------------------

RECOVERY_CONFIGS = [
    {
        "rule_id":
            "Step3D_D6BT_C1supported",

        "recovery_col":
            "R_Step3D_D6BT_C1supported",
    },

    {
        "rule_id":
            "Step3E_D6BTxD6B9",

        "recovery_col":
            "R_Step3E_D6BTxD6B9",
    },

    {
        "rule_id":
            "Step3GB_D5xD6BT",

        "recovery_col":
            "R_Step3GB_D5xD6BT",
    },

    {
        "rule_id":
            "Step3GB_D5xD6B9",

        "recovery_col":
            "R_Step3GB_D5xD6B9",
    },

    {
        "rule_id":
            "Step3GC_Triple",

        "recovery_col":
            "R_Step3GC_Triple",
    },
]


ALPHA_GRID = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]


# ------------------------------------------------------------
# 4A.6 Metric helper
# ------------------------------------------------------------

def count_metrics_4a(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )


    error = pred - gt


    return {
        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "mean_signed_error":
            float(
                np.mean(
                    error
                )
            ),

        "under_count_images":
            int(
                np.sum(
                    error < 0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    np.isclose(
                        error,
                        0.0,
                    )
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    error > 0
                )
            ),
    }


# ------------------------------------------------------------
# 4A.7 Run global scaling sweep
# ------------------------------------------------------------

long_records = []

summary_records = []

density_records = []


gt_all = (
    step4a_df[
        "gt_count"
    ]
    .to_numpy(
        dtype=float
    )
)


base_all = (
    step4a_df[
        "B0_plus_C1_raw"
    ]
    .to_numpy(
        dtype=float
    )
)


for config in RECOVERY_CONFIGS:

    rule_id = config[
        "rule_id"
    ]

    recovery_col = config[
        "recovery_col"
    ]


    recovery_all = (
        step4a_df[
            recovery_col
        ]
        .to_numpy(
            dtype=float
        )
    )


    for alpha in ALPHA_GRID:

        pred_all = (
            base_all
            +
            alpha
            *
            recovery_all
        )


        metrics = count_metrics_4a(
            gt_all,
            pred_all,
        )


        summary_records.append(
            {
                "rule_id":
                    rule_id,

                "alpha":
                    float(
                        alpha
                    ),

                "total_recovery_units":
                    int(
                        recovery_all.sum()
                    ),

                "weighted_recovery_contribution":
                    float(
                        alpha
                        *
                        recovery_all.sum()
                    ),

                "total_predicted_count":
                    float(
                        pred_all.sum()
                    ),

                **metrics,
            }
        )


        # ----------------------------------------------------
        # Per-image rows
        # ----------------------------------------------------

        signed_error = (
            pred_all
            -
            gt_all
        )


        for row_idx, row in enumerate(
            step4a_df.itertuples(
                index=False
            )
        ):

            long_records.append(
                {
                    "dataset_index":
                        int(
                            row.dataset_index
                        ),

                    "image_id":
                        row.image_id,

                    "category":
                        row.category,

                    "density_stratum":
                        row.density_stratum,

                    "gt_count":
                        float(
                            row.gt_count
                        ),

                    "B0_plus_C1_raw":
                        float(
                            row.B0_plus_C1_raw
                        ),

                    "rule_id":
                        rule_id,

                    "recovery_units":
                        float(
                            recovery_all[
                                row_idx
                            ]
                        ),

                    "alpha":
                        float(
                            alpha
                        ),

                    "weighted_recovery":
                        float(
                            alpha
                            *
                            recovery_all[
                                row_idx
                            ]
                        ),

                    "predicted_count":
                        float(
                            pred_all[
                                row_idx
                            ]
                        ),

                    "signed_error":
                        float(
                            signed_error[
                                row_idx
                            ]
                        ),

                    "abs_error":
                        float(
                            abs(
                                signed_error[
                                    row_idx
                                ]
                            )
                        ),
                }
            )


        # ----------------------------------------------------
        # Density diagnostic
        #
        # Diagnostic only; NOT used for routing.
        # ----------------------------------------------------

        temp_df = (
            step4a_df[
                [
                    "density_stratum",
                    "gt_count",
                    recovery_col,
                    "B0_plus_C1_raw",
                ]
            ]
            .copy()
        )


        temp_df[
            "predicted_count"
        ] = (
            temp_df[
                "B0_plus_C1_raw"
            ]
            +
            alpha
            *
            temp_df[
                recovery_col
            ]
        )


        for density_stratum, group in (
            temp_df.groupby(
                "density_stratum"
            )
        ):

            density_metrics = (
                count_metrics_4a(
                    group[
                        "gt_count"
                    ].to_numpy(
                        dtype=float
                    ),
                    group[
                        "predicted_count"
                    ].to_numpy(
                        dtype=float
                    ),
                )
            )


            density_records.append(
                {
                    "density_stratum":
                        density_stratum,

                    "rule_id":
                        rule_id,

                    "alpha":
                        float(
                            alpha
                        ),

                    "num_images":
                        int(
                            len(
                                group
                            )
                        ),

                    "recovery_units":
                        int(
                            group[
                                recovery_col
                            ].sum()
                        ),

                    **density_metrics,
                }
            )


step4a_long_df = pd.DataFrame(
    long_records
)


step4a_summary_df = pd.DataFrame(
    summary_records
)


step4a_density_df = pd.DataFrame(
    density_records
)


# ------------------------------------------------------------
# 4A.8 Baseline integrity check
#
# alpha=0 must give identical metrics for every rule.
# ------------------------------------------------------------

alpha0_df = (
    step4a_summary_df[
        step4a_summary_df[
            "alpha"
        ]
        ==
        0.0
    ]
)


assert (
    alpha0_df[
        "MAE"
    ].nunique()
    ==
    1
)


assert np.isclose(
    alpha0_df[
        "MAE"
    ].iloc[0],
    72.00,
)


assert np.isclose(
    alpha0_df[
        "RMSE"
    ].iloc[0],
    262.235314,
    atol=1e-5,
)


# ------------------------------------------------------------
# 4A.9 Add delta vs alpha=0 baseline
# ------------------------------------------------------------

baseline_mae = float(
    alpha0_df[
        "MAE"
    ].iloc[0]
)


baseline_rmse = float(
    alpha0_df[
        "RMSE"
    ].iloc[0]
)


step4a_summary_df[
    "delta_MAE_vs_C1"
] = (
    step4a_summary_df[
        "MAE"
    ]
    -
    baseline_mae
)


step4a_summary_df[
    "delta_RMSE_vs_C1"
] = (
    step4a_summary_df[
        "RMSE"
    ]
    -
    baseline_rmse
)


# ------------------------------------------------------------
# 4A.10 Best sampled alpha per rule
#
# Developmental diagnostic only.
# Not a final tuned hyperparameter.
# ------------------------------------------------------------

best_mae_rows = (
    step4a_summary_df
    .sort_values(
        [
            "rule_id",
            "MAE",
            "RMSE",
            "alpha",
        ]
    )
    .groupby(
        "rule_id",
        as_index=False,
    )
    .first()
)


best_rmse_rows = (
    step4a_summary_df
    .sort_values(
        [
            "rule_id",
            "RMSE",
            "MAE",
            "alpha",
        ]
    )
    .groupby(
        "rule_id",
        as_index=False,
    )
    .first()
)


# ------------------------------------------------------------
# 4A.11 Compact per-image source table
# ------------------------------------------------------------

step4a_per_image_df = (
    step4a_df.copy()
)


# ------------------------------------------------------------
# 4A.12 Save
# ------------------------------------------------------------

step4a_long_df.to_csv(
    STEP4A_LONG_FILE,
    index=False,
)


step4a_summary_df.to_csv(
    STEP4A_SUMMARY_FILE,
    index=False,
)


step4a_density_df.to_csv(
    STEP4A_DENSITY_FILE,
    index=False,
)


step4a_per_image_df.to_csv(
    STEP4A_PER_IMAGE_FILE,
    index=False,
)


step4a_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 4A - global recovery contribution scaling",

    "formula":
        (
            "pred = B0_plus_C1_raw + alpha * recovery_units"
        ),

    "alpha_grid":
        ALPHA_GRID,

    "recovery_rules": [
        config[
            "rule_id"
        ]
        for config in RECOVERY_CONFIGS
    ],

    "GT_used_for_alpha_evaluation":
        True,

    "GT_used_for_inference_routing":
        False,

    "density_used_for_routing":
        False,

    "semantic_sources_changed":
        False,

    "sam2_reprompting":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,

    "interpretation":
        (
            "Developmental sensitivity/ablation only. "
            "Best sampled alpha is not automatically treated "
            "as a final tuned hyperparameter."
        ),
}


with open(
    STEP4A_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step4a_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 4A.13 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 4A - GLOBAL RECOVERY CONTRIBUTION SCALING")
print("=" * 125)

print(
    "Images                    :",
    len(
        step4a_df
    ),
    "/ 100"
)

print(
    "Alpha grid                :",
    ALPHA_GRID
)

print(
    "Recovery rules            :",
    len(
        RECOVERY_CONFIGS
    )
)

print(
    "\nGT used for evaluation    : YES"
)

print(
    "GT used for routing       : NO"
)

print(
    "Density used for routing  : NO"
)

print(
    "Semantic evidence changed : NO"
)

print(
    "SAM2 re-prompting         : NO"
)

print(
    "Test set accessed         : NO"
)


print(
    "\nFull alpha sweep:"
)

display(
    step4a_summary_df[
        [
            "rule_id",
            "alpha",
            "total_recovery_units",
            "weighted_recovery_contribution",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_C1",
            "delta_RMSE_vs_C1",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .sort_values(
        [
            "rule_id",
            "alpha",
        ]
    )
    .round(4)
)


print(
    "\nBest sampled alpha by MAE "
    "(developmental diagnostic only):"
)

display(
    best_mae_rows[
        [
            "rule_id",
            "alpha",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_C1",
            "delta_RMSE_vs_C1",
        ]
    ]
    .sort_values(
        "MAE"
    )
    .round(4)
)


print(
    "\nBest sampled alpha by RMSE "
    "(developmental diagnostic only):"
)

display(
    best_rmse_rows[
        [
            "rule_id",
            "alpha",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_C1",
            "delta_RMSE_vs_C1",
        ]
    ]
    .sort_values(
        "RMSE"
    )
    .round(4)
)


print(
    "\nDensity-stratified sensitivity:"
)

display(
    step4a_density_df[
        [
            "density_stratum",
            "rule_id",
            "alpha",
            "num_images",
            "recovery_units",
            "MAE",
            "RMSE",
            "mean_signed_error",
        ]
    ]
    .sort_values(
        [
            "density_stratum",
            "rule_id",
            "alpha",
        ]
    )
    .round(4)
)


print("\nSaved full per-image alpha sweep:")
print(STEP4A_LONG_FILE)

print("\nSaved overall summary:")
print(STEP4A_SUMMARY_FILE)

print("\nSaved density summary:")
print(STEP4A_DENSITY_FILE)

print("\nSaved per-image recovery source table:")
print(STEP4A_PER_IMAGE_FILE)

print("\nSaved manifest:")
print(STEP4A_MANIFEST_FILE)

print("=" * 125)

STEP 4A - GLOBAL RECOVERY CONTRIBUTION SCALING
Images                    : 100 / 100
Alpha grid                : [0.0, 0.25, 0.5, 0.75, 1.0]
Recovery rules            : 5

GT used for evaluation    : YES
GT used for routing       : NO
Density used for routing  : NO
Semantic evidence changed : NO
SAM2 re-prompting         : NO
Test set accessed         : NO

Full alpha sweep:


,rule_id,alpha,total_recovery_units,weighted_recovery_contribution,MAE,RMSE,mean_signed_error,delta_MAE_vs_C1,delta_RMSE_vs_C1,under_count_images,exact_count_images,over_count_images
0,Step3D_D6BT_C1supported,0.00,805,0.00,72.0000,262.2353,-65.3200,0.0000,0.0000,52,5,43
1,Step3D_D6BT_C1supported,0.25,805,201.25,71.4025,261.2282,-63.3075,-0.5975,-1.0071,48,3,49
2,Step3D_D6BT_C1supported,0.50,805,402.50,71.0650,260.2607,-61.2950,-0.9350,-1.9746,44,3,53
3,Step3D_D6BT_C1supported,0.75,805,603.75,71.0725,259.3332,-59.2825,-0.9275,-2.9022,41,1,58
4,Step3D_D6BT_C1supported,1.00,805,805.00,71.2900,258.4460,-57.2700,-0.7100,-3.7893,37,2,61
5,Step3E_D6BTxD6B9,0.00,1355,0.00,72.0000,262.2353,-65.3200,0.0000,0.0000,52,5,43
6,Step3E_D6BTxD6B9,0.25,1355,338.75,72.2375,261.8149,-61.9325,0.2375,-0.4204,44,3,53
7,Step3E_D6BTxD6B9,0.50,1355,677.50,73.1950,261.5135,-58.5450,1.1950,-0.7218,43,2,55
8,Step3E_D6BTxD6B9,0.75,1355,1016.25,74.3025,261.3317,-55.1575,2.3025,-0.9037,40,2,58
9,Step3E_D6BTxD6B9,1.00,1355,1355.00,75.9900,261.2695,-51.7700,3.9900,-0.9658,34,4,62



Best sampled alpha by MAE (developmental diagnostic only):


,rule_id,alpha,MAE,RMSE,mean_signed_error,delta_MAE_vs_C1,delta_RMSE_vs_C1
0,Step3D_D6BT_C1supported,0.50,71.0650,260.2607,-61.2950,-0.9350,-1.9746
4,Step3GC_Triple,0.25,71.7675,262.0382,-63.9025,-0.2325,-0.1971
2,Step3GB_D5xD6B9,0.25,71.8150,262.0118,-63.6800,-0.1850,-0.2235
1,Step3E_D6BTxD6B9,0.00,72.0000,262.2353,-65.3200,0.0000,0.0000
3,Step3GB_D5xD6BT,0.00,72.0000,262.2353,-65.3200,0.0000,0.0000



Best sampled alpha by RMSE (developmental diagnostic only):


,rule_id,alpha,MAE,RMSE,mean_signed_error,delta_MAE_vs_C1,delta_RMSE_vs_C1
0,Step3D_D6BT_C1supported,1.0,71.29,258.4460,-57.27,-0.71,-3.7893
3,Step3GB_D5xD6BT,1.0,79.24,258.9390,-40.86,7.24,-3.2964
1,Step3E_D6BTxD6B9,1.0,75.99,261.2695,-51.77,3.99,-0.9658
2,Step3GB_D5xD6B9,1.0,72.82,261.4989,-58.76,0.82,-0.7364
4,Step3GC_Triple,1.0,72.39,261.5693,-59.65,0.39,-0.6660



Density-stratified sensitivity:


,density_stratum,rule_id,alpha,num_images,recovery_units,MAE,RMSE,mean_signed_error
0,S1_0_20,Step3D_D6BT_C1supported,0.00,20,76,4.550,6.1685,4.050
6,S1_0_20,Step3D_D6BT_C1supported,0.25,20,76,5.375,7.4950,5.000
12,S1_0_20,Step3D_D6BT_C1supported,0.50,20,76,6.200,8.9917,5.950
18,S1_0_20,Step3D_D6BT_C1supported,0.75,20,76,7.050,10.5865,6.900
24,S1_0_20,Step3D_D6BT_C1supported,1.00,20,76,7.950,12.2413,7.850
...,...,...,...,...,...,...,...,...
125,S6_95_100,Step3GC_Triple,0.00,10,59,588.900,825.7966,-585.500
131,S6_95_100,Step3GC_Triple,0.25,10,59,587.425,825.3263,-584.025
137,S6_95_100,Step3GC_Triple,0.50,10,59,585.950,824.8684,-582.550
143,S6_95_100,Step3GC_Triple,0.75,10,59,584.475,824.4229,-581.075



Saved full per-image alpha sweep:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_global_recovery_scaling_long.csv

Saved overall summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_global_recovery_scaling_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_global_recovery_scaling_by_density.csv

Saved per-image recovery source table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_global_recovery_scaling_per_image.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_4A_global_scaling_manifest.json


In [36]:
# ============================================================
# Step 4B-A — Inference-Available Signal Screening
#
# Goal:
# Identify inference-available image-level signals that indicate
# when Step-3D recovery should receive little vs strong count
# contribution.
#
# Recovery source under study:
#   Step 3D — C1-supported D6B-T-native
#
# GT is used ONLY to construct diagnostic targets:
#
#   - per-image oracle best alpha over the already-fixed
#     Step-4A grid {0, .25, .5, .75, 1}
#   - error improvement relative to alpha=0
#
# GT is NOT used as an inference feature.
#
# density_stratum is NOT used as an inference feature.
#
# No new semantic source.
# No SAM2 re-prompting.
# Test set untouched.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 4B-A.1 Output files
# ------------------------------------------------------------

STEP4BA_SIGNAL_FILE = (
    B2_TABLES_DIR
    / "B0_C1_step3D_adaptive_signal_table.csv"
)

STEP4BA_CORR_FILE = (
    B2_TABLES_DIR
    / "B0_C1_step3D_adaptive_signal_correlations.csv"
)

STEP4BA_GROUP_FILE = (
    B2_TABLES_DIR
    / "B0_C1_step3D_adaptive_signal_low_mid_high.csv"
)

STEP4BA_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_4BA_signal_screening_manifest.json"
)


# ------------------------------------------------------------
# 4B-A.2 Preconditions
# ------------------------------------------------------------

required_vars = [
    "step4a_df",
    "step4a_long_df",
    "step3d_count_df",
]

for var_name in required_vars:

    assert var_name in globals(), (
        f"Required variable not found: {var_name}"
    )


assert len(step4a_df) == 100
assert step4a_df["dataset_index"].nunique() == 100


# ------------------------------------------------------------
# 4B-A.3 Start from authoritative Step-4A per-image table
# ------------------------------------------------------------

signal_df = (
    step4a_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",   # retained for diagnosis only
            "gt_count",
            "B0_unfiltered",
            "B0_plus_C1_raw",
            "R_Step3D_D6BT_C1supported",
            "R_Step3E_D6BTxD6B9",
            "R_Step3GB_D5xD6BT",
            "R_Step3GB_D5xD6B9",
            "R_Step3GC_Triple",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 4B-A.4 Bring in explicit C1 added-point count
# ------------------------------------------------------------

signal_df = (
    signal_df
    .merge(
        step3d_count_df[
            [
                "dataset_index",
                "B0_plus_C1_raw__added_points",
            ]
        ]
        .rename(
            columns={
                "B0_plus_C1_raw__added_points":
                    "C1_added"
            }
        ),
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


assert signal_df["C1_added"].notna().all()


# ------------------------------------------------------------
# 4B-A.5 Construct inference-available signals
#
# No GT information enters any signal below.
# ------------------------------------------------------------

signal_df[
    "signal_B0_count"
] = (
    signal_df[
        "B0_unfiltered"
    ].astype(float)
)


signal_df[
    "signal_B0C1_count"
] = (
    signal_df[
        "B0_plus_C1_raw"
    ].astype(float)
)


signal_df[
    "signal_C1_added"
] = (
    signal_df[
        "C1_added"
    ].astype(float)
)


signal_df[
    "signal_C1_to_B0_ratio"
] = (
    signal_df[
        "C1_added"
    ]
    /
    np.maximum(
        signal_df[
            "B0_unfiltered"
        ],
        1,
    )
)


signal_df[
    "signal_Step3D_recovery"
] = (
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ].astype(float)
)


signal_df[
    "signal_Step3D_to_base_ratio"
] = (
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ]
    /
    np.maximum(
        signal_df[
            "B0_plus_C1_raw"
        ],
        1,
    )
)


signal_df[
    "signal_D6_pair_consensus"
] = (
    signal_df[
        "R_Step3E_D6BTxD6B9"
    ].astype(float)
)


signal_df[
    "signal_D5xD6B9_consensus"
] = (
    signal_df[
        "R_Step3GB_D5xD6B9"
    ].astype(float)
)


signal_df[
    "signal_triple_consensus"
] = (
    signal_df[
        "R_Step3GC_Triple"
    ].astype(float)
)


# Strict heterogeneous agreement relative to Step-3D burden

signal_df[
    "signal_D5xD6B9_per_Step3D"
] = np.where(
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ]
    >
    0,

    signal_df[
        "R_Step3GB_D5xD6B9"
    ]
    /
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ],

    np.nan,
)


signal_df[
    "signal_triple_per_Step3D"
] = np.where(
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ]
    >
    0,

    signal_df[
        "R_Step3GC_Triple"
    ]
    /
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ],

    np.nan,
)


# Triple agreement relative to stricter heterogeneous pair

signal_df[
    "signal_triple_per_D5xD6B9"
] = np.where(
    signal_df[
        "R_Step3GB_D5xD6B9"
    ]
    >
    0,

    signal_df[
        "R_Step3GC_Triple"
    ]
    /
    signal_df[
        "R_Step3GB_D5xD6B9"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 4B-A.6 Construct GT-based diagnostic target
#
# IMPORTANT:
# This is evaluation only.
#
# Find per-image best sampled alpha for Step-3D over the
# Step-4A grid.
#
# Tie-break:
# prefer the SMALLER alpha.
# ------------------------------------------------------------

step3d_alpha_df = (
    step4a_long_df[
        step4a_long_df[
            "rule_id"
        ]
        ==
        "Step3D_D6BT_C1supported"
    ]
    .copy()
)


assert len(step3d_alpha_df) == 500


oracle_alpha_df = (
    step3d_alpha_df
    .sort_values(
        [
            "dataset_index",
            "abs_error",
            "alpha",
        ]
    )
    .groupby(
        "dataset_index",
        as_index=False,
    )
    .first()
    [
        [
            "dataset_index",
            "alpha",
            "abs_error",
        ]
    ]
    .rename(
        columns={
            "alpha":
                "oracle_best_alpha",

            "abs_error":
                "oracle_best_abs_error",
        }
    )
)


signal_df = (
    signal_df
    .merge(
        oracle_alpha_df,
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


assert signal_df["oracle_best_alpha"].notna().all()


# ------------------------------------------------------------
# 4B-A.7 Error-benefit targets for fixed alpha values
# ------------------------------------------------------------

def abs_error_at_alpha(
    alpha_value
):

    subset = (
        step3d_alpha_df[
            np.isclose(
                step3d_alpha_df[
                    "alpha"
                ],
                alpha_value,
            )
        ][
            [
                "dataset_index",
                "abs_error",
            ]
        ]
        .rename(
            columns={
                "abs_error":
                    f"abs_error_alpha_{alpha_value:.2f}"
            }
        )
    )

    return subset


for alpha_value in [
    0.00,
    0.50,
    1.00,
]:

    signal_df = (
        signal_df
        .merge(
            abs_error_at_alpha(
                alpha_value
            ),
            on="dataset_index",
            how="left",
            validate="one_to_one",
        )
    )


signal_df[
    "benefit_alpha_0_50"
] = (
    signal_df[
        "abs_error_alpha_0.00"
    ]
    -
    signal_df[
        "abs_error_alpha_0.50"
    ]
)


signal_df[
    "benefit_alpha_1_00"
] = (
    signal_df[
        "abs_error_alpha_0.00"
    ]
    -
    signal_df[
        "abs_error_alpha_1.00"
    ]
)


signal_df[
    "oracle_any_recovery"
] = (
    signal_df[
        "oracle_best_alpha"
    ]
    >
    0.0
)


signal_df[
    "oracle_moderate_or_strong_recovery"
] = (
    signal_df[
        "oracle_best_alpha"
    ]
    >=
    0.50
)


# ------------------------------------------------------------
# 4B-A.8 Signal list
#
# density_stratum deliberately excluded.
# GT deliberately excluded.
# ------------------------------------------------------------

SIGNAL_COLS = [
    "signal_B0_count",
    "signal_B0C1_count",
    "signal_C1_added",
    "signal_C1_to_B0_ratio",
    "signal_Step3D_recovery",
    "signal_Step3D_to_base_ratio",
    "signal_D6_pair_consensus",
    "signal_D5xD6B9_consensus",
    "signal_triple_consensus",
    "signal_D5xD6B9_per_Step3D",
    "signal_triple_per_Step3D",
    "signal_triple_per_D5xD6B9",
]


TARGET_COLS = [
    "oracle_best_alpha",
    "benefit_alpha_0_50",
    "benefit_alpha_1_00",
]


# ------------------------------------------------------------
# 4B-A.9 Spearman signal correlations
# ------------------------------------------------------------

corr_records = []


for signal_col in SIGNAL_COLS:

    for target_col in TARGET_COLS:

        pair_df = (
            signal_df[
                [
                    signal_col,
                    target_col,
                ]
            ]
            .dropna()
        )


        if (
            len(pair_df) >= 3
            and
            pair_df[
                signal_col
            ].nunique()
            >
            1
            and
            pair_df[
                target_col
            ].nunique()
            >
            1
        ):

            rho = float(
                pair_df[
                    signal_col
                ]
                .corr(
                    pair_df[
                        target_col
                    ],
                    method="spearman",
                )
            )

        else:

            rho = np.nan


        corr_records.append(
            {
                "signal":
                    signal_col,

                "target":
                    target_col,

                "num_valid_images":
                    int(
                        len(
                            pair_df
                        )
                    ),

                "spearman_rho":
                    rho,

                "abs_spearman_rho":
                    (
                        abs(
                            rho
                        )
                        if np.isfinite(
                            rho
                        )
                        else np.nan
                    ),
            }
        )


step4ba_corr_df = pd.DataFrame(
    corr_records
)


# ------------------------------------------------------------
# 4B-A.10 Low / middle / high signal diagnostics
#
# Diagnostic only.
#
# We use percentile ranks rather than GT strata.
# ------------------------------------------------------------

group_records = []


for signal_col in SIGNAL_COLS:

    valid_df = (
        signal_df[
            [
                "dataset_index",
                signal_col,
                "oracle_best_alpha",
                "benefit_alpha_0_50",
                "benefit_alpha_1_00",
                "oracle_any_recovery",
                "oracle_moderate_or_strong_recovery",
            ]
        ]
        .dropna(
            subset=[
                signal_col
            ]
        )
        .copy()
    )


    if (
        len(
            valid_df
        )
        <
        3
        or
        valid_df[
            signal_col
        ].nunique()
        <=
        1
    ):

        continue


    valid_df[
        "signal_percentile"
    ] = (
        valid_df[
            signal_col
        ]
        .rank(
            method="average",
            pct=True,
        )
    )


    valid_df[
        "signal_level"
    ] = pd.cut(
        valid_df[
            "signal_percentile"
        ],
        bins=[
            0.0,
            1.0 / 3.0,
            2.0 / 3.0,
            1.0,
        ],
        labels=[
            "low",
            "middle",
            "high",
        ],
        include_lowest=True,
    )


    for level, group in (
        valid_df
        .groupby(
            "signal_level",
            observed=True,
        )
    ):

        group_records.append(
            {
                "signal":
                    signal_col,

                "signal_level":
                    str(
                        level
                    ),

                "num_images":
                    int(
                        len(
                            group
                        )
                    ),

                "mean_signal":
                    float(
                        group[
                            signal_col
                        ].mean()
                    ),

                "median_signal":
                    float(
                        group[
                            signal_col
                        ].median()
                    ),

                "mean_oracle_alpha":
                    float(
                        group[
                            "oracle_best_alpha"
                        ].mean()
                    ),

                "median_oracle_alpha":
                    float(
                        group[
                            "oracle_best_alpha"
                        ].median()
                    ),

                "fraction_oracle_any_recovery":
                    float(
                        group[
                            "oracle_any_recovery"
                        ].mean()
                    ),

                "fraction_oracle_alpha_ge_0_50":
                    float(
                        group[
                            "oracle_moderate_or_strong_recovery"
                        ].mean()
                    ),

                "mean_benefit_alpha_0_50":
                    float(
                        group[
                            "benefit_alpha_0_50"
                        ].mean()
                    ),

                "mean_benefit_alpha_1_00":
                    float(
                        group[
                            "benefit_alpha_1_00"
                        ].mean()
                    ),
            }
        )


step4ba_group_df = pd.DataFrame(
    group_records
)


# ------------------------------------------------------------
# 4B-A.11 Oracle-alpha distribution
# ------------------------------------------------------------

oracle_alpha_distribution_df = (
    signal_df[
        "oracle_best_alpha"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "oracle_best_alpha"
    )
    .reset_index(
        name="num_images"
    )
)


oracle_alpha_distribution_df[
    "fraction_images"
] = (
    oracle_alpha_distribution_df[
        "num_images"
    ]
    /
    len(
        signal_df
    )
)


# ------------------------------------------------------------
# 4B-A.12 Save
# ------------------------------------------------------------

signal_df.to_csv(
    STEP4BA_SIGNAL_FILE,
    index=False,
)


step4ba_corr_df.to_csv(
    STEP4BA_CORR_FILE,
    index=False,
)


step4ba_group_df.to_csv(
    STEP4BA_GROUP_FILE,
    index=False,
)


step4ba_manifest = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 4B-A - inference-available adaptive signal screening",

    "recovery_source":
        "Step3D_D6BT_C1supported",

    "oracle_alpha_grid":
        [
            0.0,
            0.25,
            0.5,
            0.75,
            1.0,
        ],

    "oracle_tie_break":
        "smaller alpha",

    "signals":
        SIGNAL_COLS,

    "GT_used_for_signal_construction":
        False,

    "GT_used_for_diagnostic_targets":
        True,

    "density_stratum_used_as_signal":
        False,

    "semantic_sources_changed":
        False,

    "sam2_reprompting":
        False,

    "test_set_accessed":
        False,

    "interpretation":
        (
            "Developmental screening only. "
            "No adaptive rule is selected in this step."
        ),
}


with open(
    STEP4BA_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        step4ba_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 4B-A.13 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 4B-A - INFERENCE-AVAILABLE SIGNAL SCREENING")
print("=" * 125)

print(
    "Images                         :",
    len(
        signal_df
    ),
    "/ 100"
)

print(
    "Recovery source                : Step3D_D6BT_C1supported"
)

print(
    "GT used to construct signals   : NO"
)

print(
    "GT used for diagnostics        : YES"
)

print(
    "Density stratum used as signal : NO"
)

print(
    "Semantic evidence changed      : NO"
)

print(
    "Test set accessed              : NO"
)


print(
    "\nPer-image oracle alpha distribution "
    "(diagnostic only):"
)

display(
    oracle_alpha_distribution_df.round(
        4
    )
)


print(
    "\nSignal correlations with oracle recovery need:"
)

display(
    step4ba_corr_df[
        [
            "signal",
            "target",
            "num_valid_images",
            "spearman_rho",
            "abs_spearman_rho",
        ]
    ]
    .sort_values(
        [
            "target",
            "abs_spearman_rho",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        4
    )
)


print(
    "\nTop signals for oracle_best_alpha:"
)

display(
    step4ba_corr_df[
        step4ba_corr_df[
            "target"
        ]
        ==
        "oracle_best_alpha"
    ]
    .sort_values(
        "abs_spearman_rho",
        ascending=False,
    )
    .head(
        12
    )
    .round(
        4
    )
)


print(
    "\nLow / middle / high signal diagnostics:"
)

display(
    step4ba_group_df[
        [
            "signal",
            "signal_level",
            "num_images",
            "mean_signal",
            "mean_oracle_alpha",
            "fraction_oracle_any_recovery",
            "fraction_oracle_alpha_ge_0_50",
            "mean_benefit_alpha_0_50",
            "mean_benefit_alpha_1_00",
        ]
    ]
    .sort_values(
        [
            "signal",
            "signal_level",
        ]
    )
    .round(
        4
    )
)


print(
    "\nSaved adaptive signal table:"
)

print(
    STEP4BA_SIGNAL_FILE
)

print(
    "\nSaved signal correlations:"
)

print(
    STEP4BA_CORR_FILE
)

print(
    "\nSaved low/middle/high diagnostics:"
)

print(
    STEP4BA_GROUP_FILE
)

print(
    "\nSaved manifest:"
)

print(
    STEP4BA_MANIFEST_FILE
)

print("=" * 125)

STEP 4B-A - INFERENCE-AVAILABLE SIGNAL SCREENING
Images                         : 100 / 100
Recovery source                : Step3D_D6BT_C1supported
GT used to construct signals   : NO
GT used for diagnostics        : YES
Density stratum used as signal : NO
Semantic evidence changed      : NO
Test set accessed              : NO

Per-image oracle alpha distribution (diagnostic only):


,oracle_best_alpha,num_images,fraction_images
0,0.00,62,0.62
1,0.25,3,0.03
2,0.50,5,0.05
3,0.75,4,0.04
4,1.00,26,0.26



Signal correlations with oracle recovery need:


,signal,target,num_valid_images,spearman_rho,abs_spearman_rho
34,signal_triple_per_D5xD6B9,benefit_alpha_0_50,75,0.1955,0.1955
10,signal_C1_to_B0_ratio,benefit_alpha_0_50,100,0.1855,0.1855
7,signal_C1_added,benefit_alpha_0_50,100,0.1842,0.1842
1,signal_B0_count,benefit_alpha_0_50,100,-0.1571,0.1571
19,signal_D6_pair_consensus,benefit_alpha_0_50,100,-0.1285,0.1285
28,signal_D5xD6B9_per_Step3D,benefit_alpha_0_50,74,-0.1139,0.1139
13,signal_Step3D_recovery,benefit_alpha_0_50,100,0.0988,0.0988
31,signal_triple_per_Step3D,benefit_alpha_0_50,74,-0.0912,0.0912
4,signal_B0C1_count,benefit_alpha_0_50,100,0.0897,0.0897
22,signal_D5xD6B9_consensus,benefit_alpha_0_50,100,-0.0825,0.0825



Top signals for oracle_best_alpha:


,signal,target,num_valid_images,spearman_rho,abs_spearman_rho
9,signal_C1_to_B0_ratio,oracle_best_alpha,100,0.4976,0.4976
6,signal_C1_added,oracle_best_alpha,100,0.4817,0.4817
12,signal_Step3D_recovery,oracle_best_alpha,100,0.4275,0.4275
15,signal_Step3D_to_base_ratio,oracle_best_alpha,100,0.3834,0.3834
33,signal_triple_per_D5xD6B9,oracle_best_alpha,75,0.2718,0.2718
0,signal_B0_count,oracle_best_alpha,100,-0.2621,0.2621
24,signal_triple_consensus,oracle_best_alpha,100,0.2528,0.2528
21,signal_D5xD6B9_consensus,oracle_best_alpha,100,0.2203,0.2203
18,signal_D6_pair_consensus,oracle_best_alpha,100,0.1735,0.1735
3,signal_B0C1_count,oracle_best_alpha,100,0.1265,0.1265



Low / middle / high signal diagnostics:


,signal,signal_level,num_images,mean_signal,mean_oracle_alpha,fraction_oracle_any_recovery,fraction_oracle_alpha_ge_0_50,mean_benefit_alpha_0_50,mean_benefit_alpha_1_00
5,signal_B0C1_count,high,33,72.5758,0.4242,0.5152,0.4242,3.4848,5.4545
3,signal_B0C1_count,low,33,10.2727,0.3333,0.3636,0.3636,0.1364,-0.0303
4,signal_B0C1_count,middle,34,22.9118,0.2132,0.2647,0.2647,-0.7647,-3.1765
2,signal_B0_count,high,33,54.5758,0.2652,0.3333,0.2424,0.5606,0.5758
0,signal_B0_count,low,32,2.5938,0.5234,0.5938,0.5938,1.9844,2.5312
1,signal_B0_count,middle,35,11.8286,0.1929,0.2286,0.2286,0.3286,-0.8286
8,signal_C1_added,high,33,29.4242,0.4773,0.5758,0.5455,3.3485,3.6061
6,signal_C1_added,low,32,0.4688,0.0938,0.0938,0.0938,-0.0312,-0.0625
7,signal_C1_added,middle,35,6.5429,0.3857,0.4571,0.4000,-0.4571,-1.3143
11,signal_C1_to_B0_ratio,high,34,13.7101,0.5809,0.7059,0.6765,3.4118,3.5294



Saved adaptive signal table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_step3D_adaptive_signal_table.csv

Saved signal correlations:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_step3D_adaptive_signal_correlations.csv

Saved low/middle/high diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_step3D_adaptive_signal_low_mid_high.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_4BA_signal_screening_manifest.json


In [37]:
# ============================================================
# Step 4B-B — Simple Inference-Time Recovery Gate Design
#
# Base counting rule:
#
#   pred =
#       B0_plus_C1_raw
#       + gate * alpha * Step3D_recovery
#
# Fixed alpha:
#   0.50
#
# Gate candidates use ONLY inference-available signals.
#
# GT is used ONLY for evaluation.
#
# No density routing.
# No new semantic source.
# No SAM2 re-prompting.
# Test set untouched.
#
# IMPORTANT:
# This is a small controlled rule-family comparison,
# not an exhaustive threshold search.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 4B-B.1 Preconditions
# ------------------------------------------------------------

required_vars = [
    "signal_df",
    "step4a_df",
]

for var_name in required_vars:

    assert var_name in globals(), (
        f"Required variable not found: {var_name}"
    )


assert len(signal_df) == 100
assert signal_df["dataset_index"].nunique() == 100


# ------------------------------------------------------------
# 4B-B.2 Fixed recovery weight
# ------------------------------------------------------------

ALPHA_4BB = 0.50


# ------------------------------------------------------------
# 4B-B.3 Gate definitions
#
# Coarse, interpretable values only.
# Not an exhaustive tuned grid.
# ------------------------------------------------------------

def gate_always(df):
    return np.ones(len(df), dtype=bool)


def gate_c1_b0_gt_025(df):
    return (
        df["signal_C1_to_B0_ratio"]
        >
        0.25
    ).to_numpy()


def gate_c1_b0_gt_050(df):
    return (
        df["signal_C1_to_B0_ratio"]
        >
        0.50
    ).to_numpy()


def gate_c1_b0_gt_100(df):
    return (
        df["signal_C1_to_B0_ratio"]
        >
        1.00
    ).to_numpy()


def gate_c1_added_ge_5(df):
    return (
        df["signal_C1_added"]
        >=
        5
    ).to_numpy()


def gate_c1_added_ge_10(df):
    return (
        df["signal_C1_added"]
        >=
        10
    ).to_numpy()


def gate_step3d_ratio_gt_010(df):
    return (
        df["signal_Step3D_to_base_ratio"]
        >
        0.10
    ).to_numpy()


def gate_step3d_ratio_gt_025(df):
    return (
        df["signal_Step3D_to_base_ratio"]
        >
        0.25
    ).to_numpy()


def gate_and_c1ratio_step3dratio(df):
    return (
        (
            df["signal_C1_to_B0_ratio"]
            >
            0.50
        )
        &
        (
            df["signal_Step3D_to_base_ratio"]
            >
            0.10
        )
    ).to_numpy()


def gate_and_c1added_step3dratio(df):
    return (
        (
            df["signal_C1_added"]
            >=
            5
        )
        &
        (
            df["signal_Step3D_to_base_ratio"]
            >
            0.10
        )
    ).to_numpy()


GATE_CONFIGS = [
    {
        "gate_id":
            "Always_ON",
        "gate_fn":
            gate_always,
    },

    {
        "gate_id":
            "C1toB0_gt_0.25",
        "gate_fn":
            gate_c1_b0_gt_025,
    },

    {
        "gate_id":
            "C1toB0_gt_0.50",
        "gate_fn":
            gate_c1_b0_gt_050,
    },

    {
        "gate_id":
            "C1toB0_gt_1.00",
        "gate_fn":
            gate_c1_b0_gt_100,
    },

    {
        "gate_id":
            "C1added_ge_5",
        "gate_fn":
            gate_c1_added_ge_5,
    },

    {
        "gate_id":
            "C1added_ge_10",
        "gate_fn":
            gate_c1_added_ge_10,
    },

    {
        "gate_id":
            "Step3Dratio_gt_0.10",
        "gate_fn":
            gate_step3d_ratio_gt_010,
    },

    {
        "gate_id":
            "Step3Dratio_gt_0.25",
        "gate_fn":
            gate_step3d_ratio_gt_025,
    },

    {
        "gate_id":
            "AND_C1toB0gt0.50_Step3Dratiogt0.10",
        "gate_fn":
            gate_and_c1ratio_step3dratio,
    },

    {
        "gate_id":
            "AND_C1addedge5_Step3Dratiogt0.10",
        "gate_fn":
            gate_and_c1added_step3dratio,
    },
]


# ------------------------------------------------------------
# 4B-B.4 Metric helper
# ------------------------------------------------------------

def count_metrics_4bb(
    gt,
    pred,
):

    gt = np.asarray(
        gt,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )

    err = pred - gt


    return {
        "MAE":
            float(
                np.mean(
                    np.abs(err)
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        err ** 2
                    )
                )
            ),

        "mean_signed_error":
            float(
                np.mean(err)
            ),

        "under_count_images":
            int(
                np.sum(
                    err < 0
                )
            ),

        "exact_count_images":
            int(
                np.sum(
                    np.isclose(
                        err,
                        0.0,
                    )
                )
            ),

        "over_count_images":
            int(
                np.sum(
                    err > 0
                )
            ),
    }


# ------------------------------------------------------------
# 4B-B.5 Authoritative vectors
# ------------------------------------------------------------

gt = (
    signal_df[
        "gt_count"
    ]
    .to_numpy(
        dtype=float
    )
)


base = (
    signal_df[
        "B0_plus_C1_raw"
    ]
    .to_numpy(
        dtype=float
    )
)


recovery = (
    signal_df[
        "R_Step3D_D6BT_C1supported"
    ]
    .to_numpy(
        dtype=float
    )
)


# ------------------------------------------------------------
# 4B-B.6 Evaluate gates
# ------------------------------------------------------------

summary_records = []
per_image_records = []


# Explicit baseline first

baseline_metrics = count_metrics_4bb(
    gt,
    base,
)


summary_records.append(
    {
        "gate_id":
            "C1_baseline_no_recovery",

        "alpha":
            0.0,

        "num_gate_ON":
            0,

        "fraction_gate_ON":
            0.0,

        "total_raw_recovery_units_on":
            0.0,

        "weighted_recovery_contribution":
            0.0,

        **baseline_metrics,
    }
)


for config in GATE_CONFIGS:

    gate_id = config[
        "gate_id"
    ]

    gate = (
        config[
            "gate_fn"
        ](
            signal_df
        )
        .astype(bool)
    )


    assert len(gate) == 100


    weighted_recovery = (
        gate.astype(float)
        *
        ALPHA_4BB
        *
        recovery
    )


    pred = (
        base
        +
        weighted_recovery
    )


    metrics = count_metrics_4bb(
        gt,
        pred,
    )


    summary_records.append(
        {
            "gate_id":
                gate_id,

            "alpha":
                ALPHA_4BB,

            "num_gate_ON":
                int(
                    gate.sum()
                ),

            "fraction_gate_ON":
                float(
                    gate.mean()
                ),

            "total_raw_recovery_units_on":
                float(
                    recovery[
                        gate
                    ].sum()
                ),

            "weighted_recovery_contribution":
                float(
                    weighted_recovery.sum()
                ),

            **metrics,
        }
    )


    signed_error = (
        pred
        -
        gt
    )


    for i, row in enumerate(
        signal_df.itertuples(
            index=False
        )
    ):

        per_image_records.append(
            {
                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    row.image_id,

                "category":
                    row.category,

                "density_stratum":
                    row.density_stratum,

                "gt_count":
                    float(
                        row.gt_count
                    ),

                "B0_plus_C1_raw":
                    float(
                        row.B0_plus_C1_raw
                    ),

                "Step3D_recovery":
                    float(
                        row.R_Step3D_D6BT_C1supported
                    ),

                "signal_C1_to_B0_ratio":
                    float(
                        row.signal_C1_to_B0_ratio
                    ),

                "signal_C1_added":
                    float(
                        row.signal_C1_added
                    ),

                "signal_Step3D_to_base_ratio":
                    float(
                        row.signal_Step3D_to_base_ratio
                    ),

                "gate_id":
                    gate_id,

                "gate_ON":
                    bool(
                        gate[i]
                    ),

                "alpha":
                    ALPHA_4BB,

                "weighted_recovery":
                    float(
                        weighted_recovery[i]
                    ),

                "predicted_count":
                    float(
                        pred[i]
                    ),

                "signed_error":
                    float(
                        signed_error[i]
                    ),

                "abs_error":
                    float(
                        abs(
                            signed_error[i]
                        )
                    ),
            }
        )


step4bb_summary_df = pd.DataFrame(
    summary_records
)


step4bb_per_image_df = pd.DataFrame(
    per_image_records
)


# ------------------------------------------------------------
# 4B-B.7 Add deltas vs C1 baseline
# ------------------------------------------------------------

baseline_mae = float(
    baseline_metrics[
        "MAE"
    ]
)

baseline_rmse = float(
    baseline_metrics[
        "RMSE"
    ]
)


step4bb_summary_df[
    "delta_MAE_vs_C1"
] = (
    step4bb_summary_df[
        "MAE"
    ]
    -
    baseline_mae
)


step4bb_summary_df[
    "delta_RMSE_vs_C1"
] = (
    step4bb_summary_df[
        "RMSE"
    ]
    -
    baseline_rmse
)


# ------------------------------------------------------------
# 4B-B.8 Compare against global-alpha Step 4A reference
# ------------------------------------------------------------

GLOBAL_ALPHA05_MAE = 71.0650
GLOBAL_ALPHA05_RMSE = 260.2607


step4bb_summary_df[
    "delta_MAE_vs_global_alpha0.5"
] = (
    step4bb_summary_df[
        "MAE"
    ]
    -
    GLOBAL_ALPHA05_MAE
)


step4bb_summary_df[
    "delta_RMSE_vs_global_alpha0.5"
] = (
    step4bb_summary_df[
        "RMSE"
    ]
    -
    GLOBAL_ALPHA05_RMSE
)


# ------------------------------------------------------------
# 4B-B.9 Density diagnostic
#
# Diagnostic only.
# Not used by any gate.
# ------------------------------------------------------------

density_records = []


for config in GATE_CONFIGS:

    gate_id = config[
        "gate_id"
    ]

    gate = (
        config[
            "gate_fn"
        ](
            signal_df
        )
        .astype(bool)
    )


    pred = (
        base
        +
        gate.astype(float)
        *
        ALPHA_4BB
        *
        recovery
    )


    temp_df = pd.DataFrame(
        {
            "density_stratum":
                signal_df[
                    "density_stratum"
                ].to_numpy(),

            "gt":
                gt,

            "pred":
                pred,

            "gate_ON":
                gate.astype(int),
        }
    )


    for density_stratum, group in (
        temp_df.groupby(
            "density_stratum"
        )
    ):

        metrics = count_metrics_4bb(
            group[
                "gt"
            ].to_numpy(),

            group[
                "pred"
            ].to_numpy(),
        )


        density_records.append(
            {
                "gate_id":
                    gate_id,

                "density_stratum":
                    density_stratum,

                "num_images":
                    int(
                        len(group)
                    ),

                "num_gate_ON":
                    int(
                        group[
                            "gate_ON"
                        ].sum()
                    ),

                **metrics,
            }
        )


step4bb_density_df = pd.DataFrame(
    density_records
)


# ------------------------------------------------------------
# 4B-B.10 Save
# ------------------------------------------------------------

STEP4BB_SUMMARY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_step3D_simple_gate_summary.csv"
)

STEP4BB_PER_IMAGE_FILE = (
    B2_TABLES_DIR
    / "B0_C1_step3D_simple_gate_per_image.csv"
)

STEP4BB_DENSITY_FILE = (
    B2_TABLES_DIR
    / "B0_C1_step3D_simple_gate_by_density.csv"
)

STEP4BB_MANIFEST_FILE = (
    B2_METADATA_DIR
    / "B0_recovery_step_4BB_simple_gate_manifest.json"
)


step4bb_summary_df.to_csv(
    STEP4BB_SUMMARY_FILE,
    index=False,
)


step4bb_per_image_df.to_csv(
    STEP4BB_PER_IMAGE_FILE,
    index=False,
)


step4bb_density_df.to_csv(
    STEP4BB_DENSITY_FILE,
    index=False,
)


manifest_4bb = {
    "experiment":
        "B2_B0_Anchored_Semantic_Validation",

    "stage":
        "Step 4B-B - simple inference-time recovery gate",

    "base_count":
        "B0_plus_C1_raw",

    "recovery_source":
        "Step3D_D6BT_C1supported",

    "alpha":
        ALPHA_4BB,

    "gate_ids": [
        x[
            "gate_id"
        ]
        for x in GATE_CONFIGS
    ],

    "GT_used_for_gate_features":
        False,

    "GT_used_for_evaluation":
        True,

    "density_used_for_gate":
        False,

    "semantic_sources_changed":
        False,

    "sam2_reprompting":
        False,

    "test_set_accessed":
        False,

    "interpretation":
        (
            "Small controlled comparison of simple "
            "inference-available gates. "
            "Not an exhaustive threshold optimisation."
        ),
}


with open(
    STEP4BB_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        manifest_4bb,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 4B-B.11 Display
# ------------------------------------------------------------

print("=" * 125)
print("STEP 4B-B - SIMPLE INFERENCE-TIME RECOVERY GATES")
print("=" * 125)

print(
    "Fixed alpha                  :",
    ALPHA_4BB
)

print(
    "Recovery source              : Step3D_D6BT_C1supported"
)

print(
    "GT used for gate features    : NO"
)

print(
    "GT used for evaluation       : YES"
)

print(
    "Density used for gate        : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nGate comparison:"
)

display(
    step4bb_summary_df[
        [
            "gate_id",
            "alpha",
            "num_gate_ON",
            "fraction_gate_ON",
            "weighted_recovery_contribution",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_C1",
            "delta_RMSE_vs_C1",
            "delta_MAE_vs_global_alpha0.5",
            "delta_RMSE_vs_global_alpha0.5",
            "under_count_images",
            "exact_count_images",
            "over_count_images",
        ]
    ]
    .sort_values(
        [
            "MAE",
            "RMSE",
        ]
    )
    .round(
        4
    )
)


print(
    "\nBest gate by MAE:"
)

display(
    step4bb_summary_df
    .sort_values(
        [
            "MAE",
            "RMSE",
        ]
    )
    .head(
        5
    )
    .round(
        4
    )
)


print(
    "\nDensity diagnostic for best 3 non-baseline gates:"
)

best_gate_ids = (
    step4bb_summary_df[
        step4bb_summary_df[
            "gate_id"
        ]
        !=
        "C1_baseline_no_recovery"
    ]
    .sort_values(
        [
            "MAE",
            "RMSE",
        ]
    )
    .head(
        3
    )[
        "gate_id"
    ]
    .tolist()
)


display(
    step4bb_density_df[
        step4bb_density_df[
            "gate_id"
        ].isin(
            best_gate_ids
        )
    ]
    .sort_values(
        [
            "gate_id",
            "density_stratum",
        ]
    )
    .round(
        4
    )
)


print(
    "\nSaved summary:"
)
print(
    STEP4BB_SUMMARY_FILE
)

print(
    "\nSaved per-image results:"
)
print(
    STEP4BB_PER_IMAGE_FILE
)

print(
    "\nSaved density diagnostics:"
)
print(
    STEP4BB_DENSITY_FILE
)

print(
    "\nSaved manifest:"
)
print(
    STEP4BB_MANIFEST_FILE
)

print("=" * 125)

STEP 4B-B - SIMPLE INFERENCE-TIME RECOVERY GATES
Fixed alpha                  : 0.5
Recovery source              : Step3D_D6BT_C1supported
GT used for gate features    : NO
GT used for evaluation       : YES
Density used for gate        : NO
Test set accessed            : NO

Gate comparison:


,gate_id,alpha,num_gate_ON,fraction_gate_ON,weighted_recovery_contribution,MAE,RMSE,mean_signed_error,delta_MAE_vs_C1,delta_RMSE_vs_C1,delta_MAE_vs_global_alpha0.5,delta_RMSE_vs_global_alpha0.5,under_count_images,exact_count_images,over_count_images
6,C1added_ge_10,0.5,36,0.36,329.0,70.900,260.2727,-62.030,-1.100,-1.9626,-0.165,0.0120,46,7,47
8,Step3Dratio_gt_0.25,0.5,45,0.45,356.0,70.930,260.2738,-61.760,-1.070,-1.9615,-0.135,0.0131,46,5,49
10,AND_C1addedge5_Step3Dratiogt0.10,0.5,54,0.54,379.5,70.945,260.2621,-61.525,-1.055,-1.9733,-0.120,0.0014,46,5,49
2,C1toB0_gt_0.25,0.5,58,0.58,377.0,70.950,260.2703,-61.550,-1.050,-1.9650,-0.115,0.0096,46,4,50
5,C1added_ge_5,0.5,59,0.59,386.0,70.970,260.2572,-61.460,-1.030,-1.9781,-0.095,-0.0035,45,5,50
4,C1toB0_gt_1.00,0.5,39,0.39,295.0,71.000,260.4104,-62.370,-1.000,-1.8250,-0.065,0.1497,46,5,49
7,Step3Dratio_gt_0.10,0.5,63,0.63,390.0,71.010,260.2631,-61.420,-0.990,-1.9722,-0.055,0.0024,45,3,52
1,Always_ON,0.5,100,1.00,402.5,71.065,260.2607,-61.295,-0.935,-1.9746,0.000,0.0000,44,3,53
9,AND_C1toB0gt0.50_Step3Dratiogt0.10,0.5,48,0.48,338.0,71.080,260.3978,-61.940,-0.920,-1.8375,0.015,0.1371,46,4,50
3,C1toB0_gt_0.50,0.5,50,0.50,339.0,71.090,260.3979,-61.930,-0.910,-1.8374,0.025,0.1372,46,4,50



Best gate by MAE:


,gate_id,alpha,num_gate_ON,fraction_gate_ON,total_raw_recovery_units_on,weighted_recovery_contribution,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images,delta_MAE_vs_C1,delta_RMSE_vs_C1,delta_MAE_vs_global_alpha0.5,delta_RMSE_vs_global_alpha0.5
6,C1added_ge_10,0.5,36,0.36,658.0,329.0,70.900,260.2727,-62.030,46,7,47,-1.100,-1.9626,-0.165,0.0120
8,Step3Dratio_gt_0.25,0.5,45,0.45,712.0,356.0,70.930,260.2738,-61.760,46,5,49,-1.070,-1.9615,-0.135,0.0131
10,AND_C1addedge5_Step3Dratiogt0.10,0.5,54,0.54,759.0,379.5,70.945,260.2621,-61.525,46,5,49,-1.055,-1.9733,-0.120,0.0014
2,C1toB0_gt_0.25,0.5,58,0.58,754.0,377.0,70.950,260.2703,-61.550,46,4,50,-1.050,-1.9650,-0.115,0.0096
5,C1added_ge_5,0.5,59,0.59,772.0,386.0,70.970,260.2572,-61.460,45,5,50,-1.030,-1.9781,-0.095,-0.0035



Density diagnostic for best 3 non-baseline gates:


,gate_id,density_stratum,num_images,num_gate_ON,MAE,RMSE,mean_signed_error,under_count_images,exact_count_images,over_count_images
54,AND_C1addedge5_Step3Dratiogt0.10,S1_0_20,20,7,6.000,8.8431,5.750,3,2,15
55,AND_C1addedge5_Step3Dratiogt0.10,S2_20_40,20,15,8.425,11.2166,7.125,3,1,16
56,AND_C1addedge5_Step3Dratiogt0.10,S3_40_60,20,12,9.525,11.9179,-0.925,9,2,9
57,AND_C1addedge5_Step3Dratiogt0.10,S4_60_80,20,9,21.100,28.8834,-14.400,14,0,6
58,AND_C1addedge5_Step3Dratiogt0.10,S5_80_95,10,6,40.300,54.5371,-34.700,8,0,2
59,AND_C1addedge5_Step3Dratiogt0.10,S6_95_100,10,5,579.050,819.7734,-575.650,9,0,1
30,C1added_ge_10,S1_0_20,20,2,5.575,8.5447,5.075,3,3,14
31,C1added_ge_10,S2_20_40,20,8,7.950,10.9567,6.250,3,2,15
32,C1added_ge_10,S3_40_60,20,8,9.950,12.2014,-1.650,9,2,9
33,C1added_ge_10,S4_60_80,20,8,21.225,28.9256,-14.525,14,0,6



Saved summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_step3D_simple_gate_summary.csv

Saved per-image results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_step3D_simple_gate_per_image.csv

Saved density diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/tables/B0_C1_step3D_simple_gate_by_density.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/metadata/B0_recovery_step_4BB_simple_gate_manifest.json


## B2 Semantic Recovery and Counting-Rule Experiments — Interim Summary

### 1. B0 as structural anchor

Unguided SAM2 B0 remains a useful structural baseline, but GT-support analysis showed that many FSC-147 objects are not represented by B0 masks. This motivated using semantic evidence primarily for validation and recovery rather than replacing B0 completely.

### 2. Semantic recovery sources

C1 was the most reliable standalone recovery source and produced the clearest improvement over B0.

D6B-T-native offered substantially greater recovery capacity but also generated many unsupported proposals. D6B9-strict behaved as a more conservative SAN-derived variant, while D5-768 provided an independent CLIP-Surgery branch.

Pairwise and triple consensus improved the quality of recovery localisation, but stronger localisation confidence did not automatically translate into better image-level count estimates.

### 3. Recovery-rule findings

The strongest direct recovery-count rule was Step 3D:

`B0 + C1 + C1-supported D6B-T`

which achieved:

- MAE = 71.29
- RMSE = 258.45

This improved over the C1 baseline but showed that dense-image recovery and sparse-image overcounting remained in tension.

Pairwise and triple consensus provided cleaner localisation signals but did not outperform Step 3D when every accepted recovery unit contributed `+1`.

Triple consensus (`D5 ∩ D6B-T ∩ D6B9`) gave the cleanest incremental localisation signal tested, with precision-like = 0.3298 and FP/TP = 2.03, but its lower recall limited direct counting benefit.

### 4. Global recovery weighting

Step 4A tested:

`Count = B0+C1 + α × Step3D recovery`

for `α ∈ {0, 0.25, 0.5, 0.75, 1.0}`.

For Step 3D, partial weighting improved MAE:

- α = 0.50 → MAE = 71.065
- α = 0.75 → MAE = 71.073
- α = 1.00 → MAE = 71.29

This indicates that one recovery proposal should not necessarily contribute one complete count.

### 5. Adaptive recovery signals

Step 4B-A examined inference-available image-level signals.

The strongest associations with per-image recovery need were:

- C1/B0 ratio
- number of C1 additions
- Step3D recovery count
- Step3D/base-count ratio

Consensus signals were less predictive of image-level count need than these simpler B0–C1 disagreement measures.

This reinforces the distinction between:

- candidate-level localisation confidence, and
- image-level need for additional count.

### 6. Simple adaptive gating

Step 4B-B tested simple inference-time gates with fixed `α = 0.5`.

The best tested rule was:

`C1_added >= 10`

which achieved:

- MAE = 70.90
- RMSE = 260.27

Several alternative simple gates produced very similar MAE values (~70.9–71.0), suggesting that selective recovery is consistently preferable to applying Step3D recovery to every image.

The exact threshold `10` should not be interpreted as a final tuned parameter; the main finding is that gating recovery using inference-available semantic disagreement signals improves MAE.

### 7. Current provisional conclusion

The semantic experiments indicate:

1. B0 should remain the structural anchor.
2. C1 provides the safest semantic recovery.
3. D6B-T adds useful recovery capacity but requires controlled contribution.
4. Pairwise/triple consensus improves localisation confidence but does not directly solve counting.
5. Recovery should be weighted and selectively activated rather than treated as `+1` per proposal.
6. The current provisional semantic counting rule is:

   `B0 + C1 + gated, weighted Step3D recovery`

7. The main remaining weakness is severe undercounting in dense scenes.

### 8. Next stages

The next experiments will:

1. integrate the Marigold geometric/depth prior;
2. investigate whether geometric information improves recovery selection or weighting;
3. revisit additional semantic evidence variants;
4. combine semantic and geometric priors into the final zero-shot multi-label counting pipeline.